In [ ]:

# ==================================================
# NBA DATA SCIENCE & ANALYTICS — STEP 1
# Installazione e configurazione
# ==================================================

!pip -q install -U nba_api pandas numpy plotly openpyxl

import pandas as pd
import numpy as np
import plotly.express as px

from datetime import datetime
from zoneinfo import ZoneInfo
from nba_api.stats.static import teams, players
from nba_api.live.nba.endpoints import scoreboard

# Stagioni da analizzare
STAGIONI = ["2024-25", "2025-26", "2026-27"]
STAGIONE_ATTUALE = "2026-27"

# Elenco delle squadre NBA
squadre = pd.DataFrame(teams.get_teams())

print("=== NBA DATA CENTER ===")
print("Stagione attuale:", STAGIONE_ATTUALE)
print("Squadre disponibili:", len(squadre))

display(
    squadre[["full_name", "abbreviation", "id"]]
)

# Recupero scoreboard NBA live
print("\n=== NBA LIVE SCOREBOARD ===")

try:
    dati_live = scoreboard.ScoreBoard().get_dict()

    partite = dati_live.get("scoreboard", {}).get(
        "games", []
    )

    righe = []

    for partita in partite:

        casa = partita.get("homeTeam", {})
        ospite = partita.get("awayTeam", {})

        righe.append({
            "Casa": casa.get("teamTricode"),
            "Trasferta": ospite.get("teamTricode"),
            "Punti Casa": casa.get("score"),
            "Punti Trasferta": ospite.get("score"),
            "Stato": partita.get("gameStatusText")
        })

    df_partite = pd.DataFrame(righe)

    if df_partite.empty:
        print("Nessuna partita nello scoreboard corrente.")
    else:
        display(df_partite)

except Exception as e:
    print("Scoreboard non disponibile:", str(e))

print("\nConfigurazione completata!")


In [ ]:

# ============================================================
# NBA DATA SCIENCE — STEP 2
# DATABASE BUILDER 2024-25 / 2025-26 / 2026-27
# ============================================================

import os
import time
import sqlite3
import pandas as pd

from datetime import datetime, timezone
from google.colab import drive
from nba_api.stats.static import teams, players
from nba_api.stats.endpoints import (
    teamgamelogs,
    playergamelogs
)

# ============================================================
# 1. COLLEGAMENTO GOOGLE DRIVE
# ============================================================

drive.mount("/content/drive")

CARTELLA = "/content/drive/MyDrive/NBA_Data_Science"
os.makedirs(CARTELLA, exist_ok=True)

DB_PATH = os.path.join(CARTELLA, "nba_database.db")

print("Database:", DB_PATH)

# ============================================================
# 2. CONFIGURAZIONE
# ============================================================

STAGIONI = [
    "2024-25",
    "2025-26",
    "2026-27"
]

TIPI_STAGIONE = [
    "Regular Season",
    "Playoffs"
]

PAUSA_API = 2
TENTATIVI = 3

# ============================================================
# 3. FUNZIONE PER LE RICHIESTE NBA
# ============================================================

def richiesta_nba(endpoint_class, **kwargs):

    ultimo_errore = None

    for tentativo in range(1, TENTATIVI + 1):

        try:
            risposta = endpoint_class(
                timeout=60,
                **kwargs
            )

            dataframe = risposta.get_data_frames()[0]

            time.sleep(PAUSA_API)

            return dataframe

        except Exception as errore:

            ultimo_errore = errore

            print(
                f"Tentativo {tentativo} fallito:",
                str(errore)[:150]
            )

            time.sleep(PAUSA_API * tentativo)

    raise RuntimeError(
        f"Richiesta NBA fallita: {ultimo_errore}"
    )

# ============================================================
# 4. DATABASE SQLITE
# ============================================================

conn = sqlite3.connect(DB_PATH)

def salva_tabella(df, nome_tabella, chiavi):

    if df.empty:
        print(f"{nome_tabella}: nessun dato ricevuto.")
        return

    df = df.copy()

    # Evita duplicati nella risposta
    df = df.drop_duplicates(subset=chiavi)

    # Crea una tabella temporanea
    nome_temp = "temp_" + nome_tabella

    df.to_sql(
        nome_temp,
        conn,
        if_exists="replace",
        index=False
    )

    # Se la tabella non esiste, la crea
    esiste = conn.execute(
        """
        SELECT name
        FROM sqlite_master
        WHERE type='table' AND name=?
        """,
        (nome_tabella,)
    ).fetchone()

    if not esiste:

        df.to_sql(
            nome_tabella,
            conn,
            if_exists="replace",
            index=False
        )

    else:

        # Aggiornamento delle righe esistenti
        # senza duplicare le partite.
        vecchio = pd.read_sql_query(
            f'SELECT * FROM "{nome_tabella}"',
            conn
        )

        combinato = pd.concat(
            [vecchio, df],
            ignore_index=True
        )

        combinato = combinato.drop_duplicates(
            subset=chiavi,
            keep="last"
        )

        combinato.to_sql(
            nome_tabella,
            conn,
            if_exists="replace",
            index=False
        )

    conn.execute(f'DROP TABLE IF EXISTS "{nome_temp}"')
    conn.commit()

    totale = pd.read_sql_query(
        f'SELECT COUNT(*) AS n FROM "{nome_tabella}"',
        conn
    )["n"].iloc[0]

    print(f"{nome_tabella}: {totale:,} righe salvate.")

# ============================================================
# 5. ANAGRAFICA SQUADRE E GIOCATORI
# ============================================================

print("\n=== ANAGRAFICA NBA ===")

df_teams = pd.DataFrame(teams.get_teams())

df_players = pd.DataFrame(players.get_players())

salva_tabella(
    df_teams,
    "teams",
    ["id"]
)

salva_tabella(
    df_players,
    "players",
    ["id"]
)

print("Squadre:", len(df_teams))
print("Giocatori in anagrafica:", len(df_players))

# ============================================================
# 6. SCARICAMENTO GAME LOGS
# ============================================================

errori = []
successi = []

for stagione in STAGIONI:

    for tipo in TIPI_STAGIONE:

        print("\n" + "=" * 60)
        print("STAGIONE:", stagione)
        print("TIPO:", tipo)
        print("=" * 60)

        # ----------------------------------------
        # TEAM GAME LOGS
        # ----------------------------------------

        try:

            df_team = richiesta_nba(
                teamgamelogs.TeamGameLogs,
                season_nullable=stagione,
                season_type_nullable=tipo,
                league_id_nullable="00"
            )

            if not df_team.empty:

                df_team["SEASON"] = stagione
                df_team["SEASON_TYPE"] = tipo

                salva_tabella(
                    df_team,
                    "team_game_logs",
                    ["GAME_ID", "TEAM_ID", "SEASON_TYPE"]
                )

            successi.append(
                (stagione, tipo, "team_game_logs")
            )

        except Exception as e:

            print("Errore TEAM:", e)

            errori.append({
                "stagione": stagione,
                "tipo": tipo,
                "tabella": "team_game_logs",
                "errore": str(e)
            })

        # ----------------------------------------
        # PLAYER GAME LOGS
        # ----------------------------------------

        try:

            df_player = richiesta_nba(
                playergamelogs.PlayerGameLogs,
                season_nullable=stagione,
                season_type_nullable=tipo,
                league_id_nullable="00"
            )

            if not df_player.empty:

                df_player["SEASON"] = stagione
                df_player["SEASON_TYPE"] = tipo

                salva_tabella(
                    df_player,
                    "player_game_logs",
                    ["GAME_ID", "PLAYER_ID", "SEASON_TYPE"]
                )

            successi.append(
                (stagione, tipo, "player_game_logs")
            )

        except Exception as e:

            print("Errore PLAYER:", e)

            errori.append({
                "stagione": stagione,
                "tipo": tipo,
                "tabella": "player_game_logs",
                "errore": str(e)
            })

# ============================================================
# 7. CONTROLLO DEL DATABASE
# ============================================================

print("\n=== DATABASE SUMMARY ===")

tabelle = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type='table'
    ORDER BY name
    """,
    conn
)

riepilogo = []

for nome in tabelle["name"]:

    totale = pd.read_sql_query(
        f'SELECT COUNT(*) AS totale FROM "{nome}"',
        conn
    )["totale"].iloc[0]

    riepilogo.append({
        "Tabella": nome,
        "Righe": totale
    })

display(pd.DataFrame(riepilogo))

# ============================================================
# 8. ESEMPIO: TOP SCORER STAGIONE CORRENTE
# ============================================================

if "player_game_logs" in tabelle["name"].values:

    query = """
    SELECT
        PLAYER_NAME,
        TEAM_ABBREVIATION,
        COUNT(*) AS GP,
        ROUND(AVG(PTS), 2) AS PPG,
        ROUND(AVG(REB), 2) AS RPG,
        ROUND(AVG(AST), 2) AS APG,
        ROUND(AVG(FG3M), 2) AS 3PM
    FROM player_game_logs
    WHERE SEASON = '2026-27'
      AND SEASON_TYPE = 'Regular Season'
    GROUP BY PLAYER_ID, PLAYER_NAME, TEAM_ABBREVIATION
    HAVING COUNT(*) >= 5
    ORDER BY PPG DESC
    LIMIT 20
    """

    top_scorer = pd.read_sql_query(query, conn)

    print("\n=== TOP 20 SCORER NBA 2026-27 ===")
    display(top_scorer)

# ============================================================
# 9. REPORT ERRORI
# ============================================================

print("\nRichieste completate:", len(successi))
print("Richieste fallite:", len(errori))

if errori:
    display(pd.DataFrame(errori))

    print(
        "ATTENZIONE: database parziale. "
        "Rieseguire lo step per riprovare le richieste."
    )
else:
    print("Tutte le richieste completate.")

# ============================================================
# 10. CHIUSURA
# ============================================================

conn.close()

print("\nDatabase salvato su Google Drive:")
print(DB_PATH)

print("\nSTEP 2 TERMINATO")


In [ ]:

print("Google Colab funziona correttamente!")

import pandas as pd
import numpy as np

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)


In [ ]:

# ============================================================
# NBA DATA SCIENCE — STEP 2.1
# DOWNLOAD STAGIONE 2025-26
# ============================================================

!pip -q install -U nba_api pandas openpyxl

import os
import time
import sqlite3
import pandas as pd

from google.colab import drive
from IPython.display import display
from nba_api.stats.static import teams, players
from nba_api.stats.endpoints import (
    teamgamelogs,
    playergamelogs
)

# 1. GOOGLE DRIVE

drive.mount("/content/drive")

CARTELLA = "/content/drive/MyDrive/NBA_Data_Science"
os.makedirs(CARTELLA, exist_ok=True)

DB_PATH = os.path.join(CARTELLA, "nba_database.db")

# 2. CONFIGURAZIONE

STAGIONE = "2025-26"
TIPO = "Regular Season"

# 3. CONNESSIONE DATABASE

conn = sqlite3.connect(DB_PATH)

# 4. FUNZIONE DI DOWNLOAD

def scarica(endpoint, **parametri):

    for tentativo in range(3):

        try:
            dati = endpoint(
                timeout=45,
                **parametri
            ).get_data_frames()[0]

            if dati.empty:
                raise ValueError(
                    "L'API ha restituito zero righe."
                )

            return dati

        except Exception as errore:

            print(
                f"Tentativo {tentativo + 1}/3:",
                str(errore)[:150]
            )

            time.sleep(3 * (tentativo + 1))

    raise RuntimeError(
        "Download fallito dopo 3 tentativi."
    )

# 5. SALVATAGGIO SICURO SENZA DUPLICATI

def salva(df, tabella, chiavi):

    if df.empty:
        raise ValueError("Dataset vuoto.")

    df = df.drop_duplicates(subset=chiavi)

    esiste = conn.execute(
        "SELECT name FROM sqlite_master "
        "WHERE type='table' AND name=?",
        (tabella,)
    ).fetchone()

    if esiste:

        vecchio = pd.read_sql_query(
            f'SELECT * FROM "{tabella}"',
            conn
        )

        df = pd.concat(
            [vecchio, df],
            ignore_index=True
        )

        df = df.drop_duplicates(
            subset=chiavi,
            keep="last"
        )

    df.to_sql(
        tabella,
        conn,
        if_exists="replace",
        index=False
    )

    print(
        f"{tabella}: {len(df):,} righe salvate."
    )

# 6. ANAGRAFICHE

print("\nScaricamento anagrafiche...")

df_teams = pd.DataFrame(teams.get_teams())
df_players = pd.DataFrame(players.get_players())

salva(df_teams, "teams", ["id"])
salva(df_players, "players", ["id"])

# 7. TEAM GAME LOGS

print("\nScaricamento statistiche squadre...")

try:

    df_team = scarica(
        teamgamelogs.TeamGameLogs,
        season_nullable=STAGIONE,
        season_type_nullable=TIPO,
        league_id_nullable="00"
    )

    df_team["SEASON"] = STAGIONE
    df_team["SEASON_TYPE"] = TIPO

    salva(
        df_team,
        "team_game_logs",
        ["GAME_ID", "TEAM_ID", "SEASON_TYPE"]
    )

except Exception as errore:
    print("ERRORE TEAM:", errore)

# 8. PLAYER GAME LOGS

print("\nScaricamento statistiche giocatori...")

try:

    df_player = scarica(
        playergamelogs.PlayerGameLogs,
        season_nullable=STAGIONE,
        season_type_nullable=TIPO,
        league_id_nullable="00"
    )

    df_player["SEASON"] = STAGIONE
    df_player["SEASON_TYPE"] = TIPO

    salva(
        df_player,
        "player_game_logs",
        ["GAME_ID", "PLAYER_ID", "SEASON_TYPE"]
    )

except Exception as errore:
    print("ERRORE PLAYER:", errore)

# 9. CONTROLLO DATABASE

print("\n========== DATABASE SUMMARY ==========")

tabelle = pd.read_sql_query(
    "SELECT name FROM sqlite_master "
    "WHERE type='table' ORDER BY name",
    conn
)

for tabella in tabelle["name"]:

    totale = pd.read_sql_query(
        f'SELECT COUNT(*) AS totale FROM "{tabella}"',
        conn
    )["totale"].iloc[0]

    print(f"{tabella}: {totale:,} righe")

# 10. TOP SCORER

if "player_game_logs" in tabelle["name"].values:

    top = pd.read_sql_query(
        """
        SELECT
            PLAYER_NAME,
            COUNT(*) AS GP,
            ROUND(AVG(PTS), 1) AS PPG,
            ROUND(AVG(REB), 1) AS RPG,
            ROUND(AVG(AST), 1) AS APG
        FROM player_game_logs
        WHERE SEASON = '2025-26'
          AND SEASON_TYPE = 'Regular Season'
        GROUP BY PLAYER_ID, PLAYER_NAME
        HAVING COUNT(*) >= 20
        ORDER BY PPG DESC
        LIMIT 20
        """,
        conn
    )

    print("\nTOP 20 SCORER — 2025-26")
    display(top)

conn.close()

print("\nDatabase:", DB_PATH)
print("STEP 2.1 TERMINATO")


In [ ]:

import requests

url = "https://site.api.espn.com/apis/site/v2/sports/basketball/nba/scoreboard"

try:
    risposta = requests.get(
        url,
        params={"dates": "20261122"},
        timeout=15
    )

    print("HTTP STATUS:", risposta.status_code)
    print("DIMENSIONE RISPOSTA:", len(risposta.content))

    risposta.raise_for_status()

    dati = risposta.json()
    partite = dati.get("events", [])

    print("CONNESSIONE ESPN RIUSCITA")
    print("PARTITE TROVATE:", len(partite))

    for partita in partite:
        print(partita.get("name"), "-", partita.get("status", {}).get("type", {}).get("description"))

except Exception as errore:
    print("ERRORE:", type(errore).__name__)
    print(str(errore))


In [ ]:

# ==========================================================
# NBA DATA SCIENCE — STEP 2.3
# ESPN GAME DATABASE BUILDER
# ==========================================================

import os
import time
import sqlite3
import requests
import pandas as pd

from datetime import date, timedelta
from google.colab import drive
from IPython.display import display

# 1. GOOGLE DRIVE

if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")

CARTELLA = "/content/drive/MyDrive/NBA_Data_Science"
os.makedirs(CARTELLA, exist_ok=True)

DB_PATH = os.path.join(CARTELLA, "nba_database.db")

# 2. CONFIGURAZIONE

URL = (
    "https://site.api.espn.com/apis/site/v2/"
    "sports/basketball/nba/scoreboard"
)

PERIODI = {
    "2025-26": ("2025-10-21", "2026-04-12"),
    "2026-27": ("2026-10-20", "2026-11-23")
}

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0"
})

conn = sqlite3.connect(DB_PATH)

# 3. TABELLE DATABASE

conn.execute("""
CREATE TABLE IF NOT EXISTS espn_games (
    game_id TEXT PRIMARY KEY,
    season TEXT,
    game_date TEXT,
    home_team TEXT,
    away_team TEXT,
    home_id TEXT,
    away_id TEXT,
    home_score INTEGER,
    away_score INTEGER,
    status TEXT,
    completed INTEGER,
    updated_at TEXT
)
""")

conn.execute("""
CREATE TABLE IF NOT EXISTS espn_download_log (
    season TEXT,
    game_date TEXT,
    games_found INTEGER,
    downloaded_at TEXT,
    PRIMARY KEY (season, game_date)
)
""")

conn.commit()

# 4. FUNZIONE DOWNLOAD

def scarica_giornata(giorno):

    params = {
        "dates": giorno.strftime("%Y%m%d"),
        "limit": 100
    }

    for tentativo in range(3):

        try:
            risposta = session.get(
                URL,
                params=params,
                timeout=20
            )

            risposta.raise_for_status()
            return risposta.json().get("events", [])

        except Exception as e:
            print("Errore:", giorno, str(e)[:100])
            time.sleep(2 * (tentativo + 1))

    return None

# 5. ESTRAZIONE PARTITE

def estrai_partita(evento, stagione):

    competizioni = evento.get("competitions", [])

    if not competizioni:
        return None

    competizione = competizioni[0]
    squadre = competizione.get("competitors", [])

    casa = next(
        (s for s in squadre if s.get("homeAway") == "home"),
        None
    )

    ospite = next(
        (s for s in squadre if s.get("homeAway") == "away"),
        None
    )

    if casa is None or ospite is None:
        return None

    stato = evento.get("status", {}).get("type", {})
    completata = int(stato.get("completed", False))

    def punteggio(s):
        valore = s.get("score")
        try:
            return int(valore)
        except (ValueError, TypeError):
            return None

    return (
        str(evento["id"]),
        stagione,
        evento.get("date"),
        casa.get("team", {}).get("displayName"),
        ospite.get("team", {}).get("displayName"),
        str(casa.get("team", {}).get("id")),
        str(ospite.get("team", {}).get("id")),
        punteggio(casa),
        punteggio(ospite),
        stato.get("name"),
        completata,
        pd.Timestamp.now(tz="UTC").isoformat()
    )

# 6. DOWNLOAD INCREMENTALE

for stagione, (inizio, fine) in PERIODI.items():

    giorno = date.fromisoformat(inizio)
    ultimo = date.fromisoformat(fine)

    print("\nSTAGIONE:", stagione)

    while giorno <= ultimo:

        data_str = giorno.isoformat()

        gia_scaricato = conn.execute(
            """
            SELECT 1 FROM espn_download_log
            WHERE season=? AND game_date=?
            """,
            (stagione, data_str)
        ).fetchone()

        # La stagione corrente viene aggiornata
        if gia_scaricato and stagione != "2026-27":
            giorno += timedelta(days=1)
            continue

        eventi = scarica_giornata(giorno)

        if eventi is None:
            print("Giornata non scaricata:", data_str)
            giorno += timedelta(days=1)
            continue

        righe = []

        for evento in eventi:
            riga = estrai_partita(evento, stagione)
            if riga is not None:
                righe.append(riga)

        with conn:
            conn.executemany(
                """
                INSERT OR REPLACE INTO espn_games
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
                """,
                righe
            )

            conn.execute(
                """
                INSERT OR REPLACE INTO espn_download_log
                VALUES (?, ?, ?, ?)
                """,
                (
                    stagione,
                    data_str,
                    len(righe),
                    pd.Timestamp.now(tz="UTC").isoformat()
                )
            )

        print(
            data_str,
            "| Partite:",
            len(righe)
        )

        time.sleep(0.35)
        giorno += timedelta(days=1)

# 7. RIEPILOGO DATABASE

riepilogo = pd.read_sql_query(
    """
    SELECT
        season AS Stagione,
        COUNT(*) AS Partite,
        SUM(completed) AS Completate
    FROM espn_games
    GROUP BY season
    ORDER BY season
    """,
    conn
)

print("\n=== DATABASE NBA ===")
display(riepilogo)

# 8. ULTIME 15 PARTITE

ultime = pd.read_sql_query(
    """
    SELECT
        game_date,
        away_team,
        away_score,
        home_team,
        home_score,
        status
    FROM espn_games
    ORDER BY game_date DESC
    LIMIT 15
    """,
    conn
)

print("\n=== ULTIME PARTITE ===")
display(ultime)

conn.close()

print("\nDATABASE SALVATO:", DB_PATH)
print("STEP 2.3 COMPLETATO!")


In [ ]:
%pip install --force-reinstall --no-deps "pandas==2.2.3"


In [ ]:

# ==================================================
# NBA DATA SCIENCE — VERIFICA DATABASE
# ==================================================

import sqlite3
import pandas as pd
from google.colab import drive
from IPython.display import display

drive.mount("/content/drive")

DB_PATH = (
    "/content/drive/MyDrive/"
    "NBA_Data_Science/nba_database.db"
)

conn = sqlite3.connect(DB_PATH)

print("VERSIONE PANDAS:", pd.__version__)

# 1. Riepilogo stagioni
riepilogo = pd.read_sql_query("""
    SELECT
        season AS Stagione,
        COUNT(*) AS Partite,
        SUM(completed) AS Completate
    FROM espn_games
    GROUP BY season
    ORDER BY season
""", conn)

print("\n=== RIEPILOGO NBA ===")
print(riepilogo.to_string(index=False))

# 2. Controllo partite non completate
incomplete = pd.read_sql_query("""
    SELECT
        season,
        game_date,
        away_team,
        home_team,
        status
    FROM espn_games
    WHERE completed = 0
    ORDER BY game_date
""", conn)

print("\n=== PARTITE NON COMPLETATE ===")
print(incomplete.to_string(index=False))

# 3. Ultime 10 partite concluse
ultime = pd.read_sql_query("""
    SELECT
        game_date,
        away_team,
        away_score,
        home_team,
        home_score
    FROM espn_games
    WHERE completed = 1
    ORDER BY game_date DESC
    LIMIT 10
""", conn)

print("\n=== ULTIME 10 PARTITE ===")
display(ultime)

conn.close()

print("\nVERIFICA COMPLETATA!")


In [ ]:

# ============================================================
# NBA DATA SCIENCE — STEP 3
# ESPN PLAYER BOX SCORE DATABASE
# ============================================================

import os
import time
import sqlite3
import requests
import pandas as pd

from google.colab import drive
from IPython.display import display

# ------------------------------------------------------------
# 1. GOOGLE DRIVE
# ------------------------------------------------------------

if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")

DB_PATH = (
    "/content/drive/MyDrive/"
    "NBA_Data_Science/nba_database.db"
)

if not os.path.isfile(DB_PATH):
    raise FileNotFoundError(
        "Database non trovato. Esegui prima lo STEP 2.3."
    )

conn = sqlite3.connect(DB_PATH)

# ------------------------------------------------------------
# 2. CONFIGURAZIONE
# ------------------------------------------------------------

URL = (
    "https://site.api.espn.com/apis/site/v2/"
    "sports/basketball/nba/summary"
)

# Prima prova con 10 partite.
# Dopo il test cambia a None per scaricare tutte le
# partite concluse non ancora elaborate.
LIMITE_PARTITE = None

PAUSA_API = 1.2
TENTATIVI = 3

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json"
})

# ------------------------------------------------------------
# 3. CREAZIONE TABELLE
# ------------------------------------------------------------

conn.execute("""
CREATE TABLE IF NOT EXISTS espn_player_boxscores (
    game_id TEXT NOT NULL,
    season TEXT,
    game_date TEXT,
    team_id TEXT NOT NULL,
    team_name TEXT,
    player_id TEXT NOT NULL,
    player_name TEXT,
    starter INTEGER,
    did_not_play INTEGER,
    minutes REAL,
    pts REAL,
    reb REAL,
    ast REAL,
    stl REAL,
    blk REAL,
    tov REAL,
    oreb REAL,
    dreb REAL,
    pf REAL,
    fgm REAL,
    fga REAL,
    fg3m REAL,
    fg3a REAL,
    ftm REAL,
    fta REAL,
    plus_minus REAL,
    PRIMARY KEY (game_id, team_id, player_id)
)
""")

conn.execute("""
CREATE TABLE IF NOT EXISTS espn_boxscore_log (
    game_id TEXT PRIMARY KEY,
    downloaded_at TEXT,
    players_found INTEGER
)
""")

conn.commit()

# ------------------------------------------------------------
# 4. FUNZIONI DI CONVERSIONE
# ------------------------------------------------------------

def numero(valore):

    if valore is None:
        return None

    testo = str(valore).strip()

    if testo in ("", "-", "--", "N/A", "DNP"):
        return None

    try:
        return float(testo.replace(",", ""))
    except (ValueError, TypeError):
        return None


def minuti(valore):

    if valore is None:
        return None

    testo = str(valore).strip()

    if ":" in testo:
        try:
            mm, ss = testo.split(":", 1)
            return int(mm) + int(ss) / 60
        except ValueError:
            return None

    return numero(testo)


def tiri(valore):

    if valore is None:
        return None, None

    testo = str(valore).strip()

    if "-" not in testo:
        return None, None

    try:
        fatti, tentati = testo.split("-", 1)
        return float(fatti), float(tentati)
    except ValueError:
        return None, None

# ------------------------------------------------------------
# 5. DOWNLOAD BOX SCORE
# ------------------------------------------------------------

def scarica_boxscore(game_id):

    for tentativo in range(1, TENTATIVI + 1):

        try:
            risposta = session.get(
                URL,
                params={"event": game_id},
                timeout=25
            )

            risposta.raise_for_status()

            dati = risposta.json()

            if not isinstance(dati, dict):
                raise ValueError("Risposta JSON non valida")

            return dati

        except Exception as errore:

            print(
                f"Errore {game_id} "
                f"({tentativo}/{TENTATIVI}): "
                f"{str(errore)[:100]}"
            )

            time.sleep(2 * tentativo)

    return None

# ------------------------------------------------------------
# 6. ESTRAZIONE DELLE STATISTICHE
# ------------------------------------------------------------

def estrai_giocatori(dati, game_id, stagione, data):

    boxscore = dati.get("boxscore", {})
    gruppi = boxscore.get("players", [])

    righe = []

    for gruppo in gruppi:

        team = gruppo.get("team", {})

        team_id = str(team.get("id", ""))
        team_name = team.get("displayName", "")

        for sezione in gruppo.get("statistics", []):

            nomi = sezione.get("names", [])
            atleti = sezione.get("athletes", [])

            for elemento in atleti:

                atleta = elemento.get("athlete", {})

                player_id = atleta.get("id")

                if player_id is None:
                    continue

                player_name = atleta.get(
                    "displayName",
                    atleta.get("fullName", "")
                )

                valori = elemento.get("stats", [])

                statistiche = dict(zip(nomi, valori))

                # ESPN può usare nomi leggermente diversi.
                def leggi(*chiavi):
                    for chiave in chiavi:
                        if chiave in statistiche:
                            return statistiche[chiave]
                    return None

                fg = leggi("FG", "fieldGoals")
                triple = leggi("3PT", "3P", "threePointFieldGoals")
                liberi = leggi("FT", "freeThrows")

                fgm, fga = tiri(fg)
                fg3m, fg3a = tiri(triple)
                ftm, fta = tiri(liberi)

                # DNP = giocatore che non ha disputato
                # minuti nella partita.
                dnp = bool(
                    elemento.get("didNotPlay", False)
                )

                riga = (
                    str(game_id),
                    stagione,
                    data,
                    team_id,
                    team_name,
                    str(player_id),
                    player_name,
                    int(bool(elemento.get("starter", False))),
                    int(dnp),
                    minuti(leggi("MIN", "minutes")),
                    numero(leggi("PTS", "points")),
                    numero(leggi("REB", "rebounds")),
                    numero(leggi("AST", "assists")),
                    numero(leggi("STL", "steals")),
                    numero(leggi("BLK", "blocks")),
                    numero(leggi("TO", "TOV", "turnovers")),
                    numero(leggi("OREB", "offensiveRebounds")),
                    numero(leggi("DREB", "defensiveRebounds")),
                    numero(leggi("PF", "fouls")),
                    fgm,
                    fga,
                    fg3m,
                    fg3a,
                    ftm,
                    fta,
                    numero(leggi("+/-", "plusMinus"))
                )

                righe.append(riga)

    return righe

# ------------------------------------------------------------
# 7. SELEZIONE PARTITE DA SCARICARE
# ------------------------------------------------------------

query = """
SELECT
    g.game_id,
    g.season,
    g.game_date
FROM espn_games g
LEFT JOIN espn_boxscore_log l
    ON g.game_id = l.game_id
WHERE g.completed = 1
  AND l.game_id IS NULL
ORDER BY g.game_date DESC
"""

partite = pd.read_sql_query(query, conn)

print("Partite ancora da elaborare:", len(partite))

if LIMITE_PARTITE is not None:
    partite = partite.head(LIMITE_PARTITE)

print("Partite selezionate:", len(partite))

# ------------------------------------------------------------
# 8. DOWNLOAD INCREMENTALE
# ------------------------------------------------------------

successi = 0
fallimenti = 0
giocatori_totali = 0

for indice, partita in enumerate(
    partite.itertuples(index=False),
    start=1
):

    game_id = str(partita.game_id)

    dati = scarica_boxscore(game_id)

    if dati is None:
        fallimenti += 1
        continue

    righe = estrai_giocatori(
        dati,
        game_id,
        partita.season,
        partita.game_date
    )

    # Non segnare come completato un box score vuoto.
    if not righe:
        print("Box score vuoto:", game_id)
        fallimenti += 1
        continue

    # Controllo minimo sulla struttura dei dati:
    # almeno una prestazione con punti e minuti.
    valide = [
        r for r in righe
        if r[9] is not None and r[10] is not None
    ]

    if not valide:
        print("Statistiche non valide:", game_id)
        fallimenti += 1
        continue

    with conn:

        conn.executemany(
            """
            INSERT OR REPLACE INTO espn_player_boxscores
            VALUES (
                ?, ?, ?, ?, ?, ?, ?, ?, ?, ?,
                ?, ?, ?, ?, ?, ?, ?, ?, ?, ?,
                ?, ?, ?, ?, ?, ?
            )
            """,
            righe
        )

        conn.execute(
            """
            INSERT OR REPLACE INTO espn_boxscore_log
            VALUES (?, ?, ?)
            """,
            (
                game_id,
                pd.Timestamp.now(tz="UTC").isoformat(),
                len(righe)
            )
        )

    successi += 1
    giocatori_totali += len(righe)

    print(
        f"[{indice}/{len(partite)}] "
        f"Partita {game_id} | "
        f"Giocatori: {len(righe)}"
    )

    time.sleep(PAUSA_API)

# ------------------------------------------------------------
# 9. CONTROLLO DATABASE
# ------------------------------------------------------------

print("\n========== STEP 3 SUMMARY ==========")

print("Partite scaricate:", successi)
print("Partite fallite:", fallimenti)
print("Righe giocatore scaricate:", giocatori_totali)

riepilogo = pd.read_sql_query(
    """
    SELECT
        season AS Stagione,
        COUNT(DISTINCT game_id) AS Partite,
        COUNT(*) AS Righe_Giocatori,
        COUNT(DISTINCT player_id) AS Giocatori_Unici
    FROM espn_player_boxscores
    GROUP BY season
    ORDER BY season
    """,
    conn
)

print("\n=== RIEPILOGO DATABASE ===")
print(riepilogo.to_string(index=False))

# ------------------------------------------------------------
# 10. TOP SCORER
# ------------------------------------------------------------

top = pd.read_sql_query(
    """
    SELECT
        player_name AS Giocatore,
        COUNT(*) AS GP,
        ROUND(AVG(pts), 1) AS PPG,
        ROUND(AVG(reb), 1) AS RPG,
        ROUND(AVG(ast), 1) AS APG
    FROM espn_player_boxscores
    WHERE season = '2026-27'
      AND did_not_play = 0
      AND minutes > 0
      AND pts IS NOT NULL
    GROUP BY player_id, player_name
    ORDER BY PPG DESC
    LIMIT 20
    """,
    conn
)

print("\n=== TOP SCORER 2026-27 ===")
print(top.to_string(index=False))

# ------------------------------------------------------------
# 11. CHIUSURA
# ------------------------------------------------------------

conn.close()

print("\nDATABASE SALVATO:", DB_PATH)
print("STEP 3 COMPLETATO!")


In [ ]:

# ============================================================
# NBA DATA SCIENCE — STEP 4
# ADVANCED ANALYTICS & PLAYER INTELLIGENCE
# ============================================================

import os
import sqlite3
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from google.colab import drive
from IPython.display import display

# ============================================================
# 1. COLLEGAMENTO DATABASE
# ============================================================

if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")

DB_PATH = (
    "/content/drive/MyDrive/"
    "NBA_Data_Science/nba_database.db"
)

if not os.path.isfile(DB_PATH):
    raise FileNotFoundError(
        "Database non trovato. Esegui prima STEP 2 e 3."
    )

conn = sqlite3.connect(DB_PATH)

tabelle = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table'",
    conn
)["name"].tolist()

necessarie = [
    "espn_games",
    "espn_player_boxscores"
]

for tabella in necessarie:
    if tabella not in tabelle:
        raise ValueError(
            f"Tabella mancante: {tabella}"
        )

# ============================================================
# 2. CARICAMENTO DATI
# ============================================================

games = pd.read_sql_query(
    "SELECT * FROM espn_games",
    conn
)

box = pd.read_sql_query(
    "SELECT * FROM espn_player_boxscores",
    conn
)

conn.close()

games["game_date"] = pd.to_datetime(
    games["game_date"],
    utc=True,
    errors="coerce"
)

box["game_date"] = pd.to_datetime(
    box["game_date"],
    utc=True,
    errors="coerce"
)

print("Partite nel database:", len(games))
print("Righe giocatori:", len(box))

# ============================================================
# 3. FILTRO STAGIONE
# ============================================================

STAGIONE = "2026-27"

df = box[
    (box["season"] == STAGIONE)
    & (box["did_not_play"] == 0)
    & (box["minutes"] > 0)
].copy()

if df.empty:
    raise ValueError(
        "Nessuna statistica disponibile per la stagione."
    )

print("\nStagione selezionata:", STAGIONE)
print("Prestazioni disponibili:", len(df))

# ============================================================
# 4. PREPARAZIONE STATISTICHE
# ============================================================

colonne_numeriche = [
    "minutes", "pts", "reb", "ast",
    "stl", "blk", "tov",
    "oreb", "dreb", "pf",
    "fgm", "fga", "fg3m",
    "fg3a", "ftm", "fta",
    "plus_minus"
]

for col in colonne_numeriche:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

# True Shooting %
den_ts = 2 * (
    df["fga"] + 0.44 * df["fta"]
)

df["TS%"] = np.where(
    den_ts > 0,
    df["pts"] / den_ts * 100,
    np.nan
)

# Effective Field Goal %
df["eFG%"] = np.where(
    df["fga"] > 0,
    (
        df["fgm"] + 0.5 * df["fg3m"]
    ) / df["fga"] * 100,
    np.nan
)

# ============================================================
# 5. CLASSIFICA GIOCATORI
# ============================================================

aggregazioni = {
    "player_name": "last",
    "game_id": "nunique",
    "minutes": "mean",
    "pts": "mean",
    "reb": "mean",
    "ast": "mean",
    "stl": "mean",
    "blk": "mean",
    "tov": "mean",
    "fg3m": "mean",
    "fgm": "sum",
    "fga": "sum",
    "fta": "sum",
    "fg3a": "sum",
    "ftm": "sum"
}

ranking = (
    df.groupby("player_id")
    .agg(aggregazioni)
    .rename(columns={
        "player_name": "Giocatore",
        "game_id": "GP",
        "minutes": "MIN",
        "pts": "PPG",
        "reb": "RPG",
        "ast": "APG",
        "stl": "SPG",
        "blk": "BPG",
        "tov": "TOPG",
        "fg3m": "3PM"
    })
    .reset_index()
)

ranking["FG%"] = np.where(
    ranking["fga"] > 0,
    ranking["fgm"] / ranking["fga"] * 100,
    np.nan
)

ranking["3P%"] = np.where(
    ranking["fg3a"] > 0,
    ranking["fg3m"] / ranking["fg3a"] * 100,
    np.nan
)

ranking["TS%"] = np.where(
    ranking["fga"] + 0.44 * ranking["fta"] > 0,
    (
        ranking["PPG"] * ranking["GP"]
        /
        (
            2 * (
                ranking["fga"]
                + 0.44 * ranking["fta"]
            )
        )
    ) * 100,
    np.nan
)

ranking["eFG%"] = np.where(
    ranking["fga"] > 0,
    (
        ranking["fgm"] + 0.5 * ranking["fg3m"]
    ) / ranking["fga"] * 100,
    np.nan
)

# Numero minimo di partite per ranking
MIN_GP = 5

ranking_validi = ranking[
    ranking["GP"] >= MIN_GP
].copy()

ranking_validi = ranking_validi.sort_values(
    "PPG",
    ascending=False
)

print("\n=== TOP 20 SCORER ===")

display(
    ranking_validi[
        [
            "Giocatore", "GP", "PPG",
            "RPG", "APG", "3PM",
            "FG%", "3P%", "TS%"
        ]
    ].head(20).round(2)
)

# ============================================================
# 6. ULTIME 5 / 10 / 20 PARTITE
# ============================================================

def ultime_partite(nome, n=10):

    dati = df[
        df["player_name"].str.lower()
        == nome.lower()
    ].sort_values(
        ["game_date", "game_id"],
        ascending=False
    )

    return dati.head(n)


def medie_recenti(nome, n=10):

    dati = ultime_partite(nome, n)

    if dati.empty:
        return None

    return {
        "Giocatore": nome,
        "Partite": len(dati),
        "PPG": dati["pts"].mean(),
        "RPG": dati["reb"].mean(),
        "APG": dati["ast"].mean(),
        "3PM": dati["fg3m"].mean(),
        "MIN": dati["minutes"].mean()
    }

# ============================================================
# 7. DASHBOARD SINGOLO GIOCATORE
# ============================================================

GIOCATORE = "Nikola Jokic"

dati_giocatore = df[
    df["player_name"].str.lower()
    == GIOCATORE.lower()
].sort_values("game_date")

if not dati_giocatore.empty:

    print("\n=== PLAYER DASHBOARD ===")
    print("Giocatore:", GIOCATORE)

    for n in [5, 10, 20]:

        statistiche = medie_recenti(
            GIOCATORE, n
        )

        print(
            f"Ultime {n}: "
            f"PPG={statistiche['PPG']:.1f} | "
            f"RPG={statistiche['RPG']:.1f} | "
            f"APG={statistiche['APG']:.1f}"
        )

    fig = go.Figure()

    for col, nome in [
        ("pts", "Punti"),
        ("reb", "Rimbalzi"),
        ("ast", "Assist")
    ]:

        fig.add_trace(
            go.Scatter(
                x=dati_giocatore["game_date"],
                y=dati_giocatore[col],
                mode="lines+markers",
                name=nome
            )
        )

    fig.update_layout(
        title=f"{GIOCATORE} — Performance Trend",
        template="plotly_dark",
        height=500,
        xaxis_title="Data",
        yaxis_title="Valore"
    )

    fig.show()

else:
    print(
        f"Giocatore {GIOCATORE} non trovato "
        "nel campione disponibile."
    )

# ============================================================
# 8. TEAM ANALYTICS
# ============================================================

partite_concluse = games[
    (games["season"] == STAGIONE)
    & (games["completed"] == 1)
].copy()

righe_team = []

for _, partita in partite_concluse.iterrows():

    casa = {
        "Squadra": partita["home_team"],
        "Data": partita["game_date"],
        "Punti Fatti": partita["home_score"],
        "Punti Subiti": partita["away_score"],
        "Vittoria": int(
            partita["home_score"] > partita["away_score"]
        )
    }

    ospite = {
        "Squadra": partita["away_team"],
        "Data": partita["game_date"],
        "Punti Fatti": partita["away_score"],
        "Punti Subiti": partita["home_score"],
        "Vittoria": int(
            partita["away_score"] > partita["home_score"]
        )
    }

    righe_team.extend([casa, ospite])

team_games = pd.DataFrame(righe_team)

team_ranking = (
    team_games.groupby("Squadra")
    .agg(
        GP=("Vittoria", "count"),
        W=("Vittoria", "sum"),
        PPG=("Punti Fatti", "mean"),
        OPP_PPG=("Punti Subiti", "mean")
    )
    .reset_index()
)

team_ranking["L"] = (
    team_ranking["GP"] - team_ranking["W"]
)

team_ranking["WIN%"] = (
    team_ranking["W"] / team_ranking["GP"] * 100
)

team_ranking["DIFF"] = (
    team_ranking["PPG"]
    - team_ranking["OPP_PPG"]
)

team_ranking = team_ranking.sort_values(
    ["WIN%", "DIFF"],
    ascending=False
)

print("\n=== TEAM POWER RANKING ===")

display(
    team_ranking[
        [
            "Squadra", "GP", "W", "L",
            "WIN%", "PPG", "OPP_PPG", "DIFF"
        ]
    ].round(2)
)

# ============================================================
# 9. TOP SCORER CHART
# ============================================================

fig = px.bar(
    ranking_validi.head(20),
    x="Giocatore",
    y="PPG",
    color="PPG",
    title="NBA — Top 20 Scorers",
    template="plotly_dark"
)

fig.update_layout(
    height=550,
    xaxis_tickangle=-45
)

fig.show()

# ============================================================
# 10. CONFRONTO GIOCATORI
# ============================================================

GIOCATORE_1 = "Nikola Jokic"
GIOCATORE_2 = "Shai Gilgeous-Alexander"

confronto = ranking[
    ranking["Giocatore"].isin([
        GIOCATORE_1,
        GIOCATORE_2
    ])
][
    ["Giocatore", "PPG", "RPG", "APG", "TS%"]
]

print("\n=== PLAYER COMPARISON ===")
display(confronto.round(2))

if len(confronto) == 2:

    confronto_long = confronto.melt(
        id_vars="Giocatore",
        var_name="Statistica",
        value_name="Valore"
    )

    fig = px.bar(
        confronto_long,
        x="Statistica",
        y="Valore",
        color="Giocatore",
        barmode="group",
        title="NBA Player Comparison",
        template="plotly_dark"
    )

    fig.show()

# ============================================================
# 11. EXPORT EXCEL
# ============================================================

OUTPUT = (
    "/content/drive/MyDrive/NBA_Data_Science/"
    "NBA_Advanced_Analytics.xlsx"
)

with pd.ExcelWriter(
    OUTPUT,
    engine="openpyxl"
) as writer:

    ranking.to_excel(
        writer,
        sheet_name="Player Rankings",
        index=False
    )

    team_ranking.to_excel(
        writer,
        sheet_name="Team Rankings",
        index=False
    )

    df.to_excel(
        writer,
        sheet_name="Player Game Logs",
        index=False
    )

print("\nExcel salvato:", OUTPUT)
print("\nSTEP 4 COMPLETATO!")


In [ ]:

# ============================================================
# NBA DATA SCIENCE & ANALYTICS
# STEP 4 — ADVANCED ANALYTICS
# VERSIONE CORRETTA
# ============================================================

import os
import sqlite3
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from google.colab import drive
from IPython.display import display

# ============================================================
# 1. COLLEGAMENTO GOOGLE DRIVE
# ============================================================

if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")

DB_PATH = (
    "/content/drive/MyDrive/"
    "NBA_Data_Science/nba_database.db"
)

if not os.path.isfile(DB_PATH):
    raise FileNotFoundError(
        "Database non trovato. Completa prima STEP 2 e STEP 3."
    )

conn = sqlite3.connect(DB_PATH)

# Verifica delle tabelle

tabelle = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table'",
    conn
)["name"].tolist()

for tabella in ["espn_games", "espn_player_boxscores"]:
    if tabella not in tabelle:
        conn.close()
        raise ValueError(f"Tabella mancante: {tabella}")

# ============================================================
# 2. CARICAMENTO DATABASE
# ============================================================

games = pd.read_sql_query(
    "SELECT * FROM espn_games",
    conn
)

box = pd.read_sql_query(
    "SELECT * FROM espn_player_boxscores",
    conn
)

conn.close()

games["game_date"] = pd.to_datetime(
    games["game_date"],
    utc=True,
    errors="coerce"
)

box["game_date"] = pd.to_datetime(
    box["game_date"],
    utc=True,
    errors="coerce"
)

print("================================")
print("NBA DATABASE CARICATO")
print("================================")
print("Partite:", len(games))
print("Prestazioni giocatori:", len(box))

# ============================================================
# 3. CONFIGURAZIONE
# ============================================================

STAGIONE = "2026-27"

GIOCATORE = "Nikola Jokic"

GIOCATORE_1 = "Nikola Jokic"
GIOCATORE_2 = "Shai Gilgeous-Alexander"

MIN_GP = 5

# ============================================================
# 4. FILTRO STAGIONE
# ============================================================

df = box[
    (box["season"] == STAGIONE)
    & (box["did_not_play"] == 0)
    & (box["minutes"] > 0)
].copy()

if df.empty:
    raise ValueError(
        f"Nessuna prestazione disponibile per {STAGIONE}"
    )

print("\nStagione:", STAGIONE)
print("Prestazioni valide:", len(df))

# ============================================================
# 5. PREPARAZIONE STATISTICHE
# ============================================================

colonne_numeriche = [
    "minutes", "pts", "reb", "ast",
    "stl", "blk", "tov", "oreb",
    "dreb", "pf", "fgm", "fga",
    "fg3m", "fg3a", "ftm", "fta",
    "plus_minus"
]

for col in colonne_numeriche:

    if col not in df.columns:
        df[col] = np.nan

    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

# True Shooting % per singola partita

den_ts = 2 * (
    df["fga"] + 0.44 * df["fta"]
)

df["TS%"] = np.where(
    den_ts > 0,
    df["pts"] / den_ts * 100,
    np.nan
)

# Effective Field Goal % per singola partita

df["eFG%"] = np.where(
    df["fga"] > 0,
    (
        df["fgm"] + 0.5 * df["fg3m"]
    ) / df["fga"] * 100,
    np.nan
)

# ============================================================
# 6. PLAYER RANKINGS
# ============================================================

ranking = (
    df.groupby("player_id")
    .agg(
        Giocatore=("player_name", "last"),
        GP=("game_id", "nunique"),
        MIN=("minutes", "mean"),
        PPG=("pts", "mean"),
        RPG=("reb", "mean"),
        APG=("ast", "mean"),
        SPG=("stl", "mean"),
        BPG=("blk", "mean"),
        TOPG=("tov", "mean"),
        TPM=("fg3m", "mean"),
        FGM=("fgm", "sum"),
        FGA=("fga", "sum"),
        FG3M=("fg3m", "sum"),
        FG3A=("fg3a", "sum"),
        FTM=("ftm", "sum"),
        FTA=("fta", "sum"),
        PTS_TOTAL=("pts", "sum")
    )
    .reset_index()
)

# Percentuale dal campo

ranking["FG%"] = np.where(
    ranking["FGA"] > 0,
    ranking["FGM"] / ranking["FGA"] * 100,
    np.nan
)

# Percentuale da tre punti

ranking["3P%"] = np.where(
    ranking["FG3A"] > 0,
    ranking["FG3M"] / ranking["FG3A"] * 100,
    np.nan
)

# True Shooting % stagionale

den_ts_ranking = 2 * (
    ranking["FGA"] + 0.44 * ranking["FTA"]
)

ranking["TS%"] = np.where(
    den_ts_ranking > 0,
    ranking["PTS_TOTAL"] / den_ts_ranking * 100,
    np.nan
)

# Effective Field Goal % stagionale

ranking["eFG%"] = np.where(
    ranking["FGA"] > 0,
    (
        ranking["FGM"] + 0.5 * ranking["FG3M"]
    ) / ranking["FGA"] * 100,
    np.nan
)

# Ranking con minimo partite

ranking_validi = ranking[
    ranking["GP"] >= MIN_GP
].copy()

ranking_validi = ranking_validi.sort_values(
    "PPG",
    ascending=False
)

print("\n================================")
print("TOP 20 NBA SCORERS")
print("================================")

colonne_ranking = [
    "Giocatore", "GP", "MIN",
    "PPG", "RPG", "APG",
    "TPM", "FG%", "3P%",
    "TS%", "eFG%"
]

display(
    ranking_validi[
        colonne_ranking
    ].head(20).round(2)
)

# ============================================================
# 7. FUNZIONI ULTIME PARTITE
# ============================================================

def ultime_partite(nome, n=10):

    dati = df[
        df["player_name"].str.casefold()
        == nome.casefold()
    ].sort_values(
        ["game_date", "game_id"],
        ascending=False
    )

    return dati.head(n)


def medie_recenti(nome, n=10):

    dati = ultime_partite(nome, n)

    if dati.empty:
        return None

    return {
        "Giocatore": nome,
        "GP": len(dati),
        "PPG": dati["pts"].mean(),
        "RPG": dati["reb"].mean(),
        "APG": dati["ast"].mean(),
        "3PM": dati["fg3m"].mean(),
        "MIN": dati["minutes"].mean()
    }

# ============================================================
# 8. PLAYER DASHBOARD
# ============================================================

print("\n================================")
print("PLAYER DASHBOARD")
print("================================")

dati_giocatore = df[
    df["player_name"].str.casefold()
    == GIOCATORE.casefold()
].sort_values("game_date")

if not dati_giocatore.empty:

    print("Giocatore:", GIOCATORE)

    for n in [5, 10, 20]:

        stats = medie_recenti(GIOCATORE, n)

        print(
            f"Ultime {n} disponibili "
            f"(GP={stats['GP']}): "
            f"PPG={stats['PPG']:.1f} | "
            f"RPG={stats['RPG']:.1f} | "
            f"APG={stats['APG']:.1f}"
        )

    fig = go.Figure()

    for col, nome in [
        ("pts", "Punti"),
        ("reb", "Rimbalzi"),
        ("ast", "Assist")
    ]:

        fig.add_trace(
            go.Scatter(
                x=dati_giocatore["game_date"],
                y=dati_giocatore[col],
                mode="lines+markers",
                name=nome
            )
        )

    fig.update_layout(
        title=f"{GIOCATORE} — Performance Trend",
        template="plotly_dark",
        height=500,
        xaxis_title="Data",
        yaxis_title="Statistica"
    )

    fig.show()

else:

    print(
        f"{GIOCATORE} non trovato "
        "nel database disponibile."
    )

# ============================================================
# 9. TEAM ANALYTICS
# ============================================================

partite_concluse = games[
    (games["season"] == STAGIONE)
    & (games["completed"] == 1)
    & games["home_score"].notna()
    & games["away_score"].notna()
].copy()

righe_team = []

for _, partita in partite_concluse.iterrows():

    casa = {
        "Squadra": partita["home_team"],
        "Data": partita["game_date"],
        "Punti Fatti": partita["home_score"],
        "Punti Subiti": partita["away_score"],
        "Vittoria": int(
            partita["home_score"] > partita["away_score"]
        )
    }

    ospite = {
        "Squadra": partita["away_team"],
        "Data": partita["game_date"],
        "Punti Fatti": partita["away_score"],
        "Punti Subiti": partita["home_score"],
        "Vittoria": int(
            partita["away_score"] > partita["home_score"]
        )
    }

    righe_team.extend([casa, ospite])

team_games = pd.DataFrame(righe_team)

if team_games.empty:
    raise ValueError(
        "Nessuna partita conclusa disponibile."
    )

team_ranking = (
    team_games.groupby("Squadra")
    .agg(
        GP=("Vittoria", "count"),
        W=("Vittoria", "sum"),
        PPG=("Punti Fatti", "mean"),
        OPP_PPG=("Punti Subiti", "mean")
    )
    .reset_index()
)

team_ranking["L"] = (
    team_ranking["GP"] - team_ranking["W"]
)

team_ranking["WIN%"] = (
    team_ranking["W"] / team_ranking["GP"] * 100
)

team_ranking["DIFF"] = (
    team_ranking["PPG"] - team_ranking["OPP_PPG"]
)

team_ranking = team_ranking.sort_values(
    ["WIN%", "DIFF"],
    ascending=False
)

print("\n================================")
print("NBA TEAM RANKINGS")
print("================================")

display(
    team_ranking[
        [
            "Squadra", "GP", "W", "L",
            "WIN%", "PPG", "OPP_PPG", "DIFF"
        ]
    ].round(2)
)

# ============================================================
# 10. TOP SCORERS CHART
# ============================================================

if not ranking_validi.empty:

    fig = px.bar(
        ranking_validi.head(20),
        x="Giocatore",
        y="PPG",
        color="PPG",
        title=f"NBA {STAGIONE} — Top 20 Scorers",
        template="plotly_dark"
    )

    fig.update_layout(
        height=550,
        xaxis_tickangle=-45
    )

    fig.show()

# ============================================================
# 11. PLAYER COMPARISON
# ============================================================

confronto = ranking[
    ranking["Giocatore"].isin([
        GIOCATORE_1,
        GIOCATORE_2
    ])
][
    ["Giocatore", "GP", "PPG", "RPG", "APG", "TS%"]
].copy()

print("\n================================")
print("PLAYER COMPARISON")
print("================================")

display(confronto.round(2))

if len(confronto) == 2:

    confronto_long = confronto.melt(
        id_vars=["Giocatore", "GP"],
        value_vars=["PPG", "RPG", "APG", "TS%"],
        var_name="Statistica",
        value_name="Valore"
    )

    fig = px.bar(
        confronto_long,
        x="Statistica",
        y="Valore",
        color="Giocatore",
        barmode="group",
        title=(
            f"{GIOCATORE_1} vs "
            f"{GIOCATORE_2}"
        ),
        template="plotly_dark"
    )

    fig.show()

# ============================================================
# 12. EXPORT EXCEL
# ============================================================

OUTPUT = (
    "/content/drive/MyDrive/NBA_Data_Science/"
    "NBA_Advanced_Analytics.xlsx"
)

with pd.ExcelWriter(
    OUTPUT,
    engine="openpyxl"
) as writer:

    ranking.to_excel(
        writer,
        sheet_name="Player Rankings",
        index=False
    )

    team_ranking.to_excel(
        writer,
        sheet_name="Team Rankings",
        index=False
    )

    df.to_excel(
        writer,
        sheet_name="Player Game Logs",
        index=False
    )

    team_games.to_excel(
        writer,
        sheet_name="Team Game Logs",
        index=False
    )

print("\n================================")
print("STEP 4 COMPLETATO!")
print("================================")

print("Stagione analizzata:", STAGIONE)
print("Giocatori analizzati:", len(ranking))
print("Partite squadre:", len(partite_concluse))
print("Excel salvato:", OUTPUT)


In [ ]:

# ============================================================
# NBA DATA SCIENCE — STEP 4
# CORREZIONE ESPORTAZIONE EXCEL
# ============================================================

import os
import pandas as pd

OUTPUT = (
    "/content/drive/MyDrive/NBA_Data_Science/"
    "NBA_Advanced_Analytics.xlsx"
)

# Funzione per rendere le date compatibili con Excel
def prepara_excel(dataframe):

    risultato = dataframe.copy()

    for colonna in risultato.columns:

        if isinstance(
            risultato[colonna].dtype,
            pd.DatetimeTZDtype
        ):

            risultato[colonna] = (
                risultato[colonna]
                .dt.tz_localize(None)
            )

    return risultato

# Preparazione dei quattro fogli
ranking_excel = prepara_excel(ranking)
team_ranking_excel = prepara_excel(team_ranking)
df_excel = prepara_excel(df)
team_games_excel = prepara_excel(team_games)

# Esportazione
with pd.ExcelWriter(
    OUTPUT,
    engine="openpyxl"
) as writer:

    ranking_excel.to_excel(
        writer,
        sheet_name="Player Rankings",
        index=False
    )

    team_ranking_excel.to_excel(
        writer,
        sheet_name="Team Rankings",
        index=False
    )

    df_excel.to_excel(
        writer,
        sheet_name="Player Game Logs",
        index=False
    )

    team_games_excel.to_excel(
        writer,
        sheet_name="Team Game Logs",
        index=False
    )

print("================================")
print("STEP 4 COMPLETATO!")
print("================================")
print("Excel salvato:", OUTPUT)
print("Dimensione file:", round(
    os.path.getsize(OUTPUT) / 1024 / 1024, 2
), "MB")


In [ ]:

# ============================================================
# NBA DATA SCIENCE & ANALYTICS
# STEP 5 — MACHINE LEARNING WIN PREDICTION
# ============================================================

import os
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from collections import defaultdict, deque

from google.colab import drive

from sklearn.base import clone
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier

from sklearn.metrics import (
    accuracy_score,
    log_loss,
    brier_score_loss,
    roc_auc_score
)

# ============================================================
# 1. CARICAMENTO DATABASE
# ============================================================

if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")

CARTELLA = "/content/drive/MyDrive/NBA_Data_Science"
DB_PATH = os.path.join(CARTELLA, "nba_database.db")

if not os.path.isfile(DB_PATH):
    raise FileNotFoundError("Database NBA non trovato.")

conn = sqlite3.connect(DB_PATH)

games = pd.read_sql_query(
    """
    SELECT
        game_id,
        season,
        game_date,
        home_id,
        away_id,
        home_team,
        away_team,
        home_score,
        away_score,
        completed
    FROM espn_games
    """,
    conn
)

conn.close()

# ============================================================
# 2. PULIZIA PARTITE
# ============================================================

games["game_date"] = pd.to_datetime(
    games["game_date"],
    utc=True,
    errors="coerce"
)

games["home_score"] = pd.to_numeric(
    games["home_score"],
    errors="coerce"
)

games["away_score"] = pd.to_numeric(
    games["away_score"],
    errors="coerce"
)

games = games[
    (games["completed"] == 1)
    & games["game_date"].notna()
    & games["home_score"].notna()
    & games["away_score"].notna()
    & games["home_id"].notna()
    & games["away_id"].notna()
    & (games["home_score"] != games["away_score"])
].copy()

games = games.sort_values(
    ["game_date", "game_id"]
).drop_duplicates("game_id")

print("Partite valide:", len(games))

if len(games) < 200:
    raise ValueError(
        "Troppo poche partite per questo esperimento."
    )

# ============================================================
# 3. FEATURE ENGINEERING
# ============================================================

# Manteniamo la cronologia di ogni squadra.
# Le feature vengono calcolate prima di aggiungere
# il risultato della partita corrente.

storici = defaultdict(list)
righe = []

def statistiche_prepartita(storico, data):

    if len(storico) == 0:
        return None

    ultime_5 = storico[-5:]
    ultime_10 = storico[-10:]

    def media(partite, chiave):
        return float(
            np.mean([p[chiave] for p in partite])
        )

    giorni_riposo = (
        data.date() - storico[-1]["data"].date()
    ).days - 1

    return {
        "win5": media(ultime_5, "win"),
        "win10": media(ultime_10, "win"),
        "pf5": media(ultime_5, "pf"),
        "pa5": media(ultime_5, "pa"),
        "diff5": media(ultime_5, "diff"),
        "diff10": media(ultime_10, "diff"),
        "rest": float(
            np.clip(giorni_riposo, 0, 7)
        ),
        "gp": len(storico)
    }

for partita in games.itertuples(index=False):

    home_id = str(partita.home_id)
    away_id = str(partita.away_id)
    data = partita.game_date

    home_hist = storici[home_id]
    away_hist = storici[away_id]

    home = statistiche_prepartita(home_hist, data)
    away = statistiche_prepartita(away_hist, data)

    # Richiediamo almeno 5 partite precedenti
    # per entrambe le squadre.
    if (
        home is not None
        and away is not None
        and home["gp"] >= 5
        and away["gp"] >= 5
    ):

        riga = {
            "game_id": partita.game_id,
            "date": data,
            "season": partita.season,
            "home_team": partita.home_team,
            "away_team": partita.away_team,
            "home_win": int(
                partita.home_score > partita.away_score
            )
        }

        for chiave in [
            "win5", "win10", "pf5", "pa5",
            "diff5", "diff10", "rest", "gp"
        ]:

            riga[f"home_{chiave}"] = home[chiave]
            riga[f"away_{chiave}"] = away[chiave]

            riga[f"delta_{chiave}"] = (
                home[chiave] - away[chiave]
            )

        righe.append(riga)

    # Aggiorniamo gli storici soltanto DOPO
    # aver calcolato le feature della partita.

    hp = float(partita.home_score)
    ap = float(partita.away_score)

    home_hist.append({
        "data": data,
        "win": int(hp > ap),
        "pf": hp,
        "pa": ap,
        "diff": hp - ap
    })

    away_hist.append({
        "data": data,
        "win": int(ap > hp),
        "pf": ap,
        "pa": hp,
        "diff": ap - hp
    })

dataset = pd.DataFrame(righe)

print("Dataset ML:", dataset.shape)
print("Feature engineering completato.")

# ============================================================
# 4. TRAIN / VALIDATION / TEST
# ============================================================

# Suddivisione cronologica:
# 70% training
# 15% validation
# 15% test

dataset = dataset.sort_values(
    ["date", "game_id"]
).reset_index(drop=True)

feature_cols = [
    col for col in dataset.columns
    if col.startswith("delta_")
]

n = len(dataset)

fine_train = int(n * 0.70)
fine_val = int(n * 0.85)

train = dataset.iloc[:fine_train]
validation = dataset.iloc[fine_train:fine_val]
test = dataset.iloc[fine_val:]

X_train = train[feature_cols]
y_train = train["home_win"]

X_val = validation[feature_cols]
y_val = validation["home_win"]

X_test = test[feature_cols]
y_test = test["home_win"]

print("\n=== DATA SPLIT ===")
print("Training:", len(train))
print("Validation:", len(validation))
print("Test:", len(test))

print(
    "Test dal:",
    test["date"].min(),
    "al:",
    test["date"].max()
)

# ============================================================
# 5. MODELLI
# ============================================================

modelli = {
    "Baseline": DummyClassifier(
        strategy="prior"
    ),

    "Logistic Regression": make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=1.0,
            max_iter=2000,
            random_state=42
        )
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=6,
        min_samples_leaf=8,
        random_state=42,
        n_jobs=-1
    )
}

# ============================================================
# 6. VALUTAZIONE
# ============================================================

def valuta_modello(y, probabilita):

    probabilita = np.clip(
        np.asarray(probabilita),
        1e-6,
        1 - 1e-6
    )

    predizioni = (
        probabilita >= 0.5
    ).astype(int)

    risultati = {
        "Accuracy": accuracy_score(
            y, predizioni
        ),
        "Log Loss": log_loss(
            y, probabilita,
            labels=[0, 1]
        ),
        "Brier Score": brier_score_loss(
            y, probabilita
        )
    }

    if len(np.unique(y)) == 2:
        risultati["ROC AUC"] = roc_auc_score(
            y, probabilita
        )
    else:
        risultati["ROC AUC"] = np.nan

    return risultati

risultati_val = []
modelli_addestrati = {}

for nome, modello in modelli.items():

    fitted = clone(modello)

    fitted.fit(X_train, y_train)

    probabilita = fitted.predict_proba(
        X_val
    )[:, 1]

    metriche = valuta_modello(
        y_val, probabilita
    )

    risultati_val.append({
        "Modello": nome,
        **metriche
    })

    modelli_addestrati[nome] = fitted

classifica = pd.DataFrame(
    risultati_val
).sort_values(
    "Log Loss"
).reset_index(drop=True)

print("\n=== VALIDATION RESULTS ===")
print(classifica.round(4).to_string(index=False))

# ============================================================
# 7. SELEZIONE MIGLIOR MODELLO
# ============================================================

nome_migliore = classifica.iloc[0]["Modello"]

print("\nMiglior modello:", nome_migliore)

# Addestramento su TRAIN + VALIDATION
train_val = dataset.iloc[:fine_val]

X_train_val = train_val[feature_cols]
y_train_val = train_val["home_win"]

modello_finale = clone(
    modelli[nome_migliore]
)

modello_finale.fit(
    X_train_val,
    y_train_val
)

# ============================================================
# 8. TEST OUT-OF-SAMPLE
# ============================================================

prob_test = modello_finale.predict_proba(
    X_test
)[:, 1]

metriche_test = valuta_modello(
    y_test, prob_test
)

print("\n=== TEST RESULTS ===")

for metrica, valore in metriche_test.items():
    print(f"{metrica}: {valore:.4f}")

risultati_test = test[
    [
        "date", "home_team",
        "away_team", "home_win"
    ]
].copy()

risultati_test["P_HOME_WIN"] = prob_test

risultati_test["P_AWAY_WIN"] = (
    1 - prob_test
)

risultati_test["Predicted_Home_Win"] = (
    prob_test >= 0.5
).astype(int)

print("\n=== ESEMPI PREVISIONI TEST ===")
print(
    risultati_test.tail(15).to_string(index=False)
)

# ============================================================
# 9. GRAFICI VALUTAZIONE
# ============================================================

fig, ax = plt.subplots(figsize=(9, 5))

ax.bar(
    classifica["Modello"],
    classifica["Log Loss"]
)

ax.set_title("NBA — Validation Log Loss")
ax.set_ylabel("Log Loss (minore è meglio)")

plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

# ============================================================
# 10. SALVATAGGIO MODELLO
# ============================================================

MODEL_PATH = os.path.join(
    CARTELLA,
    "nba_win_prediction_model.joblib"
)

DATASET_PATH = os.path.join(
    CARTELLA,
    "nba_ml_features.csv"
)

TEST_PATH = os.path.join(
    CARTELLA,
    "nba_test_predictions.csv"
)

joblib.dump(
    {
        "model": modello_finale,
        "features": feature_cols,
        "model_name": nome_migliore,
        "test_metrics": metriche_test
    },
    MODEL_PATH
)

dataset.to_csv(
    DATASET_PATH,
    index=False
)

risultati_test.to_csv(
    TEST_PATH,
    index=False
)

print("\n================================")
print("STEP 5 COMPLETATO!")
print("================================")

print("Modello:", nome_migliore)
print("File modello:", MODEL_PATH)
print("Dataset ML:", DATASET_PATH)
print("Previsioni test:", TEST_PATH)


In [ ]:

# ============================================================
# NBA DATA SCIENCE & ANALYTICS
# STEP 6 — FORECAST PROSSIME PARTITE NBA
# ============================================================

import os
import sqlite3
import requests
import joblib
import numpy as np
import pandas as pd
import plotly.express as px

from collections import defaultdict
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
from google.colab import drive
from IPython.display import display

# ============================================================
# 1. CONFIGURAZIONE
# ============================================================

if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")

CARTELLA = "/content/drive/MyDrive/NBA_Data_Science"

DB_PATH = os.path.join(
    CARTELLA, "nba_database.db"
)

MODEL_PATH = os.path.join(
    CARTELLA, "nba_win_prediction_model.joblib"
)

GIORNI_FORECAST = 3

if not os.path.isfile(DB_PATH):
    raise FileNotFoundError("Database NBA mancante.")

if not os.path.isfile(MODEL_PATH):
    raise FileNotFoundError(
        "Modello ML mancante. Completa lo STEP 5."
    )

# Caricare file joblib soltanto da fonti attendibili.
pacchetto = joblib.load(MODEL_PATH)

modello = pacchetto["model"]
feature_cols = pacchetto["features"]

print("Modello caricato:", pacchetto["model_name"])
print("Feature richieste:", len(feature_cols))

# ============================================================
# 2. CARICAMENTO RISULTATI STORICI
# ============================================================

conn = sqlite3.connect(DB_PATH)

games = pd.read_sql_query("""
    SELECT
        game_id,
        season,
        game_date,
        home_id,
        away_id,
        home_team,
        away_team,
        home_score,
        away_score,
        completed
    FROM espn_games
""", conn)

conn.close()

games["game_date"] = pd.to_datetime(
    games["game_date"],
    utc=True,
    errors="coerce"
)

for col in ["home_score", "away_score", "completed"]:
    games[col] = pd.to_numeric(
        games[col],
        errors="coerce"
    )

games = games[
    (games["completed"] == 1)
    & games["game_date"].notna()
    & games["home_score"].notna()
    & games["away_score"].notna()
    & games["home_id"].notna()
    & games["away_id"].notna()
    & (games["home_score"] != games["away_score"])
].copy()

games = games.sort_values(
    ["game_date", "game_id"]
).drop_duplicates("game_id")

print("Partite storiche:", len(games))

# ============================================================
# 3. DOWNLOAD CALENDARIO ESPN
# ============================================================

URL = (
    "https://site.api.espn.com/apis/site/v2/"
    "sports/basketball/nba/scoreboard"
)

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json"
})

oggi_ny = datetime.now(
    ZoneInfo("America/New_York")
).date()

eventi_futuri = {}

for offset in range(GIORNI_FORECAST + 1):

    giorno = oggi_ny + timedelta(days=offset)

    try:
        risposta = session.get(
            URL,
            params={
                "dates": giorno.strftime("%Y%m%d"),
                "limit": 100
            },
            timeout=20
        )

        risposta.raise_for_status()

        eventi = risposta.json().get("events", [])

        for evento in eventi:
            eventi_futuri[str(evento["id"])] = evento

        print(
            giorno,
            "| Eventi:",
            len(eventi)
        )

    except Exception as errore:
        print(
            "Errore calendario:",
            giorno,
            str(errore)[:120]
        )

# ============================================================
# 4. ESTRAZIONE PARTITE PROGRAMMATE
# ============================================================

partite_future = []

for evento in eventi_futuri.values():

    stato = evento.get("status", {}).get("type", {})

    # Solo partite non ancora iniziate
    if stato.get("name") != "STATUS_SCHEDULED":
        continue

    competizioni = evento.get("competitions", [])

    if not competizioni:
        continue

    squadre = competizioni[0].get("competitors", [])

    casa = next(
        (
            s for s in squadre
            if s.get("homeAway") == "home"
        ),
        None
    )

    ospite = next(
        (
            s for s in squadre
            if s.get("homeAway") == "away"
        ),
        None
    )

    if casa is None or ospite is None:
        continue

    data = pd.to_datetime(
        evento.get("date"),
        utc=True,
        errors="coerce"
    )

    if pd.isna(data):
        continue

    # Evita incontri il cui orario è già passato
    if data <= pd.Timestamp.now(tz="UTC"):
        continue

    partite_future.append({
        "game_id": str(evento["id"]),
        "date": data,
        "home_id": str(casa["team"]["id"]),
        "away_id": str(ospite["team"]["id"]),
        "home_team": casa["team"]["displayName"],
        "away_team": ospite["team"]["displayName"]
    })

future = pd.DataFrame(partite_future)

if future.empty:
    raise ValueError(
        "Nessuna partita futura trovata. "
        "Controlla calendario ESPN e intervallo."
    )

future = future.sort_values("date").reset_index(drop=True)

print("\nPARTITE FUTURE TROVATE:", len(future))

# ============================================================
# 5. STORICO DELLE SQUADRE
# ============================================================

storici = defaultdict(list)

# Utilizziamo solo gare già terminate prima del momento
# in cui vengono generate le previsioni.
ora_utc = pd.Timestamp.now(tz="UTC")

games = games[games["game_date"] < ora_utc].copy()

for partita in games.itertuples(index=False):

    data = partita.game_date

    hp = float(partita.home_score)
    ap = float(partita.away_score)

    storici[str(partita.home_id)].append({
        "data": data,
        "win": int(hp > ap),
        "pf": hp,
        "pa": ap,
        "diff": hp - ap
    })

    storici[str(partita.away_id)].append({
        "data": data,
        "win": int(ap > hp),
        "pf": ap,
        "pa": hp,
        "diff": ap - hp
    })

# ============================================================
# 6. FEATURE ENGINEERING
# Stesse definizioni dello STEP 5
# ============================================================

def statistiche_prepartita(storico, data):

    # Escludiamo sempre incontri non precedenti
    storico = [
        p for p in storico
        if p["data"] < data
    ]

    if len(storico) < 5:
        return None

    ultime_5 = storico[-5:]
    ultime_10 = storico[-10:]

    def media(partite, chiave):
        return float(
            np.mean([p[chiave] for p in partite])
        )

    giorni_riposo = (
        data.date() - storico[-1]["data"].date()
    ).days - 1

    return {
        "win5": media(ultime_5, "win"),
        "win10": media(ultime_10, "win"),
        "pf5": media(ultime_5, "pf"),
        "pa5": media(ultime_5, "pa"),
        "diff5": media(ultime_5, "diff"),
        "diff10": media(ultime_10, "diff"),
        "rest": float(np.clip(giorni_riposo, 0, 7)),
        "gp": len(storico)
    }

# ============================================================
# 7. COSTRUZIONE FEATURES FUTURE
# ============================================================

righe_ml = []
partite_escluse = []

for partita in future.itertuples(index=False):

    home = statistiche_prepartita(
        storici[str(partita.home_id)],
        partita.date
    )

    away = statistiche_prepartita(
        storici[str(partita.away_id)],
        partita.date
    )

    if home is None or away is None:
        partite_escluse.append(
            f"{partita.away_team} @ {partita.home_team}"
        )
        continue

    riga = {
        "game_id": partita.game_id,
        "date": partita.date,
        "home_team": partita.home_team,
        "away_team": partita.away_team
    }

    for chiave in [
        "win5", "win10", "pf5", "pa5",
        "diff5", "diff10", "rest", "gp"
    ]:
        riga[f"delta_{chiave}"] = (
            home[chiave] - away[chiave]
        )

    righe_ml.append(riga)

forecast = pd.DataFrame(righe_ml)

if forecast.empty:
    raise ValueError(
        "Nessuna partita con storico sufficiente."
    )

# Controllo delle feature richieste dal modello
mancanti = [
    col for col in feature_cols
    if col not in forecast.columns
]

if mancanti:
    raise ValueError(
        f"Feature mancanti: {mancanti}"
    )

X_future = forecast[feature_cols].copy()

# ============================================================
# 8. PREVISIONI MACHINE LEARNING
# ============================================================

prob_home = modello.predict_proba(
    X_future
)[:, 1]

forecast["P_HOME_WIN"] = prob_home
forecast["P_AWAY_WIN"] = 1 - prob_home

forecast["P_HOME_%"] = (
    forecast["P_HOME_WIN"] * 100
)

forecast["P_AWAY_%"] = (
    forecast["P_AWAY_WIN"] * 100
)

forecast["Favorita"] = np.where(
    forecast["P_HOME_WIN"] >= 0.5,
    forecast["home_team"],
    forecast["away_team"]
)

forecast["Confidenza_%"] = (
    forecast[
        ["P_HOME_%", "P_AWAY_%"]
    ].max(axis=1)
)

forecast["Data_Italia"] = (
    forecast["date"]
    .dt.tz_convert("Europe/Rome")
    .dt.strftime("%d/%m/%Y %H:%M")
)

# ============================================================
# 9. TABELLA FORECAST
# ============================================================

tabella = forecast[
    [
        "Data_Italia",
        "away_team",
        "home_team",
        "P_AWAY_%",
        "P_HOME_%",
        "Favorita",
        "Confidenza_%"
    ]
].copy()

tabella.columns = [
    "Data Italia",
    "Trasferta",
    "Casa",
    "Vittoria Trasferta %",
    "Vittoria Casa %",
    "Favorita",
    "Probabilità Favorita %"
]

tabella = tabella.round(1)

print("\n====================================")
print("NBA FORECAST — PROSSIME PARTITE")
print("====================================")

display(tabella)

# ============================================================
# 10. DASHBOARD INTERATTIVA
# ============================================================

grafico = forecast.copy()

grafico["Partita"] = (
    grafico["away_team"]
    + " @ "
    + grafico["home_team"]
)

fig = px.bar(
    grafico,
    x="Partita",
    y=["P_HOME_%", "P_AWAY_%"],
    barmode="group",
    title="NBA — Probabilità di vittoria",
    template="plotly_dark",
    labels={
        "value": "Probabilità (%)",
        "variable": "Previsione"
    }
)

fig.update_layout(
    height=550,
    xaxis_tickangle=-35,
    yaxis_range=[0, 100]
)

fig.show()

# ============================================================
# 11. ESPORTAZIONE
# ============================================================

CSV_PATH = os.path.join(
    CARTELLA,
    "NBA_Forecast_Prossime_Partite.csv"
)

EXCEL_PATH = os.path.join(
    CARTELLA,
    "NBA_Forecast_Prossime_Partite.xlsx"
)

tabella.to_csv(
    CSV_PATH,
    index=False,
    encoding="utf-8-sig"
)

with pd.ExcelWriter(
    EXCEL_PATH,
    engine="openpyxl"
) as writer:

    tabella.to_excel(
        writer,
        sheet_name="NBA Forecast",
        index=False
    )

print("\n====================================")
print("STEP 6 COMPLETATO!")
print("====================================")

print("Partite previste:", len(forecast))
print("Partite escluse:", len(partite_escluse))
print("CSV:", CSV_PATH)
print("Excel:", EXCEL_PATH)

if partite_escluse:
    print("\nPartite senza storico sufficiente:")
    for nome in partite_escluse:
        print("-", nome)


In [ ]:

# ============================================================
# NBA DATA SCIENCE & ANALYTICS
# STEP 7 — PLAYER MACHINE LEARNING FORECAST
# ============================================================

import os
import sqlite3
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import drive
from IPython.display import display

from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

# ============================================================
# 1. COLLEGAMENTO DATABASE
# ============================================================

if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")

CARTELLA = "/content/drive/MyDrive/NBA_Data_Science"
DB_PATH = os.path.join(CARTELLA, "nba_database.db")

if not os.path.isfile(DB_PATH):
    raise FileNotFoundError("Database NBA non trovato.")

conn = sqlite3.connect(DB_PATH)

box = pd.read_sql_query(
    "SELECT * FROM espn_player_boxscores",
    conn
)

conn.close()

print("Righe disponibili:", len(box))
print("Colonne:", box.columns.tolist())

# ============================================================
# 2. PULIZIA DATI
# ============================================================

richieste = [
    "game_id", "game_date", "season",
    "player_id", "player_name",
    "minutes", "pts", "reb", "ast", "fg3m"
]

mancanti = [
    c for c in richieste if c not in box.columns
]

if mancanti:
    raise ValueError(
        f"Colonne mancanti nel database: {mancanti}"
    )

box["game_date"] = pd.to_datetime(
    box["game_date"],
    utc=True,
    errors="coerce"
)

numeriche = [
    "minutes", "pts", "reb", "ast", "fg3m"
]

for col in numeriche:
    box[col] = pd.to_numeric(
        box[col],
        errors="coerce"
    )

if "did_not_play" in box.columns:
    box["did_not_play"] = pd.to_numeric(
        box["did_not_play"],
        errors="coerce"
    ).fillna(0)

    box = box[box["did_not_play"] == 0]

box = box[
    box["game_date"].notna()
    & box["player_id"].notna()
    & box["game_id"].notna()
    & (box["minutes"] > 0)
].copy()

# Evitiamo di interpretare dati assenti come zero.
box = box.dropna(subset=[
    "pts", "reb", "ast", "fg3m"
])

box = box.drop_duplicates(
    ["game_id", "player_id"],
    keep="last"
)

box["PRA"] = (
    box["pts"] + box["reb"] + box["ast"]
)

box = box.sort_values(
    ["player_id", "game_date", "game_id"]
).reset_index(drop=True)

print("Prestazioni valide:", len(box))
print("Giocatori unici:", box["player_id"].nunique())

# ============================================================
# 3. FEATURE ENGINEERING
# ============================================================

TARGETS = {
    "PTS": "pts",
    "REB": "reb",
    "AST": "ast",
    "3PM": "fg3m",
    "PRA": "PRA"
}

MIN_STORICO = 10

righe = []

for player_id, gruppo in box.groupby("player_id"):

    gruppo = gruppo.sort_values(
        ["game_date", "game_id"]
    ).reset_index(drop=True)

    for i in range(MIN_STORICO, len(gruppo)):

        storico = gruppo.iloc[:i]
        corrente = gruppo.iloc[i]

        ultime5 = storico.tail(5)
        ultime10 = storico.tail(10)

        riga = {
            "player_id": player_id,
            "player_name": corrente["player_name"],
            "game_id": corrente["game_id"],
            "date": corrente["game_date"],
            "season": corrente["season"],
            "GP_precedenti": len(storico)
        }

        # Giorni di riposo
        giorni = (
            corrente["game_date"].date()
            - storico.iloc[-1]["game_date"].date()
        ).days - 1

        riga["rest_days"] = float(
            np.clip(giorni, 0, 7)
        )

        for nome, colonna in TARGETS.items():

            riga[f"{nome}_last1"] = float(
                storico.iloc[-1][colonna]
            )

            riga[f"{nome}_avg5"] = float(
                ultime5[colonna].mean()
            )

            riga[f"{nome}_avg10"] = float(
                ultime10[colonna].mean()
            )

            riga[f"{nome}_std10"] = float(
                ultime10[colonna].std(ddof=0)
            )

            # Valore da prevedere
            riga[f"TARGET_{nome}"] = float(
                corrente[colonna]
            )

        riga["MIN_avg5"] = float(
            ultime5["minutes"].mean()
        )

        riga["MIN_avg10"] = float(
            ultime10["minutes"].mean()
        )

        riga["MIN_last1"] = float(
            storico.iloc[-1]["minutes"]
        )

        righe.append(riga)

dataset = pd.DataFrame(righe)

if len(dataset) < 100:
    raise ValueError(
        "Dataset insufficiente. Completa prima "
        "il download dei box score dello STEP 3."
    )

dataset = dataset.sort_values(
    ["date", "game_id", "player_id"]
).reset_index(drop=True)

feature_cols = [
    c for c in dataset.columns
    if (
        c.endswith("_last1")
        or c.endswith("_avg5")
        or c.endswith("_avg10")
        or c.endswith("_std10")
        or c in ["rest_days", "GP_precedenti"]
    )
]

print("\n=== DATASET MACHINE LEARNING ===")
print("Esempi:", len(dataset))
print("Feature:", len(feature_cols))

# ============================================================
# 4. TRAIN / VALIDATION / TEST
# ============================================================

# Usiamo date distinte per evitare di dividere
# le prestazioni della stessa giornata tra i set.

date_uniche = np.array(
    sorted(dataset["date"].unique())
)

if len(date_uniche) < 20:
    raise ValueError(
        "Servono più giornate distinte per "
        "una validazione temporale attendibile."
    )

taglio_train = date_uniche[
    int(len(date_uniche) * 0.70)
]

taglio_test = date_uniche[
    int(len(date_uniche) * 0.85)
]

train = dataset[
    dataset["date"] < taglio_train
].copy()

validation = dataset[
    (dataset["date"] >= taglio_train)
    & (dataset["date"] < taglio_test)
].copy()

test = dataset[
    dataset["date"] >= taglio_test
].copy()

if min(len(train), len(validation), len(test)) == 0:
    raise ValueError("Uno dei set risulta vuoto.")

print("\n=== DATA SPLIT ===")
print("Training:", len(train))
print("Validation:", len(validation))
print("Test:", len(test))

X_train = train[feature_cols]
X_val = validation[feature_cols]
X_test = test[feature_cols]

# ============================================================
# 5. MODELLI DI REGRESSIONE
# ============================================================

modelli = {
    "Baseline Media": DummyRegressor(
        strategy="mean"
    ),

    "Ridge Regression": make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        Ridge(alpha=20.0)
    ),

    "Random Forest": make_pipeline(
        SimpleImputer(strategy="median"),
        RandomForestRegressor(
            n_estimators=150,
            max_depth=9,
            min_samples_leaf=10,
            random_state=42,
            n_jobs=-1
        )
    )
}

def metriche(y_true, y_pred):

    y_pred = np.maximum(
        np.asarray(y_pred, dtype=float),
        0
    )

    return {
        "MAE": mean_absolute_error(
            y_true, y_pred
        ),
        "RMSE": np.sqrt(
            mean_squared_error(y_true, y_pred)
        )
    }

# ============================================================
# 6. TRAINING E SELEZIONE
# ============================================================

risultati_validazione = []
modelli_finali = {}
risultati_test = []

for target_nome in TARGETS:

    col_target = f"TARGET_{target_nome}"

    y_train = train[col_target]
    y_val = validation[col_target]
    y_test = test[col_target]

    valutazioni = []

    for nome_modello, modello in modelli.items():

        fitted = clone(modello)
        fitted.fit(X_train, y_train)

        pred_val = fitted.predict(X_val)

        m = metriche(y_val, pred_val)

        valutazioni.append({
            "Target": target_nome,
            "Modello": nome_modello,
            **m
        })

    tab_val = pd.DataFrame(
        valutazioni
    ).sort_values("MAE")

    risultati_validazione.append(tab_val)

    migliore = tab_val.iloc[0]["Modello"]

    print(
        f"{target_nome}: miglior modello = {migliore}"
    )

    # Riaddestramento su train + validation
    train_val = pd.concat(
        [train, validation],
        ignore_index=True
    )

    modello_finale = clone(modelli[migliore])

    modello_finale.fit(
        train_val[feature_cols],
        train_val[col_target]
    )

    modelli_finali[target_nome] = modello_finale

    pred_test = np.maximum(
        modello_finale.predict(X_test),
        0
    )

    m_test = metriche(y_test, pred_test)

    risultati_test.append({
        "Target": target_nome,
        "Modello": migliore,
        **m_test
    })

tab_validazione = pd.concat(
    risultati_validazione,
    ignore_index=True
)

tab_test = pd.DataFrame(risultati_test)

print("\n=== VALIDATION RESULTS ===")
display(tab_validazione.round(3))

print("\n=== TEST RESULTS ===")
display(tab_test.round(3))

# ============================================================
# 7. FORECAST PROSSIMA PRESTAZIONE
# ============================================================

# ATTENZIONE:
# Questa previsione è per la prossima presenza
# ipotetica del giocatore, non per una partita
# specifica già verificata nel calendario.

STAGIONE = "2026-27"

storico_recente = box[
    box["season"] == STAGIONE
].copy()

# Fallback: se la stagione è incompleta, usiamo
# tutto lo storico disponibile del giocatore.
if storico_recente.empty:
    storico_recente = box.copy()

def crea_features_prossima(storico):

    storico = storico.sort_values(
        ["game_date", "game_id"]
    )

    if len(storico) < MIN_STORICO:
        return None

    ultime5 = storico.tail(5)
    ultime10 = storico.tail(10)

    # Non conosciamo ancora la data effettiva
    # della prossima partita: ipotesi 1 giorno
    # di riposo, da sostituire nello STEP 8.
    riga = {
        "GP_precedenti": len(storico),
        "rest_days": 1.0
    }

    for nome, colonna in TARGETS.items():

        riga[f"{nome}_last1"] = float(
            storico.iloc[-1][colonna]
        )

        riga[f"{nome}_avg5"] = float(
            ultime5[colonna].mean()
        )

        riga[f"{nome}_avg10"] = float(
            ultime10[colonna].mean()
        )

        riga[f"{nome}_std10"] = float(
            ultime10[colonna].std(ddof=0)
        )

    riga["MIN_avg5"] = float(
        ultime5["minutes"].mean()
    )

    riga["MIN_avg10"] = float(
        ultime10["minutes"].mean()
    )

    riga["MIN_last1"] = float(
        storico.iloc[-1]["minutes"]
    )

    return riga

righe_forecast = []

for player_id, gruppo in storico_recente.groupby(
    "player_id"
):

    if len(gruppo) < MIN_STORICO:
        continue

    features = crea_features_prossima(gruppo)

    if features is None:
        continue

    riga = {
        "player_id": player_id,
        "Giocatore": gruppo.iloc[-1]["player_name"],
        **features
    }

    righe_forecast.append(riga)

future = pd.DataFrame(righe_forecast)

if future.empty:
    raise ValueError(
        "Nessun giocatore con almeno 10 partite."
    )

X_future = future[feature_cols]

for target_nome, modello in modelli_finali.items():

    future[f"Forecast_{target_nome}"] = np.maximum(
        modello.predict(X_future),
        0
    )

colonne_output = [
    "Giocatore",
    "Forecast_PTS",
    "Forecast_REB",
    "Forecast_AST",
    "Forecast_3PM",
    "Forecast_PRA"
]

forecast_giocatori = future[
    colonne_output
].sort_values(
    "Forecast_PTS",
    ascending=False
)

print("\n=== PLAYER FORECAST ===")
display(
    forecast_giocatori.head(30).round(2)
)

# ============================================================
# 8. GRAFICO FORECAST
# ============================================================

top20 = forecast_giocatori.head(20)

plt.figure(figsize=(12, 6))

plt.bar(
    top20["Giocatore"],
    top20["Forecast_PTS"]
)

plt.title("NBA — Forecast punti giocatori")
plt.ylabel("Punti previsti")
plt.xticks(rotation=75, ha="right")
plt.tight_layout()
plt.show()

# ============================================================
# 9. SALVATAGGIO MODELLI
# ============================================================

MODEL_PATH = os.path.join(
    CARTELLA,
    "nba_player_forecast_models.joblib"
)

FORECAST_PATH = os.path.join(
    CARTELLA,
    "NBA_Player_Forecast.xlsx"
)

joblib.dump({
    "models": modelli_finali,
    "features": feature_cols,
    "targets": TARGETS,
    "test_metrics": tab_test.to_dict("records")
}, MODEL_PATH)

with pd.ExcelWriter(
    FORECAST_PATH,
    engine="openpyxl"
) as writer:

    forecast_giocatori.to_excel(
        writer,
        sheet_name="Player Forecast",
        index=False
    )

    tab_validazione.to_excel(
        writer,
        sheet_name="Validation",
        index=False
    )

    tab_test.to_excel(
        writer,
        sheet_name="Test",
        index=False
    )

print("\n================================")
print("STEP 7 COMPLETATO!")
print("================================")
print("Modelli salvati:", MODEL_PATH)
print("Excel salvato:", FORECAST_PATH)
print("Giocatori previsti:", len(forecast_giocatori))


In [ ]:

# ============================================================
# NBA DATA SCIENCE & ANALYTICS
# STEP 8 — MATCHUP INTELLIGENCE
# ============================================================

import os
import sqlite3
import requests
import joblib
import numpy as np
import pandas as pd
import plotly.express as px

from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
from google.colab import drive
from IPython.display import display

# ============================================================
# 1. CONFIGURAZIONE
# ============================================================

if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")

CARTELLA = "/content/drive/MyDrive/NBA_Data_Science"

DB_PATH = os.path.join(
    CARTELLA, "nba_database.db"
)

MODEL_PATH = os.path.join(
    CARTELLA, "nba_player_forecast_models.joblib"
)

GIORNI_FORECAST = 3
MIN_STORICO = 10

for percorso in [DB_PATH, MODEL_PATH]:
    if not os.path.isfile(percorso):
        raise FileNotFoundError(
            f"File mancante: {percorso}"
        )

# Carica joblib solo da file attendibili
pacchetto = joblib.load(MODEL_PATH)

modelli = pacchetto["models"]
feature_cols = pacchetto["features"]
targets = pacchetto["targets"]

print("Modelli caricati:", list(modelli.keys()))

# ============================================================
# 2. CARICAMENTO DATABASE
# ============================================================

conn = sqlite3.connect(DB_PATH)

box = pd.read_sql_query(
    "SELECT * FROM espn_player_boxscores",
    conn
)

games = pd.read_sql_query(
    "SELECT * FROM espn_games",
    conn
)

conn.close()

print("Box score:", len(box))
print("Partite:", len(games))

# ============================================================
# 3. CONTROLLO SCHEMA
# ============================================================

richieste_box = [
    "game_id", "game_date",
    "player_id", "player_name",
    "minutes", "pts", "reb",
    "ast", "fg3m"
]

richieste_games = [
    "game_id", "game_date",
    "home_id", "away_id",
    "home_score", "away_score",
    "completed"
]

for col in richieste_box:
    if col not in box.columns:
        raise ValueError(
            f"Colonna box score mancante: {col}"
        )

for col in richieste_games:
    if col not in games.columns:
        raise ValueError(
            f"Colonna partite mancante: {col}"
        )

# Individuazione identificativo squadra del giocatore
possibili_team_id = [
    "team_id",
    "teamId",
    "teamID"
]

TEAM_COL = next(
    (c for c in possibili_team_id if c in box.columns),
    None
)

if TEAM_COL is None:
    raise ValueError(
        "Manca team_id nei box score. "
        "Non posso associare con sicurezza "
        "i giocatori alle squadre. "
        f"Colonne disponibili: {box.columns.tolist()}"
    )

# ============================================================
# 4. PULIZIA DATI
# ============================================================

box["game_date"] = pd.to_datetime(
    box["game_date"],
    utc=True,
    errors="coerce"
)

games["game_date"] = pd.to_datetime(
    games["game_date"],
    utc=True,
    errors="coerce"
)

for col in [
    "minutes", "pts", "reb", "ast", "fg3m"
]:
    box[col] = pd.to_numeric(
        box[col],
        errors="coerce"
    )

for col in [
    "home_score", "away_score", "completed"
]:
    games[col] = pd.to_numeric(
        games[col],
        errors="coerce"
    )

if "did_not_play" in box.columns:
    box = box[
        pd.to_numeric(
            box["did_not_play"],
            errors="coerce"
        ).fillna(0) == 0
    ].copy()

box = box[
    box["game_date"].notna()
    & box["player_id"].notna()
    & box[TEAM_COL].notna()
    & (box["minutes"] > 0)
].dropna(
    subset=["pts", "reb", "ast", "fg3m"]
).copy()

box["game_id"] = box["game_id"].astype(str)
box["player_id"] = box["player_id"].astype(str)
box["team_id"] = box[TEAM_COL].astype(str)

games["game_id"] = games["game_id"].astype(str)

for col in ["home_id", "away_id"]:
    games[col] = games[col].astype(str)

box = box.drop_duplicates(
    ["game_id", "player_id"],
    keep="last"
)

box["PRA"] = (
    box["pts"] + box["reb"] + box["ast"]
)

ora_utc = pd.Timestamp.now(tz="UTC")

box = box[
    box["game_date"] < ora_utc
].copy()

games = games[
    (games["completed"] == 1)
    & games["game_date"].notna()
    & (games["game_date"] < ora_utc)
    & games["home_score"].notna()
    & games["away_score"].notna()
].copy()

games = games.sort_values(
    ["game_date", "game_id"]
).drop_duplicates("game_id")

# ============================================================
# 5. CALENDARIO ESPN
# ============================================================

URL = (
    "https://site.api.espn.com/apis/site/v2/"
    "sports/basketball/nba/scoreboard"
)

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json"
})

oggi = datetime.now(
    ZoneInfo("America/New_York")
).date()

eventi = {}

for offset in range(GIORNI_FORECAST + 1):

    giorno = oggi + timedelta(days=offset)

    try:
        response = session.get(
            URL,
            params={
                "dates": giorno.strftime("%Y%m%d"),
                "limit": 100
            },
            timeout=25
        )

        response.raise_for_status()

        for evento in response.json().get(
            "events", []
        ):
            eventi[str(evento["id"])] = evento

        print("Calendario letto:", giorno)

    except Exception as errore:
        print(
            "Errore calendario:",
            giorno,
            str(errore)[:120]
        )

# ============================================================
# 6. ESTRAZIONE PROSSIME PARTITE
# ============================================================

partite = []

for evento in eventi.values():

    status = (
        evento.get("status", {})
        .get("type", {})
        .get("name")
    )

    if status != "STATUS_SCHEDULED":
        continue

    competizioni = evento.get("competitions", [])

    if not competizioni:
        continue

    competitors = competizioni[0].get(
        "competitors", []
    )

    casa = next(
        (
            x for x in competitors
            if x.get("homeAway") == "home"
        ),
        None
    )

    ospite = next(
        (
            x for x in competitors
            if x.get("homeAway") == "away"
        ),
        None
    )

    if casa is None or ospite is None:
        continue

    data = pd.to_datetime(
        evento.get("date"),
        utc=True,
        errors="coerce"
    )

    if pd.isna(data) or data <= ora_utc:
        continue

    partite.append({
        "game_id": str(evento["id"]),
        "date": data,
        "home_id": str(casa["team"]["id"]),
        "away_id": str(ospite["team"]["id"]),
        "home_team": casa["team"]["displayName"],
        "away_team": ospite["team"]["displayName"]
    })

future_games = pd.DataFrame(partite)

if future_games.empty:
    raise ValueError(
        "Nessuna partita futura trovata."
    )

future_games = future_games.sort_values(
    "date"
).drop_duplicates("game_id")

print("\nProssime partite:", len(future_games))

# ============================================================
# 7. DIFESA AVVERSARIA
# ============================================================

def difesa_avversaria(team_id, data):

    storico = games[
        (games["game_date"] < data)
        & (
            (games["home_id"] == team_id)
            | (games["away_id"] == team_id)
        )
    ].sort_values("game_date").tail(10)

    if storico.empty:
        return np.nan

    concessi = np.where(
        storico["home_id"] == team_id,
        storico["away_score"],
        storico["home_score"]
    )

    return float(np.mean(concessi))

# ============================================================
# 8. FEATURE PER IL MODELLO
# ============================================================

def crea_features_player(storico, data):

    storico = storico[
        storico["game_date"] < data
    ].sort_values(
        ["game_date", "game_id"]
    )

    if len(storico) < MIN_STORICO:
        return None

    ultime5 = storico.tail(5)
    ultime10 = storico.tail(10)

    giorni = (
        data.date()
        - storico.iloc[-1]["game_date"].date()
    ).days - 1

    riga = {
        "GP_precedenti": len(storico),
        "rest_days": float(
            np.clip(giorni, 0, 7)
        )
    }

    for nome, colonna in targets.items():

        riga[f"{nome}_last1"] = float(
            storico.iloc[-1][colonna]
        )

        riga[f"{nome}_avg5"] = float(
            ultime5[colonna].mean()
        )

        riga[f"{nome}_avg10"] = float(
            ultime10[colonna].mean()
        )

        riga[f"{nome}_std10"] = float(
            ultime10[colonna].std(ddof=0)
        )

    riga["MIN_avg5"] = float(
        ultime5["minutes"].mean()
    )

    riga["MIN_avg10"] = float(
        ultime10["minutes"].mean()
    )

    riga["MIN_last1"] = float(
        storico.iloc[-1]["minutes"]
    )

    return riga

# ============================================================
# 9. ASSOCIAZIONE GIOCATORI / PARTITE
# ============================================================

# Per ogni giocatore, usiamo l'ultima squadra
# registrata nel database.

ultime_squadre = (
    box.sort_values(
        ["game_date", "game_id"]
    )
    .drop_duplicates(
        "player_id",
        keep="last"
    )[
        ["player_id", "player_name", "team_id"]
    ]
)

righe_forecast = []

for partita in future_games.itertuples(index=False):

    for team_id, team_name, avversario, sede in [
        (
            partita.home_id,
            partita.home_team,
            partita.away_team,
            "Casa"
        ),
        (
            partita.away_id,
            partita.away_team,
            partita.home_team,
            "Trasferta"
        )
    ]:

        giocatori = ultime_squadre[
            ultime_squadre["team_id"] == team_id
        ]

        avversario_id = (
            partita.away_id
            if sede == "Casa"
            else partita.home_id
        )

        punti_concessi = difesa_avversaria(
            avversario_id,
            partita.date
        )

        for player in giocatori.itertuples(index=False):

            storico = box[
                box["player_id"] == player.player_id
            ]

            features = crea_features_player(
                storico,
                partita.date
            )

            if features is None:
                continue

            riga = {
                "game_id": partita.game_id,
                "Data": partita.date,
                "Giocatore": player.player_name,
                "Squadra": team_name,
                "Avversario": avversario,
                "Sede": sede,
                "Punti concessi avversario L10":
                    punti_concessi,
                **features
            }

            righe_forecast.append(riga)

forecast = pd.DataFrame(righe_forecast)

if forecast.empty:
    raise ValueError(
        "Nessun giocatore con storico sufficiente. "
        "Verifica team_id e copertura box score."
    )

print("Giocatori/partite analizzati:", len(forecast))

# ============================================================
# 10. MACHINE LEARNING FORECAST
# ============================================================

mancanti = [
    c for c in feature_cols
    if c not in forecast.columns
]

if mancanti:
    raise ValueError(
        f"Feature mancanti: {mancanti}"
    )

X_future = forecast[feature_cols]

for nome, modello in modelli.items():

    forecast[f"Forecast_{nome}"] = np.maximum(
        modello.predict(X_future),
        0
    )

# PRA coerente con la somma dei tre forecast
forecast["Forecast_PRA"] = (
    forecast["Forecast_PTS"]
    + forecast["Forecast_REB"]
    + forecast["Forecast_AST"]
)

# Selezione colonne
forecast["Data Italia"] = (
    forecast["Data"]
    .dt.tz_convert("Europe/Rome")
    .dt.strftime("%d/%m/%Y %H:%M")
)

output_cols = [
    "Data Italia",
    "Giocatore",
    "Squadra",
    "Avversario",
    "Sede",
    "Forecast_PTS",
    "Forecast_REB",
    "Forecast_AST",
    "Forecast_3PM",
    "Forecast_PRA",
    "Punti concessi avversario L10",
    "MIN_avg10",
    "rest_days"
]

tabella = forecast[output_cols].copy()

tabella = tabella.sort_values(
    ["Data Italia", "Forecast_PTS"],
    ascending=[True, False]
)

print("\n================================")
print("NBA MATCHUP PLAYER FORECAST")
print("================================")

display(tabella.round(2))

# ============================================================
# 11. DASHBOARD INTERATTIVA
# ============================================================

grafico = forecast.sort_values(
    "Forecast_PTS",
    ascending=False
).head(30)

fig = px.bar(
    grafico,
    x="Giocatore",
    y="Forecast_PTS",
    color="Squadra",
    hover_data=[
        "Avversario",
        "Sede",
        "Forecast_REB",
        "Forecast_AST",
        "Forecast_3PM"
    ],
    title="NBA — Top 30 Player Points Forecast",
    template="plotly_dark"
)

fig.update_layout(
    height=600,
    xaxis_tickangle=-45
)

fig.show()

# ============================================================
# 12. EXPORT EXCEL
# ============================================================

EXCEL_PATH = os.path.join(
    CARTELLA,
    "NBA_Matchup_Player_Forecast.xlsx"
)

CSV_PATH = os.path.join(
    CARTELLA,
    "NBA_Matchup_Player_Forecast.csv"
)

tabella.to_csv(
    CSV_PATH,
    index=False,
    encoding="utf-8-sig"
)

with pd.ExcelWriter(
    EXCEL_PATH,
    engine="openpyxl"
) as writer:

    tabella.to_excel(
        writer,
        sheet_name="Player Matchup Forecast",
        index=False
    )

    future_games_excel = future_games.copy()

    future_games_excel["date"] = (
        future_games_excel["date"]
        .dt.tz_convert("Europe/Rome")
        .dt.tz_localize(None)
    )

    future_games_excel.to_excel(
        writer,
        sheet_name="Upcoming Games",
        index=False
    )

print("\n================================")
print("STEP 8 COMPLETATO!")
print("================================")

print("Partite:", len(future_games))
print("Previsioni giocatori:", len(tabella))
print("Excel:", EXCEL_PATH)
print("CSV:", CSV_PATH)


In [ ]:

# ============================================================
# NBA DATA SCIENCE & ANALYTICS
# STEP 9 — ADVANCED MODEL TRAINING & BACKTESTING
# ============================================================

import os
import sqlite3
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import drive
from IPython.display import display

from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

# ============================================================
# 1. GOOGLE DRIVE E DATABASE
# ============================================================

if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")

CARTELLA = "/content/drive/MyDrive/NBA_Data_Science"
DB_PATH = os.path.join(CARTELLA, "nba_database.db")

if not os.path.isfile(DB_PATH):
    raise FileNotFoundError("Database NBA non trovato.")

conn = sqlite3.connect(DB_PATH)

box = pd.read_sql_query(
    "SELECT * FROM espn_player_boxscores",
    conn
)

games = pd.read_sql_query(
    "SELECT * FROM espn_games",
    conn
)

conn.close()

print("Box score caricati:", len(box))
print("Partite caricate:", len(games))

# ============================================================
# 2. CONFIGURAZIONE
# ============================================================

MIN_STORICO = 10
MIN_PARTITE_DIFESA = 5

TARGETS = {
    "PTS": "pts",
    "REB": "reb",
    "AST": "ast",
    "3PM": "fg3m",
    "PRA": "PRA"
}

# ============================================================
# 3. PULIZIA DATI
# ============================================================

required_box = [
    "game_id", "game_date", "season",
    "player_id", "player_name", "team_id",
    "minutes", "pts", "reb", "ast", "fg3m"
]

required_games = [
    "game_id", "game_date", "season",
    "home_id", "away_id",
    "home_score", "away_score", "completed"
]

for c in required_box:
    if c not in box.columns:
        raise ValueError(f"Colonna box mancante: {c}")

for c in required_games:
    if c not in games.columns:
        raise ValueError(f"Colonna games mancante: {c}")

for frame in [box, games]:
    frame["game_id"] = frame["game_id"].astype(str)
    frame["game_date"] = pd.to_datetime(
        frame["game_date"],
        utc=True,
        errors="coerce"
    )

for c in ["player_id", "team_id"]:
    box[c] = box[c].astype(str)

for c in ["home_id", "away_id"]:
    games[c] = games[c].astype(str)

for c in ["minutes", "pts", "reb", "ast", "fg3m"]:
    box[c] = pd.to_numeric(box[c], errors="coerce")

for c in ["home_score", "away_score", "completed"]:
    games[c] = pd.to_numeric(games[c], errors="coerce")

if "did_not_play" in box.columns:
    box = box[
        pd.to_numeric(
            box["did_not_play"], errors="coerce"
        ).fillna(0) == 0
    ].copy()

box = box[
    box["game_date"].notna()
    & (box["minutes"] > 0)
].dropna(
    subset=["pts", "reb", "ast", "fg3m"]
).copy()

box = box.drop_duplicates(
    ["game_id", "player_id"],
    keep="last"
)

box["PRA"] = box["pts"] + box["reb"] + box["ast"]

games = games[
    (games["completed"] == 1)
    & games["game_date"].notna()
    & games["home_score"].notna()
    & games["away_score"].notna()
    & (games["home_score"] != games["away_score"])
].drop_duplicates("game_id").copy()

# Verifica che i dati non siano troppo incompleti
copertura = (
    box["game_id"].nunique()
    / max(games["game_id"].nunique(), 1)
)

print("\nCopertura box score:", f"{copertura:.1%}")

if copertura < 0.70:
    print(
        "ATTENZIONE: box score incompleti. "
        "Le medie e le previsioni potrebbero "
        "essere poco rappresentative."
    )

# ============================================================
# 4. IDENTIFICAZIONE MATCHUP
# ============================================================

matchups = games[
    [
        "game_id", "home_id", "away_id",
        "game_date", "season"
    ]
].rename(columns={
    "game_date": "game_date_match",
    "season": "game_season"
})

box = box.merge(
    matchups,
    on="game_id",
    how="inner",
    validate="many_to_one"
)

box["home"] = np.where(
    box["team_id"] == box["home_id"],
    1,
    np.where(
        box["team_id"] == box["away_id"],
        0,
        np.nan
    )
)

box["opponent_id"] = np.where(
    box["home"] == 1,
    box["away_id"],
    box["home_id"]
)

box = box.dropna(subset=["home"]).copy()

print("Prestazioni collegate alle partite:", len(box))

# ============================================================
# 5. STORICO DIFENSIVO DELLE SQUADRE
# ============================================================

team_rows = []

for g in games.itertuples(index=False):

    team_rows.append({
        "team_id": g.home_id,
        "date": g.game_date,
        "game_id": g.game_id,
        "allowed": float(g.away_score)
    })

    team_rows.append({
        "team_id": g.away_id,
        "date": g.game_date,
        "game_id": g.game_id,
        "allowed": float(g.home_score)
    })

team_history = pd.DataFrame(team_rows).sort_values(
    ["team_id", "date", "game_id"]
)

# Calcoliamo la difesa usando solo le partite
# precedenti alla giornata corrente.
# Non usiamo il punteggio della gara da prevedere.

team_history["game_day"] = (
    team_history["date"]
    .dt.tz_convert("America/New_York")
    .dt.date
)

team_daily = (
    team_history.groupby(
        ["team_id", "game_day"],
        as_index=False
    )["allowed"].mean()
)

team_daily = team_daily.sort_values(
    ["team_id", "game_day"]
)

team_daily["opp_allowed_l10"] = (
    team_daily.groupby("team_id")["allowed"]
    .transform(
        lambda s: s.shift(1).rolling(
            10, min_periods=MIN_PARTITE_DIFESA
        ).mean()
    )
)

# Se una squadra ha più gare nello stesso giorno,
# usiamo la media della giornata, senza includere
# nessun risultato di quel giorno nelle feature.

# ============================================================
# 6. FEATURE ENGINEERING GIOCATORI
# ============================================================

box["game_day"] = (
    box["game_date"]
    .dt.tz_convert("America/New_York")
    .dt.date
)

box = box.sort_values(
    ["player_id", "game_date", "game_id"]
)

features_rows = []

for player_id, gruppo in box.groupby("player_id"):

    gruppo = gruppo.sort_values(
        ["game_date", "game_id"]
    ).reset_index(drop=True)

    # Raggruppamento per giornata per evitare
    # che due gare nello stesso giorno si
    # influenzino reciprocamente.

    for giorno, gare_giorno in gruppo.groupby(
        "game_day", sort=True
    ):

        storico = gruppo[
            gruppo["game_day"] < giorno
        ].sort_values("game_date")

        if len(storico) < MIN_STORICO:
            continue

        last5 = storico.tail(5)
        last10 = storico.tail(10)

        ultima_data = storico.iloc[-1]["game_day"]

        rest = max(
            (giorno - ultima_data).days - 1,
            0
        )

        for corrente in gare_giorno.itertuples(index=False):

            r = {
                "game_id": corrente.game_id,
                "date": corrente.game_date,
                "game_day": giorno,
                "season": corrente.season,
                "player_id": player_id,
                "player_name": corrente.player_name,
                "team_id": corrente.team_id,
                "opponent_id": corrente.opponent_id,
                "home": float(corrente.home),
                "rest_days": float(np.clip(rest, 0, 7)),
                "GP_precedenti": len(storico),
                "MIN_avg5": float(last5["minutes"].mean()),
                "MIN_avg10": float(last10["minutes"].mean()),
                "MIN_last1": float(
                    storico.iloc[-1]["minutes"]
                )
            }

            for nome, col in TARGETS.items():

                r[f"{nome}_last1"] = float(
                    storico.iloc[-1][col]
                )

                r[f"{nome}_avg5"] = float(
                    last5[col].mean()
                )

                r[f"{nome}_avg10"] = float(
                    last10[col].mean()
                )

                r[f"{nome}_std10"] = float(
                    last10[col].std(ddof=0)
                )

                r[f"TARGET_{nome}"] = float(
                    getattr(corrente, col)
                )

            features_rows.append(r)

dataset = pd.DataFrame(features_rows)

if dataset.empty:
    raise ValueError(
        "Dataset vuoto. Verifica il download dello STEP 3."
    )

# ============================================================
# 7. MATCHUP FEATURES SENZA LEAKAGE
# ============================================================

dataset = dataset.merge(
    team_daily[
        ["team_id", "game_day", "opp_allowed_l10"]
    ].rename(columns={
        "team_id": "opponent_id"
    }),
    on=["opponent_id", "game_day"],
    how="left",
    validate="many_to_one"
)

# Per la prima partita del giorno, il valore
# proviene esclusivamente dalle giornate precedenti.

dataset = dataset.sort_values(
    ["date", "game_id", "player_id"]
).reset_index(drop=True)

feature_cols = [
    c for c in dataset.columns
    if (
        c.endswith("_last1")
        or c.endswith("_avg5")
        or c.endswith("_avg10")
        or c.endswith("_std10")
        or c in [
            "home",
            "rest_days",
            "GP_precedenti",
            "opp_allowed_l10"
        ]
    )
]

print("\nDataset ML:", dataset.shape)
print("Numero feature:", len(feature_cols))

# ============================================================
# 8. TRAIN / VALIDATION / TEST
# ============================================================

giornate = np.array(
    sorted(dataset["game_day"].unique())
)

if len(giornate) < 30:
    raise ValueError(
        "Servono almeno 30 giornate distinte."
    )

cut_train = giornate[int(len(giornate) * 0.70)]
cut_test = giornate[int(len(giornate) * 0.85)]

train = dataset[
    dataset["game_day"] < cut_train
].copy()

val = dataset[
    (dataset["game_day"] >= cut_train)
    & (dataset["game_day"] < cut_test)
].copy()

test = dataset[
    dataset["game_day"] >= cut_test
].copy()

if min(len(train), len(val), len(test)) == 0:
    raise ValueError("Uno dei set è vuoto.")

print("\n=== DATA SPLIT ===")
print("Train:", len(train))
print("Validation:", len(val))
print("Test:", len(test))
print("Inizio test:", cut_test)

X_train = train[feature_cols]
X_val = val[feature_cols]
X_test = test[feature_cols]

# ============================================================
# 9. MODELLI
# ============================================================

modelli = {
    "Ridge": make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        Ridge(alpha=20)
    ),

    "Random Forest": make_pipeline(
        SimpleImputer(strategy="median"),
        RandomForestRegressor(
            n_estimators=120,
            max_depth=8,
            min_samples_leaf=12,
            random_state=42,
            n_jobs=-1
        )
    )
}

def valuta(y, pred):

    pred = np.maximum(
        np.asarray(pred, dtype=float), 0
    )

    return {
        "MAE": mean_absolute_error(y, pred),
        "RMSE": np.sqrt(
            mean_squared_error(y, pred)
        ),
        "Bias": float(
            np.mean(pred - np.asarray(y))
        )
    }

# ============================================================
# 10. VALIDAZIONE E SELEZIONE
# ============================================================

validazioni = []
test_results = []
predizioni_test = []
modelli_finali = {}

train_val = pd.concat(
    [train, val],
    ignore_index=True
)

for target_nome in TARGETS:

    target_col = f"TARGET_{target_nome}"

    y_train = train[target_col]
    y_val = val[target_col]
    y_test = test[target_col]

    valutazioni = []

    # Baseline: media delle ultime 10 partite
    baseline_val = np.maximum(
        val[f"{target_nome}_avg10"].to_numpy(),
        0
    )

    m = valuta(y_val, baseline_val)

    valutazioni.append({
        "Target": target_nome,
        "Modello": "Last 10 Baseline",
        **m
    })

    for nome, modello in modelli.items():

        fitted = clone(modello)
        fitted.fit(X_train, y_train)

        pred = fitted.predict(X_val)

        m = valuta(y_val, pred)

        valutazioni.append({
            "Target": target_nome,
            "Modello": nome,
            **m
        })

    tab = pd.DataFrame(valutazioni).sort_values(
        ["MAE", "RMSE"]
    )

    validazioni.append(tab)

    migliore = tab.iloc[0]["Modello"]

    print(
        f"{target_nome} | Modello selezionato: {migliore}"
    )

    if migliore == "Last 10 Baseline":

        # La baseline non richiede addestramento.
        modello_finale = None

        pred_test = test[
            f"{target_nome}_avg10"
        ].to_numpy()

    else:

        modello_finale = clone(modelli[migliore])

        modello_finale.fit(
            train_val[feature_cols],
            train_val[target_col]
        )

        pred_test = modello_finale.predict(X_test)

    modelli_finali[target_nome] = {
        "name": migliore,
        "model": modello_finale
    }

    pred_test = np.maximum(pred_test, 0)

    test_results.append({
        "Target": target_nome,
        "Modello": migliore,
        **valuta(y_test, pred_test)
    })

    predizioni_test.append(
        pd.DataFrame({
            "game_id": test["game_id"].values,
            "date": test["date"].values,
            "player_id": test["player_id"].values,
            "player_name": test["player_name"].values,
            "Target": target_nome,
            "Reale": y_test.values,
            "Previsto": pred_test,
            "Errore": pred_test - y_test.values,
            "Errore Assoluto": np.abs(
                pred_test - y_test.values
            )
        })
    )

tab_validazione = pd.concat(
    validazioni,
    ignore_index=True
)

tab_test = pd.DataFrame(test_results)

dettaglio_test = pd.concat(
    predizioni_test,
    ignore_index=True
)

print("\n=== VALIDATION RESULTS ===")
display(tab_validazione.round(3))

print("\n=== TEST RESULTS ===")
display(tab_test.round(3))

# ============================================================
# 11. ERROR ANALYSIS PER GIOCATORE
# ============================================================

errori_giocatore = (
    dettaglio_test.groupby(
        ["player_id", "player_name", "Target"]
    )
    .agg(
        GP=("Errore Assoluto", "count"),
        MAE=("Errore Assoluto", "mean"),
        Bias=("Errore", "mean")
    )
    .reset_index()
)

errori_giocatore = errori_giocatore[
    errori_giocatore["GP"] >= 5
].sort_values(
    ["Target", "MAE"],
    ascending=[True, False]
)

print("\n=== ERROR ANALYSIS PER GIOCATORE ===")
display(errori_giocatore.head(30).round(2))

# ============================================================
# 12. GRAFICO CONFRONTO MODELLI
# ============================================================

for target_nome in TARGETS:

    subset = tab_validazione[
        tab_validazione["Target"] == target_nome
    ]

    plt.figure(figsize=(8, 4))

    plt.bar(
        subset["Modello"],
        subset["MAE"]
    )

    plt.title(
        f"NBA {target_nome} — Validation MAE"
    )

    plt.ylabel("Errore assoluto medio")
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.show()

# ============================================================
# 13. SALVATAGGIO MODELLI
# ============================================================

MODEL_PATH = os.path.join(
    CARTELLA,
    "nba_player_models_v2.joblib"
)

joblib.dump({
    "models": modelli_finali,
    "features": feature_cols,
    "targets": TARGETS,
    "minimum_history": MIN_STORICO,
    "validation": tab_validazione.to_dict("records"),
    "test": tab_test.to_dict("records"),
    "training_cutoff": str(cut_test),
    "model_version": "2.0"
}, MODEL_PATH)

# ============================================================
# 14. EXPORT EXCEL
# ============================================================

EXCEL_PATH = os.path.join(
    CARTELLA,
    "NBA_Model_Accuracy_Report.xlsx"
)

# Excel non supporta i timezone.
dettaglio_excel = dettaglio_test.copy()

dettaglio_excel["date"] = (
    pd.to_datetime(
        dettaglio_excel["date"],
        utc=True
    )
    .dt.tz_localize(None)
)

with pd.ExcelWriter(
    EXCEL_PATH,
    engine="openpyxl"
) as writer:

    tab_validazione.to_excel(
        writer,
        sheet_name="Validation",
        index=False
    )

    tab_test.to_excel(
        writer,
        sheet_name="Test",
        index=False
    )

    dettaglio_excel.to_excel(
        writer,
        sheet_name="Predictions",
        index=False
    )

    errori_giocatore.to_excel(
        writer,
        sheet_name="Player Errors",
        index=False
    )

print("\n====================================")
print("STEP 9 COMPLETATO!")
print("====================================")
print("Modelli:", MODEL_PATH)
print("Report:", EXCEL_PATH)
print("Esempi training:", len(train))
print("Esempi validation:", len(val))
print("Esempi test:", len(test))


In [ ]:

# ============================================================
# NBA DATA SCIENCE & ANALYTICS
# STEP 10 — FORECAST ENGINE V2 + DASHBOARD
# ============================================================

import os
import sqlite3
import requests
import joblib
import numpy as np
import pandas as pd
import plotly.express as px
import ipywidgets as widgets

from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
from IPython.display import display, clear_output
from google.colab import drive

# ============================================================
# 1. CONFIGURAZIONE
# ============================================================

if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")

CARTELLA = "/content/drive/MyDrive/NBA_Data_Science"

DB_PATH = os.path.join(CARTELLA, "nba_database.db")

MODEL_PATH = os.path.join(
    CARTELLA, "nba_player_models_v2.joblib"
)

GIORNI_FORECAST = 3
MIN_STORICO = 10

for path in [DB_PATH, MODEL_PATH]:
    if not os.path.isfile(path):
        raise FileNotFoundError(
            f"File mancante: {path}"
        )

# Caricare joblib solo da file attendibili.
pacchetto = joblib.load(MODEL_PATH)

modelli = pacchetto["models"]
feature_cols = pacchetto["features"]
targets = pacchetto["targets"]

print("Versione modello:", pacchetto["model_version"])
print("Target disponibili:", list(modelli.keys()))

# ============================================================
# 2. CARICAMENTO DATABASE
# ============================================================

conn = sqlite3.connect(DB_PATH)

box = pd.read_sql_query(
    "SELECT * FROM espn_player_boxscores",
    conn
)

games = pd.read_sql_query(
    "SELECT * FROM espn_games",
    conn
)

conn.close()

required_box = [
    "game_id", "game_date",
    "player_id", "player_name", "team_id",
    "minutes", "pts", "reb", "ast", "fg3m"
]

required_games = [
    "game_id", "game_date",
    "home_id", "away_id",
    "home_score", "away_score", "completed"
]

for c in required_box:
    if c not in box.columns:
        raise ValueError(f"Colonna box mancante: {c}")

for c in required_games:
    if c not in games.columns:
        raise ValueError(f"Colonna games mancante: {c}")

for frame in [box, games]:
    frame["game_id"] = frame["game_id"].astype(str)
    frame["game_date"] = pd.to_datetime(
        frame["game_date"],
        utc=True,
        errors="coerce"
    )

for c in ["player_id", "team_id"]:
    box[c] = box[c].astype(str)

for c in ["home_id", "away_id"]:
    games[c] = games[c].astype(str)

for c in ["minutes", "pts", "reb", "ast", "fg3m"]:
    box[c] = pd.to_numeric(box[c], errors="coerce")

for c in ["home_score", "away_score", "completed"]:
    games[c] = pd.to_numeric(games[c], errors="coerce")

if "did_not_play" in box.columns:
    box = box[
        pd.to_numeric(
            box["did_not_play"],
            errors="coerce"
        ).fillna(0) == 0
    ].copy()

box = box[
    box["game_date"].notna()
    & (box["minutes"] > 0)
].dropna(
    subset=["pts", "reb", "ast", "fg3m"]
).copy()

box = box.drop_duplicates(
    ["game_id", "player_id"],
    keep="last"
)

box["PRA"] = box["pts"] + box["reb"] + box["ast"]

ora_utc = pd.Timestamp.now(tz="UTC")

box = box[
    box["game_date"] < ora_utc
].copy()

games = games[
    (games["completed"] == 1)
    & games["game_date"].notna()
    & (games["game_date"] < ora_utc)
    & games["home_score"].notna()
    & games["away_score"].notna()
].sort_values(
    ["game_date", "game_id"]
).drop_duplicates("game_id").copy()

print("Prestazioni disponibili:", len(box))
print("Partite concluse:", len(games))

# ============================================================
# 3. CALENDARIO PROSSIME PARTITE ESPN
# ============================================================

URL = (
    "https://site.api.espn.com/apis/site/v2/"
    "sports/basketball/nba/scoreboard"
)

session = requests.Session()

session.headers.update({
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json"
})

oggi = datetime.now(
    ZoneInfo("America/New_York")
).date()

eventi = {}

for offset in range(GIORNI_FORECAST + 1):

    giorno = oggi + timedelta(days=offset)

    try:
        risposta = session.get(
            URL,
            params={
                "dates": giorno.strftime("%Y%m%d"),
                "limit": 100
            },
            timeout=25
        )

        risposta.raise_for_status()

        for evento in risposta.json().get(
            "events", []
        ):
            eventi[str(evento["id"])] = evento

        print("Calendario:", giorno, "OK")

    except Exception as errore:
        print("Errore ESPN:", giorno, errore)

partite = []

for evento in eventi.values():

    status = (
        evento.get("status", {})
        .get("type", {})
        .get("name")
    )

    if status != "STATUS_SCHEDULED":
        continue

    competizioni = evento.get("competitions", [])

    if not competizioni:
        continue

    competitors = competizioni[0].get(
        "competitors", []
    )

    casa = next(
        (
            x for x in competitors
            if x.get("homeAway") == "home"
        ),
        None
    )

    ospite = next(
        (
            x for x in competitors
            if x.get("homeAway") == "away"
        ),
        None
    )

    if casa is None or ospite is None:
        continue

    data = pd.to_datetime(
        evento.get("date"),
        utc=True,
        errors="coerce"
    )

    if pd.isna(data) or data <= ora_utc:
        continue

    partite.append({
        "game_id": str(evento["id"]),
        "date": data,
        "home_id": str(casa["team"]["id"]),
        "away_id": str(ospite["team"]["id"]),
        "home_team": casa["team"]["displayName"],
        "away_team": ospite["team"]["displayName"]
    })

future_games = pd.DataFrame(partite)

if future_games.empty:
    raise ValueError(
        "Nessuna partita futura trovata nel calendario."
    )

future_games = future_games.sort_values(
    "date"
).drop_duplicates("game_id")

print("Partite future:", len(future_games))

# ============================================================
# 4. STORICO DIFENSIVO
# ============================================================

def punti_concessi_ultime10(team_id, data):

    storico = games[
        (games["game_date"] < data)
        & (
            (games["home_id"] == team_id)
            | (games["away_id"] == team_id)
        )
    ].sort_values(
        ["game_date", "game_id"]
    )

    if len(storico) < 5:
        return np.nan

    ultime10 = storico.tail(10)

    concessi = np.where(
        ultime10["home_id"] == team_id,
        ultime10["away_score"],
        ultime10["home_score"]
    )

    return float(np.mean(concessi))

# ============================================================
# 5. SQUADRA ATTUALE DEI GIOCATORI
# ============================================================

ultime_squadre = (
    box.sort_values(
        ["game_date", "game_id"]
    )
    .drop_duplicates(
        "player_id",
        keep="last"
    )[
        ["player_id", "player_name", "team_id"]
    ]
)

# ============================================================
# 6. CREAZIONE FEATURE PREPARTITA
# ============================================================

def crea_features(storico, data, home, opp_allowed):

    storico = storico[
        storico["game_date"] < data
    ].sort_values(
        ["game_date", "game_id"]
    )

    if len(storico) < MIN_STORICO:
        return None

    last5 = storico.tail(5)
    last10 = storico.tail(10)

    # Coerente con lo STEP 9:
    # giorni tra le date delle due gare,
    # meno il giorno della partita corrente.
    data_ny = data.tz_convert(
        "America/New_York"
    ).date()

    ultima_ny = (
        storico.iloc[-1]["game_date"]
        .tz_convert("America/New_York")
        .date()
    )

    rest = max(
        (data_ny - ultima_ny).days - 1,
        0
    )

    r = {
        "home": float(home),
        "rest_days": float(np.clip(rest, 0, 7)),
        "GP_precedenti": len(storico),
        "MIN_avg5": float(last5["minutes"].mean()),
        "MIN_avg10": float(last10["minutes"].mean()),
        "MIN_last1": float(
            storico.iloc[-1]["minutes"]
        ),
        "opp_allowed_l10": opp_allowed
    }

    for nome, col in targets.items():

        r[f"{nome}_last1"] = float(
            storico.iloc[-1][col]
        )

        r[f"{nome}_avg5"] = float(
            last5[col].mean()
        )

        r[f"{nome}_avg10"] = float(
            last10[col].mean()
        )

        r[f"{nome}_std10"] = float(
            last10[col].std(ddof=0)
        )

    return r

# ============================================================
# 7. GENERAZIONE MATCHUP
# ============================================================

righe = []

for partita in future_games.itertuples(index=False):

    for team_id, squadra, opponent_id, avversario, home in [
        (
            partita.home_id,
            partita.home_team,
            partita.away_id,
            partita.away_team,
            1
        ),
        (
            partita.away_id,
            partita.away_team,
            partita.home_id,
            partita.home_team,
            0
        )
    ]:

        giocatori = ultime_squadre[
            ultime_squadre["team_id"] == team_id
        ]

        opp_allowed = punti_concessi_ultime10(
            opponent_id,
            partita.date
        )

        for player in giocatori.itertuples(index=False):

            storico = box[
                box["player_id"] == player.player_id
            ]

            f = crea_features(
                storico,
                partita.date,
                home,
                opp_allowed
            )

            if f is None:
                continue

            righe.append({
                "game_id": partita.game_id,
                "date": partita.date,
                "player_id": player.player_id,
                "Giocatore": player.player_name,
                "Squadra": squadra,
                "Avversario": avversario,
                "Sede": (
                    "Casa" if home == 1
                    else "Trasferta"
                ),
                **f
            })

forecast = pd.DataFrame(righe)

if forecast.empty:
    raise ValueError(
        "Nessun giocatore con storico sufficiente."
    )

print("Forecast da generare:", len(forecast))

# ============================================================
# 8. PREVISIONI MODELLI V2
# ============================================================

mancanti = [
    c for c in feature_cols
    if c not in forecast.columns
]

if mancanti:
    raise ValueError(
        f"Feature mancanti: {mancanti}"
    )

X_future = forecast[feature_cols]

for target_nome, info in modelli.items():

    nome_modello = info["name"]
    modello = info["model"]

    if nome_modello == "Last 10 Baseline":

        pred = forecast[
            f"{target_nome}_avg10"
        ].to_numpy()

    else:

        pred = modello.predict(X_future)

    forecast[f"Forecast_{target_nome}"] = np.maximum(
        np.asarray(pred, dtype=float),
        0
    )

# Manteniamo PRA coerente con le sue componenti.
forecast["Forecast_PRA"] = (
    forecast["Forecast_PTS"]
    + forecast["Forecast_REB"]
    + forecast["Forecast_AST"]
)

forecast["Data Italia"] = (
    forecast["date"]
    .dt.tz_convert("Europe/Rome")
    .dt.strftime("%d/%m/%Y %H:%M")
)

forecast["Partita"] = (
    forecast["Squadra"]
    + " vs "
    + forecast["Avversario"]
)

# Etichetta univoca per la partita
forecast["Matchup"] = (
    forecast["game_id"]
    + " | "
    + forecast["Partita"]
)

print("\n=== NBA FORECAST ENGINE V2 ===")

display(
    forecast[
        [
            "Data Italia",
            "Giocatore",
            "Squadra",
            "Avversario",
            "Sede",
            "Forecast_PTS",
            "Forecast_REB",
            "Forecast_AST",
            "Forecast_3PM",
            "Forecast_PRA"
        ]
    ].head(30).round(2)
)

# ============================================================
# 9. DASHBOARD INTERATTIVA
# ============================================================

from google.colab import output
output.enable_custom_widget_manager()

scelta_partita = widgets.Dropdown(
    options=["Tutte"] + sorted(
        forecast["Matchup"].unique().tolist()
    ),
    value="Tutte",
    description="Partita:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="550px")
)

scelta_statistica = widgets.Dropdown(
    options=["PTS", "REB", "AST", "3PM", "PRA"],
    value="PTS",
    description="Statistica:",
    style={"description_width": "initial"}
)

scelta_squadra = widgets.Dropdown(
    options=["Tutte"] + sorted(
        forecast["Squadra"].unique().tolist()
    ),
    value="Tutte",
    description="Squadra:",
    style={"description_width": "initial"}
)

scelta_giocatore = widgets.Dropdown(
    options=["Tutti"] + sorted(
        forecast["Giocatore"].unique().tolist()
    ),
    value="Tutti",
    description="Giocatore:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="400px")
)

area_output = widgets.Output()

def aggiorna_dashboard(change=None):

    with area_output:

        clear_output(wait=True)

        dati = forecast.copy()

        if scelta_partita.value != "Tutte":
            dati = dati[
                dati["Matchup"] == scelta_partita.value
            ]

        if scelta_squadra.value != "Tutte":
            dati = dati[
                dati["Squadra"] == scelta_squadra.value
            ]

        if scelta_giocatore.value != "Tutti":
            dati = dati[
                dati["Giocatore"] == scelta_giocatore.value
            ]

        col = f"Forecast_{scelta_statistica.value}"

        dati = dati.sort_values(
            col,
            ascending=False
        )

        print("Giocatori selezionati:", len(dati))

        if dati.empty:
            print("Nessun risultato per i filtri scelti.")
            return

        display(
            dati[
                [
                    "Data Italia",
                    "Giocatore",
                    "Squadra",
                    "Avversario",
                    "Sede",
                    col
                ]
            ].round(2)
        )

        grafico = dati.head(30)

        fig = px.bar(
            grafico,
            x="Giocatore",
            y=col,
            color="Squadra",
            hover_data=[
                "Avversario",
                "Sede",
                "Data Italia"
            ],
            title=(
                "NBA Forecast V2 — "
                + scelta_statistica.value
            ),
            template="plotly_dark"
        )

        fig.update_layout(
            height=550,
            xaxis_tickangle=-45
        )

        fig.show()

for controllo in [
    scelta_partita,
    scelta_statistica,
    scelta_squadra,
    scelta_giocatore
]:
    controllo.observe(
        aggiorna_dashboard,
        names="value"
    )

print("\n=== DASHBOARD INTERATTIVA ===")

display(
    widgets.VBox([
        scelta_partita,
        scelta_squadra,
        scelta_giocatore,
        scelta_statistica,
        area_output
    ])
)

aggiorna_dashboard()

# ============================================================
# 10. EXPORT EXCEL
# ============================================================

EXCEL_PATH = os.path.join(
    CARTELLA,
    "NBA_Forecast_Engine_V2.xlsx"
)

CSV_PATH = os.path.join(
    CARTELLA,
    "NBA_Forecast_Engine_V2.csv"
)

colonne_export = [
    "Data Italia",
    "Giocatore",
    "Squadra",
    "Avversario",
    "Sede",
    "Forecast_PTS",
    "Forecast_REB",
    "Forecast_AST",
    "Forecast_3PM",
    "Forecast_PRA",
    "MIN_avg10",
    "rest_days",
    "opp_allowed_l10"
]

export = forecast[colonne_export].copy()

export.to_csv(
    CSV_PATH,
    index=False,
    encoding="utf-8-sig"
)

with pd.ExcelWriter(
    EXCEL_PATH,
    engine="openpyxl"
) as writer:

    export.to_excel(
        writer,
        sheet_name="Player Forecast",
        index=False
    )

    metriche = pd.DataFrame(
        pacchetto.get("test", [])
    )

    if not metriche.empty:
        metriche.to_excel(
            writer,
            sheet_name="Model Accuracy",
            index=False
        )

print("\n====================================")
print("STEP 10 COMPLETATO!")
print("====================================")
print("Partite:", len(future_games))
print("Forecast:", len(forecast))
print("Excel:", EXCEL_PATH)
print("CSV:", CSV_PATH)


In [ ]:
!pip -q install gradio plotly openpyxl

In [ ]:

# ============================================================
# NBA DATA SCIENCE & ANALYTICS
# STEP 11 — NBA PREDICTION CENTER
# GRADIO WEB DASHBOARD
# ============================================================

import os
import sqlite3
import pandas as pd
import numpy as np
import plotly.express as px
import gradio as gr

from google.colab import drive

# ============================================================
# 1. GOOGLE DRIVE
# ============================================================

if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")

CARTELLA = "/content/drive/MyDrive/NBA_Data_Science"

PLAYER_FILE = os.path.join(
    CARTELLA,
    "NBA_Forecast_Engine_V2.csv"
)

GAME_FILE = os.path.join(
    CARTELLA,
    "nba_test_predictions.csv"
)

ACCURACY_FILE = os.path.join(
    CARTELLA,
    "NBA_Model_Accuracy_Report.xlsx"
)

DB_PATH = os.path.join(
    CARTELLA,
    "nba_database.db"
)

if not os.path.isfile(PLAYER_FILE):
    raise FileNotFoundError(
        "Manca NBA_Forecast_Engine_V2.csv. "
        "Esegui prima lo STEP 10."
    )

# ============================================================
# 2. CARICAMENTO DATASET
# ============================================================

players = pd.read_csv(PLAYER_FILE)

for c in [
    "Forecast_PTS",
    "Forecast_REB",
    "Forecast_AST",
    "Forecast_3PM",
    "Forecast_PRA"
]:
    if c not in players.columns:
        raise ValueError(
            f"Colonna forecast mancante: {c}"
        )

    players[c] = pd.to_numeric(
        players[c],
        errors="coerce"
    )

if os.path.isfile(GAME_FILE):

    games_predictions = pd.read_csv(GAME_FILE)

else:
    games_predictions = pd.DataFrame()

if os.path.isfile(ACCURACY_FILE):

    accuracy = pd.read_excel(
        ACCURACY_FILE,
        sheet_name="Test"
    )

else:
    accuracy = pd.DataFrame()

print("Player forecast caricati:", len(players))
print("Game predictions:", len(games_predictions))
print("Metriche accuracy:", len(accuracy))

# ============================================================
# 3. CONFIGURAZIONE DASHBOARD
# ============================================================

STATISTICHE = {
    "Punti": "Forecast_PTS",
    "Rimbalzi": "Forecast_REB",
    "Assist": "Forecast_AST",
    "Triple": "Forecast_3PM",
    "PRA": "Forecast_PRA"
}

squadre = sorted(
    players["Squadra"].dropna().unique().tolist()
)

giocatori = sorted(
    players["Giocatore"].dropna().unique().tolist()
)

# ============================================================
# 4. FUNZIONE PLAYER FORECAST
# ============================================================

def player_dashboard(
    squadra,
    giocatore,
    statistica,
    numero
):

    dati = players.copy()

    if squadra != "Tutte":
        dati = dati[
            dati["Squadra"] == squadra
        ]

    if giocatore != "Tutti":
        dati = dati[
            dati["Giocatore"] == giocatore
        ]

    col = STATISTICHE[statistica]

    dati = dati.sort_values(
        col,
        ascending=False
    )

    dati = dati.head(int(numero))

    colonne = [
        "Data Italia",
        "Giocatore",
        "Squadra",
        "Avversario",
        "Sede",
        "Forecast_PTS",
        "Forecast_REB",
        "Forecast_AST",
        "Forecast_3PM",
        "Forecast_PRA"
    ]

    tabella = dati[
        [c for c in colonne if c in dati.columns]
    ].round(2)

    if dati.empty:
        return tabella, None

    fig = px.bar(
        dati,
        x="Giocatore",
        y=col,
        color="Squadra",
        hover_data=[
            "Avversario",
            "Sede"
        ],
        title=f"NBA Forecast — {statistica}",
        template="plotly_dark"
    )

    fig.update_layout(
        height=500,
        xaxis_tickangle=-40
    )

    return tabella, fig

# ============================================================
# 5. CONFRONTO GIOCATORI
# ============================================================

def confronta_giocatori(
    giocatore1,
    giocatore2
):

    dati = players[
        players["Giocatore"].isin(
            [giocatore1, giocatore2]
        )
    ].copy()

    if dati.empty:
        return pd.DataFrame(), None

    colonne = [
        "Giocatore",
        "Forecast_PTS",
        "Forecast_REB",
        "Forecast_AST",
        "Forecast_3PM",
        "Forecast_PRA"
    ]

    # Se un giocatore ha più partite nel periodo,
    # confrontiamo la prima in ordine di data.
    if "Data Italia" in dati.columns:
        dati["_data"] = pd.to_datetime(
            dati["Data Italia"],
            dayfirst=True,
            errors="coerce"
        )
        dati = dati.sort_values("_data")

    dati = dati.drop_duplicates(
        "Giocatore",
        keep="first"
    )

    tabella = dati[colonne].round(2)

    long = tabella.melt(
        id_vars="Giocatore",
        var_name="Statistica",
        value_name="Forecast"
    )

    fig = px.bar(
        long,
        x="Statistica",
        y="Forecast",
        color="Giocatore",
        barmode="group",
        title="Confronto giocatori",
        template="plotly_dark"
    )

    return tabella, fig

# ============================================================
# 6. GAME PREDICTIONS
# ============================================================

def game_dashboard():

    if games_predictions.empty:

        return (
            pd.DataFrame(),
            "File delle previsioni squadre non disponibile."
        )

    dati = games_predictions.copy()

    # Questo file contiene il test storico dello STEP 5,
    # non necessariamente le prossime partite.
    return (
        dati.tail(100),
        "Previsioni storiche out-of-sample "
        "del modello squadre (STEP 5)."
    )

# ============================================================
# 7. MODEL ACCURACY
# ============================================================

def accuracy_dashboard():

    if accuracy.empty:
        return pd.DataFrame(), None

    dati = accuracy.copy()

    if "Target" not in dati.columns:
        return dati, None

    if "MAE" not in dati.columns:
        return dati, None

    fig = px.bar(
        dati,
        x="Target",
        y="MAE",
        color="Modello",
        title="NBA Player Forecast — Test MAE",
        template="plotly_dark"
    )

    return dati, fig

# ============================================================
# 8. CREAZIONE INTERFACCIA WEB
# ============================================================

with gr.Blocks(
    title="NBA Prediction Center"
) as app:

    gr.Markdown(
        """
        # NBA PREDICTION CENTER

        ### Machine Learning · Player Analytics · Forecasting

        Dashboard basata sui modelli NBA
        sviluppati in Google Colab.
        """
    )

    with gr.Tab("Player Forecast"):

        with gr.Row():

            squadra_input = gr.Dropdown(
                choices=["Tutte"] + squadre,
                value="Tutte",
                label="Squadra"
            )

            giocatore_input = gr.Dropdown(
                choices=["Tutti"] + giocatori,
                value="Tutti",
                label="Giocatore"
            )

        with gr.Row():

            statistica_input = gr.Dropdown(
                choices=list(STATISTICHE.keys()),
                value="Punti",
                label="Statistica"
            )

            numero_input = gr.Slider(
                minimum=5,
                maximum=100,
                value=30,
                step=5,
                label="Numero risultati"
            )

        aggiorna_btn = gr.Button(
            "Genera Player Forecast",
            variant="primary"
        )

        player_table = gr.Dataframe(
            label="Previsioni giocatori"
        )

        player_plot = gr.Plot(
            label="Grafico previsioni"
        )

        aggiorna_btn.click(
            fn=player_dashboard,
            inputs=[
                squadra_input,
                giocatore_input,
                statistica_input,
                numero_input
            ],
            outputs=[
                player_table,
                player_plot
            ]
        )

    with gr.Tab("Player Comparison"):

        with gr.Row():

            p1 = gr.Dropdown(
                choices=giocatori,
                value=giocatori[0] if giocatori else None,
                label="Giocatore 1"
            )

            p2 = gr.Dropdown(
                choices=giocatori,
                value=(
                    giocatori[1]
                    if len(giocatori) > 1
                    else giocatori[0] if giocatori else None
                ),
                label="Giocatore 2"
            )

        compare_btn = gr.Button(
            "Confronta giocatori"
        )

        compare_table = gr.Dataframe()
        compare_plot = gr.Plot()

        compare_btn.click(
            fn=confronta_giocatori,
            inputs=[p1, p2],
            outputs=[compare_table, compare_plot]
        )

    with gr.Tab("Game Predictions"):

        gr.Markdown(
            "### Storico previsioni squadre"
        )

        gr.Markdown(
            "Questa sezione mostra il test storico "
            "dello STEP 5, non il calendario futuro."
        )

        game_btn = gr.Button(
            "Mostra Game Predictions"
        )

        game_table = gr.Dataframe()
        game_info = gr.Textbox(
            label="Informazioni"
        )

        game_btn.click(
            fn=game_dashboard,
            outputs=[game_table, game_info]
        )

    with gr.Tab("Model Accuracy"):

        accuracy_btn = gr.Button(
            "Mostra accuratezza modelli"
        )

        accuracy_table = gr.Dataframe()
        accuracy_plot = gr.Plot()

        accuracy_btn.click(
            fn=accuracy_dashboard,
            outputs=[
                accuracy_table,
                accuracy_plot
            ]
        )

    gr.Markdown(
        """
        ---
        **Nota:** i forecast sono stime statistiche.
        Non costituiscono garanzie di prestazione.
        La disponibilità effettiva dei giocatori
        non è verificata automaticamente.
        """
    )

# ============================================================
# 9. AVVIO DASHBOARD
# ============================================================

print("\n====================================")
print("NBA PREDICTION CENTER")
print("====================================")
print("Dashboard pronta.")
print("Avvio interfaccia Gradio...")

app.launch(
    share=False,
    debug=True
)


In [ ]:
!pip -q install --upgrade gradio plotly openpyxl

In [ ]:

# ============================================================
# STEP 11 — NBA PREDICTION CENTER
# DASHBOARD GRADIO COMPLETA
# ============================================================

import os
import pandas as pd
import numpy as np
import plotly.express as px
import gradio as gr

from google.colab import drive

# ============================================================
# 1. GOOGLE DRIVE
# ============================================================

if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")

CARTELLA = "/content/drive/MyDrive/NBA_Data_Science"

PLAYER_FILE = os.path.join(
    CARTELLA,
    "NBA_Forecast_Engine_V2.csv"
)

GAME_FILE = os.path.join(
    CARTELLA,
    "nba_test_predictions.csv"
)

ACCURACY_FILE = os.path.join(
    CARTELLA,
    "NBA_Model_Accuracy_Report.xlsx"
)

if not os.path.isfile(PLAYER_FILE):
    raise FileNotFoundError(
        "File forecast non trovato. "
        "Esegui prima lo STEP 10."
    )

# ============================================================
# 2. CARICAMENTO DATI
# ============================================================

players = pd.read_csv(PLAYER_FILE)

colonne_richieste = [
    "Giocatore",
    "Squadra",
    "Avversario",
    "Sede",
    "Forecast_PTS",
    "Forecast_REB",
    "Forecast_AST",
    "Forecast_3PM",
    "Forecast_PRA"
]

mancanti = [
    c for c in colonne_richieste
    if c not in players.columns
]

if mancanti:
    raise ValueError(
        f"Colonne mancanti: {mancanti}"
    )

for col in [
    "Forecast_PTS",
    "Forecast_REB",
    "Forecast_AST",
    "Forecast_3PM",
    "Forecast_PRA"
]:
    players[col] = pd.to_numeric(
        players[col],
        errors="coerce"
    )

if os.path.isfile(GAME_FILE):
    games_predictions = pd.read_csv(GAME_FILE)
else:
    games_predictions = pd.DataFrame()

if os.path.isfile(ACCURACY_FILE):
    accuracy = pd.read_excel(
        ACCURACY_FILE,
        sheet_name="Test"
    )
else:
    accuracy = pd.DataFrame()

print("Giocatori/partite:", len(players))
print("Previsioni storiche:", len(games_predictions))
print("Metriche disponibili:", len(accuracy))

# ============================================================
# 3. OPZIONI DASHBOARD
# ============================================================

STATISTICHE = {
    "Punti": "Forecast_PTS",
    "Rimbalzi": "Forecast_REB",
    "Assist": "Forecast_AST",
    "Triple": "Forecast_3PM",
    "PRA": "Forecast_PRA"
}

squadre = sorted(
    players["Squadra"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

giocatori = sorted(
    players["Giocatore"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

# ============================================================
# 4. PLAYER FORECAST
# ============================================================

def player_dashboard(
    squadra,
    giocatore,
    statistica,
    numero
):

    dati = players.copy()

    if squadra != "Tutte":
        dati = dati[
            dati["Squadra"] == squadra
        ]

    if giocatore != "Tutti":
        dati = dati[
            dati["Giocatore"] == giocatore
        ]

    col = STATISTICHE[statistica]

    dati = dati.sort_values(
        col,
        ascending=False
    ).head(int(numero))

    colonne = [
        "Data Italia",
        "Giocatore",
        "Squadra",
        "Avversario",
        "Sede",
        "Forecast_PTS",
        "Forecast_REB",
        "Forecast_AST",
        "Forecast_3PM",
        "Forecast_PRA"
    ]

    colonne = [
        c for c in colonne
        if c in dati.columns
    ]

    tabella = dati[colonne].round(2)

    if dati.empty:
        return tabella, None

    fig = px.bar(
        dati,
        x="Giocatore",
        y=col,
        color="Squadra",
        hover_data=[
            "Avversario",
            "Sede"
        ],
        title=f"NBA Forecast — {statistica}",
        template="plotly_dark"
    )

    fig.update_layout(
        height=550,
        xaxis_tickangle=-45
    )

    return tabella, fig

# ============================================================
# 5. CONFRONTO GIOCATORI
# ============================================================

def confronta_giocatori(p1, p2):

    dati = players[
        players["Giocatore"].isin([p1, p2])
    ].copy()

    if dati.empty:
        return pd.DataFrame(), None

    if "Data Italia" in dati.columns:
        dati["_data"] = pd.to_datetime(
            dati["Data Italia"],
            dayfirst=True,
            errors="coerce"
        )
        dati = dati.sort_values("_data")

    dati = dati.drop_duplicates(
        "Giocatore",
        keep="first"
    )

    colonne = [
        "Giocatore",
        "Forecast_PTS",
        "Forecast_REB",
        "Forecast_AST",
        "Forecast_3PM",
        "Forecast_PRA"
    ]

    tabella = dati[colonne].round(2)

    grafico = tabella.melt(
        id_vars="Giocatore",
        var_name="Statistica",
        value_name="Forecast"
    )

    fig = px.bar(
        grafico,
        x="Statistica",
        y="Forecast",
        color="Giocatore",
        barmode="group",
        title="NBA — Confronto giocatori",
        template="plotly_dark"
    )

    return tabella, fig

# ============================================================
# 6. GAME PREDICTIONS
# ============================================================

def game_dashboard():

    if games_predictions.empty:
        return (
            pd.DataFrame(),
            "Nessun file di previsioni squadre disponibile."
        )

    return (
        games_predictions.tail(100),
        "Questi sono i risultati del test storico "
        "dello STEP 5, non forecast futuri."
    )

# ============================================================
# 7. MODEL ACCURACY
# ============================================================

def accuracy_dashboard():

    if accuracy.empty:
        return pd.DataFrame(), None

    dati = accuracy.copy()

    if not {"Target", "MAE"}.issubset(dati.columns):
        return dati, None

    fig = px.bar(
        dati,
        x="Target",
        y="MAE",
        color="Modello" if "Modello" in dati.columns else None,
        title="NBA Player Forecast — Test MAE",
        template="plotly_dark"
    )

    return dati, fig

# ============================================================
# 8. DASHBOARD GRADIO
# ============================================================

with gr.Blocks(
    title="NBA Prediction Center",
    theme=gr.themes.Soft()
) as app:

    gr.Markdown("""
    # 🏀 NBA PREDICTION CENTER

    **Machine Learning · NBA Analytics · Forecasting**

    Dashboard per l'analisi delle prestazioni
    e delle previsioni NBA.
    """)

    # --------------------------------------------------------
    # TAB 1 — PLAYER FORECAST
    # --------------------------------------------------------

    with gr.Tab("Player Forecast"):

        with gr.Row():

            squadra_input = gr.Dropdown(
                choices=["Tutte"] + squadre,
                value="Tutte",
                label="Squadra"
            )

            giocatore_input = gr.Dropdown(
                choices=["Tutti"] + giocatori,
                value="Tutti",
                label="Giocatore"
            )

        with gr.Row():

            statistica_input = gr.Dropdown(
                choices=list(STATISTICHE.keys()),
                value="Punti",
                label="Statistica"
            )

            numero_input = gr.Slider(
                minimum=5,
                maximum=100,
                value=30,
                step=5,
                label="Numero risultati"
            )

        player_btn = gr.Button(
            "Genera Player Forecast",
            variant="primary"
        )

        player_table = gr.Dataframe(
            label="Previsioni giocatori",
            interactive=False
        )

        player_plot = gr.Plot(
            label="Grafico previsioni"
        )

        player_btn.click(
            fn=player_dashboard,
            inputs=[
                squadra_input,
                giocatore_input,
                statistica_input,
                numero_input
            ],
            outputs=[
                player_table,
                player_plot
            ]
        )

    # --------------------------------------------------------
    # TAB 2 — PLAYER COMPARISON
    # --------------------------------------------------------

    with gr.Tab("Player Comparison"):

        with gr.Row():

            p1 = gr.Dropdown(
                choices=giocatori,
                value=giocatori[0] if giocatori else None,
                label="Giocatore 1"
            )

            p2 = gr.Dropdown(
                choices=giocatori,
                value=giocatori[1] if len(giocatori) > 1 else None,
                label="Giocatore 2"
            )

        compare_btn = gr.Button(
            "Confronta giocatori",
            variant="primary"
        )

        compare_table = gr.Dataframe(
            interactive=False
        )

        compare_plot = gr.Plot()

        compare_btn.click(
            fn=confronta_giocatori,
            inputs=[p1, p2],
            outputs=[compare_table, compare_plot]
        )

    # --------------------------------------------------------
    # TAB 3 — GAME PREDICTIONS
    # --------------------------------------------------------

    with gr.Tab("Game Predictions"):

        gr.Markdown("""
        ### Storico previsioni squadre

        Visualizza le previsioni out-of-sample
        generate nello STEP 5.
        """)

        game_btn = gr.Button(
            "Mostra Game Predictions"
        )

        game_table = gr.Dataframe(
            interactive=False
        )

        game_info = gr.Textbox(
            label="Informazioni"
        )

        game_btn.click(
            fn=game_dashboard,
            outputs=[
                game_table,
                game_info
            ]
        )

    # --------------------------------------------------------
    # TAB 4 — MODEL ACCURACY
    # --------------------------------------------------------

    with gr.Tab("Model Accuracy"):

        accuracy_btn = gr.Button(
            "Mostra accuratezza modelli"
        )

        accuracy_table = gr.Dataframe(
            interactive=False
        )

        accuracy_plot = gr.Plot()

        accuracy_btn.click(
            fn=accuracy_dashboard,
            outputs=[
                accuracy_table,
                accuracy_plot
            ]
        )

    gr.Markdown("""
    ---
    **NBA Prediction Center v1.1**

    Le previsioni sono stime statistiche.
    La disponibilità effettiva dei giocatori
    non è verificata automaticamente.
    """)

# ============================================================
# 9. AVVIO CORRETTO SU GOOGLE COLAB
# ============================================================

print("\n====================================")
print("NBA PREDICTION CENTER v1.1")
print("====================================")
print("Avvio dashboard Gradio...")

# Chiude eventuali istanze precedenti
try:
    app.close()
except Exception:
    pass

# Genera un collegamento esterno temporaneo.
# ATTENZIONE: il link è accessibile a chiunque lo possieda.

app.launch(
    share=True,
    inline=False,
    debug=True,
    show_error=True
)


In [ ]:
import gradio as gr
import pandas as pd

print("Gradio:", gr.__version__)
print("Pandas:", pd.__version__)

print("Librerie caricate correttamente!")

In [ ]:
# ==================================================
# RIAVVIO NBA PREDICTION CENTER
# ==================================================

import gradio as gr

# Chiude eventuali istanze precedenti
try:
    app.close()
except Exception:
    pass

print("Riavvio NBA Prediction Center...")

app.launch(
    share=True,
    inline=False,
    debug=False,
    show_error=True,
    prevent_thread_lock=False
)

In [ ]:
%pip install -q gradio plotly openpyxl


In [ ]:

# ============================================================
# STEP 11 — NBA PREDICTION CENTER
# DASHBOARD GRADIO COMPLETA — VERSIONE CORRETTA
# ============================================================

import os
import pandas as pd
import numpy as np
import plotly.express as px
import gradio as gr

from google.colab import drive

# ============================================================
# 1. GOOGLE DRIVE
# ============================================================

CARTELLA = "/content/drive/MyDrive/NBA_Data_Science"

if not os.path.isdir(CARTELLA):
    drive.mount("/content/drive")

PLAYER_FILE = os.path.join(
    CARTELLA,
    "NBA_Forecast_Engine_V2.csv"
)

GAME_FILE = os.path.join(
    CARTELLA,
    "nba_test_predictions.csv"
)

ACCURACY_FILE = os.path.join(
    CARTELLA,
    "NBA_Model_Accuracy_Report.xlsx"
)

if not os.path.isfile(PLAYER_FILE):
    raise FileNotFoundError(
        "Manca NBA_Forecast_Engine_V2.csv. "
        "Controlla Google Drive oppure esegui STEP 10."
    )

# ============================================================
# 2. CARICAMENTO DATI
# ============================================================

players = pd.read_csv(PLAYER_FILE)

required_cols = [
    "Giocatore",
    "Squadra",
    "Avversario",
    "Sede",
    "Forecast_PTS",
    "Forecast_REB",
    "Forecast_AST",
    "Forecast_3PM",
    "Forecast_PRA"
]

missing = [
    c for c in required_cols
    if c not in players.columns
]

if missing:
    raise ValueError(
        f"Colonne mancanti nel forecast: {missing}"
    )

for col in [
    "Forecast_PTS",
    "Forecast_REB",
    "Forecast_AST",
    "Forecast_3PM",
    "Forecast_PRA"
]:
    players[col] = pd.to_numeric(
        players[col],
        errors="coerce"
    )

if "Data Italia" not in players.columns:
    players["Data Italia"] = ""

if os.path.isfile(GAME_FILE):
    games_predictions = pd.read_csv(GAME_FILE)
else:
    games_predictions = pd.DataFrame()

if os.path.isfile(ACCURACY_FILE):
    try:
        accuracy = pd.read_excel(
            ACCURACY_FILE,
            sheet_name="Test"
        )
    except Exception as e:
        print("Avviso accuracy:", e)
        accuracy = pd.DataFrame()
else:
    accuracy = pd.DataFrame()

print("====================================")
print("NBA PREDICTION CENTER")
print("====================================")
print("Player forecast:", len(players))
print("Game predictions storiche:", len(games_predictions))
print("Metriche accuracy:", len(accuracy))

# ============================================================
# 3. OPZIONI
# ============================================================

STATISTICHE = {
    "Punti": "Forecast_PTS",
    "Rimbalzi": "Forecast_REB",
    "Assist": "Forecast_AST",
    "Triple": "Forecast_3PM",
    "PRA": "Forecast_PRA"
}

squadre = sorted(
    players["Squadra"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

giocatori = sorted(
    players["Giocatore"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

# ============================================================
# 4. PLAYER FORECAST
# ============================================================

def player_dashboard(
    squadra,
    giocatore,
    statistica,
    numero
):

    dati = players.copy()

    if squadra != "Tutte":
        dati = dati[
            dati["Squadra"] == squadra
        ]

    if giocatore != "Tutti":
        dati = dati[
            dati["Giocatore"] == giocatore
        ]

    col = STATISTICHE[statistica]

    dati = dati.sort_values(
        col,
        ascending=False
    ).head(int(numero))

    colonne = [
        "Data Italia",
        "Giocatore",
        "Squadra",
        "Avversario",
        "Sede",
        "Forecast_PTS",
        "Forecast_REB",
        "Forecast_AST",
        "Forecast_3PM",
        "Forecast_PRA"
    ]

    tabella = dati[colonne].round(2)

    if dati.empty:
        return tabella, None

    fig = px.bar(
        dati,
        x="Giocatore",
        y=col,
        color="Squadra",
        hover_data=[
            "Avversario",
            "Sede",
            "Data Italia"
        ],
        title=f"NBA Player Forecast — {statistica}",
        template="plotly_dark"
    )

    fig.update_layout(
        height=550,
        xaxis_tickangle=-45
    )

    return tabella, fig

# ============================================================
# 5. CONFRONTO GIOCATORI
# ============================================================

def confronta_giocatori(p1, p2):

    if not p1 or not p2:
        return pd.DataFrame(), None

    dati = players[
        players["Giocatore"].isin([p1, p2])
    ].copy()

    if dati.empty:
        return pd.DataFrame(), None

    dati["_data"] = pd.to_datetime(
        dati["Data Italia"],
        dayfirst=True,
        errors="coerce"
    )

    dati = dati.sort_values("_data")

    dati = dati.drop_duplicates(
        "Giocatore",
        keep="first"
    )

    colonne = [
        "Giocatore",
        "Forecast_PTS",
        "Forecast_REB",
        "Forecast_AST",
        "Forecast_3PM",
        "Forecast_PRA"
    ]

    tabella = dati[colonne].round(2)

    grafico = tabella.melt(
        id_vars="Giocatore",
        var_name="Statistica",
        value_name="Forecast"
    )

    fig = px.bar(
        grafico,
        x="Statistica",
        y="Forecast",
        color="Giocatore",
        barmode="group",
        title="NBA — Confronto giocatori",
        template="plotly_dark"
    )

    fig.update_layout(height=500)

    return tabella, fig

# ============================================================
# 6. GAME PREDICTIONS STORICHE
# ============================================================

def game_dashboard():

    if games_predictions.empty:
        return (
            pd.DataFrame(),
            "File nba_test_predictions.csv non disponibile."
        )

    dati = games_predictions.tail(100).copy()

    return (
        dati,
        "Questi sono i risultati storici del test "
        "dello STEP 5. Non sono previsioni future."
    )

# ============================================================
# 7. MODEL ACCURACY
# ============================================================

def accuracy_dashboard():

    if accuracy.empty:
        return pd.DataFrame(), None

    dati = accuracy.copy()

    if "Target" not in dati.columns:
        return dati, None

    if "MAE" not in dati.columns:
        return dati, None

    fig = px.bar(
        dati,
        x="Target",
        y="MAE",
        color=(
            "Modello"
            if "Modello" in dati.columns
            else None
        ),
        title="NBA Player Forecast — Test MAE",
        template="plotly_dark"
    )

    fig.update_layout(height=500)

    return dati, fig

# ============================================================
# 8. CREAZIONE DASHBOARD GRADIO
# ============================================================

with gr.Blocks(
    title="NBA Prediction Center",
    theme=gr.themes.Soft()
) as app:

    gr.Markdown("""
    # 🏀 NBA PREDICTION CENTER

    **NBA Analytics · Machine Learning · Forecasting**

    Dashboard basata sui dati NBA e sui modelli
    di Machine Learning sviluppati in Google Colab.
    """)

    # --------------------------------------------------------
    # TAB 1 — PLAYER FORECAST
    # --------------------------------------------------------

    with gr.Tab("Player Forecast"):

        with gr.Row():

            squadra_input = gr.Dropdown(
                choices=["Tutte"] + squadre,
                value="Tutte",
                label="Squadra"
            )

            giocatore_input = gr.Dropdown(
                choices=["Tutti"] + giocatori,
                value="Tutti",
                label="Giocatore"
            )

        with gr.Row():

            statistica_input = gr.Dropdown(
                choices=list(STATISTICHE.keys()),
                value="Punti",
                label="Statistica"
            )

            numero_input = gr.Slider(
                minimum=5,
                maximum=100,
                value=30,
                step=5,
                label="Numero risultati"
            )

        player_btn = gr.Button(
            "Genera Player Forecast",
            variant="primary"
        )

        player_table = gr.Dataframe(
            label="Previsioni giocatori",
            interactive=False
        )

        player_plot = gr.Plot(
            label="Grafico previsioni"
        )

        player_btn.click(
            fn=player_dashboard,
            inputs=[
                squadra_input,
                giocatore_input,
                statistica_input,
                numero_input
            ],
            outputs=[
                player_table,
                player_plot
            ]
        )

    # --------------------------------------------------------
    # TAB 2 — PLAYER COMPARISON
    # --------------------------------------------------------

    with gr.Tab("Player Comparison"):

        with gr.Row():

            p1 = gr.Dropdown(
                choices=giocatori,
                value=(
                    giocatori[0]
                    if giocatori else None
                ),
                label="Giocatore 1"
            )

            p2 = gr.Dropdown(
                choices=giocatori,
                value=(
                    giocatori[1]
                    if len(giocatori) > 1
                    else giocatori[0]
                    if giocatori else None
                ),
                label="Giocatore 2"
            )

        compare_btn = gr.Button(
            "Confronta giocatori",
            variant="primary"
        )

        compare_table = gr.Dataframe(
            label="Confronto statistiche",
            interactive=False
        )

        compare_plot = gr.Plot(
            label="Grafico confronto"
        )

        compare_btn.click(
            fn=confronta_giocatori,
            inputs=[p1, p2],
            outputs=[
                compare_table,
                compare_plot
            ]
        )

    # --------------------------------------------------------
    # TAB 3 — GAME PREDICTIONS
    # --------------------------------------------------------

    with gr.Tab("Game Predictions"):

        gr.Markdown("""
        ### Previsioni storiche delle squadre

        Risultati out-of-sample del modello
        sviluppato nello STEP 5.
        """)

        game_btn = gr.Button(
            "Mostra Game Predictions",
            variant="primary"
        )

        game_table = gr.Dataframe(
            label="Previsioni partite",
            interactive=False
        )

        game_info = gr.Textbox(
            label="Informazioni",
            interactive=False
        )

        game_btn.click(
            fn=game_dashboard,
            outputs=[
                game_table,
                game_info
            ]
        )

    # --------------------------------------------------------
    # TAB 4 — MODEL ACCURACY
    # --------------------------------------------------------

    with gr.Tab("Model Accuracy"):

        gr.Markdown("""
        ### Accuratezza dei modelli NBA

        MAE e RMSE calcolati sulle partite
        del test storico dello STEP 9.
        """)

        accuracy_btn = gr.Button(
            "Mostra accuratezza modelli",
            variant="primary"
        )

        accuracy_table = gr.Dataframe(
            label="Metriche modelli",
            interactive=False
        )

        accuracy_plot = gr.Plot(
            label="Grafico accuratezza"
        )

        accuracy_btn.click(
            fn=accuracy_dashboard,
            outputs=[
                accuracy_table,
                accuracy_plot
            ]
        )

    gr.Markdown("""
    ---
    **NBA Prediction Center v1.2**

    Le previsioni sono stime statistiche.
    Non vengono verificate automaticamente
    le disponibilità effettive dei giocatori.
    """)

# ============================================================
# 9. AVVIO DASHBOARD
# ============================================================

print("\n====================================")
print("NBA PREDICTION CENTER v1.2")
print("====================================")
print("Dashboard creata correttamente.")
print("Avvio server Gradio...")

# Se il notebook viene rieseguito, prova a
# chiudere un'eventuale vecchia interfaccia.
# Non chiudiamo la nuova app appena creata.

app.launch(
    share=True,
    inline=False,
    debug=False,
    show_error=True
)


In [ ]:
%pip install -q requests pandas tqdm

In [ ]:

# ============================================================
# NBA DATA SCIENCE ANALYTICS
# STEP 12 — AUTOMATED DATA PIPELINE
# ESPN -> SQLITE -> PLAYER BOXSCORES
# ============================================================

import os
import time
import sqlite3
import requests
import pandas as pd

from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
from tqdm.auto import tqdm
from google.colab import drive

# ============================================================
# 1. CONFIGURAZIONE
# ============================================================

DB_PATH = (
    "/content/drive/MyDrive/"
    "NBA_Data_Science/nba_database.db"
)

GIORNI_PASSATI = 7
GIORNI_FUTURI = 3

PAUSA_RICHIESTE = 0.25

BASE = (
    "https://site.api.espn.com/apis/"
    "site/v2/sports/basketball/nba"
)

if not os.path.isfile(DB_PATH):
    drive.mount("/content/drive")

if not os.path.isfile(DB_PATH):
    raise FileNotFoundError(
        "Database NBA non trovato su Google Drive."
    )

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json"
})

conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA busy_timeout = 10000")

# ============================================================
# 2. CONTROLLO SCHEMA SQLITE
# ============================================================

def colonne_tabella(nome):
    return {
        row[1]
        for row in conn.execute(
            f"PRAGMA table_info({nome})"
        ).fetchall()
    }

def aggiungi_colonna(tabella, nome, tipo):
    if nome not in colonne_tabella(tabella):
        conn.execute(
            f"ALTER TABLE {tabella} "
            f"ADD COLUMN {nome} {tipo}"
        )

conn.execute("""
CREATE TABLE IF NOT EXISTS espn_games (
    game_id TEXT PRIMARY KEY,
    season TEXT,
    game_date TEXT,
    home_team TEXT,
    away_team TEXT,
    home_score REAL,
    away_score REAL,
    completed INTEGER,
    status TEXT
)
""")

for nome, tipo in {
    "home_id": "TEXT",
    "away_id": "TEXT",
    "season": "TEXT",
    "game_date": "TEXT",
    "home_team": "TEXT",
    "away_team": "TEXT",
    "home_score": "REAL",
    "away_score": "REAL",
    "completed": "INTEGER",
    "status": "TEXT"
}.items():
    aggiungi_colonna("espn_games", nome, tipo)

conn.execute("""
CREATE TABLE IF NOT EXISTS espn_player_boxscores (
    game_id TEXT NOT NULL,
    season TEXT,
    game_date TEXT,
    team_id TEXT NOT NULL,
    team_name TEXT,
    player_id TEXT NOT NULL,
    player_name TEXT,
    starter INTEGER,
    did_not_play INTEGER,
    minutes REAL,
    pts REAL,
    reb REAL,
    ast REAL,
    stl REAL,
    blk REAL,
    tov REAL,
    oreb REAL,
    dreb REAL,
    pf REAL,
    fgm REAL,
    fga REAL,
    fg3m REAL,
    fg3a REAL,
    ftm REAL,
    fta REAL,
    plus_minus REAL,
    PRIMARY KEY (game_id, team_id, player_id)
)
""")

conn.execute("""
CREATE TABLE IF NOT EXISTS espn_boxscore_log (
    game_id TEXT PRIMARY KEY,
    downloaded_at TEXT,
    players_found INTEGER
)
""")

conn.commit()

print("Schema database verificato.")

# ============================================================
# 3. FUNZIONI DI SUPPORTO
# ============================================================

def fetch_json(url, params=None, retries=3):

    for tentativo in range(retries):
        try:
            r = session.get(
                url,
                params=params,
                timeout=30
            )
            r.raise_for_status()
            return r.json()

        except requests.RequestException as e:
            print(
                f"Richiesta fallita "
                f"({tentativo + 1}/{retries}): {e}"
            )
            time.sleep(2 * (tentativo + 1))

    return None


def numero(valore):

    try:
        if valore is None:
            return None

        s = str(valore).strip()

        if s in ["", "--", "DNP", "None"]:
            return None

        return float(s.replace("+", ""))

    except (ValueError, TypeError):
        return None


def minuti(valore):

    if valore is None:
        return None

    s = str(valore).strip()

    if ":" in s:
        try:
            mm, ss = s.split(":", 1)
            return float(mm) + float(ss) / 60
        except ValueError:
            return None

    return numero(s)


def frazione(valore):

    if valore is None:
        return None, None

    s = str(valore)

    if "-" not in s:
        return None, None

    try:
        a, b = s.split("-", 1)
        return float(a), float(b)
    except ValueError:
        return None, None


def stagione_nba(data_iso):

    data = pd.to_datetime(
        data_iso,
        utc=True
    ).tz_convert("America/New_York")

    anno = data.year

    if data.month < 7:
        anno -= 1

    return f"{anno}-{str(anno + 1)[-2:]}"


# ============================================================
# 4. AGGIORNAMENTO CALENDARIO
# ============================================================

oggi = datetime.now(
    ZoneInfo("America/New_York")
).date()

giorni = [
    oggi + timedelta(days=i)
    for i in range(
        -GIORNI_PASSATI,
        GIORNI_FUTURI + 1
    )
]

eventi = {}

print("\nAggiornamento calendario ESPN...")

for giorno in tqdm(giorni):

    dati = fetch_json(
        BASE + "/scoreboard",
        params={
            "dates": giorno.strftime("%Y%m%d"),
            "limit": 100
        }
    )

    if dati is None:
        continue

    for evento in dati.get("events", []):
        eventi[str(evento["id"])] = evento

    time.sleep(PAUSA_RICHIESTE)

print("Eventi ESPN individuati:", len(eventi))

# ============================================================
# 5. SALVATAGGIO PARTITE
# ============================================================

aggiornate = 0
completate = []

for evento in eventi.values():

    competizioni = evento.get(
        "competitions", []
    )

    if not competizioni:
        continue

    competitors = competizioni[0].get(
        "competitors", []
    )

    home = next(
        (
            x for x in competitors
            if x.get("homeAway") == "home"
        ),
        None
    )

    away = next(
        (
            x for x in competitors
            if x.get("homeAway") == "away"
        ),
        None
    )

    if home is None or away is None:
        continue

    game_id = str(evento["id"])
    data = evento.get("date")

    if not data:
        continue

    status_obj = evento.get(
        "status", {}
    ).get("type", {})

    status = status_obj.get(
        "name", "UNKNOWN"
    )

    completed = int(
        status_obj.get("completed", False)
    )

    home_score = numero(
        home.get("score")
    )

    away_score = numero(
        away.get("score")
    )

    season = stagione_nba(data)

    # Non sovrascrive una gara conclusa con
    # una versione successiva incompleta.
    precedente = conn.execute(
        """
        SELECT completed
        FROM espn_games
        WHERE game_id = ?
        """,
        (game_id,)
    ).fetchone()

    if precedente and precedente[0] == 1:
        if not completed:
            continue

    conn.execute("""
        INSERT INTO espn_games (
            game_id, season, game_date,
            home_team, away_team,
            home_id, away_id,
            home_score, away_score,
            completed, status
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)

        ON CONFLICT(game_id) DO UPDATE SET
            season = excluded.season,
            game_date = excluded.game_date,
            home_team = excluded.home_team,
            away_team = excluded.away_team,
            home_id = excluded.home_id,
            away_id = excluded.away_id,
            home_score = excluded.home_score,
            away_score = excluded.away_score,
            completed = excluded.completed,
            status = excluded.status
    """, (
        game_id,
        season,
        data,
        home["team"].get("displayName"),
        away["team"].get("displayName"),
        str(home["team"]["id"]),
        str(away["team"]["id"]),
        home_score,
        away_score,
        completed,
        status
    ))

    aggiornate += 1

    if completed:
        completate.append(game_id)

conn.commit()

print("Partite inserite/aggiornate:", aggiornate)
print("Partite concluse nel periodo:", len(completate))

# ============================================================
# 6. DOWNLOAD INCREMENTALE BOX SCORE
# ============================================================

gia_scaricate = {
    row[0]
    for row in conn.execute("""
        SELECT game_id
        FROM espn_boxscore_log
        WHERE players_found > 0
    """)
}

da_scaricare = [
    gid for gid in completate
    if gid not in gia_scaricate
]

print("\nNuovi box score da scaricare:", len(da_scaricare))

STAT_MAP = {
    "MIN": "minutes",
    "PTS": "pts",
    "REB": "reb",
    "AST": "ast",
    "STL": "stl",
    "BLK": "blk",
    "TO": "tov",
    "TOV": "tov",
    "OREB": "oreb",
    "DREB": "dreb",
    "PF": "pf",
    "+/-": "plus_minus",
    "FG": "fg",
    "3PT": "fg3",
    "FT": "ft"
}

BOX_COLS = [
    "game_id", "season", "game_date",
    "team_id", "team_name",
    "player_id", "player_name",
    "starter", "did_not_play",
    "minutes", "pts", "reb", "ast",
    "stl", "blk", "tov", "oreb",
    "dreb", "pf", "fgm", "fga",
    "fg3m", "fg3a", "ftm", "fta",
    "plus_minus"
]


def estrai_boxscore(game_id):

    dati = fetch_json(
        BASE + "/summary",
        params={"event": game_id}
    )

    if not dati:
        return []

    info = conn.execute("""
        SELECT season, game_date
        FROM espn_games
        WHERE game_id = ?
    """, (game_id,)).fetchone()

    if not info:
        return []

    season, game_date = info

    righe = []

    for squadra in dati.get(
        "boxscore", {}
    ).get("players", []):

        team = squadra.get("team", {})

        team_id = str(team.get("id", ""))
        team_name = team.get(
            "displayName", ""
        )

        for gruppo in squadra.get(
            "statistics", []
        ):

            labels = gruppo.get(
                "names", []
            )

            if not labels:
                labels = gruppo.get(
                    "labels", []
                )

            for atleta in gruppo.get(
                "athletes", []
            ):

                player = atleta.get(
                    "athlete", {}
                )

                player_id = str(
                    player.get("id", "")
                )

                if not player_id or not team_id:
                    continue

                valori = atleta.get(
                    "stats", []
                )

                stats = {
                    STAT_MAP.get(
                        str(label).upper(), label
                    ): valore
                    for label, valore
                    in zip(labels, valori)
                }

                fgm, fga = frazione(
                    stats.get("fg")
                )

                fg3m, fg3a = frazione(
                    stats.get("fg3")
                )

                ftm, fta = frazione(
                    stats.get("ft")
                )

                riga = {
                    "game_id": game_id,
                    "season": season,
                    "game_date": game_date,
                    "team_id": team_id,
                    "team_name": team_name,
                    "player_id": player_id,
                    "player_name": player.get(
                        "displayName", ""
                    ),
                    "starter": int(
                        bool(atleta.get("starter"))
                    ),
                    "did_not_play": int(
                        bool(
                            atleta.get(
                                "didNotPlay", False
                            )
                        )
                    ),
                    "minutes": minuti(
                        stats.get("minutes")
                    ),
                    "pts": numero(stats.get("pts")),
                    "reb": numero(stats.get("reb")),
                    "ast": numero(stats.get("ast")),
                    "stl": numero(stats.get("stl")),
                    "blk": numero(stats.get("blk")),
                    "tov": numero(stats.get("tov")),
                    "oreb": numero(stats.get("oreb")),
                    "dreb": numero(stats.get("dreb")),
                    "pf": numero(stats.get("pf")),
                    "fgm": fgm,
                    "fga": fga,
                    "fg3m": fg3m,
                    "fg3a": fg3a,
                    "ftm": ftm,
                    "fta": fta,
                    "plus_minus": numero(
                        stats.get("plus_minus")
                    )
                }

                righe.append(riga)

    return righe


scaricate = 0
fallite = []

for game_id in tqdm(da_scaricare):

    try:
        righe = estrai_boxscore(game_id)

        # Evitiamo di registrare come riuscito
        # un download senza statistiche valide.
        valide = [
            r for r in righe
            if r["minutes"] is not None
            and r["pts"] is not None
        ]

        if not valide:
            fallite.append(game_id)
            continue

        placeholders = ",".join(
            ["?"] * len(BOX_COLS)
        )

        sql = (
            "INSERT OR REPLACE INTO "
            "espn_player_boxscores ("
            + ",".join(BOX_COLS)
            + ") VALUES ("
            + placeholders
            + ")"
        )

        with conn:

            conn.executemany(
                sql,
                [
                    tuple(r[c] for c in BOX_COLS)
                    for r in righe
                ]
            )

            conn.execute("""
                INSERT OR REPLACE
                INTO espn_boxscore_log (
                    game_id,
                    downloaded_at,
                    players_found
                )
                VALUES (?, ?, ?)
            """, (
                game_id,
                datetime.now(
                    ZoneInfo("UTC")
                ).isoformat(),
                len(righe)
            ))

        scaricate += 1

    except Exception as errore:
        fallite.append(game_id)
        print(
            f"Errore partita {game_id}: {errore}"
        )

    time.sleep(PAUSA_RICHIESTE)

# ============================================================
# 7. CONTROLLO QUALITA'
# ============================================================

tot_games = conn.execute("""
    SELECT COUNT(*) FROM espn_games
""").fetchone()[0]

tot_completed = conn.execute("""
    SELECT COUNT(*)
    FROM espn_games
    WHERE completed = 1
""").fetchone()[0]

tot_box = conn.execute("""
    SELECT COUNT(*)
    FROM espn_player_boxscores
""").fetchone()[0]

tot_players = conn.execute("""
    SELECT COUNT(DISTINCT player_id)
    FROM espn_player_boxscores
""").fetchone()[0]

tot_logged = conn.execute("""
    SELECT COUNT(*)
    FROM espn_boxscore_log
    WHERE players_found > 0
""").fetchone()[0]

# ============================================================
# 8. SALVATAGGIO REPORT
# ============================================================

REPORT_PATH = (
    "/content/drive/MyDrive/"
    "NBA_Data_Science/"
    "NBA_Pipeline_Status.csv"
)

report = pd.DataFrame([{
    "Aggiornamento UTC": datetime.now(
        ZoneInfo("UTC")
    ).isoformat(),
    "Partite database": tot_games,
    "Partite concluse": tot_completed,
    "Player box score": tot_box,
    "Giocatori unici": tot_players,
    "Partite con box score": tot_logged,
    "Box score nuovi": scaricate,
    "Download falliti": len(fallite)
}])

report.to_csv(
    REPORT_PATH,
    index=False
)

conn.close()
session.close()

print("\n======================================")
print("STEP 12 COMPLETATO")
print("======================================")

print("Partite nel database:", tot_games)
print("Partite concluse:", tot_completed)
print("Player box score:", tot_box)
print("Giocatori unici:", tot_players)
print("Partite con box score:", tot_logged)
print("Nuovi box score:", scaricate)
print("Download falliti:", len(fallite))

if fallite:
    print("Partite da riprovare:", fallite)

print("\nDatabase:", DB_PATH)
print("Report:", REPORT_PATH)


In [ ]:

# ============================================================
# STEP 13 — NBA AUTOMATIC FORECAST REFRESH
# ============================================================

import os
import json
import sqlite3
import requests
import joblib
import numpy as np
import pandas as pd

from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
from google.colab import drive
from IPython.display import display

# ============================================================
# 1. CONFIGURAZIONE
# ============================================================

BASE_DIR = "/content/drive/MyDrive/NBA_Data_Science"

if not os.path.isdir(BASE_DIR):
    drive.mount("/content/drive")

DB_PATH = os.path.join(
    BASE_DIR, "nba_database.db"
)

MODEL_PATH = os.path.join(
    BASE_DIR, "nba_player_models_v2.joblib"
)

CSV_PATH = os.path.join(
    BASE_DIR, "NBA_Forecast_Engine_V2.csv"
)

EXCEL_PATH = os.path.join(
    BASE_DIR, "NBA_Forecast_Engine_V2.xlsx"
)

REPORT_PATH = os.path.join(
    BASE_DIR, "NBA_Forecast_Refresh_Log.csv"
)

GIORNI_FUTURI = 3
MIN_STORICO = 10
MAX_GIORNI_SQUADRA = 90

for path in [DB_PATH, MODEL_PATH]:
    if not os.path.isfile(path):
        raise FileNotFoundError(
            f"File necessario non trovato: {path}"
        )

# ============================================================
# 2. CARICAMENTO MODELLI
# ============================================================

# Caricare joblib solo da file attendibili.

package = joblib.load(MODEL_PATH)

models = package["models"]
feature_cols = package["features"]
targets = package["targets"]

MIN_STORICO = package.get(
    "minimum_history", MIN_STORICO
)

print("Modello:", package.get("model_version"))
print("Target:", list(models.keys()))

# ============================================================
# 3. CARICAMENTO DATABASE
# ============================================================

conn = sqlite3.connect(DB_PATH)

box = pd.read_sql_query(
    "SELECT * FROM espn_player_boxscores",
    conn
)

games = pd.read_sql_query(
    "SELECT * FROM espn_games",
    conn
)

conn.close()

required_box = [
    "game_id", "game_date",
    "player_id", "player_name", "team_id",
    "minutes", "pts", "reb", "ast", "fg3m"
]

required_games = [
    "game_id", "game_date",
    "home_id", "away_id",
    "home_score", "away_score", "completed"
]

for col in required_box:
    if col not in box.columns:
        raise ValueError(
            f"Colonna box mancante: {col}"
        )

for col in required_games:
    if col not in games.columns:
        raise ValueError(
            f"Colonna games mancante: {col}"
        )

for df in [box, games]:
    df["game_id"] = df["game_id"].astype(str)
    df["game_date"] = pd.to_datetime(
        df["game_date"],
        utc=True,
        errors="coerce"
    )

for col in ["player_id", "team_id"]:
    box[col] = box[col].astype(str)

for col in ["home_id", "away_id"]:
    games[col] = games[col].astype(str)

for col in ["minutes", "pts", "reb", "ast", "fg3m"]:
    box[col] = pd.to_numeric(
        box[col],
        errors="coerce"
    )

for col in ["home_score", "away_score", "completed"]:
    games[col] = pd.to_numeric(
        games[col],
        errors="coerce"
    )

if "did_not_play" in box.columns:
    box = box[
        pd.to_numeric(
            box["did_not_play"],
            errors="coerce"
        ).fillna(0) == 0
    ].copy()

NOW = pd.Timestamp.now(tz="UTC")

box = box[
    box["game_date"].notna()
    & (box["game_date"] < NOW)
    & (box["minutes"] > 0)
].dropna(
    subset=["pts", "reb", "ast", "fg3m"]
).drop_duplicates(
    ["game_id", "player_id"]
).copy()

box["PRA"] = (
    box["pts"] + box["reb"] + box["ast"]
)

games = games[
    (games["completed"] == 1)
    & games["game_date"].notna()
    & (games["game_date"] < NOW)
    & games["home_score"].notna()
    & games["away_score"].notna()
].drop_duplicates("game_id").copy()

print("Box score:", len(box))
print("Partite completate:", len(games))

# ============================================================
# 4. CALENDARIO ESPN
# ============================================================

BASE_URL = (
    "https://site.api.espn.com/apis/"
    "site/v2/sports/basketball/nba"
)

session = requests.Session()

session.headers.update({
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json"
})

today = datetime.now(
    ZoneInfo("America/New_York")
).date()

events = {}

for offset in range(GIORNI_FUTURI + 1):

    day = today + timedelta(days=offset)

    try:
        response = session.get(
            BASE_URL + "/scoreboard",
            params={
                "dates": day.strftime("%Y%m%d"),
                "limit": 100
            },
            timeout=30
        )

        response.raise_for_status()

        for event in response.json().get(
            "events", []
        ):
            events[str(event["id"])] = event

    except requests.RequestException as e:
        print("Errore calendario:", day, e)

session.close()

future_rows = []

for event in events.values():

    status = event.get(
        "status", {}
    ).get("type", {})

    if status.get("name") != "STATUS_SCHEDULED":
        continue

    competitions = event.get(
        "competitions", []
    )

    if not competitions:
        continue

    competitors = competitions[0].get(
        "competitors", []
    )

    home = next(
        (
            x for x in competitors
            if x.get("homeAway") == "home"
        ),
        None
    )

    away = next(
        (
            x for x in competitors
            if x.get("homeAway") == "away"
        ),
        None
    )

    if not home or not away:
        continue

    date = pd.to_datetime(
        event.get("date"),
        utc=True,
        errors="coerce"
    )

    if pd.isna(date) or date <= NOW:
        continue

    future_rows.append({
        "game_id": str(event["id"]),
        "date": date,
        "home_id": str(home["team"]["id"]),
        "away_id": str(away["team"]["id"]),
        "home_team": home["team"]["displayName"],
        "away_team": away["team"]["displayName"]
    })

future = pd.DataFrame(future_rows)

if future.empty:
    raise ValueError(
        "Nessuna partita futura trovata."
    )

future = future.sort_values(
    "date"
).drop_duplicates("game_id")

print("Partite future:", len(future))

# ============================================================
# 5. DIFESA AVVERSARIA
# ============================================================

def opponent_defense(team_id, date):

    storico = games[
        (games["game_date"] < date)
        & (
            (games["home_id"] == team_id)
            | (games["away_id"] == team_id)
        )
    ].sort_values(
        ["game_date", "game_id"]
    )

    if len(storico) < 5:
        return np.nan

    last10 = storico.tail(10)

    allowed = np.where(
        last10["home_id"] == team_id,
        last10["away_score"],
        last10["home_score"]
    )

    return float(np.mean(allowed))

# ============================================================
# 6. GIOCATORI ASSOCIATI ALLE SQUADRE
# ============================================================

box = box.sort_values(
    ["game_date", "game_id"]
)

latest_team = box.drop_duplicates(
    "player_id",
    keep="last"
)[
    ["player_id", "player_name", "team_id", "game_date"]
].copy()

# Esclude associazioni molto vecchie.
latest_team = latest_team[
    latest_team["game_date"] >= (
        NOW - pd.Timedelta(
            days=MAX_GIORNI_SQUADRA
        )
    )
].copy()

# ============================================================
# 7. FEATURE ENGINEERING
# ============================================================

def build_features(history, date, home, allowed):

    history = history[
        history["game_date"] < date
    ].sort_values(
        ["game_date", "game_id"]
    )

    if len(history) < MIN_STORICO:
        return None

    last5 = history.tail(5)
    last10 = history.tail(10)

    date_ny = date.tz_convert(
        "America/New_York"
    ).date()

    last_ny = (
        history.iloc[-1]["game_date"]
        .tz_convert("America/New_York")
        .date()
    )

    rest = max(
        (date_ny - last_ny).days - 1,
        0
    )

    row = {
        "home": float(home),
        "rest_days": float(np.clip(rest, 0, 7)),
        "GP_precedenti": len(history),
        "MIN_avg5": float(
            last5["minutes"].mean()
        ),
        "MIN_avg10": float(
            last10["minutes"].mean()
        ),
        "MIN_last1": float(
            history.iloc[-1]["minutes"]
        ),
        "opp_allowed_l10": allowed
    }

    for name, col in targets.items():

        row[f"{name}_last1"] = float(
            history.iloc[-1][col]
        )

        row[f"{name}_avg5"] = float(
            last5[col].mean()
        )

        row[f"{name}_avg10"] = float(
            last10[col].mean()
        )

        row[f"{name}_std10"] = float(
            last10[col].std(ddof=0)
        )

    return row

# ============================================================
# 8. GENERAZIONE FORECAST
# ============================================================

rows = []

for game in future.itertuples(index=False):

    matchups = [
        (
            game.home_id,
            game.home_team,
            game.away_id,
            game.away_team,
            1
        ),
        (
            game.away_id,
            game.away_team,
            game.home_id,
            game.home_team,
            0
        )
    ]

    for (
        team_id,
        team_name,
        opp_id,
        opp_name,
        home
    ) in matchups:

        roster = latest_team[
            latest_team["team_id"] == team_id
        ]

        allowed = opponent_defense(
            opp_id,
            game.date
        )

        for player in roster.itertuples(index=False):

            history = box[
                box["player_id"] == player.player_id
            ]

            features = build_features(
                history,
                game.date,
                home,
                allowed
            )

            if features is None:
                continue

            rows.append({
                "game_id": game.game_id,
                "date": game.date,
                "player_id": player.player_id,
                "Giocatore": player.player_name,
                "Squadra": team_name,
                "Avversario": opp_name,
                "Sede": (
                    "Casa" if home else "Trasferta"
                ),
                **features
            })

forecast = pd.DataFrame(rows)

if forecast.empty:
    raise ValueError(
        "Nessun forecast generato. "
        "Controlla lo storico dei giocatori."
    )

missing_features = [
    col for col in feature_cols
    if col not in forecast.columns
]

if missing_features:
    raise ValueError(
        f"Feature mancanti: {missing_features}"
    )

X = forecast[feature_cols]

for target, info in models.items():

    if info["name"] == "Last 10 Baseline":

        prediction = forecast[
            f"{target}_avg10"
        ].to_numpy()

    else:

        prediction = info["model"].predict(X)

    forecast[f"Forecast_{target}"] = np.maximum(
        np.asarray(prediction, dtype=float),
        0
    )

# PRA coerente con punti + rimbalzi + assist.
forecast["Forecast_PRA"] = (
    forecast["Forecast_PTS"]
    + forecast["Forecast_REB"]
    + forecast["Forecast_AST"]
)

forecast["Data Italia"] = (
    forecast["date"]
    .dt.tz_convert("Europe/Rome")
    .dt.strftime("%d/%m/%Y %H:%M")
)

# ============================================================
# 9. ESPORTAZIONE
# ============================================================

export_cols = [
    "Data Italia",
    "Giocatore",
    "Squadra",
    "Avversario",
    "Sede",
    "Forecast_PTS",
    "Forecast_REB",
    "Forecast_AST",
    "Forecast_3PM",
    "Forecast_PRA",
    "MIN_avg10",
    "rest_days",
    "opp_allowed_l10"
]

export = forecast[export_cols].copy()

# Scrittura temporanea prima della sostituzione.
TEMP_CSV = CSV_PATH + ".tmp"
TEMP_XLSX = EXCEL_PATH + ".tmp.xlsx"

export.to_csv(
    TEMP_CSV,
    index=False,
    encoding="utf-8-sig"
)

with pd.ExcelWriter(
    TEMP_XLSX,
    engine="openpyxl"
) as writer:

    export.to_excel(
        writer,
        sheet_name="Player Forecast",
        index=False
    )

    accuracy = pd.DataFrame(
        package.get("test", [])
    )

    if not accuracy.empty:
        accuracy.to_excel(
            writer,
            sheet_name="Model Accuracy",
            index=False
        )

os.replace(TEMP_CSV, CSV_PATH)
os.replace(TEMP_XLSX, EXCEL_PATH)

# ============================================================
# 10. LOG E RISULTATI
# ============================================================

timestamp = datetime.now(
    ZoneInfo("Europe/Rome")
).isoformat()

log_row = pd.DataFrame([{
    "Timestamp": timestamp,
    "Partite": len(future),
    "Forecast": len(forecast),
    "Model Version": package.get(
        "model_version", "unknown"
    ),
    "Status": "SUCCESS"
}])

if os.path.isfile(REPORT_PATH):
    old_log = pd.read_csv(REPORT_PATH)
    log_row = pd.concat(
        [old_log, log_row],
        ignore_index=True
    )

log_row.to_csv(
    REPORT_PATH,
    index=False
)

print("\n====================================")
print("STEP 13 COMPLETATO!")
print("====================================")

print("Partite future:", len(future))
print("Forecast generati:", len(forecast))
print("CSV:", CSV_PATH)
print("Excel:", EXCEL_PATH)
print("Log:", REPORT_PATH)

display(
    export.head(25).round(2)
)


In [ ]:

# ============================================================
# STEP 13.1 — FORECAST HISTORY PREPARATION
# Preparazione dati per STEP 14
# ============================================================

import os
import pandas as pd
import numpy as np

from IPython.display import display

# ============================================================
# 1. CONFIGURAZIONE
# ============================================================

BASE_DIR = "/content/drive/MyDrive/NBA_Data_Science"

CSV_PATH = os.path.join(
    BASE_DIR,
    "NBA_Forecast_Engine_V2.csv"
)

EXCEL_PATH = os.path.join(
    BASE_DIR,
    "NBA_Forecast_Engine_V2.xlsx"
)

ARCHIVE_DIR = os.path.join(
    BASE_DIR,
    "Forecast_Archive"
)

os.makedirs(ARCHIVE_DIR, exist_ok=True)

# ============================================================
# 2. CONTROLLO STEP 13
# ============================================================

if "forecast" not in globals():
    raise RuntimeError(
        "Esegui prima lo STEP 13 nello stesso runtime."
    )

if "NOW" not in globals():
    raise RuntimeError(
        "Timestamp originale mancante. "
        "Riesegui STEP 13."
    )

if not isinstance(forecast, pd.DataFrame):
    raise TypeError(
        "La variabile forecast non è un DataFrame."
    )

if forecast.empty:
    raise ValueError(
        "Nessuna previsione disponibile."
    )

data = forecast.copy()

# ============================================================
# 3. IDENTIFICATIVI E TIMESTAMP
# ============================================================

required_cols = [
    "game_id",
    "player_id",
    "date",
    "Giocatore",
    "Squadra",
    "Avversario",
    "Sede",
    "Forecast_PTS",
    "Forecast_REB",
    "Forecast_AST",
    "Forecast_3PM",
    "Forecast_PRA"
]

missing = [
    col for col in required_cols
    if col not in data.columns
]

if missing:
    raise ValueError(
        f"Colonne mancanti: {missing}"
    )

data["date"] = pd.to_datetime(
    data["date"],
    utc=True,
    errors="coerce"
)

# Utilizziamo il timestamp originale dello STEP 13.
# Non inventiamo una nuova data di generazione.

generated_at = pd.to_datetime(
    NOW,
    utc=True
)

data["generated_at"] = generated_at

data = data.dropna(
    subset=[
        "game_id",
        "player_id",
        "date"
    ]
).copy()

data["game_id"] = data["game_id"].astype(str)
data["player_id"] = data["player_id"].astype(str)

# ============================================================
# 4. CONTROLLO QUALITA'
# ============================================================

TARGETS = [
    "PTS",
    "REB",
    "AST",
    "3PM",
    "PRA"
]

for target in TARGETS:

    col = f"Forecast_{target}"

    data[col] = pd.to_numeric(
        data[col],
        errors="coerce"
    )

    data[col] = data[col].replace(
        [np.inf, -np.inf],
        np.nan
    )

before = len(data)

data = data.dropna(
    subset=[
        f"Forecast_{t}"
        for t in TARGETS
    ]
)

# Solo previsioni generate prima della partita.

data = data[
    data["generated_at"] < data["date"]
].copy()

data = data.drop_duplicates(
    subset=[
        "game_id",
        "player_id",
        "generated_at"
    ]
)

if data.empty:
    raise ValueError(
        "Nessun forecast valido da archiviare."
    )

# ============================================================
# 5. DATA ITALIANA E VERSIONE MODELLO
# ============================================================

data["Data Italia"] = (
    data["date"]
    .dt.tz_convert("Europe/Rome")
    .dt.strftime("%d/%m/%Y %H:%M")
)

if "package" in globals():

    data["Model_Version"] = str(
        package.get(
            "model_version",
            "unknown"
        )
    )

else:

    data["Model_Version"] = "unknown"

# ============================================================
# 6. COLONNE DA ESPORTARE
# ============================================================

export_cols = [
    "game_id",
    "player_id",
    "date",
    "generated_at",
    "Data Italia",
    "Giocatore",
    "Squadra",
    "Avversario",
    "Sede",
    "Forecast_PTS",
    "Forecast_REB",
    "Forecast_AST",
    "Forecast_3PM",
    "Forecast_PRA",
    "MIN_avg10",
    "rest_days",
    "opp_allowed_l10",
    "Model_Version"
]

export_cols = [
    col for col in export_cols
    if col in data.columns
]

export = data[export_cols].copy()

export = export.sort_values(
    ["date", "Forecast_PTS"],
    ascending=[True, False]
)

# Excel non supporta timestamp con timezone.
# Convertiamo le date in stringhe ISO.

for col in ["date", "generated_at"]:

    export[col] = export[col].map(
        lambda x: x.isoformat()
    )

# ============================================================
# 7. ARCHIVIAZIONE PERMANENTE
# ============================================================

timestamp = generated_at.strftime(
    "%Y%m%d_%H%M%S_%f"
)

ARCHIVE_PATH = os.path.join(
    ARCHIVE_DIR,
    f"NBA_Forecast_{timestamp}.csv"
)

if not os.path.exists(ARCHIVE_PATH):

    export.to_csv(
        ARCHIVE_PATH,
        index=False,
        encoding="utf-8-sig"
    )

# ============================================================
# 8. AGGIORNAMENTO CSV PRINCIPALE
# ============================================================

TEMP_CSV = CSV_PATH + ".tmp"

export.to_csv(
    TEMP_CSV,
    index=False,
    encoding="utf-8-sig"
)

os.replace(
    TEMP_CSV,
    CSV_PATH
)

# ============================================================
# 9. AGGIORNAMENTO EXCEL
# ============================================================

TEMP_EXCEL = EXCEL_PATH + ".tmp.xlsx"

# Conserviamo gli altri fogli Excel eventualmente presenti.

other_sheets = {}

if os.path.isfile(EXCEL_PATH):

    other_sheets = pd.read_excel(
        EXCEL_PATH,
        sheet_name=None
    )

with pd.ExcelWriter(
    TEMP_EXCEL,
    engine="openpyxl"
) as writer:

    export.to_excel(
        writer,
        sheet_name="Player Forecast",
        index=False
    )

    for sheet_name, sheet_data in other_sheets.items():

        if sheet_name != "Player Forecast":

            sheet_data.to_excel(
                writer,
                sheet_name=sheet_name,
                index=False
            )

os.replace(
    TEMP_EXCEL,
    EXCEL_PATH
)

# ============================================================
# 10. RIEPILOGO
# ============================================================

print("\n====================================")
print("STEP 13.1 COMPLETATO!")
print("====================================")

print(
    "Partite:",
    export["game_id"].nunique()
)

print(
    "Giocatori:",
    export["player_id"].nunique()
)

print(
    "Forecast validi:",
    len(export)
)

print(
    "Forecast esclusi:",
    before - len(export)
)

print(
    "Generazione:",
    generated_at.tz_convert("Europe/Rome")
)

print("\nCSV aggiornato:")
print(CSV_PATH)

print("\nExcel aggiornato:")
print(EXCEL_PATH)

print("\nArchivio permanente:")
print(ARCHIVE_PATH)

print("\nANTEPRIMA FORECAST")

display(
    export.head(20).round(2)
)

print("\nPRONTO PER STEP 14!")


In [ ]:
%pip install -q pandas numpy requests joblib scikit-learn plotly openpyxl gradio tqdm

import os
from google.colab import drive

drive.mount("/content/drive")

BASE_DIR = "/content/drive/MyDrive/NBA_Data_Science"

files = [
    "nba_database.db",
    "nba_player_models_v2.joblib",
    "NBA_Forecast_Engine_V2.csv",
    "NBA_Model_Accuracy_Report.xlsx"
]

for file in files:
    path = os.path.join(BASE_DIR, file)

    if os.path.isfile(path):
        print(f"OK - {file}")
    else:
        print(f"MANCANTE - {file}")

In [ ]:

import os
from google.colab import drive

# Collegamento sicuro a Google Drive

MOUNT = "/content/drive"

if os.path.ismount(MOUNT):
    print("Google Drive già collegato.")

elif os.path.isdir(MOUNT) and os.listdir(MOUNT):
    MOUNT = "/content/gdrive_nba"

    if not os.path.ismount(MOUNT):
        drive.mount(MOUNT)

else:
    drive.mount(MOUNT)

BASE_DIR = os.path.join(
    MOUNT,
    "MyDrive",
    "NBA_Data_Science"
)

print("\nCartella progetto:", BASE_DIR)

files = [
    "nba_database.db",
    "nba_player_models_v2.joblib",
    "NBA_Forecast_Engine_V2.csv",
    "NBA_Model_Accuracy_Report.xlsx"
]

print("\nVERIFICA FILE NBA")
print("=" * 45)

for file in files:

    path = os.path.join(BASE_DIR, file)

    if os.path.isfile(path):

        size = os.path.getsize(path) / 1024**2

        print(f"OK - {file} ({size:.2f} MB)")

    else:

        print(f"MANCANTE - {file}")

print("=" * 45)


In [ ]:

# ============================================================
# STEP 12 — NBA DATABASE UPDATER
# ============================================================

import os
import time
import sqlite3
import requests
import pandas as pd

from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
from tqdm.auto import tqdm

DB_PATH = os.path.join(BASE_DIR, "nba_database.db")

if not os.path.isfile(DB_PATH):
    raise FileNotFoundError(DB_PATH)

API = (
    "https://site.api.espn.com/apis/"
    "site/v2/sports/basketball/nba"
)

session = requests.Session()

session.headers.update({
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json"
})

conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA busy_timeout = 10000")

def fetch(endpoint, params):

    for attempt in range(3):

        try:

            response = session.get(
                API + endpoint,
                params=params,
                timeout=30
            )

            response.raise_for_status()
            return response.json()

        except (requests.RequestException, ValueError):

            time.sleep(2 * (attempt + 1))

    return None


def number(value):

    try:
        return float(value)
    except (ValueError, TypeError):
        return None


def minutes(value):

    if ":" in str(value):

        try:
            m, s = str(value).split(":")
            return float(m) + float(s) / 60
        except ValueError:
            return None

    return number(value)


def split_stat(value):

    try:
        a, b = str(value).split("-")
        return float(a), float(b)
    except (ValueError, TypeError):
        return None, None


def season_from_date(value):

    d = pd.Timestamp(value).tz_convert(
        "America/New_York"
    )

    y = d.year if d.month >= 7 else d.year - 1

    return f"{y}-{str(y + 1)[-2:]}"


# ============================================================
# CONTROLLO SCHEMA
# ============================================================

game_columns = {
    row[1]
    for row in conn.execute(
        "PRAGMA table_info(espn_games)"
    )
}

for column in ["home_id", "away_id"]:

    if column not in game_columns:

        conn.execute(
            f"ALTER TABLE espn_games "
            f"ADD COLUMN {column} TEXT"
        )

conn.commit()

# ============================================================
# AGGIORNAMENTO PARTITE
# ============================================================

today = datetime.now(
    ZoneInfo("America/New_York")
).date()

events = {}
failed_dates = []

for offset in tqdm(
    range(-7, 4),
    desc="Calendario NBA"
):

    day = today + timedelta(days=offset)

    result = fetch(
        "/scoreboard",
        {
            "dates": day.strftime("%Y%m%d"),
            "limit": 100
        }
    )

    if result is None:
        failed_dates.append(str(day))
        continue

    for event in result.get("events", []):

        events[str(event["id"])] = event

    time.sleep(0.2)

updated = 0

for gid, event in events.items():

    competitions = event.get("competitions", [])

    if not competitions:
        continue

    teams = competitions[0].get("competitors", [])

    home = next(
        (t for t in teams if t["homeAway"] == "home"),
        None
    )

    away = next(
        (t for t in teams if t["homeAway"] == "away"),
        None
    )

    if not home or not away:
        continue

    status = event.get("status", {}).get("type", {})
    completed = int(status.get("completed", False))

    previous = conn.execute(
        "SELECT completed FROM espn_games WHERE game_id=?",
        (gid,)
    ).fetchone()

    if previous and previous[0] == 1 and not completed:
        continue

    values = (
        season_from_date(event["date"]),
        event["date"],
        str(home["team"]["id"]),
        str(away["team"]["id"]),
        home["team"]["displayName"],
        away["team"]["displayName"],
        number(home.get("score")),
        number(away.get("score")),
        completed,
        status.get("name", "UNKNOWN")
    )

    with conn:

        if previous:

            conn.execute("""
                UPDATE espn_games SET
                    season=?,
                    game_date=?,
                    home_id=?,
                    away_id=?,
                    home_team=?,
                    away_team=?,
                    home_score=?,
                    away_score=?,
                    completed=?,
                    status=?
                WHERE game_id=?
            """, values + (gid,))

        else:

            conn.execute("""
                INSERT INTO espn_games (
                    season, game_date,
                    home_id, away_id,
                    home_team, away_team,
                    home_score, away_score,
                    completed, status, game_id
                )
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """, values + (gid,))

    updated += 1

# ============================================================
# INDIVIDUAZIONE BOXSCORE MANCANTI
# ============================================================

missing_games = conn.execute("""
    SELECT g.game_id, g.season, g.game_date
    FROM espn_games g
    WHERE g.completed = 1
    AND (
        SELECT COUNT(DISTINCT b.team_id)
        FROM espn_player_boxscores b
        WHERE b.game_id = g.game_id
        AND b.minutes > 0
        AND b.pts IS NOT NULL
    ) < 2
    ORDER BY g.game_date
""").fetchall()

print("Box score mancanti:", len(missing_games))

BOX_COLS = [
    "game_id", "season", "game_date",
    "team_id", "team_name",
    "player_id", "player_name",
    "starter", "did_not_play",
    "minutes", "pts", "reb", "ast",
    "stl", "blk", "tov", "oreb",
    "dreb", "pf", "fgm", "fga",
    "fg3m", "fg3a", "ftm", "fta",
    "plus_minus"
]

box_schema = {
    row[1]
    for row in conn.execute(
        "PRAGMA table_info(espn_player_boxscores)"
    )
}

missing_cols = set(BOX_COLS) - box_schema

if missing_cols:
    raise ValueError(
        f"Schema boxscore incompatibile: {missing_cols}"
    )

# ============================================================
# DOWNLOAD STATISTICHE
# ============================================================

downloaded = 0
failed = []

for gid, season, game_date in tqdm(
    missing_games,
    desc="Player Box Scores"
):

    result = fetch(
        "/summary",
        {"event": gid}
    )

    if not result:
        failed.append(gid)
        continue

    rows = []
    valid_teams = set()

    for team_data in result.get(
        "boxscore", {}
    ).get("players", []):

        team = team_data.get("team", {})
        team_id = str(team.get("id", ""))

        for group in team_data.get("statistics", []):

            labels = group.get("labels") or group.get("names", [])

            labels = [
                str(x).upper()
                for x in labels
            ]

            for athlete in group.get("athletes", []):

                player = athlete.get("athlete", {})
                player_id = str(player.get("id", ""))

                if not player_id or not team_id:
                    continue

                stats = dict(zip(
                    labels,
                    athlete.get("stats", [])
                ))

                fgm, fga = split_stat(stats.get("FG"))
                fg3m, fg3a = split_stat(stats.get("3PT"))
                ftm, fta = split_stat(stats.get("FT"))

                mins = minutes(stats.get("MIN"))
                pts = number(stats.get("PTS"))

                if mins is not None and mins > 0 and pts is not None:
                    valid_teams.add(team_id)

                rows.append((
                    str(gid),
                    season,
                    game_date,
                    team_id,
                    team.get("displayName", ""),
                    player_id,
                    player.get("displayName", ""),
                    int(bool(athlete.get("starter"))),
                    int(bool(athlete.get("didNotPlay"))),
                    mins,
                    pts,
                    number(stats.get("REB")),
                    number(stats.get("AST")),
                    number(stats.get("STL")),
                    number(stats.get("BLK")),
                    number(stats.get("TO", stats.get("TOV"))),
                    number(stats.get("OREB")),
                    number(stats.get("DREB")),
                    number(stats.get("PF")),
                    fgm, fga,
                    fg3m, fg3a,
                    ftm, fta,
                    number(stats.get("+/-"))
                ))

    if len(valid_teams) != 2:
        failed.append(gid)
        continue

    sql = (
        "INSERT INTO espn_player_boxscores ("
        + ",".join(BOX_COLS)
        + ") VALUES ("
        + ",".join(["?"] * len(BOX_COLS))
        + ")"
    )

    try:

        with conn:

            conn.execute("""
                DELETE FROM espn_player_boxscores
                WHERE game_id = ?
            """, (gid,))

            conn.executemany(sql, rows)

        downloaded += 1

    except sqlite3.Error as e:

        failed.append(gid)
        print("Errore:", gid, e)

    time.sleep(0.25)

# ============================================================
# REPORT
# ============================================================

total_games = conn.execute(
    "SELECT COUNT(*) FROM espn_games"
).fetchone()[0]

total_box = conn.execute(
    "SELECT COUNT(*) FROM espn_player_boxscores"
).fetchone()[0]

conn.close()
session.close()

report = pd.DataFrame([{
    "Timestamp": pd.Timestamp.now(tz="UTC").isoformat(),
    "Partite database": total_games,
    "Partite aggiornate": updated,
    "Box score totali": total_box,
    "Nuovi box score": downloaded,
    "Download falliti": len(failed),
    "Date non recuperate": len(failed_dates)
}])

report.to_csv(
    os.path.join(BASE_DIR, "NBA_Pipeline_Status.csv"),
    index=False
)

print("=" * 45)
print("STEP 12 TERMINATO")
print("=" * 45)

print("Partite totali:", total_games)
print("Statistiche individuali:", total_box)
print("Nuovi box score:", downloaded)
print("Download falliti:", failed)
print("Date non recuperate:", failed_dates)

if failed or failed_dates:
    print("ATTENZIONE: aggiornamento parziale.")
else:
    print("Aggiornamento completato.")


In [ ]:

# ============================================================
# STEP 13 — NBA FORECAST ENGINE
# CON ARCHIVIAZIONE PERMANENTE
# ============================================================

import os
import sqlite3
import joblib
import numpy as np
import pandas as pd

from IPython.display import display

DB_PATH = os.path.join(
    BASE_DIR, "nba_database.db"
)

MODEL_PATH = os.path.join(
    BASE_DIR, "nba_player_models_v2.joblib"
)

CSV_PATH = os.path.join(
    BASE_DIR, "NBA_Forecast_Engine_V2.csv"
)

EXCEL_PATH = os.path.join(
    BASE_DIR, "NBA_Forecast_Engine_V2.xlsx"
)

ARCHIVE_DIR = os.path.join(
    BASE_DIR, "Forecast_Archive"
)

os.makedirs(ARCHIVE_DIR, exist_ok=True)

NOW = pd.Timestamp.now(tz="UTC")

package = joblib.load(MODEL_PATH)

models = package["models"]
feature_cols = package["features"]
targets = package["targets"]

MIN_HISTORY = package.get(
    "minimum_history", 10
)

print("Feature richieste dal modello:")
print(feature_cols)

# ============================================================
# CARICAMENTO DATABASE
# ============================================================

with sqlite3.connect(DB_PATH) as conn:

    box = pd.read_sql_query(
        "SELECT * FROM espn_player_boxscores",
        conn
    )

    games = pd.read_sql_query(
        "SELECT * FROM espn_games",
        conn
    )

for df in [box, games]:

    df["game_id"] = df["game_id"].astype(str)

    df["game_date"] = pd.to_datetime(
        df["game_date"],
        utc=True,
        errors="coerce",
        format="mixed"
    )

for c in ["player_id", "team_id"]:
    box[c] = box[c].astype(str)

for c in ["home_id", "away_id"]:
    games[c] = games[c].astype(str)

for c in ["minutes", "pts", "reb", "ast", "fg3m"]:
    box[c] = pd.to_numeric(box[c], errors="coerce")

for c in ["home_score", "away_score", "completed"]:
    games[c] = pd.to_numeric(games[c], errors="coerce")

completed = games[
    (games["completed"] == 1)
    & (games["game_date"] < NOW)
].dropna(
    subset=["home_score", "away_score"]
).copy()

box = box[
    (box["minutes"] > 0)
    & (box["game_date"] < NOW)
    & box["game_id"].isin(completed["game_id"])
].dropna(
    subset=["pts", "reb", "ast", "fg3m"]
).drop_duplicates(
    ["game_id", "player_id"]
).copy()

box["PRA"] = (
    box["pts"] + box["reb"] + box["ast"]
)

box = box.sort_values(
    ["game_date", "game_id"]
)

# ============================================================
# PARTITE FUTURE
# ============================================================

future = games[
    (games["completed"] == 0)
    & (games["status"] == "STATUS_SCHEDULED")
    & (games["game_date"] > NOW)
    & (
        games["game_date"]
        <= NOW + pd.Timedelta(days=3)
    )
].drop_duplicates(
    "game_id"
).sort_values("game_date")

if future.empty:

    raise RuntimeError(
        "Nessuna partita futura disponibile. "
        "Verifica STEP 12."
    )

print("Partite future:", len(future))

# ============================================================
# ROSTER STIMATI
# ============================================================

latest_team = box.drop_duplicates(
    "player_id",
    keep="last"
)

latest_team = latest_team[
    latest_team["game_date"]
    >= NOW - pd.Timedelta(days=90)
]

histories = {
    pid: group.copy()
    for pid, group in box.groupby("player_id")
}

# ============================================================
# DIFESA AVVERSARIA
# ============================================================

def opponent_defense(team_id, date):

    history = completed[
        (completed["game_date"] < date)
        & (
            (completed["home_id"] == team_id)
            | (completed["away_id"] == team_id)
        )
    ].sort_values("game_date").tail(10)

    if len(history) < 5:
        return np.nan

    allowed = np.where(
        history["home_id"] == team_id,
        history["away_score"],
        history["home_score"]
    )

    return float(np.mean(allowed))

# ============================================================
# FEATURE ENGINEERING
# ============================================================

def build_features(history, date, home, allowed):

    history = history[
        history["game_date"] < date
    ].sort_values("game_date")

    if len(history) < MIN_HISTORY:
        return None

    last5 = history.tail(5)
    last10 = history.tail(10)

    game_day = date.tz_convert(
        "America/New_York"
    ).date()

    last_day = history.iloc[-1][
        "game_date"
    ].tz_convert(
        "America/New_York"
    ).date()

    rest = max(
        (game_day - last_day).days - 1,
        0
    )

    row = {
        "home": float(home),
        "rest_days": float(np.clip(rest, 0, 7)),
        "GP_precedenti": len(history),
        "MIN_avg5": last5["minutes"].mean(),
        "MIN_avg10": last10["minutes"].mean(),
        "MIN_last1": history.iloc[-1]["minutes"],
        "opp_allowed_l10": allowed
    }

    for target, column in targets.items():

        row[f"{target}_last1"] = history.iloc[-1][column]
        row[f"{target}_avg5"] = last5[column].mean()
        row[f"{target}_avg10"] = last10[column].mean()
        row[f"{target}_std10"] = last10[column].std(ddof=0)

    return row

# ============================================================
# GENERAZIONE DATASET FUTURO
# ============================================================

rows = []

for game in future.itertuples(index=False):

    matchups = [
        (
            game.home_id,
            game.home_team,
            game.away_id,
            game.away_team,
            1
        ),
        (
            game.away_id,
            game.away_team,
            game.home_id,
            game.home_team,
            0
        )
    ]

    for tid, team, oid, opponent, home in matchups:

        roster = latest_team[
            latest_team["team_id"] == tid
        ]

        allowed = opponent_defense(
            oid,
            game.game_date
        )

        for player in roster.itertuples(index=False):

            history = histories[player.player_id]

            features = build_features(
                history,
                game.game_date,
                home,
                allowed
            )

            if features is None:
                continue

            rows.append({
                "game_id": game.game_id,
                "player_id": player.player_id,
                "date": game.game_date,
                "Giocatore": player.player_name,
                "Squadra": team,
                "Avversario": opponent,
                "Sede": "Casa" if home else "Trasferta",
                **features
            })

forecast = pd.DataFrame(rows)

if forecast.empty:
    raise ValueError("Nessun forecast disponibile.")

missing_features = [
    c for c in feature_cols
    if c not in forecast.columns
]

if missing_features:

    raise ValueError(
        "Feature incompatibili con STEP 9: "
        + str(missing_features)
    )

X = forecast[feature_cols].replace(
    [np.inf, -np.inf],
    np.nan
)

valid = X.notna().all(axis=1)

print("Righe escluse per feature mancanti:", (~valid).sum())

forecast = forecast.loc[valid].copy()
X = X.loc[valid]

if forecast.empty:
    raise ValueError("Nessuna riga con feature complete.")

# ============================================================
# MACHINE LEARNING
# ============================================================

for target, info in models.items():

    if info["name"] == "Last 10 Baseline":

        prediction = forecast[
            f"{target}_avg10"
        ].to_numpy()

    else:

        prediction = info["model"].predict(X)

    forecast[f"Forecast_{target}"] = np.maximum(
        np.asarray(prediction, dtype=float),
        0
    )

forecast["Forecast_PRA"] = (
    forecast["Forecast_PTS"]
    + forecast["Forecast_REB"]
    + forecast["Forecast_AST"]
)

# Timestamp effettivo dopo il calcolo.
generated_at = pd.Timestamp.now(tz="UTC")

forecast = forecast[
    forecast["date"] > generated_at
].copy()

forecast["generated_at"] = generated_at

forecast["Data Italia"] = (
    forecast["date"]
    .dt.tz_convert("Europe/Rome")
    .dt.strftime("%d/%m/%Y %H:%M")
)

forecast["Model_Version"] = str(
    package.get("model_version", "unknown")
)

TARGETS = ["PTS", "REB", "AST", "3PM", "PRA"]

forecast = forecast.replace(
    [np.inf, -np.inf],
    np.nan
).dropna(
    subset=[f"Forecast_{t}" for t in TARGETS]
).drop_duplicates(
    ["game_id", "player_id"]
)

if forecast.empty:
    raise ValueError("Nessuna previsione futura valida.")

# ============================================================
# ARCHIVIAZIONE SQLITE
# ============================================================

with sqlite3.connect(DB_PATH) as conn:

    conn.execute("""
        CREATE TABLE IF NOT EXISTS nba_forecast_history (
            game_id TEXT NOT NULL,
            player_id TEXT NOT NULL,
            generated_at TEXT NOT NULL,
            game_date TEXT NOT NULL,
            player_name TEXT,
            team_name TEXT,
            opponent TEXT,
            forecast_pts REAL,
            forecast_reb REAL,
            forecast_ast REAL,
            forecast_3pm REAL,
            forecast_pra REAL,
            PRIMARY KEY (
                game_id, player_id, generated_at
            )
        )
    """)

    records = []

    for _, r in forecast.iterrows():

        records.append((
            str(r["game_id"]),
            str(r["player_id"]),
            r["generated_at"].isoformat(),
            r["date"].isoformat(),
            r["Giocatore"],
            r["Squadra"],
            r["Avversario"],
            float(r["Forecast_PTS"]),
            float(r["Forecast_REB"]),
            float(r["Forecast_AST"]),
            float(r["Forecast_3PM"]),
            float(r["Forecast_PRA"])
        ))

    conn.executemany("""
        INSERT OR IGNORE INTO nba_forecast_history
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, records)

    total_history = conn.execute("""
        SELECT COUNT(*)
        FROM nba_forecast_history
    """).fetchone()[0]

# ============================================================
# ESPORTAZIONE
# ============================================================

columns = [
    "game_id",
    "player_id",
    "date",
    "generated_at",
    "Data Italia",
    "Giocatore",
    "Squadra",
    "Avversario",
    "Sede",
    "Forecast_PTS",
    "Forecast_REB",
    "Forecast_AST",
    "Forecast_3PM",
    "Forecast_PRA",
    "MIN_avg10",
    "rest_days",
    "opp_allowed_l10",
    "Model_Version"
]

export = forecast[columns].copy()

export = export.sort_values(
    ["date", "Forecast_PTS"],
    ascending=[True, False]
)

for c in ["date", "generated_at"]:
    export[c] = export[c].map(
        lambda x: x.isoformat()
    )

stamp = generated_at.strftime(
    "%Y%m%d_%H%M%S_%f"
)

archive = os.path.join(
    ARCHIVE_DIR,
    f"NBA_Forecast_{stamp}.csv"
)

export.to_csv(
    archive,
    index=False,
    encoding="utf-8-sig"
)

export.to_csv(
    CSV_PATH + ".tmp",
    index=False,
    encoding="utf-8-sig"
)

with pd.ExcelWriter(
    EXCEL_PATH + ".tmp.xlsx",
    engine="openpyxl"
) as writer:

    export.to_excel(
        writer,
        sheet_name="Player Forecast",
        index=False
    )

    accuracy = pd.DataFrame(
        package.get("test", [])
    )

    if not accuracy.empty:

        accuracy.to_excel(
            writer,
            sheet_name="Model Accuracy",
            index=False
        )

os.replace(CSV_PATH + ".tmp", CSV_PATH)
os.replace(EXCEL_PATH + ".tmp.xlsx", EXCEL_PATH)

# ============================================================
# REPORT
# ============================================================

log_path = os.path.join(
    BASE_DIR,
    "NBA_Forecast_Refresh_Log.csv"
)

pd.DataFrame([{
    "Timestamp": generated_at.isoformat(),
    "Partite": export["game_id"].nunique(),
    "Forecast": len(export),
    "Model Version": package.get("model_version"),
    "Status": "SUCCESS"
}]).to_csv(
    log_path,
    mode="a",
    header=not os.path.exists(log_path),
    index=False
)

print("=" * 45)
print("STEP 13 COMPLETATO!")
print("=" * 45)

print("Partite future:", export["game_id"].nunique())
print("Giocatori:", export["player_id"].nunique())
print("Forecast:", len(export))
print("Forecast storici:", total_history)
print("Archivio:", archive)

display(export.head(20).round(2))


In [ ]:

# ============================================================
# STEP 14 — NBA PREDICTION ACCURACY TRACKER
# ============================================================

import os
import sqlite3
import numpy as np
import pandas as pd
import plotly.express as px

from IPython.display import display

DB_PATH = os.path.join(
    BASE_DIR, "nba_database.db"
)

REPORT_PATH = os.path.join(
    BASE_DIR, "NBA_Live_Accuracy_Report.xlsx"
)

HISTORY_PATH = os.path.join(
    BASE_DIR, "NBA_Historical_Forecasts.csv"
)

# ============================================================
# CARICAMENTO DATI
# ============================================================

with sqlite3.connect(DB_PATH) as conn:

    history = pd.read_sql_query("""
        SELECT * FROM nba_forecast_history
    """, conn)

    actual = pd.read_sql_query("""
        SELECT
            b.game_id,
            b.player_id,
            b.minutes,
            b.pts,
            b.reb,
            b.ast,
            b.fg3m,
            b.did_not_play,
            g.game_date AS actual_game_date
        FROM espn_player_boxscores b
        JOIN espn_games g
            ON b.game_id = g.game_id
        WHERE g.completed = 1
    """, conn)

for df in [history, actual]:

    df["game_id"] = df["game_id"].astype(str)
    df["player_id"] = df["player_id"].astype(str)

for col in ["generated_at", "game_date"]:

    history[col] = pd.to_datetime(
        history[col],
        utc=True,
        errors="coerce",
        format="mixed"
    )

actual["actual_game_date"] = pd.to_datetime(
    actual["actual_game_date"],
    utc=True,
    errors="coerce",
    format="mixed"
)

for col in [
    "minutes", "pts", "reb",
    "ast", "fg3m", "did_not_play"
]:

    actual[col] = pd.to_numeric(
        actual[col],
        errors="coerce"
    )

actual = actual[
    (actual["minutes"] > 0)
    & (actual["did_not_play"] == 0)
].dropna(
    subset=["pts", "reb", "ast", "fg3m"]
).drop_duplicates(
    ["game_id", "player_id"]
).copy()

actual["pra"] = (
    actual["pts"] + actual["reb"] + actual["ast"]
)

# ============================================================
# FORECAST VS ACTUAL
# ============================================================

evaluation = history.merge(
    actual,
    on=["game_id", "player_id"],
    how="inner"
)

evaluation = evaluation[
    evaluation["generated_at"]
    < evaluation["actual_game_date"]
].sort_values(
    "generated_at"
).drop_duplicates(
    ["game_id", "player_id"],
    keep="last"
).copy()

mapping = {
    "PTS": ("forecast_pts", "pts"),
    "REB": ("forecast_reb", "reb"),
    "AST": ("forecast_ast", "ast"),
    "3PM": ("forecast_3pm", "fg3m"),
    "PRA": ("forecast_pra", "pra")
}

# ============================================================
# METRICHE GLOBALI
# ============================================================

metrics_rows = []

for target, (pred, real) in mapping.items():

    evaluation[pred] = pd.to_numeric(
        evaluation[pred],
        errors="coerce"
    )

    evaluation[f"Error_{target}"] = (
        evaluation[pred] - evaluation[real]
    )

    evaluation[f"Abs_Error_{target}"] = (
        evaluation[f"Error_{target}"].abs()
    )

    error = evaluation[
        f"Error_{target}"
    ].dropna()

    if error.empty:
        continue

    metrics_rows.append({
        "Target": target,
        "Campione": len(error),
        "MAE": error.abs().mean(),
        "RMSE": np.sqrt((error ** 2).mean()),
        "Bias": error.mean(),
        "Median Error": error.abs().median()
    })

metrics = pd.DataFrame(
    metrics_rows,
    columns=[
        "Target", "Campione",
        "MAE", "RMSE", "Bias",
        "Median Error"
    ]
)

# ============================================================
# ACCURATEZZA INDIVIDUALE
# ============================================================

player_rows = []

for pid, group in evaluation.groupby("player_id"):

    if len(group) < 5:
        continue

    row = {
        "Player ID": pid,
        "Giocatore": group["player_name"].iloc[-1],
        "Partite": len(group)
    }

    for target in mapping:

        row[f"MAE_{target}"] = group[
            f"Abs_Error_{target}"
        ].mean()

        row[f"Bias_{target}"] = group[
            f"Error_{target}"
        ].mean()

    player_rows.append(row)

player_accuracy = pd.DataFrame(player_rows)

if not player_accuracy.empty:

    player_accuracy = player_accuracy.sort_values(
        "MAE_PTS"
    )

# ============================================================
# PERFORMANCE GIORNALIERA
# ============================================================

evaluation["Giorno"] = (
    evaluation["actual_game_date"]
    .dt.tz_convert("Europe/Rome")
    .dt.strftime("%Y-%m-%d")
)

daily = evaluation.groupby("Giorno").agg(
    Forecast_verificati=("player_id", "size"),
    MAE_PTS=("Abs_Error_PTS", "mean"),
    MAE_REB=("Abs_Error_REB", "mean"),
    MAE_AST=("Abs_Error_AST", "mean"),
    MAE_3PM=("Abs_Error_3PM", "mean"),
    MAE_PRA=("Abs_Error_PRA", "mean")
).reset_index()

# ============================================================
# ESPORTAZIONE EXCEL
# ============================================================

def excel_safe(df):

    result = df.copy()

    for c in result.columns:

        if isinstance(
            result[c].dtype,
            pd.DatetimeTZDtype
        ):

            result[c] = result[c].astype(str)

    return result

with pd.ExcelWriter(
    REPORT_PATH,
    engine="openpyxl"
) as writer:

    sheets = {
        "Model Accuracy": metrics,
        "Player Accuracy": player_accuracy,
        "Forecast vs Actual": evaluation,
        "Daily Performance": daily,
        "Forecast History": history
    }

    for name, df in sheets.items():

        excel_safe(df).to_excel(
            writer,
            sheet_name=name,
            index=False
        )

history.to_csv(
    HISTORY_PATH,
    index=False
)

# ============================================================
# GRAFICI
# ============================================================

if not metrics.empty:

    display(metrics.round(3))

    fig = px.bar(
        metrics,
        x="Target",
        y=["MAE", "RMSE"],
        barmode="group",
        title="NBA Live Model Accuracy",
        template="plotly_dark"
    )

    fig.show()

if not daily.empty:

    fig = px.line(
        daily,
        x="Giorno",
        y=[
            "MAE_PTS",
            "MAE_REB",
            "MAE_AST",
            "MAE_3PM"
        ],
        markers=True,
        title="Daily Model Performance",
        template="plotly_dark"
    )

    fig.show()

if not player_accuracy.empty:

    display(
        player_accuracy.head(30).round(3)
    )

print("=" * 45)
print("STEP 14 COMPLETATO!")
print("=" * 45)

print("Forecast archiviati:", len(history))
print("Forecast verificati:", len(evaluation))
print("Giocatori valutati:", len(player_accuracy))
print("Report:", REPORT_PATH)

if evaluation.empty:

    print(
        "Nessun forecast ancora verificabile. "
        "Attendi le prossime partite NBA."
    )


In [ ]:
%pip install -q gradio plotly openpyxl

In [ ]:

# ============================================================
# NBA PREDICTION CENTER V3
# ============================================================

import os
import pandas as pd
import plotly.express as px
import gradio as gr

PLAYER_PATH = os.path.join(
    BASE_DIR, "NBA_Forecast_Engine_V2.csv"
)

GAME_PATH = os.path.join(
    BASE_DIR, "NBA_Forecast_Prossime_Partite.csv"
)

ACCURACY_PATH = os.path.join(
    BASE_DIR, "NBA_Live_Accuracy_Report.xlsx"
)

STATS = {
    "Punti": "Forecast_PTS",
    "Rimbalzi": "Forecast_REB",
    "Assist": "Forecast_AST",
    "Triple": "Forecast_3PM",
    "PRA": "Forecast_PRA"
}

def load_players():

    df = pd.read_csv(PLAYER_PATH)

    if "date" in df.columns:

        dates = pd.to_datetime(
            df["date"],
            utc=True,
            errors="coerce",
            format="mixed"
        )

        df = df[
            dates > pd.Timestamp.now(tz="UTC")
        ]

    return df


initial = load_players()

teams = sorted(
    initial["Squadra"].dropna().unique().tolist()
)

players = sorted(
    initial["Giocatore"].dropna().unique().tolist()
)


def refresh_options():

    df = load_players()

    teams = sorted(
        df["Squadra"].dropna().unique().tolist()
    )

    players = sorted(
        df["Giocatore"].dropna().unique().tolist()
    )

    return (
        gr.Dropdown(choices=["Tutte"] + teams, value="Tutte"),
        gr.Dropdown(choices=["Tutti"] + players, value="Tutti"),
        gr.Dropdown(choices=players, value=players[0] if players else None),
        gr.Dropdown(choices=players, value=players[-1] if players else None)
    )


def player_forecast(team, player, stat, limit):

    df = load_players()

    if team != "Tutte":
        df = df[df["Squadra"] == team]

    if player != "Tutti":
        df = df[df["Giocatore"] == player]

    col = STATS[stat]

    df = df.sort_values(
        col,
        ascending=False
    ).head(int(limit))

    columns = [
        "Data Italia",
        "Giocatore",
        "Squadra",
        "Avversario",
        "Sede",
        *STATS.values()
    ]

    table = df[columns].round(2)

    if df.empty:
        return table, None

    fig = px.bar(
        df,
        x="Giocatore",
        y=col,
        color="Squadra",
        hover_data=["Avversario", "Data Italia"],
        title=f"NBA Forecast — {stat}",
        template="plotly_dark"
    )

    fig.update_layout(
        height=550,
        xaxis_tickangle=-45
    )

    return table, fig


def compare_players(p1, p2):

    df = load_players()

    df = df[
        df["Giocatore"].isin([p1, p2])
    ].copy()

    df["_date"] = pd.to_datetime(
        df["date"],
        utc=True,
        errors="coerce",
        format="mixed"
    )

    df = df.sort_values(
        "_date"
    ).drop_duplicates("Giocatore")

    table = df[
        ["Giocatore", *STATS.values()]
    ].round(2)

    if table.empty:
        return table, None

    melted = table.melt(
        id_vars="Giocatore",
        var_name="Statistica",
        value_name="Forecast"
    )

    fig = px.bar(
        melted,
        x="Statistica",
        y="Forecast",
        color="Giocatore",
        barmode="group",
        template="plotly_dark",
        title="Player Comparison"
    )

    return table, fig


def game_predictions():

    if not os.path.isfile(GAME_PATH):
        return pd.DataFrame(), "Esegui STEP 6."

    df = pd.read_csv(GAME_PATH)

    updated = pd.Timestamp(
        os.path.getmtime(GAME_PATH),
        unit="s",
        tz="UTC"
    ).tz_convert("Europe/Rome")

    return (
        df,
        f"Ultima modifica file: {updated}. "
        "Per aggiornare le previsioni delle squadre "
        "riesegui STEP 6."
    )


def live_accuracy():

    if not os.path.isfile(ACCURACY_PATH):
        return pd.DataFrame(), None

    df = pd.read_excel(
        ACCURACY_PATH,
        sheet_name="Model Accuracy"
    )

    if df.empty:
        return df, None

    fig = px.bar(
        df,
        x="Target",
        y=["MAE", "RMSE"],
        barmode="group",
        template="plotly_dark",
        title="Live Model Accuracy"
    )

    return df, fig


with gr.Blocks(
    title="NBA Prediction Center V3"
) as app:

    gr.Markdown("""
    # 🏀 NBA PREDICTION CENTER V3
    ### Machine Learning · Player Analytics · Forecasting
    """)

    refresh = gr.Button(
        "Aggiorna elenco giocatori e squadre"
    )

    with gr.Tab("Player Forecast"):

        with gr.Row():

            team = gr.Dropdown(
                ["Tutte"] + teams,
                value="Tutte",
                label="Squadra"
            )

            player = gr.Dropdown(
                ["Tutti"] + players,
                value="Tutti",
                label="Giocatore"
            )

        with gr.Row():

            stat = gr.Dropdown(
                list(STATS.keys()),
                value="Punti",
                label="Statistica"
            )

            limit = gr.Slider(
                5, 100,
                value=30,
                step=5,
                label="Risultati"
            )

        btn = gr.Button(
            "Genera Forecast",
            variant="primary"
        )

        table = gr.Dataframe(interactive=False)
        chart = gr.Plot()

        btn.click(
            player_forecast,
            [team, player, stat, limit],
            [table, chart]
        )

    with gr.Tab("Player Comparison"):

        with gr.Row():

            p1 = gr.Dropdown(
                players,
                label="Giocatore 1"
            )

            p2 = gr.Dropdown(
                players,
                label="Giocatore 2"
            )

        compare = gr.Button(
            "Confronta",
            variant="primary"
        )

        comparison_table = gr.Dataframe(
            interactive=False
        )

        comparison_chart = gr.Plot()

        compare.click(
            compare_players,
            [p1, p2],
            [comparison_table, comparison_chart]
        )

    with gr.Tab("Game Predictions"):

        gr.Markdown(
            "Previsioni delle squadre generate dallo STEP 6."
        )

        game_btn = gr.Button("Mostra partite")
        game_table = gr.Dataframe(interactive=False)
        game_info = gr.Textbox(interactive=False)

        game_btn.click(
            game_predictions,
            outputs=[game_table, game_info]
        )

    with gr.Tab("Live Model Accuracy"):

        accuracy_btn = gr.Button(
            "Mostra accuratezza"
        )

        accuracy_table = gr.Dataframe(
            interactive=False
        )

        accuracy_chart = gr.Plot()

        accuracy_btn.click(
            live_accuracy,
            outputs=[accuracy_table, accuracy_chart]
        )

    refresh.click(
        refresh_options,
        outputs=[team, player, p1, p2]
    )

    gr.Markdown("""
    ---
    Le previsioni sono stime statistiche.
    La disponibilità effettiva dei giocatori
    non è ancora verificata automaticamente.
    """)

app.launch(
    share=True,
    inline=False,
    debug=False,
    show_error=True
)


In [ ]:

# ============================================================
# STEP 15 — ADVANCED PLAYER INTELLIGENCE
# NBA DATA SCIENCE ANALYTICS
# ============================================================

import os
import sqlite3
import numpy as np
import pandas as pd
import plotly.express as px

from IPython.display import display

# ============================================================
# 1. CONFIGURAZIONE
# ============================================================

if "BASE_DIR" not in globals():
    raise RuntimeError("Esegui prima STEP 0.")

DB_PATH = os.path.join(
    BASE_DIR, "nba_database.db"
)

CSV_PATH = os.path.join(
    BASE_DIR, "NBA_Advanced_ML_Features.csv"
)

EXCEL_PATH = os.path.join(
    BASE_DIR, "NBA_Advanced_Player_Intelligence.xlsx"
)

# ============================================================
# 2. CARICAMENTO DATABASE
# ============================================================

with sqlite3.connect(DB_PATH) as conn:

    box = pd.read_sql_query(
        "SELECT * FROM espn_player_boxscores",
        conn
    )

    games = pd.read_sql_query(
        "SELECT * FROM espn_games",
        conn
    )

print("Box score originali:", len(box))
print("Partite originali:", len(games))

# ============================================================
# 3. PULIZIA DATI
# ============================================================

for df in [box, games]:

    df["game_id"] = df["game_id"].astype(str)

    df["game_date"] = pd.to_datetime(
        df["game_date"],
        utc=True,
        errors="coerce",
        format="mixed"
    )

for c in ["player_id", "team_id"]:
    box[c] = box[c].astype(str)

for c in ["home_id", "away_id"]:
    games[c] = games[c].astype(str)

numeric_cols = [
    "minutes", "pts", "reb", "ast",
    "stl", "blk", "tov",
    "oreb", "dreb", "pf",
    "fgm", "fga", "fg3m", "fg3a",
    "ftm", "fta", "plus_minus",
    "did_not_play"
]

for c in numeric_cols:

    if c not in box.columns:
        raise ValueError(
            f"Colonna mancante: {c}"
        )

    box[c] = pd.to_numeric(
        box[c],
        errors="coerce"
    )

for c in [
    "home_score",
    "away_score",
    "completed"
]:

    games[c] = pd.to_numeric(
        games[c],
        errors="coerce"
    )

games = games[
    (games["completed"] == 1)
    & games["game_date"].notna()
].dropna(
    subset=["home_score", "away_score"]
).drop_duplicates(
    "game_id"
).copy()

box = box[
    (box["minutes"] > 0)
    & (box["did_not_play"] == 0)
    & box["game_id"].isin(games["game_id"])
    & box["game_date"].notna()
].dropna(
    subset=[
        "pts", "reb", "ast", "fg3m"
    ]
).drop_duplicates(
    ["game_id", "player_id"]
).copy()

box["PRA"] = (
    box["pts"]
    + box["reb"]
    + box["ast"]
)

print("Box score validi:", len(box))

# ============================================================
# 4. FUNZIONE DIVISIONE SICURA
# ============================================================

def safe_divide(a, b):

    return a / b.replace(0, np.nan)

# ============================================================
# 5. ADVANCED PLAYER STATISTICS
# ============================================================

# True Shooting Percentage

box["TS_pct"] = 100 * safe_divide(
    box["pts"],
    2 * (
        box["fga"]
        + 0.44 * box["fta"]
    )
)

# Effective Field Goal Percentage

box["eFG_pct"] = 100 * safe_divide(
    box["fgm"] + 0.5 * box["fg3m"],
    box["fga"]
)

# Statistiche per minuto

box["PTS_per_min"] = safe_divide(
    box["pts"],
    box["minutes"]
)

box["REB_per_min"] = safe_divide(
    box["reb"],
    box["minutes"]
)

box["AST_per_min"] = safe_divide(
    box["ast"],
    box["minutes"]
)

box["PRA_per_min"] = safe_divide(
    box["PRA"],
    box["minutes"]
)

# Coinvolgimento offensivo stimato

box["Offensive_Load"] = (
    box["fga"]
    + 0.44 * box["fta"]
    + box["tov"]
)

# ============================================================
# 6. TEAM STATISTICS
# ============================================================

team_stats = box.groupby(
    ["game_id", "team_id"],
    as_index=False
).agg(
    Team_PTS=("pts", "sum"),
    Team_FGA=("fga", lambda s: s.sum(min_count=1)),
    Team_FTA=("fta", lambda s: s.sum(min_count=1)),
    Team_OREB=("oreb", lambda s: s.sum(min_count=1)),
    Team_TOV=("tov", lambda s: s.sum(min_count=1)),
    Team_MIN=("minutes", "sum")
)

# Stima dei possessi

team_stats["Possessions"] = (
    team_stats["Team_FGA"]
    + 0.44 * team_stats["Team_FTA"]
    - team_stats["Team_OREB"]
    + team_stats["Team_TOV"]
)

team_stats["Possessions"] = (
    team_stats["Possessions"]
    .where(team_stats["Possessions"] > 0)
)

team_stats["Offensive_Rating"] = (
    100 * safe_divide(
        team_stats["Team_PTS"],
        team_stats["Possessions"]
    )
)

team_stats["Team_Offensive_Load"] = (
    team_stats["Team_FGA"]
    + 0.44 * team_stats["Team_FTA"]
    + team_stats["Team_TOV"]
)

# ============================================================
# 7. CONTESTO PARTITA
# ============================================================

context = games[[
    "game_id",
    "home_id",
    "away_id",
    "home_score",
    "away_score"
]].copy()

box = box.merge(
    context,
    on="game_id",
    how="inner",
    validate="many_to_one"
)

box = box[
    (box["team_id"] == box["home_id"])
    | (box["team_id"] == box["away_id"])
].copy()

box["home"] = (
    box["team_id"] == box["home_id"]
).astype(int)

box["opponent_id"] = np.where(
    box["home"] == 1,
    box["away_id"],
    box["home_id"]
)

box["Team_Score"] = np.where(
    box["home"] == 1,
    box["home_score"],
    box["away_score"]
)

box["Opponent_Score"] = np.where(
    box["home"] == 1,
    box["away_score"],
    box["home_score"]
)

box = box.merge(
    team_stats,
    on=["game_id", "team_id"],
    how="left",
    validate="many_to_one"
)

opponent_stats = team_stats[[
    "game_id",
    "team_id",
    "Possessions"
]].rename(columns={
    "team_id": "opponent_id",
    "Possessions": "Opponent_Possessions"
})

box = box.merge(
    opponent_stats,
    on=["game_id", "opponent_id"],
    how="left",
    validate="many_to_one"
)

box["Defensive_Rating"] = (
    100 * safe_divide(
        box["Opponent_Score"],
        box["Opponent_Possessions"]
    )
)

box["Pace"] = safe_divide(
    48 * (
        box["Possessions"]
        + box["Opponent_Possessions"]
    ) / 2,
    box["Team_MIN"] / 5
)

# Usage proxy normalizzato per minuti.

box["Usage_Proxy"] = (
    100 * safe_divide(
        box["Offensive_Load"]
        * (box["Team_MIN"] / 5),
        box["minutes"]
        * box["Team_Offensive_Load"]
    )
)

# ============================================================
# 8. ORDINAMENTO TEMPORALE
# ============================================================

box = box.sort_values(
    ["player_id", "game_date", "game_id"]
).reset_index(drop=True)

group = box.groupby(
    "player_id",
    sort=False
)

# ============================================================
# 9. ROLLING FEATURES SENZA DATA LEAKAGE
# ============================================================

# shift(1) esclude la partita che vogliamo prevedere.

rolling_stats = [
    "pts", "reb", "ast", "fg3m",
    "PRA", "minutes",
    "TS_pct", "eFG_pct",
    "PTS_per_min",
    "REB_per_min",
    "AST_per_min",
    "PRA_per_min",
    "Usage_Proxy",
    "Pace",
    "Offensive_Rating",
    "Defensive_Rating"
]

feature_data = {}

for stat in rolling_stats:

    for window in [5, 10, 20]:

        feature_data[
            f"{stat}_avg{window}"
        ] = group[stat].transform(
            lambda s: (
                s.shift(1)
                .rolling(
                    window,
                    min_periods=window
                )
                .mean()
            )
        )

    feature_data[
        f"{stat}_std10"
    ] = group[stat].transform(
        lambda s: (
            s.shift(1)
            .rolling(
                10,
                min_periods=10
            )
            .std()
        )
    )

    feature_data[
        f"{stat}_last1"
    ] = group[stat].shift(1)

features = pd.DataFrame(
    feature_data,
    index=box.index
)

box = pd.concat(
    [box, features],
    axis=1
)

# ============================================================
# 10. MOMENTUM E CONSISTENCY
# ============================================================

for stat in [
    "pts", "reb", "ast",
    "fg3m", "PRA", "minutes"
]:

    box[f"{stat}_momentum"] = (
        box[f"{stat}_avg5"]
        - box[f"{stat}_avg20"]
    )

    box[f"{stat}_momentum_pct"] = (
        100 * safe_divide(
            box[f"{stat}_momentum"],
            box[f"{stat}_avg20"]
        )
    )

    box[f"{stat}_variation"] = safe_divide(
        box[f"{stat}_std10"],
        box[f"{stat}_avg10"]
    )

    box[f"{stat}_consistency"] = (
        100 / (
            1 + box[f"{stat}_variation"]
        )
    )

# ============================================================
# 11. RIPOSO E BACK-TO-BACK
# ============================================================

box["local_day"] = (
    box["game_date"]
    .dt.tz_convert("America/New_York")
    .dt.tz_localize(None)
    .dt.normalize()
)

previous_day = box.groupby(
    "player_id"
)["local_day"].shift(1)

box["Days_Between_Games"] = (
    box["local_day"] - previous_day
).dt.days

box["Rest_Days"] = (
    box["Days_Between_Games"] - 1
).clip(lower=0, upper=30)

box["Back_to_Back"] = (
    box["Days_Between_Games"] == 1
).astype(int)

box["Games_Previous"] = (
    box.groupby("player_id").cumcount()
)

# ============================================================
# 12. HOME / AWAY PERFORMANCE
# ============================================================

for stat in ["pts", "reb", "ast", "PRA"]:

    box[f"{stat}_venue_avg10"] = (
        box.groupby(
            ["player_id", "home"]
        )[stat].transform(
            lambda s: (
                s.shift(1)
                .rolling(
                    10,
                    min_periods=3
                )
                .mean()
            )
        )
    )

# ============================================================
# 13. DIFESA E PACE AVVERSARIA
# ============================================================

team_context = box[[
    "game_id",
    "game_date",
    "team_id",
    "Team_Score",
    "Opponent_Score",
    "Offensive_Rating",
    "Defensive_Rating",
    "Pace"
]].drop_duplicates(
    ["game_id", "team_id"]
).sort_values(
    ["team_id", "game_date"]
).copy()

team_features = [
    "Team_Score",
    "Opponent_Score",
    "Offensive_Rating",
    "Defensive_Rating",
    "Pace"
]

for stat in team_features:

    team_context[f"pre_{stat}_avg10"] = (
        team_context.groupby(
            "team_id"
        )[stat].transform(
            lambda s: (
                s.shift(1)
                .rolling(
                    10,
                    min_periods=5
                )
                .mean()
            )
        )
    )

opp_columns = [
    f"pre_{s}_avg10"
    for s in team_features
]

opp_context = team_context[
    ["game_id", "team_id"] + opp_columns
].rename(columns={
    "team_id": "opponent_id",
    **{
        c: f"Opp_{c}"
        for c in opp_columns
    }
})

box = box.merge(
    opp_context,
    on=["game_id", "opponent_id"],
    how="left",
    validate="many_to_one"
)

# ============================================================
# 14. SELEZIONE FEATURE MACHINE LEARNING
# ============================================================

feature_cols = list(feature_data.keys())

feature_cols += [
    "home",
    "Rest_Days",
    "Back_to_Back",
    "Games_Previous"
]

for stat in [
    "pts", "reb", "ast",
    "fg3m", "PRA", "minutes"
]:

    feature_cols += [
        f"{stat}_momentum",
        f"{stat}_momentum_pct",
        f"{stat}_variation",
        f"{stat}_consistency"
    ]

for stat in ["pts", "reb", "ast", "PRA"]:

    feature_cols.append(
        f"{stat}_venue_avg10"
    )

feature_cols += [
    f"Opp_{c}"
    for c in opp_columns
]

feature_cols = list(
    dict.fromkeys(feature_cols)
)

target_cols = [
    "pts", "reb", "ast",
    "fg3m", "PRA"
]

metadata = [
    "game_id",
    "game_date",
    "player_id",
    "player_name",
    "team_id",
    "team_name",
    "opponent_id"
]

dataset = box[
    metadata
    + feature_cols
    + target_cols
].copy()

dataset = dataset.replace(
    [np.inf, -np.inf],
    np.nan
)

dataset = dataset[
    dataset["Games_Previous"] >= 20
].dropna(
    subset=target_cols
).sort_values(
    "game_date"
)

if dataset.empty:
    raise ValueError(
        "Dataset avanzato vuoto."
    )

# Le feature mancanti vengono conservate.
# L'imputazione sarà addestrata solo
# sul training set nello STEP 16.

# ============================================================
# 15. PLAYER INTELLIGENCE RANKING
# ============================================================

recent = box[
    box["game_date"]
    >= box["game_date"].max()
    - pd.Timedelta(days=30)
].copy()

ranking = recent.groupby(
    ["player_id", "player_name"],
    as_index=False
).agg(
    Partite=("game_id", "nunique"),
    PTS=("pts", "mean"),
    REB=("reb", "mean"),
    AST=("ast", "mean"),
    PRA=("PRA", "mean"),
    MIN=("minutes", "mean"),
    PTS_per_min=("PTS_per_min", "mean"),
    Usage_Proxy=("Usage_Proxy", "mean"),
    PTS_Momentum=("pts_momentum", "last"),
    PTS_Consistency=("pts_consistency", "last")
)

ranking = ranking[
    ranking["Partite"] >= 5
].sort_values(
    "PRA",
    ascending=False
)

# ============================================================
# 16. CONTROLLO QUALITA'
# ============================================================

quality = pd.DataFrame({
    "Feature": feature_cols,
    "Missing_pct": [
        100 * dataset[c].isna().mean()
        for c in feature_cols
    ]
}).sort_values(
    "Missing_pct",
    ascending=False
)

# ============================================================
# 17. SALVATAGGIO DATASET
# ============================================================

dataset.to_csv(
    CSV_PATH,
    index=False,
    encoding="utf-8-sig"
)

import json

FEATURE_PATH = os.path.join(
    BASE_DIR,
    "NBA_Advanced_Feature_List.json"
)

with open(
    FEATURE_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump({
        "features": feature_cols,
        "targets": target_cols,
        "minimum_history": 20
    }, f, indent=2)

with pd.ExcelWriter(
    EXCEL_PATH,
    engine="openpyxl"
) as writer:

    ranking.to_excel(
        writer,
        sheet_name="Player Ranking",
        index=False
    )

    quality.to_excel(
        writer,
        sheet_name="Feature Quality",
        index=False
    )

    sample = dataset.tail(500).copy()

    sample["game_date"] = (
        sample["game_date"].astype(str)
    )

    sample.to_excel(
        writer,
        sheet_name="ML Dataset Sample",
        index=False
    )

# ============================================================
# 18. VISUALIZZAZIONE
# ============================================================

print("\nTOP 20 PLAYER INTELLIGENCE")

display(
    ranking.head(20).round(2)
)

if not ranking.empty:

    fig = px.bar(
        ranking.head(20),
        x="player_name",
        y="PRA",
        color="PTS_Consistency",
        hover_data=[
            "PTS",
            "REB",
            "AST",
            "MIN",
            "Usage_Proxy"
        ],
        title="NBA Advanced Player Ranking",
        template="plotly_dark"
    )

    fig.show()

    fig = px.scatter(
        ranking,
        x="Usage_Proxy",
        y="PTS",
        size="MIN",
        color="PTS_Consistency",
        hover_name="player_name",
        title="Offensive Load vs Scoring",
        template="plotly_dark"
    )

    fig.show()

    momentum = ranking.dropna(
        subset=["PTS_Momentum"]
    ).sort_values(
        "PTS_Momentum",
        ascending=False
    ).head(20)

    if not momentum.empty:

        fig = px.bar(
            momentum,
            x="player_name",
            y="PTS_Momentum",
            title="NBA Player Momentum",
            template="plotly_dark"
        )

        fig.show()

# ============================================================
# 19. RIEPILOGO FINALE
# ============================================================

print("\n" + "=" * 50)
print("STEP 15 COMPLETATO!")
print("=" * 50)

print("Righe dataset ML:", len(dataset))
print("Feature avanzate:", len(feature_cols))
print("Giocatori:", dataset["player_id"].nunique())
print("Partite:", dataset["game_id"].nunique())

print("\nPeriodo storico:")
print(dataset["game_date"].min())
print(dataset["game_date"].max())

print("\nFile salvati:")
print(CSV_PATH)
print(EXCEL_PATH)
print(FEATURE_PATH)

print("\nCONTROLLO FEATURE MANCANTI")

display(
    quality.head(15).round(2)
)

print("\nPRONTO PER STEP 16!")


In [ ]:
%pip install -q xgboost lightgbm scikit-learn joblib openpyxl plotly

In [ ]:

# ============================================================
# STEP 16 — ADVANCED MACHINE LEARNING & ENSEMBLE
# NBA DATA SCIENCE ANALYTICS
# MODEL VERSION 3.0
# ============================================================

import os
import json
import time
import joblib
import warnings
import numpy as np
import pandas as pd
import plotly.express as px

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import Ridge

from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from IPython.display import display

# ============================================================
# 1. CONFIGURAZIONE
# ============================================================

if "BASE_DIR" not in globals():
    raise RuntimeError("Esegui prima STEP 0.")

DATA_PATH = os.path.join(
    BASE_DIR,
    "NBA_Advanced_ML_Features.csv"
)

FEATURE_PATH = os.path.join(
    BASE_DIR,
    "NBA_Advanced_Feature_List.json"
)

MODEL_PATH = os.path.join(
    BASE_DIR,
    "nba_player_models_v3.joblib"
)

REPORT_PATH = os.path.join(
    BASE_DIR,
    "NBA_Advanced_Model_Accuracy.xlsx"
)

PREDICTIONS_PATH = os.path.join(
    BASE_DIR,
    "NBA_Advanced_Test_Predictions.csv"
)

RANDOM_STATE = 42

TARGETS = {
    "PTS": "pts",
    "REB": "reb",
    "AST": "ast",
    "3PM": "fg3m",
    "PRA": "PRA"
}

for path in [DATA_PATH, FEATURE_PATH]:

    if not os.path.isfile(path):
        raise FileNotFoundError(path)

START_TIME = time.time()

# ============================================================
# 2. CARICAMENTO DATASET
# ============================================================

df = pd.read_csv(
    DATA_PATH,
    dtype={
        "game_id": str,
        "player_id": str,
        "team_id": str,
        "opponent_id": str
    }
)

with open(
    FEATURE_PATH,
    "r",
    encoding="utf-8"
) as f:

    config = json.load(f)

feature_cols = config["features"]

df["game_date"] = pd.to_datetime(
    df["game_date"],
    utc=True,
    errors="coerce",
    format="mixed"
)

required = (
    feature_cols
    + list(TARGETS.values())
    + ["game_date", "game_id", "player_id"]
)

missing = [
    c for c in required
    if c not in df.columns
]

if missing:
    raise ValueError(
        f"Colonne mancanti: {missing}"
    )

for c in feature_cols + list(TARGETS.values()):

    df[c] = pd.to_numeric(
        df[c],
        errors="coerce"
    )

df = df.replace(
    [np.inf, -np.inf],
    np.nan
)

df = df.dropna(
    subset=[
        "game_date",
        "game_id",
        "player_id",
        *TARGETS.values()
    ]
).drop_duplicates(
    ["game_id", "player_id"]
).sort_values(
    "game_date"
).reset_index(drop=True)

print("Dataset:", len(df))
print("Feature iniziali:", len(feature_cols))
print("Giocatori:", df["player_id"].nunique())

# ============================================================
# 3. TRAIN / VALIDATION / TEST
# SUDDIVISIONE CRONOLOGICA PER GIORNATA NBA
# ============================================================

df["NBA_Day"] = (
    df["game_date"]
    .dt.tz_convert("America/New_York")
    .dt.strftime("%Y-%m-%d")
)

days = sorted(
    df["NBA_Day"].unique()
)

if len(days) < 30:
    raise ValueError(
        "Storico insufficiente per il backtest."
    )

train_end = int(len(days) * 0.70)
val_end = int(len(days) * 0.85)

train_days = days[:train_end]

val_days = days[
    train_end:val_end
]

test_days = days[val_end:]

train = df[
    df["NBA_Day"].isin(train_days)
].copy()

val = df[
    df["NBA_Day"].isin(val_days)
].copy()

test = df[
    df["NBA_Day"].isin(test_days)
].copy()

if min(len(train), len(val), len(test)) == 0:
    raise ValueError("Split temporale non valido.")

# Rimuoviamo feature completamente vuote
# utilizzando esclusivamente il training set.

feature_cols = [
    c for c in feature_cols
    if train[c].notna().any()
]

# Evitiamo che le statistiche target
# siano incluse direttamente tra le feature.

leakage = set(feature_cols).intersection(
    TARGETS.values()
)

if leakage:
    raise ValueError(
        f"Possibile data leakage: {leakage}"
    )

X_train = train[feature_cols]
X_val = val[feature_cols]
X_test = test[feature_cols]

print("\nTEMPORAL SPLIT")

print("Training:", len(train))
print("Validation:", len(val))
print("Test:", len(test))

print("\nTraining:", train_days[0], train_days[-1])
print("Validation:", val_days[0], val_days[-1])
print("Test:", test_days[0], test_days[-1])

print("Feature utilizzate:", len(feature_cols))

# ============================================================
# 4. DEFINIZIONE MODELLI
# ============================================================

def build_models():

    return {

        "Ridge": Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="median")
            ),
            (
                "scaler",
                StandardScaler()
            ),
            (
                "model",
                Ridge(alpha=10.0)
            )
        ]),

        "Random Forest": Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="median")
            ),
            (
                "model",
                RandomForestRegressor(
                    n_estimators=150,
                    max_depth=12,
                    min_samples_leaf=8,
                    max_features=0.7,
                    random_state=RANDOM_STATE,
                    n_jobs=2
                )
            )
        ]),

        "Extra Trees": Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="median")
            ),
            (
                "model",
                ExtraTreesRegressor(
                    n_estimators=150,
                    max_depth=14,
                    min_samples_leaf=8,
                    max_features=0.8,
                    random_state=RANDOM_STATE,
                    n_jobs=2
                )
            )
        ]),

        "XGBoost": XGBRegressor(
            n_estimators=250,
            learning_rate=0.04,
            max_depth=5,
            subsample=0.85,
            colsample_bytree=0.8,
            reg_lambda=5,
            objective="reg:squarederror",
            tree_method="hist",
            random_state=RANDOM_STATE,
            n_jobs=2
        ),

        "LightGBM": LGBMRegressor(
            n_estimators=250,
            learning_rate=0.04,
            num_leaves=31,
            max_depth=7,
            min_child_samples=30,
            subsample=0.85,
            subsample_freq=1,
            colsample_bytree=0.8,
            reg_lambda=5,
            random_state=RANDOM_STATE,
            n_jobs=2,
            verbosity=-1
        )
    }

# ============================================================
# 5. METRICHE DI VALUTAZIONE
# ============================================================

def evaluate(y, prediction):

    y = np.asarray(y, dtype=float)

    prediction = np.maximum(
        np.asarray(prediction, dtype=float),
        0
    )

    error = prediction - y

    return {
        "MAE": mean_absolute_error(
            y, prediction
        ),
        "RMSE": np.sqrt(
            mean_squared_error(y, prediction)
        ),
        "Bias": float(error.mean()),
        "Median_Error": float(
            np.median(np.abs(error))
        ),
        "R2": r2_score(y, prediction),
        "Campione": len(y)
    }

# ============================================================
# 6. BASELINE
# ============================================================

def baseline_predict(data, target, fallback):

    col = f"{target}_avg10"

    if col not in data.columns:

        raise ValueError(
            f"Baseline mancante: {col}"
        )

    return np.maximum(
        data[col]
        .fillna(fallback)
        .to_numpy(dtype=float),
        0
    )

# ============================================================
# 7. TRAINING E VALIDATION
# ============================================================

validation_results = []
test_results = []
selected_models = {}
all_predictions = []
importance_rows = []

for target_name, target_col in TARGETS.items():

    print("\n" + "=" * 55)
    print("TRAINING TARGET:", target_name)
    print("=" * 55)

    y_train = train[target_col]
    y_val = val[target_col]
    y_test = test[target_col]

    fallback = float(y_train.mean())

    candidates = build_models()

    fitted_models = {
        "Last 10 Baseline": None
    }

    val_predictions = {}
    val_scores = {}

    # --------------------------------------------------------
    # BASELINE
    # --------------------------------------------------------

    baseline_val = baseline_predict(
        val,
        target_col,
        fallback
    )

    val_predictions[
        "Last 10 Baseline"
    ] = baseline_val

    score = evaluate(
        y_val,
        baseline_val
    )

    val_scores[
        "Last 10 Baseline"
    ] = score["MAE"]

    validation_results.append({
        "Target": target_name,
        "Model": "Last 10 Baseline",
        **score
    })

    print(
        "Baseline MAE:",
        round(score["MAE"], 4)
    )

    # --------------------------------------------------------
    # ADDESTRAMENTO MODELLI
    # --------------------------------------------------------

    for name, model in candidates.items():

        model_start = time.time()

        print(f"\nTraining {name}...")

        model.fit(
            X_train,
            y_train
        )

        prediction = np.maximum(
            model.predict(X_val),
            0
        )

        score = evaluate(
            y_val,
            prediction
        )

        fitted_models[name] = model

        val_predictions[name] = prediction

        val_scores[name] = score["MAE"]

        validation_results.append({
            "Target": target_name,
            "Model": name,
            **score
        })

        print(
            f"{name} MAE:",
            round(score["MAE"], 4)
        )

        print(
            "Tempo:",
            round(time.time() - model_start, 1),
            "secondi"
        )

    # ========================================================
    # 8. WEIGHTED ENSEMBLE
    # ========================================================

    ranked = sorted(
        val_scores,
        key=val_scores.get
    )

    top_models = ranked[:3]

    inverse_errors = np.array([
        1 / max(val_scores[name], 1e-8)
        for name in top_models
    ])

    weights = (
        inverse_errors
        / inverse_errors.sum()
    )

    ensemble_val = np.zeros(
        len(val)
    )

    for name, weight in zip(
        top_models,
        weights
    ):

        ensemble_val += (
            weight
            * val_predictions[name]
        )

    ensemble_score = evaluate(
        y_val,
        ensemble_val
    )

    val_scores[
        "Weighted Ensemble"
    ] = ensemble_score["MAE"]

    validation_results.append({
        "Target": target_name,
        "Model": "Weighted Ensemble",
        **ensemble_score
    })

    print(
        "\nEnsemble MAE:",
        round(
            ensemble_score["MAE"],
            4
        )
    )

    # ========================================================
    # 9. SELEZIONE MODELLO
    # SOLO SULLA VALIDATION
    # ========================================================

    winner = min(
        val_scores,
        key=val_scores.get
    )

    print("\nMODELLO SELEZIONATO:", winner)

    # ========================================================
    # 10. REFIT TRAIN + VALIDATION
    # ========================================================

    development = pd.concat(
        [train, val],
        ignore_index=True
    )

    X_dev = development[feature_cols]
    y_dev = development[target_col]

    final_fallback = float(y_dev.mean())

    if winner == "Weighted Ensemble":

        members = top_models

        final_weights = [
            float(w) for w in weights
        ]

    else:

        members = [winner]
        final_weights = [1.0]

    final_models = {}

    test_prediction = np.zeros(
        len(test)
    )

    for name, weight in zip(
        members,
        final_weights
    ):

        if name == "Last 10 Baseline":

            prediction = baseline_predict(
                test,
                target_col,
                final_fallback
            )

            final_models[name] = None

        else:

            model = build_models()[name]

            model.fit(
                X_dev,
                y_dev
            )

            prediction = np.maximum(
                model.predict(X_test),
                0
            )

            final_models[name] = model

        test_prediction += (
            weight * prediction
        )

    # ========================================================
    # 11. TEST FINALE OUT-OF-SAMPLE
    # ========================================================

    test_score = evaluate(
        y_test,
        test_prediction
    )

    baseline_test = baseline_predict(
        test,
        target_col,
        final_fallback
    )

    baseline_score = evaluate(
        y_test,
        baseline_test
    )

    improvement = 100 * (
        baseline_score["MAE"]
        - test_score["MAE"]
    ) / max(
        baseline_score["MAE"],
        1e-8
    )

    test_results.append({
        "Target": target_name,
        "Selected_Model": winner,
        **test_score,
        "Baseline_MAE": baseline_score["MAE"],
        "Improvement_pct": improvement
    })

    selected_models[target_name] = {
        "name": winner,
        "members": members,
        "weights": final_weights,
        "models": final_models,
        "target_column": target_col,
        "baseline_column": f"{target_col}_avg10",
        "fallback": final_fallback
    }

    # ========================================================
    # 12. SALVATAGGIO PREVISIONI TEST
    # ========================================================

    pred_df = test[[
        "game_id",
        "game_date",
        "player_id",
        "player_name",
        "team_name"
    ]].copy()

    pred_df["Target"] = target_name

    pred_df["Actual"] = (
        y_test.to_numpy()
    )

    pred_df["Forecast"] = (
        test_prediction
    )

    pred_df["Baseline"] = (
        baseline_test
    )

    pred_df["Error"] = (
        pred_df["Forecast"]
        - pred_df["Actual"]
    )

    pred_df["Abs_Error"] = (
        pred_df["Error"].abs()
    )

    pred_df["Model"] = winner

    all_predictions.append(pred_df)

    # ========================================================
    # 13. FEATURE IMPORTANCE
    # ========================================================

    for name, model in final_models.items():

        if model is None:
            continue

        estimator = model

        if isinstance(model, Pipeline):

            estimator = model.named_steps[
                "model"
            ]

        if hasattr(
            estimator,
            "feature_importances_"
        ):

            importance = (
                estimator.feature_importances_
            )

            total = np.sum(importance)

            if total > 0:
                importance = importance / total

            if len(importance) != len(feature_cols):
                continue

            for feature, value in zip(
                feature_cols,
                importance
            ):

                importance_rows.append({
                    "Target": target_name,
                    "Model": name,
                    "Feature": feature,
                    "Importance": float(value)
                })

    print("\nTEST MAE:", round(test_score["MAE"], 4))

    print(
        "Miglioramento vs baseline:",
        round(improvement, 2),
        "%"
    )

# ============================================================
# 14. CREAZIONE REPORT
# ============================================================

validation_df = pd.DataFrame(
    validation_results
).sort_values(
    ["Target", "MAE"]
)

test_df = pd.DataFrame(
    test_results
)

predictions = pd.concat(
    all_predictions,
    ignore_index=True
)

importance_df = pd.DataFrame(
    importance_rows,
    columns=[
        "Target",
        "Model",
        "Feature",
        "Importance"
    ]
)

if not importance_df.empty:

    importance_df = importance_df.sort_values(
        ["Target", "Model", "Importance"],
        ascending=[True, True, False]
    )

# ============================================================
# 15. PERFORMANCE PER GIOCATORE
# ============================================================

player_accuracy = predictions.groupby(
    [
        "player_id",
        "player_name",
        "Target"
    ],
    as_index=False
).agg(
    Partite=("game_id", "nunique"),
    MAE=("Abs_Error", "mean"),
    Bias=("Error", "mean"),
    Actual_Mean=("Actual", "mean"),
    Forecast_Mean=("Forecast", "mean")
)

player_accuracy = player_accuracy[
    player_accuracy["Partite"] >= 5
].sort_values(
    ["Target", "MAE"]
)

# ============================================================
# 16. ENSEMBLE WEIGHTS
# ============================================================

weight_rows = []

for target, info in selected_models.items():

    for name, weight in zip(
        info["members"],
        info["weights"]
    ):

        weight_rows.append({
            "Target": target,
            "Selected_Model": info["name"],
            "Member": name,
            "Weight": weight
        })

weights_df = pd.DataFrame(
    weight_rows
)

# ============================================================
# 17. SALVATAGGIO MODELLO V3
# ============================================================

package_v3 = {
    "model_version": "3.0",
    "models": selected_models,
    "features": feature_cols,
    "targets": TARGETS,
    "minimum_history": config.get(
        "minimum_history", 20
    ),
    "training_cutoff": (
        development["game_date"]
        .max()
        .isoformat()
    ),
    "train_period": [
        train_days[0],
        train_days[-1]
    ],
    "validation_period": [
        val_days[0],
        val_days[-1]
    ],
    "test_period": [
        test_days[0],
        test_days[-1]
    ],
    "validation": (
        validation_df.to_dict("records")
    ),
    "test": test_df.to_dict("records"),
    "created_at": (
        pd.Timestamp.now(tz="UTC")
        .isoformat()
    )
}

joblib.dump(
    package_v3,
    MODEL_PATH + ".tmp",
    compress=3
)

os.replace(
    MODEL_PATH + ".tmp",
    MODEL_PATH
)

# ============================================================
# 18. ESPORTAZIONE CSV ED EXCEL
# ============================================================

predictions["game_date"] = (
    predictions["game_date"].astype(str)
)

predictions.to_csv(
    PREDICTIONS_PATH,
    index=False,
    encoding="utf-8-sig"
)

split_report = pd.DataFrame([
    {
        "Dataset": "Training",
        "Rows": len(train),
        "Start": train_days[0],
        "End": train_days[-1]
    },
    {
        "Dataset": "Validation",
        "Rows": len(val),
        "Start": val_days[0],
        "End": val_days[-1]
    },
    {
        "Dataset": "Test",
        "Rows": len(test),
        "Start": test_days[0],
        "End": test_days[-1]
    }
])

with pd.ExcelWriter(
    REPORT_PATH,
    engine="openpyxl"
) as writer:

    sheets = {
        "Validation": validation_df,
        "Test": test_df,
        "Predictions": predictions,
        "Player Accuracy": player_accuracy,
        "Feature Importance": importance_df,
        "Ensemble Weights": weights_df,
        "Temporal Split": split_report
    }

    for name, data in sheets.items():

        data.to_excel(
            writer,
            sheet_name=name,
            index=False
        )

# ============================================================
# 19. VISUALIZZAZIONE
# ============================================================

print("\nVALIDATION RESULTS")

display(
    validation_df.round(4)
)

print("\nFINAL TEST RESULTS")

display(
    test_df.round(4)
)

fig = px.bar(
    validation_df,
    x="Model",
    y="MAE",
    color="Model",
    facet_col="Target",
    facet_col_wrap=2,
    title="NBA V3 — Model Validation",
    template="plotly_dark"
)

fig.update_yaxes(matches=None)

fig.update_layout(
    height=1000,
    showlegend=False
)

fig.show()

# Baseline vs modello selezionato

fig = px.bar(
    test_df,
    x="Target",
    y=["Baseline_MAE", "MAE"],
    barmode="group",
    title="Baseline vs Advanced ML",
    template="plotly_dark"
)

fig.show()

# Miglioramento percentuale

fig = px.bar(
    test_df,
    x="Target",
    y="Improvement_pct",
    color="Improvement_pct",
    title="Model Improvement vs Baseline (%)",
    template="plotly_dark"
)

fig.add_hline(y=0)

fig.show()

# Feature importance per punti

if not importance_df.empty:

    pts_importance = importance_df[
        importance_df["Target"] == "PTS"
    ].copy()

    if not pts_importance.empty:

        pts_importance = (
            pts_importance
            .sort_values(
                "Importance",
                ascending=False
            )
            .groupby("Model")
            .head(15)
        )

        fig = px.bar(
            pts_importance,
            x="Importance",
            y="Feature",
            color="Model",
            orientation="h",
            barmode="group",
            title="PTS — Feature Importance",
            template="plotly_dark"
        )

        fig.update_layout(height=700)

        fig.show()

# ============================================================
# 20. RIEPILOGO FINALE
# ============================================================

elapsed = (
    time.time() - START_TIME
) / 60

print("\n" + "=" * 55)
print("STEP 16 COMPLETATO!")
print("=" * 55)

print("Versione modello: 3.0")
print("Feature:", len(feature_cols))
print("Training rows:", len(train))
print("Validation rows:", len(val))
print("Test rows:", len(test))

print(
    "Tempo totale:",
    round(elapsed, 2),
    "minuti"
)

print("\nMODELLI SELEZIONATI")

for target, info in selected_models.items():

    print(
        target,
        "->",
        info["name"]
    )

print("\nFILE SALVATI")

print(MODEL_PATH)
print(REPORT_PATH)
print(PREDICTIONS_PATH)

print("\nPRONTO PER STEP 17!")


In [ ]:

# ============================================================
# STEP 16.0 — RIPRISTINO AUTOMATICO AMBIENTE
# ============================================================

import os
from google.colab import drive

mounts = [
    "/content/drive",
    "/content/gdrive_nba",
    "/content/nba_drive"
]

BASE_DIR = None

for mount in mounts:

    if not os.path.ismount(mount):

        if os.path.isdir(mount) and os.listdir(mount):
            continue

        drive.mount(mount)

    candidate = os.path.join(
        mount,
        "MyDrive",
        "NBA_Data_Science"
    )

    if os.path.isfile(
        os.path.join(candidate, "nba_database.db")
    ):
        BASE_DIR = candidate
        break

if BASE_DIR is None:
    raise FileNotFoundError(
        "Cartella NBA_Data_Science non trovata."
    )

print("Google Drive collegato.")
print("Cartella progetto:", BASE_DIR)

required_files = [
    "nba_database.db",
    "NBA_Advanced_ML_Features.csv",
    "NBA_Advanced_Feature_List.json"
]

missing = []

print("\nCONTROLLO FILE")
print("=" * 50)

for filename in required_files:

    path = os.path.join(BASE_DIR, filename)

    if os.path.isfile(path):

        size = os.path.getsize(path) / 1024**2

        print(f"OK - {filename} ({size:.2f} MB)")

    else:

        missing.append(filename)
        print(f"MANCANTE - {filename}")

print("=" * 50)

if missing:
    print("Riesegui STEP 15.")
else:
    print("AMBIENTE RIPRISTINATO!")
    print("PRONTO PER STEP 16!")


In [ ]:
import os
import joblib
import pandas as pd

model_path = os.path.join(
    BASE_DIR, "nba_player_models_v3.joblib"
)

if not os.path.isfile(model_path):
    raise FileNotFoundError(
        "STEP 16 non ancora completato."
    )

package_v3 = joblib.load(model_path)

print("MODELLO V3 CARICATO")
print("Versione:", package_v3["model_version"])
print("Numero feature:", len(package_v3["features"]))

print("\nMODELLI SELEZIONATI")

for target, info in package_v3["models"].items():
    print(target, "->", info["name"])

print("\nRISULTATI REALI DEL TEST")

display(
    pd.DataFrame(package_v3["test"]).round(3)
)

print("\nPRONTO PER STEP 17!")

In [ ]:

# ============================================================
# STEP 17 — ADVANCED FORECAST ENGINE V3
# NBA DATA SCIENCE ANALYTICS
# ============================================================

import os
import sqlite3
import joblib
import numpy as np
import pandas as pd
import plotly.express as px

from google.colab import drive
from IPython.display import display

# ============================================================
# 1. RIPRISTINO AUTOMATICO AMBIENTE
# ============================================================

mounts = [
    "/content/drive",
    "/content/gdrive_nba",
    "/content/nba_drive"
]

BASE_DIR = None

for mount in mounts:

    candidate = os.path.join(
        mount, "MyDrive", "NBA_Data_Science"
    )

    if os.path.ismount(mount):

        if os.path.isfile(
            os.path.join(candidate, "nba_database.db")
        ):
            BASE_DIR = candidate
            break

if BASE_DIR is None:

    for mount in mounts:

        if os.path.ismount(mount):
            continue

        if os.path.isdir(mount) and os.listdir(mount):
            continue

        drive.mount(mount)

        candidate = os.path.join(
            mount, "MyDrive", "NBA_Data_Science"
        )

        if os.path.isfile(
            os.path.join(candidate, "nba_database.db")
        ):
            BASE_DIR = candidate
            break

if BASE_DIR is None:
    raise FileNotFoundError(
        "Cartella NBA_Data_Science non trovata."
    )

DB_PATH = os.path.join(
    BASE_DIR, "nba_database.db"
)

MODEL_PATH = os.path.join(
    BASE_DIR, "nba_player_models_v3.joblib"
)

CSV_PATH = os.path.join(
    BASE_DIR, "NBA_Forecast_Engine_V3.csv"
)

EXCEL_PATH = os.path.join(
    BASE_DIR, "NBA_Forecast_Engine_V3.xlsx"
)

ARCHIVE_DIR = os.path.join(
    BASE_DIR, "Forecast_Archive_V3"
)

os.makedirs(ARCHIVE_DIR, exist_ok=True)

if not os.path.isfile(MODEL_PATH):
    raise FileNotFoundError(
        "Modello V3 mancante. Completa STEP 16."
    )

NOW = pd.Timestamp.now(tz="UTC")

# ============================================================
# 2. CARICAMENTO MODELLO V3
# ============================================================

package_v3 = joblib.load(MODEL_PATH)

feature_cols = package_v3["features"]
models = package_v3["models"]

MIN_HISTORY = package_v3.get(
    "minimum_history", 20
)

print("Modello:", package_v3["model_version"])
print("Feature richieste:", len(feature_cols))

# ============================================================
# 3. CARICAMENTO DATABASE
# ============================================================

with sqlite3.connect(DB_PATH) as conn:

    box = pd.read_sql_query(
        "SELECT * FROM espn_player_boxscores",
        conn
    )

    games = pd.read_sql_query(
        "SELECT * FROM espn_games",
        conn
    )

for df in [box, games]:

    df["game_id"] = df["game_id"].astype(str)

    df["game_date"] = pd.to_datetime(
        df["game_date"],
        utc=True,
        errors="coerce",
        format="mixed"
    )

for c in ["player_id", "team_id"]:
    box[c] = box[c].astype(str)

for c in ["home_id", "away_id"]:
    games[c] = games[c].astype(str)

numeric = [
    "minutes", "pts", "reb", "ast",
    "fg3m", "fgm", "fga", "fta",
    "oreb", "tov", "did_not_play"
]

for c in numeric:

    box[c] = pd.to_numeric(
        box[c], errors="coerce"
    )

for c in [
    "home_score", "away_score", "completed"
]:

    games[c] = pd.to_numeric(
        games[c], errors="coerce"
    )

games = games.drop_duplicates("game_id")

completed = games[
    (games["completed"] == 1)
    & (games["game_date"] < NOW)
].dropna(
    subset=["home_score", "away_score"]
).copy()

box = box[
    (box["minutes"] > 0)
    & (box["did_not_play"] == 0)
    & box["game_id"].isin(completed["game_id"])
    & (box["game_date"] < NOW)
].dropna(
    subset=["pts", "reb", "ast", "fg3m"]
).drop_duplicates(
    ["game_id", "player_id"]
).copy()

# ============================================================
# 4. STATISTICHE AVANZATE
# ============================================================

def safe_divide(a, b):

    return a / b.replace(0, np.nan)

box["PRA"] = (
    box["pts"] + box["reb"] + box["ast"]
)

box["TS_pct"] = 100 * safe_divide(
    box["pts"],
    2 * (box["fga"] + 0.44 * box["fta"])
)

box["eFG_pct"] = 100 * safe_divide(
    box["fgm"] + 0.5 * box["fg3m"],
    box["fga"]
)

for stat, name in [
    ("pts", "PTS"),
    ("reb", "REB"),
    ("ast", "AST"),
    ("PRA", "PRA")
]:

    box[f"{name}_per_min"] = safe_divide(
        box[stat], box["minutes"]
    )

box["Offensive_Load"] = (
    box["fga"]
    + 0.44 * box["fta"]
    + box["tov"]
)

# ============================================================
# 5. TEAM ANALYTICS
# ============================================================

team_stats = box.groupby(
    ["game_id", "team_id"],
    as_index=False
).agg(
    Team_PTS=("pts", "sum"),
    Team_FGA=("fga", lambda s: s.sum(min_count=1)),
    Team_FTA=("fta", lambda s: s.sum(min_count=1)),
    Team_OREB=("oreb", lambda s: s.sum(min_count=1)),
    Team_TOV=("tov", lambda s: s.sum(min_count=1)),
    Team_MIN=("minutes", "sum")
)

team_stats["Possessions"] = (
    team_stats["Team_FGA"]
    + 0.44 * team_stats["Team_FTA"]
    - team_stats["Team_OREB"]
    + team_stats["Team_TOV"]
)

team_stats["Possessions"] = (
    team_stats["Possessions"]
    .where(team_stats["Possessions"] > 0)
)

team_stats["Offensive_Rating"] = (
    100 * safe_divide(
        team_stats["Team_PTS"],
        team_stats["Possessions"]
    )
)

team_stats["Team_Offensive_Load"] = (
    team_stats["Team_FGA"]
    + 0.44 * team_stats["Team_FTA"]
    + team_stats["Team_TOV"]
)

context = completed[[
    "game_id", "home_id", "away_id",
    "home_score", "away_score"
]]

box = box.merge(
    context,
    on="game_id",
    how="inner"
)

box = box[
    (box["team_id"] == box["home_id"])
    | (box["team_id"] == box["away_id"])
].copy()

box["home"] = (
    box["team_id"] == box["home_id"]
).astype(int)

box["opponent_id"] = np.where(
    box["home"] == 1,
    box["away_id"],
    box["home_id"]
)

box["Team_Score"] = np.where(
    box["home"] == 1,
    box["home_score"],
    box["away_score"]
)

box["Opponent_Score"] = np.where(
    box["home"] == 1,
    box["away_score"],
    box["home_score"]
)

box = box.merge(
    team_stats,
    on=["game_id", "team_id"],
    how="left"
)

opp_stats = team_stats[[
    "game_id", "team_id", "Possessions"
]].rename(columns={
    "team_id": "opponent_id",
    "Possessions": "Opponent_Possessions"
})

box = box.merge(
    opp_stats,
    on=["game_id", "opponent_id"],
    how="left"
)

box["Defensive_Rating"] = (
    100 * safe_divide(
        box["Opponent_Score"],
        box["Opponent_Possessions"]
    )
)

box["Pace"] = safe_divide(
    48 * (
        box["Possessions"]
        + box["Opponent_Possessions"]
    ) / 2,
    box["Team_MIN"] / 5
)

box["Usage_Proxy"] = (
    100 * safe_divide(
        box["Offensive_Load"]
        * (box["Team_MIN"] / 5),
        box["minutes"]
        * box["Team_Offensive_Load"]
    )
)

box = box.sort_values(
    ["game_date", "game_id"]
)

# ============================================================
# 6. STORICO GIOCATORI E SQUADRE
# ============================================================

histories = {
    pid: group.copy()
    for pid, group in box.groupby("player_id")
}

latest_team = box.drop_duplicates(
    "player_id",
    keep="last"
)

latest_team = latest_team[
    latest_team["game_date"]
    >= NOW - pd.Timedelta(days=90)
]

team_context = box[[
    "game_id",
    "game_date",
    "team_id",
    "Team_Score",
    "Opponent_Score",
    "Offensive_Rating",
    "Defensive_Rating",
    "Pace"
]].drop_duplicates(
    ["game_id", "team_id"]
).sort_values("game_date")

team_histories = {
    tid: group.copy()
    for tid, group in team_context.groupby("team_id")
}

# ============================================================
# 7. PARTITE FUTURE
# ============================================================

future = games[
    (games["completed"] == 0)
    & (games["status"] == "STATUS_SCHEDULED")
    & (games["game_date"] > NOW)
    & (
        games["game_date"]
        <= NOW + pd.Timedelta(days=3)
    )
].sort_values("game_date")

if future.empty:

    raise RuntimeError(
        "Nessuna partita futura. Esegui STEP 12."
    )

print("Partite future:", len(future))

# ============================================================
# 8. FEATURE ENGINEERING V3
# ============================================================

rolling_stats = [
    "pts", "reb", "ast", "fg3m",
    "PRA", "minutes",
    "TS_pct", "eFG_pct",
    "PTS_per_min",
    "REB_per_min",
    "AST_per_min",
    "PRA_per_min",
    "Usage_Proxy",
    "Pace",
    "Offensive_Rating",
    "Defensive_Rating"
]

momentum_stats = [
    "pts", "reb", "ast",
    "fg3m", "PRA", "minutes"
]

team_features = [
    "Team_Score",
    "Opponent_Score",
    "Offensive_Rating",
    "Defensive_Rating",
    "Pace"
]

def scalar_divide(a, b):

    if pd.isna(b) or b == 0:
        return np.nan

    return a / b


def build_advanced_features(
    history, opponent_id, date, home
):

    history = history[
        history["game_date"] < NOW
    ].sort_values("game_date")

    if len(history) < MIN_HISTORY:
        return None

    row = {}

    for stat in rolling_stats:

        for window in [5, 10, 20]:

            values = history[stat].tail(window)

            row[f"{stat}_avg{window}"] = (
                values.mean()
                if values.count() >= window
                else np.nan
            )

        values = history[stat].tail(10)

        row[f"{stat}_std10"] = (
            values.std(ddof=1)
            if values.count() >= 10
            else np.nan
        )

        row[f"{stat}_last1"] = (
            history[stat].iloc[-1]
        )

    # Momentum e consistency

    for stat in momentum_stats:

        momentum = (
            row[f"{stat}_avg5"]
            - row[f"{stat}_avg20"]
        )

        variation = scalar_divide(
            row[f"{stat}_std10"],
            row[f"{stat}_avg10"]
        )

        row[f"{stat}_momentum"] = momentum

        row[f"{stat}_momentum_pct"] = (
            100 * scalar_divide(
                momentum,
                row[f"{stat}_avg20"]
            )
        )

        row[f"{stat}_variation"] = variation

        row[f"{stat}_consistency"] = (
            100 / (1 + variation)
        )

    # Riposo e fattore campo

    game_day = date.tz_convert(
        "America/New_York"
    ).date()

    last_day = history.iloc[-1][
        "game_date"
    ].tz_convert(
        "America/New_York"
    ).date()

    days = (game_day - last_day).days

    row["home"] = int(home)

    row["Rest_Days"] = np.clip(
        days - 1, 0, 30
    )

    row["Back_to_Back"] = int(days == 1)

    row["Games_Previous"] = len(history)

    # Performance casa / trasferta

    venue = history[
        history["home"] == home
    ].tail(10)

    for stat in ["pts", "reb", "ast", "PRA"]:

        row[f"{stat}_venue_avg10"] = (
            venue[stat].mean()
            if venue[stat].count() >= 3
            else np.nan
        )

    # Contesto avversario

    opponent = team_histories.get(opponent_id)

    for stat in team_features:

        key = f"Opp_pre_{stat}_avg10"

        if opponent is None:

            row[key] = np.nan

        else:

            values = opponent[stat].tail(10)

            row[key] = (
                values.mean()
                if values.count() >= 5
                else np.nan
            )

    return row

# ============================================================
# 9. GENERAZIONE DATASET FUTURO
# ============================================================

rows = []

for game in future.itertuples(index=False):

    matchups = [
        (
            game.home_id,
            game.home_team,
            game.away_id,
            game.away_team,
            1
        ),
        (
            game.away_id,
            game.away_team,
            game.home_id,
            game.home_team,
            0
        )
    ]

    for tid, team, oid, opponent, home in matchups:

        roster = latest_team[
            latest_team["team_id"] == tid
        ]

        for player in roster.itertuples(index=False):

            features = build_advanced_features(
                histories[player.player_id],
                oid,
                game.game_date,
                home
            )

            if features is None:
                continue

            rows.append({
                "game_id": game.game_id,
                "player_id": player.player_id,
                "date": game.game_date,
                "Giocatore": player.player_name,
                "Squadra": team,
                "Avversario": opponent,
                "Sede": "Casa" if home else "Trasferta",
                **features
            })

forecast_v3 = pd.DataFrame(rows)

if forecast_v3.empty:
    raise ValueError("Nessun giocatore idoneo.")

missing = [
    c for c in feature_cols
    if c not in forecast_v3.columns
]

if missing:

    raise ValueError(
        f"Feature incompatibili: {missing}"
    )

X = forecast_v3[feature_cols].replace(
    [np.inf, -np.inf],
    np.nan
)

print("Player-game forecast:", len(X))
print("Feature disponibili:", len(X.columns))

# ============================================================
# 10. MACHINE LEARNING ED ENSEMBLE
# ============================================================

for target, info in models.items():

    prediction = np.zeros(
        len(forecast_v3)
    )

    for name, weight in zip(
        info["members"],
        info["weights"]
    ):

        if name == "Last 10 Baseline":

            member_prediction = (
                forecast_v3[info["baseline_column"]]
                .fillna(info["fallback"])
                .to_numpy(dtype=float)
            )

        else:

            model = info["models"][name]

            member_prediction = model.predict(X)

        prediction += (
            weight * np.maximum(
                member_prediction, 0
            )
        )

    forecast_v3[f"Forecast_{target}"] = prediction

    forecast_v3[f"Model_{target}"] = info["name"]

# Manteniamo il PRA previsto dal modello V3.
# Aggiungiamo anche la somma delle previsioni individuali.

forecast_v3["PRA_Sum"] = (
    forecast_v3["Forecast_PTS"]
    + forecast_v3["Forecast_REB"]
    + forecast_v3["Forecast_AST"]
)

# ============================================================
# 11. CONTROLLO QUALITA'
# ============================================================

TARGETS = ["PTS", "REB", "AST", "3PM", "PRA"]

generated_at = pd.Timestamp.now(tz="UTC")

forecast_v3["generated_at"] = generated_at

forecast_v3 = forecast_v3[
    forecast_v3["date"] > generated_at
].copy()

forecast_v3 = forecast_v3.replace(
    [np.inf, -np.inf],
    np.nan
).dropna(
    subset=[
        f"Forecast_{t}"
        for t in TARGETS
    ]
).drop_duplicates(
    ["game_id", "player_id"]
)

if forecast_v3.empty:
    raise ValueError("Nessun forecast valido.")

forecast_v3["Data Italia"] = (
    forecast_v3["date"]
    .dt.tz_convert("Europe/Rome")
    .dt.strftime("%d/%m/%Y %H:%M")
)

forecast_v3["Model_Version"] = "3.0"

forecast_v3["Roster_Status"] = (
    "Stimato - disponibilità non verificata"
)

# ============================================================
# 12. ARCHIVIAZIONE SQLITE SEPARATA V3
# ============================================================

with sqlite3.connect(DB_PATH) as conn:

    conn.execute("""
        CREATE TABLE IF NOT EXISTS nba_forecast_history_v3 (
            game_id TEXT NOT NULL,
            player_id TEXT NOT NULL,
            generated_at TEXT NOT NULL,
            game_date TEXT NOT NULL,
            player_name TEXT,
            team_name TEXT,
            opponent TEXT,
            forecast_pts REAL,
            forecast_reb REAL,
            forecast_ast REAL,
            forecast_3pm REAL,
            forecast_pra REAL,
            model_version TEXT,
            PRIMARY KEY (
                game_id, player_id, generated_at
            )
        )
    """)

    records = []

    for _, r in forecast_v3.iterrows():

        records.append((
            str(r["game_id"]),
            str(r["player_id"]),
            r["generated_at"].isoformat(),
            r["date"].isoformat(),
            r["Giocatore"],
            r["Squadra"],
            r["Avversario"],
            float(r["Forecast_PTS"]),
            float(r["Forecast_REB"]),
            float(r["Forecast_AST"]),
            float(r["Forecast_3PM"]),
            float(r["Forecast_PRA"]),
            "3.0"
        ))

    conn.executemany("""
        INSERT OR IGNORE INTO nba_forecast_history_v3
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, records)

    total_history = conn.execute("""
        SELECT COUNT(*)
        FROM nba_forecast_history_v3
    """).fetchone()[0]

# ============================================================
# 13. ESPORTAZIONE FORECAST
# ============================================================

columns = [
    "game_id",
    "player_id",
    "date",
    "generated_at",
    "Data Italia",
    "Giocatore",
    "Squadra",
    "Avversario",
    "Sede",
    "Forecast_PTS",
    "Forecast_REB",
    "Forecast_AST",
    "Forecast_3PM",
    "Forecast_PRA",
    "PRA_Sum",
    "minutes_avg10",
    "pts_avg10",
    "pts_momentum",
    "pts_consistency",
    "Usage_Proxy_avg10",
    "TS_pct_avg10",
    "Rest_Days",
    "Back_to_Back",
    "Model_Version",
    "Roster_Status"
]

columns += [
    f"Model_{t}"
    for t in TARGETS
]

export_v3 = forecast_v3[columns].copy()

export_v3 = export_v3.sort_values(
    ["date", "Forecast_PTS"],
    ascending=[True, False]
)

for c in ["date", "generated_at"]:

    export_v3[c] = export_v3[c].map(
        lambda x: x.isoformat()
    )

stamp = generated_at.strftime(
    "%Y%m%d_%H%M%S_%f"
)

archive_path = os.path.join(
    ARCHIVE_DIR,
    f"NBA_V3_{stamp}.csv"
)

export_v3.to_csv(
    archive_path,
    index=False,
    encoding="utf-8-sig"
)

export_v3.to_csv(
    CSV_PATH + ".tmp",
    index=False,
    encoding="utf-8-sig"
)

with pd.ExcelWriter(
    EXCEL_PATH + ".tmp.xlsx",
    engine="openpyxl"
) as writer:

    export_v3.to_excel(
        writer,
        sheet_name="Player Forecast",
        index=False
    )

    pd.DataFrame(
        package_v3["test"]
    ).to_excel(
        writer,
        sheet_name="Model Accuracy",
        index=False
    )

os.replace(CSV_PATH + ".tmp", CSV_PATH)

os.replace(
    EXCEL_PATH + ".tmp.xlsx",
    EXCEL_PATH
)

# ============================================================
# 14. CONFRONTO FORECAST V2 VS V3
# ============================================================

V2_PATH = os.path.join(
    BASE_DIR, "NBA_Forecast_Engine_V2.csv"
)

if os.path.isfile(V2_PATH):

    v2 = pd.read_csv(
        V2_PATH,
        dtype={
            "game_id": str,
            "player_id": str
        }
    )

    keys = ["game_id", "player_id"]

    if all(c in v2.columns for c in keys):

        common = v2.merge(
            export_v3,
            on=keys,
            suffixes=("_V2", "_V3")
        )

        for target in TARGETS:

            common[f"Delta_{target}"] = (
                common[f"Forecast_{target}_V3"]
                - common[f"Forecast_{target}_V2"]
            )

        comparison_path = os.path.join(
            BASE_DIR,
            "NBA_V2_vs_V3_Forecasts.csv"
        )

        common.to_csv(
            comparison_path,
            index=False
        )

        print(
            "Forecast confrontabili V2/V3:",
            len(common)
        )

# ============================================================
# 15. VISUALIZZAZIONE
# ============================================================

print("\nTOP 20 — POINTS FORECAST")

top_pts = export_v3.sort_values(
    "Forecast_PTS",
    ascending=False
).head(20)

display(
    top_pts[[
        "Giocatore",
        "Squadra",
        "Avversario",
        "Forecast_PTS",
        "Forecast_REB",
        "Forecast_AST",
        "Forecast_3PM",
        "Forecast_PRA"
    ]].round(2)
)

fig = px.bar(
    top_pts,
    x="Giocatore",
    y="Forecast_PTS",
    color="Squadra",
    hover_data=[
        "Avversario",
        "Data Italia",
        "Forecast_REB",
        "Forecast_AST",
        "Forecast_PRA"
    ],
    title="NBA V3 — Top Points Forecast",
    template="plotly_dark"
)

fig.update_layout(
    height=550,
    xaxis_tickangle=-45
)

fig.show()

top_pra = export_v3.sort_values(
    "Forecast_PRA",
    ascending=False
).head(20)

fig = px.bar(
    top_pra,
    x="Giocatore",
    y="Forecast_PRA",
    color="Squadra",
    title="NBA V3 — Top PRA Forecast",
    template="plotly_dark"
)

fig.show()

# ============================================================
# 16. RIEPILOGO FINALE
# ============================================================

print("\n" + "=" * 55)
print("STEP 17 COMPLETATO!")
print("=" * 55)

print("Versione: V3.0")
print("Partite future:", export_v3["game_id"].nunique())
print("Giocatori:", export_v3["player_id"].nunique())
print("Forecast generati:", len(export_v3))
print("Forecast storici V3:", total_history)
print("Feature ML:", len(feature_cols))

print("\nMODELLI UTILIZZATI")

for target, info in models.items():

    print(target, "->", info["name"])

print("\nFILE SALVATI")

print(CSV_PATH)
print(EXCEL_PATH)
print(archive_path)

print("\nPRONTO PER STEP 18!")


In [ ]:

# ============================================================
# STEP 18 — PROBABILITY DISTRIBUTION ENGINE
# NBA DATA SCIENCE ANALYTICS
# ============================================================

import os
import sqlite3
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from google.colab import drive
from IPython.display import display

# ============================================================
# 1. RIPRISTINO AUTOMATICO AMBIENTE
# ============================================================

mounts = [
    "/content/drive",
    "/content/gdrive_nba",
    "/content/nba_drive"
]

BASE_DIR = None

for mount in mounts:

    candidate = os.path.join(
        mount, "MyDrive", "NBA_Data_Science"
    )

    if os.path.isfile(
        os.path.join(candidate, "nba_database.db")
    ):
        BASE_DIR = candidate
        break

if BASE_DIR is None:

    for mount in mounts:

        if os.path.isdir(mount) and os.listdir(mount):
            continue

        drive.mount(mount)

        candidate = os.path.join(
            mount, "MyDrive", "NBA_Data_Science"
        )

        if os.path.isfile(
            os.path.join(candidate, "nba_database.db")
        ):
            BASE_DIR = candidate
            break

if BASE_DIR is None:
    raise FileNotFoundError(
        "Cartella NBA_Data_Science non trovata."
    )

# ============================================================
# 2. CONFIGURAZIONE
# ============================================================

HISTORY_PATH = os.path.join(
    BASE_DIR,
    "NBA_Advanced_Test_Predictions.csv"
)

FORECAST_PATH = os.path.join(
    BASE_DIR,
    "NBA_Forecast_Engine_V3.csv"
)

OUTPUT_PATH = os.path.join(
    BASE_DIR,
    "NBA_Probability_Engine_V3.xlsx"
)

CSV_PATH = os.path.join(
    BASE_DIR,
    "NBA_Probability_Engine_V3.csv"
)

DB_PATH = os.path.join(
    BASE_DIR,
    "nba_database.db"
)

ARCHIVE_DIR = os.path.join(
    BASE_DIR,
    "Probability_Archive_V3"
)

os.makedirs(ARCHIVE_DIR, exist_ok=True)

TARGETS = [
    "PTS", "REB", "AST", "3PM", "PRA"
]

THRESHOLDS = {
    "PTS": [5, 10, 15, 20, 25, 30, 35, 40, 45, 50],
    "REB": [2, 4, 6, 8, 10, 12, 15, 20],
    "AST": [2, 4, 6, 8, 10, 12, 15],
    "3PM": [1, 2, 3, 4, 5, 6, 7],
    "PRA": [10, 15, 20, 25, 30, 35, 40, 45, 50, 60]
}

INTERVALS = [50, 80, 90, 95]

MIN_PLAYER_SAMPLE = 10
MIN_GROUP_SAMPLE = 50

NOW = pd.Timestamp.now(tz="UTC")

for path in [HISTORY_PATH, FORECAST_PATH]:

    if not os.path.isfile(path):
        raise FileNotFoundError(path)

# ============================================================
# 3. CARICAMENTO DATI
# ============================================================

history = pd.read_csv(
    HISTORY_PATH,
    dtype={
        "game_id": str,
        "player_id": str
    }
)

forecast = pd.read_csv(
    FORECAST_PATH,
    dtype={
        "game_id": str,
        "player_id": str
    }
)

history["game_date"] = pd.to_datetime(
    history["game_date"],
    utc=True,
    errors="coerce",
    format="mixed"
)

forecast["date"] = pd.to_datetime(
    forecast["date"],
    utc=True,
    errors="coerce",
    format="mixed"
)

for col in ["Actual", "Forecast"]:

    history[col] = pd.to_numeric(
        history[col],
        errors="coerce"
    )

history = history.replace(
    [np.inf, -np.inf],
    np.nan
).dropna(
    subset=[
        "game_date",
        "Actual",
        "Forecast",
        "player_id",
        "Target"
    ]
)

history = history[
    history["game_date"] < NOW
].drop_duplicates(
    ["game_id", "player_id", "Target"]
).sort_values(
    "game_date"
)

forecast = forecast[
    forecast["date"] > NOW
].drop_duplicates(
    ["game_id", "player_id"]
).copy()

if history.empty:
    raise ValueError("Storico errori vuoto.")

if forecast.empty:
    raise ValueError(
        "Forecast scaduti. Riesegui STEP 17."
    )

# Residuo = risultato reale - previsione.

history["Residual"] = (
    history["Actual"]
    - history["Forecast"]
)

print("Errori storici:", len(history))
print("Forecast futuri:", len(forecast))

# ============================================================
# 4. SUDDIVISIONE TEMPORALE
# ============================================================

history["NBA_Day"] = (
    history["game_date"]
    .dt.tz_convert("America/New_York")
    .dt.strftime("%Y-%m-%d")
)

days = sorted(
    history["NBA_Day"].unique()
)

if len(days) < 10:
    raise ValueError(
        "Storico insufficiente per la calibrazione."
    )

split = int(len(days) * 0.60)

calibration = history[
    history["NBA_Day"].isin(days[:split])
].copy()

evaluation = history[
    history["NBA_Day"].isin(days[split:])
].copy()

print("\nCALIBRATION:", len(calibration))
print("EVALUATION:", len(evaluation))

# ============================================================
# 5. GRUPPI DI LIVELLO STATISTICO
# ============================================================

BINS = {
    "PTS": [0, 5, 10, 15, 20, 25, 30, 40, np.inf],
    "REB": [0, 2, 4, 6, 8, 10, 15, np.inf],
    "AST": [0, 2, 4, 6, 8, 10, 15, np.inf],
    "3PM": [0, 1, 2, 3, 4, 6, np.inf],
    "PRA": [0, 10, 20, 30, 40, 50, 60, np.inf]
}

def get_bucket(target, prediction):

    edges = BINS[target]

    return int(
        np.clip(
            np.searchsorted(
                edges,
                prediction,
                side="right"
            ) - 1,
            0,
            len(edges) - 2
        )
    )

# ============================================================
# 6. COSTRUZIONE DISTRIBUZIONI EMPIRICHE
# ============================================================

def build_error_pools(data):

    pools = {
        "global": {},
        "group": {},
        "player": {}
    }

    temp = data[
        data["Target"].isin(TARGETS)
    ].copy()

    temp["Bucket"] = [
        get_bucket(t, p)
        for t, p in zip(
            temp["Target"],
            temp["Forecast"]
        )
    ]

    for target, g in temp.groupby("Target"):

        pools["global"][target] = (
            g["Residual"].to_numpy(dtype=float)
        )

    for key, g in temp.groupby(
        ["Target", "Bucket"]
    ):

        pools["group"][key] = (
            g["Residual"].to_numpy(dtype=float)
        )

    for key, g in temp.groupby(
        ["Target", "player_id"]
    ):

        pools["player"][key] = (
            g["Residual"].to_numpy(dtype=float)
        )

    return pools

# ============================================================
# 7. DISTRIBUZIONE PERSONALIZZATA
# ============================================================

def get_distribution(
    pools,
    target,
    player_id,
    prediction
):

    global_errors = pools["global"].get(
        target,
        np.array([])
    )

    if len(global_errors) == 0:
        raise ValueError(
            f"Errori storici mancanti: {target}"
        )

    bucket = get_bucket(
        target, prediction
    )

    group_errors = pools["group"].get(
        (target, bucket),
        np.array([])
    )

    player_errors = pools["player"].get(
        (target, str(player_id)),
        np.array([])
    )

    if len(group_errors) >= MIN_GROUP_SAMPLE:

        base_errors = group_errors
        source = "Forecast Group"

    else:

        base_errors = global_errors
        source = "Global"

    # Shrinkage:
    # Combiniamo gli errori individuali
    # con quelli del gruppo statistico.

    if len(player_errors) >= MIN_PLAYER_SAMPLE:

        alpha = len(player_errors) / (
            len(player_errors) + 30
        )

        errors = np.concatenate([
            player_errors,
            base_errors
        ])

        weights = np.concatenate([
            np.full(
                len(player_errors),
                alpha / len(player_errors)
            ),
            np.full(
                len(base_errors),
                (1 - alpha) / len(base_errors)
            )
        ])

        source = "Player + " + source

    else:

        errors = base_errors

        weights = np.full(
            len(errors),
            1 / len(errors)
        )

    # Le statistiche NBA sono intere
    # e non possono essere negative.

    outcomes = np.maximum(
        np.floor(prediction + errors + 0.5),
        0
    )

    order = np.argsort(outcomes)

    outcomes = outcomes[order]
    weights = weights[order]

    weights = weights / weights.sum()

    return (
        outcomes,
        weights,
        source,
        len(player_errors),
        len(base_errors)
    )

# ============================================================
# 8. FUNZIONI STATISTICHE
# ============================================================

def weighted_quantile(
    values,
    weights,
    q
):

    cumulative = np.cumsum(weights)

    index = np.searchsorted(
        cumulative,
        q,
        side="left"
    )

    index = min(
        index,
        len(values) - 1
    )

    return float(values[index])


def probability_at_least(
    values,
    weights,
    threshold
):

    return float(
        weights[values >= threshold].sum()
    )


def prediction_interval(
    values,
    weights,
    level
):

    alpha = 1 - level / 100

    lower = weighted_quantile(
        values, weights, alpha / 2
    )

    upper = weighted_quantile(
        values, weights, 1 - alpha / 2
    )

    return lower, upper

# ============================================================
# 9. BACKTEST PROBABILISTICO
# ============================================================

calibration_pools = build_error_pools(
    calibration
)

coverage_rows = []
probability_rows = []

print("\nBACKTEST PROBABILISTICO...")

for r in evaluation.itertuples(index=False):

    target = r.Target

    if target not in TARGETS:
        continue

    values, weights, source, n_player, n_base = (
        get_distribution(
            calibration_pools,
            target,
            r.player_id,
            r.Forecast
        )
    )

    for level in INTERVALS:

        lower, upper = prediction_interval(
            values, weights, level
        )

        coverage_rows.append({
            "Target": target,
            "Level": level,
            "Covered": int(
                lower <= r.Actual <= upper
            ),
            "Width": upper - lower
        })

    for threshold in THRESHOLDS[target]:

        p = probability_at_least(
            values,
            weights,
            threshold
        )

        actual_event = int(
            r.Actual >= threshold
        )

        probability_rows.append({
            "Target": target,
            "Threshold": threshold,
            "Probability": p,
            "Actual_Event": actual_event,
            "Brier_Error": (
                p - actual_event
            ) ** 2
        })

coverage = pd.DataFrame(coverage_rows)

coverage_report = coverage.groupby(
    ["Target", "Level"],
    as_index=False
).agg(
    Observed_Coverage=("Covered", "mean"),
    Average_Width=("Width", "mean"),
    Sample=("Covered", "size")
)

coverage_report["Observed_Coverage"] *= 100

coverage_report["Coverage_Error"] = (
    coverage_report["Observed_Coverage"]
    - coverage_report["Level"]
)

probability_backtest = pd.DataFrame(
    probability_rows
)

brier_report = probability_backtest.groupby(
    ["Target", "Threshold"],
    as_index=False
).agg(
    Brier_Score=("Brier_Error", "mean"),
    Predicted_Probability=("Probability", "mean"),
    Actual_Frequency=("Actual_Event", "mean"),
    Sample=("Actual_Event", "size")
)

# ============================================================
# 10. RELIABILITY DIAGRAM
# ============================================================

probability_backtest["Probability_Bin"] = pd.cut(
    probability_backtest["Probability"],
    bins=np.linspace(0, 1, 11),
    include_lowest=True
)

reliability = probability_backtest.groupby(
    ["Target", "Probability_Bin"],
    observed=True,
    as_index=False
).agg(
    Mean_Probability=("Probability", "mean"),
    Actual_Frequency=("Actual_Event", "mean"),
    Sample=("Actual_Event", "size")
)

reliability["Calibration_Error"] = (
    reliability["Mean_Probability"]
    - reliability["Actual_Frequency"]
).abs()

reliability["Probability_Bin"] = (
    reliability["Probability_Bin"].astype(str)
)

ece_rows = []

for target, g in reliability.groupby("Target"):

    ece = np.average(
        g["Calibration_Error"],
        weights=g["Sample"]
    )

    ece_rows.append({
        "Target": target,
        "ECE_pct": 100 * ece,
        "Sample": g["Sample"].sum()
    })

ece_report = pd.DataFrame(ece_rows)

# ============================================================
# 11. DISTRIBUZIONI PER LE PROSSIME PARTITE
# ============================================================

# Dopo il backtest utilizziamo tutto
# lo storico disponibile per le previsioni future.

final_pools = build_error_pools(history)

summary_rows = []
threshold_rows = []

print("\nGENERAZIONE PROBABILITA' FUTURE...")

for _, r in forecast.iterrows():

    for target in TARGETS:

        prediction = float(
            r[f"Forecast_{target}"]
        )

        if not np.isfinite(prediction):
            continue

        values, weights, source, n_player, n_base = (
            get_distribution(
                final_pools,
                target,
                r["player_id"],
                prediction
            )
        )

        expected = float(
            np.average(
                values,
                weights=weights
            )
        )

        std = float(
            np.sqrt(
                np.average(
                    (values - expected) ** 2,
                    weights=weights
                )
            )
        )

        metadata = {
            "game_id": r["game_id"],
            "player_id": r["player_id"],
            "date": r["date"].isoformat(),
            "generated_at": NOW.isoformat(),
            "Giocatore": r["Giocatore"],
            "Squadra": r["Squadra"],
            "Avversario": r["Avversario"],
            "Target": target,
            "Forecast": prediction,
            "Distribution_Source": source,
            "Player_Sample": n_player,
            "Reference_Sample": n_base,
            "Roster_Status": r.get(
                "Roster_Status",
                "Disponibilita non verificata"
            )
        }

        summary = {
            **metadata,
            "Expected_Value": expected,
            "Median": weighted_quantile(
                values, weights, 0.5
            ),
            "Std_Dev": std,
            "P10": weighted_quantile(
                values, weights, 0.10
            ),
            "P90": weighted_quantile(
                values, weights, 0.90
            )
        }

        for level in INTERVALS:

            lower, upper = prediction_interval(
                values, weights, level
            )

            summary[f"Lower_{level}"] = lower
            summary[f"Upper_{level}"] = upper

        summary_rows.append(summary)

        for threshold in THRESHOLDS[target]:

            p = probability_at_least(
                values,
                weights,
                threshold
            )

            threshold_rows.append({
                **metadata,
                "Threshold": threshold,
                "Probability_At_Least": p,
                "Probability_Below": 1 - p,
                "Probability_pct": 100 * p
            })

summary_df = pd.DataFrame(summary_rows)

probabilities = pd.DataFrame(threshold_rows)

if summary_df.empty or probabilities.empty:

    raise ValueError(
        "Nessuna distribuzione valida."
    )

# ============================================================
# 12. CONTROLLO QUALITA'
# ============================================================

assert probabilities[
    "Probability_At_Least"
].between(0, 1).all()

assert np.allclose(
    probabilities["Probability_At_Least"]
    + probabilities["Probability_Below"],
    1
)

for level in INTERVALS:

    assert (
        summary_df[f"Lower_{level}"]
        <= summary_df[f"Upper_{level}"]
    ).all()

assert (
    summary_df["Lower_95"]
    <= summary_df["Lower_50"]
).all()

assert (
    summary_df["Upper_95"]
    >= summary_df["Upper_50"]
).all()

# ============================================================
# 13. ARCHIVIAZIONE PERMANENTE SQLITE
# ============================================================

with sqlite3.connect(DB_PATH) as conn:

    conn.execute("""
        CREATE TABLE IF NOT EXISTS
        nba_probability_history_v3 (
            game_id TEXT NOT NULL,
            player_id TEXT NOT NULL,
            generated_at TEXT NOT NULL,
            game_date TEXT NOT NULL,
            player_name TEXT,
            target TEXT NOT NULL,
            threshold REAL NOT NULL,
            forecast REAL,
            probability REAL,
            distribution_source TEXT,
            PRIMARY KEY (
                game_id,
                player_id,
                generated_at,
                target,
                threshold
            )
        )
    """)

    records = []

    for r in probabilities.itertuples(index=False):

        records.append((
            str(r.game_id),
            str(r.player_id),
            r.generated_at,
            r.date,
            r.Giocatore,
            r.Target,
            float(r.Threshold),
            float(r.Forecast),
            float(r.Probability_At_Least),
            r.Distribution_Source
        ))

    conn.executemany("""
        INSERT OR IGNORE INTO
        nba_probability_history_v3
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, records)

    total_history = conn.execute("""
        SELECT COUNT(*)
        FROM nba_probability_history_v3
    """).fetchone()[0]

# ============================================================
# 14. ESPORTAZIONE FILE
# ============================================================

stamp = NOW.strftime(
    "%Y%m%d_%H%M%S_%f"
)

archive_path = os.path.join(
    ARCHIVE_DIR,
    f"NBA_Probabilities_{stamp}.csv"
)

probabilities.to_csv(
    CSV_PATH,
    index=False,
    encoding="utf-8-sig"
)

probabilities.to_csv(
    archive_path,
    index=False,
    encoding="utf-8-sig"
)

summary_path = os.path.join(
    BASE_DIR,
    "NBA_Probability_Summary_V3.csv"
)

summary_df.to_csv(
    summary_path,
    index=False,
    encoding="utf-8-sig"
)

with pd.ExcelWriter(
    OUTPUT_PATH,
    engine="openpyxl"
) as writer:

    sheets = {
        "Player Distributions": summary_df,
        "Threshold Probabilities": probabilities,
        "Interval Coverage": coverage_report,
        "Brier Score": brier_report,
        "Reliability": reliability,
        "Calibration ECE": ece_report
    }

    for name, data in sheets.items():

        data.to_excel(
            writer,
            sheet_name=name,
            index=False
        )

        sheet = writer.sheets[name]
        sheet.freeze_panes = "A2"
        sheet.auto_filter.ref = sheet.dimensions

# ============================================================
# 15. ANALISI CALIBRAZIONE
# ============================================================

print("\nINTERVAL COVERAGE")

display(
    coverage_report.round(3)
)

print("\nEXPECTED CALIBRATION ERROR")

display(
    ece_report.round(3)
)

fig = px.scatter(
    reliability,
    x="Mean_Probability",
    y="Actual_Frequency",
    color="Target",
    size="Sample",
    title="NBA V3 — Probability Calibration",
    template="plotly_dark"
)

fig.add_shape(
    type="line",
    x0=0,
    y0=0,
    x1=1,
    y1=1,
    line=dict(
        color="white",
        dash="dash"
    )
)

fig.update_xaxes(
    range=[0, 1],
    tickformat=".0%"
)

fig.update_yaxes(
    range=[0, 1],
    tickformat=".0%"
)

fig.show()

fig = px.bar(
    coverage_report,
    x="Level",
    y="Observed_Coverage",
    color="Target",
    barmode="group",
    title="NBA V3 — Prediction Interval Coverage",
    template="plotly_dark"
)

fig.show()

# ============================================================
# 16. PLAYER PROBABILITY EXPLORER
# ============================================================

import ipywidgets as widgets

from IPython.display import clear_output

players = sorted(
    summary_df["Giocatore"].unique()
)

default_player = next(
    (
        p for p in players
        if "jokic" in p.lower()
        or "jokić" in p.lower()
    ),
    players[0]
)

player_widget = widgets.Dropdown(
    options=players,
    value=default_player,
    description="Player:"
)

target_widget = widgets.Dropdown(
    options=TARGETS,
    value="PTS",
    description="Stat:"
)

output = widgets.Output()

def show_player(change=None):

    with output:

        clear_output(wait=True)

        player = player_widget.value
        target = target_widget.value

        selected = summary_df[
            (summary_df["Giocatore"] == player)
            & (summary_df["Target"] == target)
        ].sort_values("date")

        if selected.empty:
            print("Nessuna previsione.")
            return

        r = selected.iloc[0]

        selected_probs = probabilities[
            (probabilities["player_id"] == r["player_id"])
            & (probabilities["game_id"] == r["game_id"])
            & (probabilities["Target"] == target)
        ].sort_values("Threshold")

        print("=" * 55)
        print(player.upper())
        print(r["Squadra"], "vs", r["Avversario"])
        print("Target:", target)
        print("Forecast:", round(r["Forecast"], 2))
        print(
            "Expected Value:",
            round(r["Expected_Value"], 2)
        )
        print(
            "Deviazione standard:",
            round(r["Std_Dev"], 2)
        )
        print(
            "Campione individuale:",
            r["Player_Sample"]
        )
        print("Distribuzione:", r["Distribution_Source"])
        print("Disponibilita:", r["Roster_Status"])
        print("=" * 55)

        print("\nINTERVALLI PREDITTIVI")

        for level in INTERVALS:

            print(
                f"{level}%:",
                r[f"Lower_{level}"],
                "-",
                r[f"Upper_{level}"]
            )

        display(
            selected_probs[[
                "Threshold",
                "Probability_pct"
            ]].round(2)
        )

        values, weights, _, _, _ = (
            get_distribution(
                final_pools,
                target,
                r["player_id"],
                r["Forecast"]
            )
        )

        distribution = pd.DataFrame({
            "Outcome": values,
            "Probability": weights
        }).groupby(
            "Outcome",
            as_index=False
        ).sum()

        fig = px.bar(
            distribution,
            x="Outcome",
            y="Probability",
            title=f"{player} — {target} Distribution",
            template="plotly_dark"
        )

        fig.add_vline(
            x=r["Forecast"],
            line_dash="dash",
            line_color="orange"
        )

        fig.update_yaxes(
            tickformat=".0%"
        )

        fig.show()

        fig = px.bar(
            selected_probs,
            x="Threshold",
            y="Probability_pct",
            color="Probability_pct",
            title=f"{player} — Threshold Probabilities",
            template="plotly_dark"
        )

        fig.update_yaxes(
            range=[0, 100],
            ticksuffix="%"
        )

        fig.show()

player_widget.observe(
    show_player,
    names="value"
)

target_widget.observe(
    show_player,
    names="value"
)

display(
    widgets.HBox([
        player_widget,
        target_widget
    ]),
    output
)

show_player()

# ============================================================
# 17. RIEPILOGO FINALE
# ============================================================

print("\n" + "=" * 55)
print("STEP 18 COMPLETATO!")
print("=" * 55)

print(
    "Partite future:",
    summary_df["game_id"].nunique()
)

print(
    "Giocatori:",
    summary_df["player_id"].nunique()
)

print(
    "Distribuzioni generate:",
    len(summary_df)
)

print(
    "Probabilita calcolate:",
    len(probabilities)
)

print(
    "Probabilita archiviate:",
    total_history
)

print("\nFILE SALVATI")

print(OUTPUT_PATH)
print(CSV_PATH)
print(summary_path)
print(archive_path)

print("\nPRONTO PER STEP 19!")


In [ ]:

# ============================================================
# STEP 18 — PROBABILITY DISTRIBUTION ENGINE
# NBA DATA SCIENCE ANALYTICS
# VERSION 3.1 — FIXED
# ============================================================

import os
import sqlite3
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets

from google.colab import drive
from IPython.display import display, clear_output

# ============================================================
# 1. RIPRISTINO AUTOMATICO AMBIENTE
# ============================================================

mounts = [
    "/content/drive",
    "/content/gdrive_nba",
    "/content/nba_drive"
]

BASE_DIR = None

for mount in mounts:

    candidate = os.path.join(
        mount, "MyDrive", "NBA_Data_Science"
    )

    if os.path.isfile(
        os.path.join(candidate, "nba_database.db")
    ):
        BASE_DIR = candidate
        break

if BASE_DIR is None:

    for mount in mounts:

        if os.path.isdir(mount) and os.listdir(mount):
            continue

        drive.mount(mount)

        candidate = os.path.join(
            mount, "MyDrive", "NBA_Data_Science"
        )

        if os.path.isfile(
            os.path.join(candidate, "nba_database.db")
        ):
            BASE_DIR = candidate
            break

if BASE_DIR is None:

    raise FileNotFoundError(
        "Cartella NBA_Data_Science non trovata."
    )

print("Google Drive collegato.")
print("Cartella progetto:", BASE_DIR)

# ============================================================
# 2. CONFIGURAZIONE
# ============================================================

HISTORY_PATH = os.path.join(
    BASE_DIR,
    "NBA_Advanced_Test_Predictions.csv"
)

FORECAST_PATH = os.path.join(
    BASE_DIR,
    "NBA_Forecast_Engine_V3.csv"
)

OUTPUT_PATH = os.path.join(
    BASE_DIR,
    "NBA_Probability_Engine_V3.xlsx"
)

CSV_PATH = os.path.join(
    BASE_DIR,
    "NBA_Probability_Engine_V3.csv"
)

SUMMARY_PATH = os.path.join(
    BASE_DIR,
    "NBA_Probability_Summary_V3.csv"
)

DB_PATH = os.path.join(
    BASE_DIR,
    "nba_database.db"
)

ARCHIVE_DIR = os.path.join(
    BASE_DIR,
    "Probability_Archive_V3"
)

os.makedirs(
    ARCHIVE_DIR,
    exist_ok=True
)

TARGETS = [
    "PTS", "REB", "AST", "3PM", "PRA"
]

THRESHOLDS = {
    "PTS": [5, 10, 15, 20, 25, 30, 35, 40, 45, 50],
    "REB": [2, 4, 6, 8, 10, 12, 15, 20],
    "AST": [2, 4, 6, 8, 10, 12, 15],
    "3PM": [1, 2, 3, 4, 5, 6, 7],
    "PRA": [10, 15, 20, 25, 30, 35, 40, 45, 50, 60]
}

INTERVALS = [50, 80, 90, 95]

MIN_PLAYER_SAMPLE = 10
MIN_GROUP_SAMPLE = 50

TOLERANCE = 1e-10

NOW = pd.Timestamp.now(tz="UTC")

for path in [HISTORY_PATH, FORECAST_PATH]:

    if not os.path.isfile(path):

        raise FileNotFoundError(
            f"File mancante: {path}"
        )

# ============================================================
# 3. CARICAMENTO DATI
# ============================================================

history = pd.read_csv(
    HISTORY_PATH,
    dtype={
        "game_id": str,
        "player_id": str
    }
)

forecast = pd.read_csv(
    FORECAST_PATH,
    dtype={
        "game_id": str,
        "player_id": str
    }
)

history["game_date"] = pd.to_datetime(
    history["game_date"],
    utc=True,
    errors="coerce",
    format="mixed"
)

forecast["date"] = pd.to_datetime(
    forecast["date"],
    utc=True,
    errors="coerce",
    format="mixed"
)

for col in ["Actual", "Forecast"]:

    history[col] = pd.to_numeric(
        history[col],
        errors="coerce"
    )

for target in TARGETS:

    col = f"Forecast_{target}"

    if col not in forecast.columns:
        raise ValueError(f"Colonna mancante: {col}")

    forecast[col] = pd.to_numeric(
        forecast[col],
        errors="coerce"
    )

history = history.replace(
    [np.inf, -np.inf],
    np.nan
).dropna(
    subset=[
        "game_date",
        "game_id",
        "Actual",
        "Forecast",
        "player_id",
        "Target"
    ]
)

history = history[
    (history["game_date"] < NOW)
    & history["Target"].isin(TARGETS)
    & (history["Actual"] >= 0)
    & (history["Forecast"] >= 0)
].drop_duplicates(
    ["game_id", "player_id", "Target"]
).sort_values(
    "game_date"
)

forecast = forecast[
    forecast["date"] > NOW
].drop_duplicates(
    ["game_id", "player_id"]
).copy()

if history.empty:
    raise ValueError("Storico errori vuoto.")

if forecast.empty:

    raise ValueError(
        "Forecast scaduti. Riesegui STEP 17."
    )

history["Residual"] = (
    history["Actual"]
    - history["Forecast"]
)

print("\nDATI CARICATI")

print("Errori storici:", len(history))
print("Forecast futuri:", len(forecast))

# ============================================================
# 4. SUDDIVISIONE TEMPORALE
# ============================================================

history["NBA_Day"] = (
    history["game_date"]
    .dt.tz_convert("America/New_York")
    .dt.strftime("%Y-%m-%d")
)

days = sorted(
    history["NBA_Day"].unique()
)

if len(days) < 10:

    raise ValueError(
        "Storico insufficiente per calibrazione."
    )

split = int(len(days) * 0.60)

calibration = history[
    history["NBA_Day"].isin(days[:split])
].copy()

evaluation = history[
    history["NBA_Day"].isin(days[split:])
].copy()

print("\nTEMPORAL SPLIT")

print("CALIBRATION:", len(calibration))
print("EVALUATION:", len(evaluation))

# ============================================================
# 5. GRUPPI STATISTICI
# ============================================================

BINS = {
    "PTS": [0, 5, 10, 15, 20, 25, 30, 40, np.inf],
    "REB": [0, 2, 4, 6, 8, 10, 15, np.inf],
    "AST": [0, 2, 4, 6, 8, 10, 15, np.inf],
    "3PM": [0, 1, 2, 3, 4, 6, np.inf],
    "PRA": [0, 10, 20, 30, 40, 50, 60, np.inf]
}

def get_bucket(target, prediction):

    edges = BINS[target]

    return int(
        np.clip(
            np.searchsorted(
                edges,
                prediction,
                side="right"
            ) - 1,
            0,
            len(edges) - 2
        )
    )

# ============================================================
# 6. COSTRUZIONE DISTRIBUZIONI EMPIRICHE
# ============================================================

def build_error_pools(data):

    pools = {
        "global": {},
        "group": {},
        "player": {}
    }

    temp = data[
        data["Target"].isin(TARGETS)
    ].copy()

    temp["Bucket"] = [
        get_bucket(t, p)
        for t, p in zip(
            temp["Target"],
            temp["Forecast"]
        )
    ]

    for target, g in temp.groupby("Target"):

        pools["global"][target] = (
            g["Residual"].to_numpy(dtype=float)
        )

    for key, g in temp.groupby(
        ["Target", "Bucket"]
    ):

        pools["group"][key] = (
            g["Residual"].to_numpy(dtype=float)
        )

    for key, g in temp.groupby(
        ["Target", "player_id"]
    ):

        pools["player"][key] = (
            g["Residual"].to_numpy(dtype=float)
        )

    return pools

# ============================================================
# 7. NORMALIZZAZIONE SICURA DEI PESI
# ============================================================

def normalize_weights(weights):

    weights = np.asarray(
        weights,
        dtype=np.float64
    )

    if len(weights) == 0:
        raise ValueError("Distribuzione vuota.")

    if not np.isfinite(weights).all():
        raise ValueError("Pesi non finiti.")

    if (weights < 0).any():
        raise ValueError("Pesi negativi.")

    total = weights.sum(dtype=np.float64)

    if not np.isfinite(total) or total <= 0:
        raise ValueError("Somma pesi non valida.")

    weights = weights / total

    if not np.isclose(
        weights.sum(),
        1.0,
        atol=TOLERANCE,
        rtol=0
    ):

        raise ValueError(
            "Normalizzazione pesi fallita."
        )

    return weights

# ============================================================
# 8. DISTRIBUZIONE PERSONALIZZATA
# ============================================================

def get_distribution(
    pools,
    target,
    player_id,
    prediction
):

    if not np.isfinite(prediction):
        raise ValueError("Forecast non valido.")

    global_errors = pools["global"].get(
        target,
        np.array([])
    )

    if len(global_errors) == 0:

        raise ValueError(
            f"Errori storici mancanti: {target}"
        )

    bucket = get_bucket(
        target,
        prediction
    )

    group_errors = pools["group"].get(
        (target, bucket),
        np.array([])
    )

    player_errors = pools["player"].get(
        (target, str(player_id)),
        np.array([])
    )

    if len(group_errors) >= MIN_GROUP_SAMPLE:

        base_errors = group_errors
        source = "Forecast Group"

    else:

        base_errors = global_errors
        source = "Global"

    # Shrinkage statistico

    if len(player_errors) >= MIN_PLAYER_SAMPLE:

        alpha = len(player_errors) / (
            len(player_errors) + 30
        )

        errors = np.concatenate([
            player_errors,
            base_errors
        ])

        weights = np.concatenate([
            np.full(
                len(player_errors),
                alpha / len(player_errors)
            ),
            np.full(
                len(base_errors),
                (1 - alpha) / len(base_errors)
            )
        ])

        source = "Player + " + source

    else:

        errors = base_errors

        weights = np.full(
            len(errors),
            1 / len(errors)
        )

    outcomes = np.maximum(
        np.floor(
            prediction + errors + 0.5
        ),
        0
    )

    if not np.isfinite(outcomes).all():
        raise ValueError("Outcome non valido.")

    order = np.argsort(outcomes)

    outcomes = outcomes[order]
    weights = weights[order]

    weights = normalize_weights(weights)

    return (
        outcomes,
        weights,
        source,
        len(player_errors),
        len(base_errors)
    )

# ============================================================
# 9. FUNZIONI STATISTICHE — FIXED
# ============================================================

def weighted_quantile(
    values,
    weights,
    q
):

    if not 0 <= q <= 1:
        raise ValueError("Quantile non valido.")

    cumulative = np.cumsum(
        weights,
        dtype=np.float64
    )

    cumulative = np.clip(
        cumulative,
        0.0,
        1.0
    )

    cumulative[-1] = 1.0

    index = np.searchsorted(
        cumulative,
        q,
        side="left"
    )

    index = min(
        index,
        len(values) - 1
    )

    return float(values[index])


def probability_at_least(
    values,
    weights,
    threshold
):

    p = float(
        weights[values >= threshold].sum(
            dtype=np.float64
        )
    )

    if not np.isfinite(p):

        raise ValueError(
            f"Probabilita non valida: {p}"
        )

    if p < -TOLERANCE or p > 1 + TOLERANCE:

        raise ValueError(
            f"Probabilita fuori intervallo: {p}"
        )

    # FIX: corregge esclusivamente
    # gli errori numerici floating point.

    return float(
        np.clip(p, 0.0, 1.0)
    )


def prediction_interval(
    values,
    weights,
    level
):

    alpha = 1 - level / 100

    lower = weighted_quantile(
        values,
        weights,
        alpha / 2
    )

    upper = weighted_quantile(
        values,
        weights,
        1 - alpha / 2
    )

    return lower, upper

# ============================================================
# 10. BACKTEST PROBABILISTICO
# ============================================================

calibration_pools = build_error_pools(
    calibration
)

coverage_rows = []
probability_rows = []

print("\nBACKTEST PROBABILISTICO...")

for r in evaluation.itertuples(index=False):

    target = r.Target

    if target not in TARGETS:
        continue

    values, weights, source, n_player, n_base = (
        get_distribution(
            calibration_pools,
            target,
            r.player_id,
            r.Forecast
        )
    )

    for level in INTERVALS:

        lower, upper = prediction_interval(
            values,
            weights,
            level
        )

        coverage_rows.append({
            "Target": target,
            "Level": level,
            "Covered": int(
                lower <= r.Actual <= upper
            ),
            "Width": upper - lower
        })

    for threshold in THRESHOLDS[target]:

        p = probability_at_least(
            values,
            weights,
            threshold
        )

        actual_event = int(
            r.Actual >= threshold
        )

        probability_rows.append({
            "Target": target,
            "Threshold": threshold,
            "Probability": p,
            "Actual_Event": actual_event,
            "Brier_Error": (
                p - actual_event
            ) ** 2
        })

coverage = pd.DataFrame(
    coverage_rows
)

coverage_report = coverage.groupby(
    ["Target", "Level"],
    as_index=False
).agg(
    Observed_Coverage=("Covered", "mean"),
    Average_Width=("Width", "mean"),
    Sample=("Covered", "size")
)

coverage_report["Observed_Coverage"] *= 100

coverage_report["Coverage_Error"] = (
    coverage_report["Observed_Coverage"]
    - coverage_report["Level"]
)

probability_backtest = pd.DataFrame(
    probability_rows
)

brier_report = probability_backtest.groupby(
    ["Target", "Threshold"],
    as_index=False
).agg(
    Brier_Score=("Brier_Error", "mean"),
    Predicted_Probability=("Probability", "mean"),
    Actual_Frequency=("Actual_Event", "mean"),
    Sample=("Actual_Event", "size")
)

print("Backtest completato.")

# ============================================================
# 11. RELIABILITY DIAGRAM
# ============================================================

probability_backtest["Probability_Bin"] = pd.cut(
    probability_backtest["Probability"],
    bins=np.linspace(0, 1, 11),
    include_lowest=True
)

reliability = probability_backtest.groupby(
    ["Target", "Probability_Bin"],
    observed=True,
    as_index=False
).agg(
    Mean_Probability=("Probability", "mean"),
    Actual_Frequency=("Actual_Event", "mean"),
    Sample=("Actual_Event", "size")
)

reliability["Calibration_Error"] = (
    reliability["Mean_Probability"]
    - reliability["Actual_Frequency"]
).abs()

reliability["Probability_Bin"] = (
    reliability["Probability_Bin"].astype(str)
)

ece_rows = []

for target, g in reliability.groupby("Target"):

    ece = np.average(
        g["Calibration_Error"],
        weights=g["Sample"]
    )

    ece_rows.append({
        "Target": target,
        "ECE_pct": 100 * ece,
        "Sample": g["Sample"].sum()
    })

ece_report = pd.DataFrame(
    ece_rows
)

# ============================================================
# 12. DISTRIBUZIONI FUTURE
# ============================================================

final_pools = build_error_pools(
    history
)

summary_rows = []
threshold_rows = []

print("\nGENERAZIONE PROBABILITA' FUTURE...")

for _, r in forecast.iterrows():

    for target in TARGETS:

        prediction = float(
            r[f"Forecast_{target}"]
        )

        if not np.isfinite(prediction):
            continue

        if prediction < 0:
            raise ValueError("Forecast negativo.")

        values, weights, source, n_player, n_base = (
            get_distribution(
                final_pools,
                target,
                r["player_id"],
                prediction
            )
        )

        expected = float(
            np.average(
                values,
                weights=weights
            )
        )

        std = float(
            np.sqrt(
                np.average(
                    (values - expected) ** 2,
                    weights=weights
                )
            )
        )

        metadata = {
            "game_id": r["game_id"],
            "player_id": r["player_id"],
            "date": r["date"].isoformat(),
            "generated_at": NOW.isoformat(),
            "Forecast_Generated_At": r.get(
                "generated_at",
                ""
            ),
            "Giocatore": r["Giocatore"],
            "Squadra": r["Squadra"],
            "Avversario": r["Avversario"],
            "Target": target,
            "Forecast": prediction,
            "Distribution_Source": source,
            "Player_Sample": n_player,
            "Reference_Sample": n_base,
            "Roster_Status": r.get(
                "Roster_Status",
                "Disponibilita non verificata"
            ),
            "Probability_Version": "3.1"
        }

        summary = {
            **metadata,
            "Expected_Value": expected,
            "Median": weighted_quantile(
                values, weights, 0.50
            ),
            "Std_Dev": std,
            "P10": weighted_quantile(
                values, weights, 0.10
            ),
            "P90": weighted_quantile(
                values, weights, 0.90
            )
        }

        for level in INTERVALS:

            lower, upper = prediction_interval(
                values,
                weights,
                level
            )

            summary[f"Lower_{level}"] = lower
            summary[f"Upper_{level}"] = upper

        summary_rows.append(summary)

        for threshold in THRESHOLDS[target]:

            p = probability_at_least(
                values,
                weights,
                threshold
            )

            threshold_rows.append({
                **metadata,
                "Threshold": threshold,
                "Probability_At_Least": p,
                "Probability_Below": 1 - p,
                "Probability_pct": 100 * p
            })

summary_df = pd.DataFrame(
    summary_rows
)

probabilities = pd.DataFrame(
    threshold_rows
)

if summary_df.empty or probabilities.empty:

    raise ValueError(
        "Nessuna distribuzione valida."
    )

# ============================================================
# 13. CONTROLLO QUALITA' — FIXED
# ============================================================

print("\nCONTROLLO QUALITA'...")

p = probabilities[
    "Probability_At_Least"
].to_numpy(dtype=float)

if not np.isfinite(p).all():

    raise ValueError(
        "Probabilita NaN o infinite."
    )

if (
    (p < -TOLERANCE)
    | (p > 1 + TOLERANCE)
).any():

    raise ValueError(
        "Probabilita realmente fuori intervallo."
    )

# Correzione finale floating point.

probabilities["Probability_At_Least"] = (
    np.clip(p, 0.0, 1.0)
)

probabilities["Probability_Below"] = (
    1 - probabilities["Probability_At_Least"]
)

probabilities["Probability_pct"] = (
    100 * probabilities["Probability_At_Least"]
)

assert probabilities[
    "Probability_At_Least"
].between(0, 1).all()

assert np.allclose(
    probabilities["Probability_At_Least"]
    + probabilities["Probability_Below"],
    1.0,
    atol=TOLERANCE,
    rtol=0
)

# Controllo intervalli predittivi.

for level in INTERVALS:

    assert (
        summary_df[f"Lower_{level}"]
        <= summary_df[f"Upper_{level}"]
    ).all()

for small, large in zip(
    INTERVALS[:-1],
    INTERVALS[1:]
):

    assert (
        summary_df[f"Lower_{large}"]
        <= summary_df[f"Lower_{small}"]
    ).all()

    assert (
        summary_df[f"Upper_{large}"]
        >= summary_df[f"Upper_{small}"]
    ).all()

# Controllo monotonicita:
# P(X >= 30) non puo superare P(X >= 20).

probabilities = probabilities.sort_values(
    [
        "game_id",
        "player_id",
        "Target",
        "Threshold"
    ]
).reset_index(drop=True)

diff = probabilities.groupby(
    ["game_id", "player_id", "Target"]
)["Probability_At_Least"].diff()

if (diff > TOLERANCE).any():

    raise ValueError(
        "Errore monotonicita probabilita."
    )

# Rimuove eventuali partite iniziate
# durante il calcolo.

FINISHED_AT = pd.Timestamp.now(tz="UTC")

summary_df = summary_df[
    pd.to_datetime(
        summary_df["date"],
        utc=True
    ) > FINISHED_AT
].copy()

probabilities = probabilities[
    pd.to_datetime(
        probabilities["date"],
        utc=True
    ) > FINISHED_AT
].copy()

if probabilities.empty or summary_df.empty:

    raise ValueError(
        "Forecast scaduti durante il calcolo."
    )

print("Probabilita min:", probabilities[
    "Probability_At_Least"
].min())

print("Probabilita max:", probabilities[
    "Probability_At_Least"
].max())

print("Controllo intervalli: OK")
print("Controllo monotonicita: OK")
print("CONTROLLO QUALITA' SUPERATO!")

# ============================================================
# 14. ARCHIVIAZIONE PERMANENTE SQLITE
# ============================================================

with sqlite3.connect(
    DB_PATH,
    timeout=60
) as conn:

    conn.execute("""
        CREATE TABLE IF NOT EXISTS
        nba_probability_history_v3 (
            game_id TEXT NOT NULL,
            player_id TEXT NOT NULL,
            generated_at TEXT NOT NULL,
            game_date TEXT NOT NULL,
            player_name TEXT,
            target TEXT NOT NULL,
            threshold REAL NOT NULL,
            forecast REAL,
            probability REAL,
            distribution_source TEXT,
            PRIMARY KEY (
                game_id,
                player_id,
                generated_at,
                target,
                threshold
            )
        )
    """)

    records = []

    for r in probabilities.itertuples(index=False):

        records.append((
            str(r.game_id),
            str(r.player_id),
            r.generated_at,
            r.date,
            r.Giocatore,
            r.Target,
            float(r.Threshold),
            float(r.Forecast),
            float(r.Probability_At_Least),
            r.Distribution_Source
        ))

    conn.executemany("""
        INSERT OR IGNORE INTO
        nba_probability_history_v3 (
            game_id,
            player_id,
            generated_at,
            game_date,
            player_name,
            target,
            threshold,
            forecast,
            probability,
            distribution_source
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, records)

    total_history = conn.execute("""
        SELECT COUNT(*)
        FROM nba_probability_history_v3
    """).fetchone()[0]

print("\nARCHIVIAZIONE SQLITE COMPLETATA")

# ============================================================
# 15. ESPORTAZIONE FILE
# ============================================================

stamp = NOW.strftime(
    "%Y%m%d_%H%M%S_%f"
)

archive_path = os.path.join(
    ARCHIVE_DIR,
    f"NBA_Probabilities_{stamp}.csv"
)

summary_archive_path = os.path.join(
    ARCHIVE_DIR,
    f"NBA_Distributions_{stamp}.csv"
)

probabilities.to_csv(
    archive_path,
    index=False,
    encoding="utf-8-sig"
)

summary_df.to_csv(
    summary_archive_path,
    index=False,
    encoding="utf-8-sig"
)

probabilities.to_csv(
    CSV_PATH + ".tmp",
    index=False,
    encoding="utf-8-sig"
)

summary_df.to_csv(
    SUMMARY_PATH + ".tmp",
    index=False,
    encoding="utf-8-sig"
)

with pd.ExcelWriter(
    OUTPUT_PATH + ".tmp.xlsx",
    engine="openpyxl"
) as writer:

    sheets = {
        "Player Distributions": summary_df,
        "Threshold Probabilities": probabilities,
        "Interval Coverage": coverage_report,
        "Brier Score": brier_report,
        "Reliability": reliability,
        "Calibration ECE": ece_report
    }

    for name, data in sheets.items():

        data.to_excel(
            writer,
            sheet_name=name,
            index=False
        )

        sheet = writer.sheets[name]

        sheet.freeze_panes = "A2"

        sheet.auto_filter.ref = (
            sheet.dimensions
        )

os.replace(
    CSV_PATH + ".tmp",
    CSV_PATH
)

os.replace(
    SUMMARY_PATH + ".tmp",
    SUMMARY_PATH
)

os.replace(
    OUTPUT_PATH + ".tmp.xlsx",
    OUTPUT_PATH
)

print("FILE SALVATI CORRETTAMENTE")

# ============================================================
# 16. ANALISI CALIBRAZIONE
# ============================================================

print("\nINTERVAL COVERAGE")

display(
    coverage_report.round(3)
)

print("\nEXPECTED CALIBRATION ERROR")

display(
    ece_report.round(3)
)

print("\nBRIER SCORE")

display(
    brier_report.round(4)
)

fig = px.scatter(
    reliability,
    x="Mean_Probability",
    y="Actual_Frequency",
    color="Target",
    size="Sample",
    title="NBA V3.1 — Probability Calibration",
    template="plotly_dark"
)

fig.add_shape(
    type="line",
    x0=0,
    y0=0,
    x1=1,
    y1=1,
    line=dict(
        color="white",
        dash="dash"
    )
)

fig.update_xaxes(
    range=[0, 1],
    tickformat=".0%"
)

fig.update_yaxes(
    range=[0, 1],
    tickformat=".0%"
)

fig.show()

fig = px.bar(
    coverage_report,
    x="Level",
    y="Observed_Coverage",
    color="Target",
    barmode="group",
    title="NBA V3.1 — Prediction Interval Coverage",
    template="plotly_dark"
)

fig.update_yaxes(
    range=[0, 100],
    ticksuffix="%"
)

fig.show()

fig = px.bar(
    ece_report,
    x="Target",
    y="ECE_pct",
    color="Target",
    title="NBA V3.1 — Calibration Error",
    template="plotly_dark"
)

fig.update_yaxes(
    ticksuffix="%"
)

fig.show()

# ============================================================
# 17. PLAYER PROBABILITY EXPLORER
# ============================================================

player_options = (
    summary_df[["player_id", "Giocatore"]]
    .drop_duplicates("player_id")
    .sort_values("Giocatore")
)

players = [
    (
        f"{r.Giocatore} ({r.player_id})",
        r.player_id
    )
    for r in player_options.itertuples(index=False)
]

default_player = next(
    (
        pid for name, pid in players
        if "jokic" in name.lower()
        or "jokić" in name.lower()
    ),
    players[0][1]
)

player_widget = widgets.Dropdown(
    options=players,
    value=default_player,
    description="Player:"
)

target_widget = widgets.Dropdown(
    options=TARGETS,
    value="PTS",
    description="Stat:"
)

game_widget = widgets.Dropdown(
    options=[],
    description="Game:"
)

threshold_widget = widgets.IntSlider(
    value=20,
    min=0,
    max=100,
    step=1,
    description="Soglia:",
    continuous_update=False
)

output = widgets.Output()

def update_games():

    selected = summary_df[
        summary_df["player_id"]
        == player_widget.value
    ].drop_duplicates(
        "game_id"
    ).sort_values("date")

    options = []

    for r in selected.itertuples(index=False):

        date_it = pd.Timestamp(
            r.date
        ).tz_convert(
            "Europe/Rome"
        ).strftime(
            "%d/%m %H:%M"
        )

        label = (
            f"{date_it} — vs {r.Avversario}"
        )

        options.append(
            (label, r.game_id)
        )

    game_widget.options = options

    if options:
        game_widget.value = options[0][1]


def show_player(change=None):

    with output:

        clear_output(wait=True)

        player_id = player_widget.value
        target = target_widget.value
        game_id = game_widget.value

        selected = summary_df[
            (summary_df["player_id"] == player_id)
            & (summary_df["Target"] == target)
            & (summary_df["game_id"] == game_id)
        ]

        if selected.empty:

            print("Nessuna previsione.")
            return

        r = selected.iloc[0]

        selected_probs = probabilities[
            (probabilities["player_id"] == player_id)
            & (probabilities["game_id"] == game_id)
            & (probabilities["Target"] == target)
        ].sort_values("Threshold")

        values, weights, _, _, _ = (
            get_distribution(
                final_pools,
                target,
                player_id,
                r["Forecast"]
            )
        )

        custom_threshold = (
            threshold_widget.value
        )

        custom_probability = (
            probability_at_least(
                values,
                weights,
                custom_threshold
            )
        )

        print("=" * 60)

        print(
            r["Giocatore"].upper()
        )

        print(
            r["Squadra"],
            "vs",
            r["Avversario"]
        )

        print("Target:", target)

        print(
            "Forecast:",
            round(r["Forecast"], 2)
        )

        print(
            "Expected Value:",
            round(r["Expected_Value"], 2)
        )

        print(
            "Mediana:",
            r["Median"]
        )

        print(
            "Deviazione standard:",
            round(r["Std_Dev"], 2)
        )

        print(
            "Campione individuale:",
            r["Player_Sample"]
        )

        print(
            "Campione riferimento:",
            r["Reference_Sample"]
        )

        print(
            "Distribuzione:",
            r["Distribution_Source"]
        )

        print(
            "Disponibilita:",
            r["Roster_Status"]
        )

        print("=" * 60)

        print("\nSOGLIA PERSONALIZZATA")

        print(
            f"P({target} >= {custom_threshold}): "
            f"{100 * custom_probability:.2f}%"
        )

        print(
            f"P({target} < {custom_threshold}): "
            f"{100 * (1-custom_probability):.2f}%"
        )

        print("\nINTERVALLI PREDITTIVI")

        for level in INTERVALS:

            print(
                f"{level}%:",
                r[f"Lower_{level}"],
                "-",
                r[f"Upper_{level}"]
            )

        print("\nPROBABILITA' PER SOGLIA")

        display(
            selected_probs[[
                "Threshold",
                "Probability_pct"
            ]].round(2)
        )

        distribution = pd.DataFrame({
            "Outcome": values,
            "Probability": weights
        }).groupby(
            "Outcome",
            as_index=False
        ).sum()

        distribution["Threshold_Status"] = np.where(
            distribution["Outcome"] >= custom_threshold,
            "Soglia raggiunta",
            "Sotto soglia"
        )

        fig = px.bar(
            distribution,
            x="Outcome",
            y="Probability",
            color="Threshold_Status",
            title=(
                f"{r['Giocatore']} — "
                f"{target} Distribution"
            ),
            template="plotly_dark",
            color_discrete_map={
                "Soglia raggiunta": "#22c55e",
                "Sotto soglia": "#64748b"
            }
        )

        fig.add_vline(
            x=r["Forecast"],
            line_dash="dash",
            line_color="orange",
            annotation_text="Forecast"
        )

        fig.add_vline(
            x=custom_threshold,
            line_dash="dot",
            line_color="white",
            annotation_text="Soglia"
        )

        fig.update_yaxes(
            tickformat=".0%"
        )

        fig.show()

        fig = px.bar(
            selected_probs,
            x="Threshold",
            y="Probability_pct",
            color="Probability_pct",
            title=(
                f"{r['Giocatore']} — "
                f"{target} Threshold Probabilities"
            ),
            template="plotly_dark"
        )

        fig.update_yaxes(
            range=[0, 100],
            ticksuffix="%"
        )

        fig.show()


def on_player_change(change):

    update_games()
    show_player()


def on_target_change(change):

    defaults = {
        "PTS": 20,
        "REB": 8,
        "AST": 5,
        "3PM": 2,
        "PRA": 30
    }

    threshold_widget.value = (
        defaults[target_widget.value]
    )

    show_player()


update_games()

player_widget.observe(
    on_player_change,
    names="value"
)

target_widget.observe(
    on_target_change,
    names="value"
)

game_widget.observe(
    show_player,
    names="value"
)

threshold_widget.observe(
    show_player,
    names="value"
)

display(
    widgets.VBox([
        widgets.HBox([
            player_widget,
            target_widget
        ]),
        game_widget,
        threshold_widget,
        output
    ])
)

show_player()

# ============================================================
# 18. RIEPILOGO FINALE
# ============================================================

print("\n" + "=" * 60)

print("STEP 18 COMPLETATO!")
print("PROBABILITY ENGINE V3.1")

print("=" * 60)

print(
    "Errori storici:",
    len(history)
)

print(
    "Partite future:",
    summary_df["game_id"].nunique()
)

print(
    "Giocatori:",
    summary_df["player_id"].nunique()
)

print(
    "Distribuzioni generate:",
    len(summary_df)
)

print(
    "Probabilita calcolate:",
    len(probabilities)
)

print(
    "Probabilita archiviate:",
    total_history
)

print("\nFILE SALVATI")

print(OUTPUT_PATH)
print(CSV_PATH)
print(SUMMARY_PATH)
print(archive_path)
print(summary_archive_path)

print("\nTUTTI I CONTROLLI SUPERATI")

print("\nPRONTO PER STEP 19!")


In [ ]:

# ============================================================
# STEP 19 — INJURY & LINEUP INTELLIGENCE
# NBA DATA SCIENCE ANALYTICS
# VERSION 3.2
# ============================================================

import os
import sqlite3
import requests
import numpy as np
import pandas as pd
import plotly.express as px
import ipywidgets as widgets

from google.colab import drive
from IPython.display import display, clear_output
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# ============================================================
# 1. RIPRISTINO AUTOMATICO AMBIENTE
# ============================================================

mounts = [
    "/content/drive",
    "/content/gdrive_nba",
    "/content/nba_drive"
]

BASE_DIR = None

for mount in mounts:

    candidate = os.path.join(
        mount, "MyDrive", "NBA_Data_Science"
    )

    if os.path.isfile(
        os.path.join(candidate, "nba_database.db")
    ):
        BASE_DIR = candidate
        break

if BASE_DIR is None:

    for mount in mounts:

        if os.path.isdir(mount) and os.listdir(mount):
            continue

        drive.mount(mount)

        candidate = os.path.join(
            mount, "MyDrive", "NBA_Data_Science"
        )

        if os.path.isfile(
            os.path.join(candidate, "nba_database.db")
        ):
            BASE_DIR = candidate
            break

if BASE_DIR is None:
    raise FileNotFoundError(
        "Cartella NBA_Data_Science non trovata."
    )

NOW = pd.Timestamp.now(tz="UTC")

DB_PATH = os.path.join(
    BASE_DIR, "nba_database.db"
)

FORECAST_PATH = os.path.join(
    BASE_DIR, "NBA_Forecast_Engine_V3.csv"
)

PROBABILITY_PATH = os.path.join(
    BASE_DIR, "NBA_Probability_Engine_V3.csv"
)

OUTPUT_PATH = os.path.join(
    BASE_DIR, "NBA_Context_Aware_Forecast.xlsx"
)

CSV_PATH = os.path.join(
    BASE_DIR, "NBA_Context_Aware_Forecast.csv"
)

INJURY_PATH = os.path.join(
    BASE_DIR, "NBA_Injury_Report.csv"
)

ARCHIVE_DIR = os.path.join(
    BASE_DIR, "Injury_Archive"
)

os.makedirs(ARCHIVE_DIR, exist_ok=True)

print("Cartella progetto:", BASE_DIR)

# ============================================================
# 2. CONFIGURAZIONE
# ============================================================

TARGETS = {
    "PTS": "pts",
    "REB": "reb",
    "AST": "ast",
    "3PM": "fg3m",
    "PRA": "PRA"
}

MIN_IMPACT_SAMPLE = 5

HISTORY_DAYS = 365

ESPN_INJURIES_URL = (
    "https://site.api.espn.com/apis/site/v2/"
    "sports/basketball/nba/injuries"
)

session = requests.Session()

retry = Retry(
    total=3,
    backoff_factor=1,
    status_forcelist=[
        429, 500, 502, 503, 504
    ],
    allowed_methods=["GET"]
)

session.mount(
    "https://",
    HTTPAdapter(max_retries=retry)
)

session.headers.update({
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json"
})

# ============================================================
# 3. CARICAMENTO FORECAST
# ============================================================

if not os.path.isfile(FORECAST_PATH):

    raise FileNotFoundError(
        "Completa prima STEP 17."
    )

forecast = pd.read_csv(
    FORECAST_PATH,
    dtype={
        "game_id": str,
        "player_id": str
    }
)

forecast["date"] = pd.to_datetime(
    forecast["date"],
    utc=True,
    errors="coerce",
    format="mixed"
)

forecast = forecast[
    forecast["date"] > NOW
].drop_duplicates(
    ["game_id", "player_id"]
).copy()

if forecast.empty:

    raise ValueError(
        "Forecast scaduti. Riesegui STEP 17."
    )

print("Forecast futuri:", len(forecast))

# ============================================================
# 4. DOWNLOAD INJURY REPORT ESPN
# ============================================================

def normalize_status(value):

    value = str(value).strip().lower()

    mapping = {
        "out": "Out",
        "doubtful": "Doubtful",
        "questionable": "Questionable",
        "day-to-day": "Questionable",
        "day to day": "Questionable",
        "probable": "Probable",
        "available": "Available",
        "active": "Available",
        "game time decision": "Questionable",
        "gtd": "Questionable"
    }

    return mapping.get(value, "Unknown")


injury_rows = []

injury_feed_ok = False

print("\nDOWNLOAD INJURY REPORT ESPN...")

try:

    response = session.get(
        ESPN_INJURIES_URL,
        timeout=30
    )

    response.raise_for_status()

    payload = response.json()

    groups = payload.get("injuries")

    if not isinstance(groups, list):

        raise ValueError(
            "Struttura ESPN non riconosciuta."
        )

    for group in groups:

        team = group.get("displayName", "")

        team_info = group.get("team", {})

        if isinstance(team_info, dict):

            team = team_info.get(
                "displayName", team
            )

        items = group.get("injuries", [])

        if not isinstance(items, list):
            continue

        for injury in items:

            athlete = injury.get("athlete", {})

            if not isinstance(athlete, dict):
                continue

            player_id = athlete.get("id")

            if player_id is None:
                continue

            details = injury.get("details") or {}

            if not isinstance(details, dict):
                details = {}

            raw_status = injury.get("status", "")

            if isinstance(raw_status, dict):

                raw_status = raw_status.get(
                    "name",
                    raw_status.get("description", "")
                )

            status = normalize_status(
                raw_status
            )

            injury_rows.append({
                "player_id": str(player_id),
                "Giocatore": athlete.get(
                    "displayName",
                    athlete.get("fullName", "")
                ),
                "Squadra_ESPN": team,
                "Injury_Status": status,
                "Raw_Status": raw_status,
                "Injury_Type": details.get(
                    "type", ""
                ),
                "Injury_Location": details.get(
                    "location", ""
                ),
                "Return_Date": details.get(
                    "returnDate", ""
                ),
                "Comment": injury.get(
                    "longComment",
                    injury.get("shortComment", "")
                ),
                "Report_Date": injury.get(
                    "date", ""
                ),
                "Fetched_At": NOW.isoformat(),
                "Source": ESPN_INJURIES_URL
            })

    injury_feed_ok = True

except Exception as e:

    print("ATTENZIONE: injury feed non disponibile.")
    print(type(e).__name__, str(e))

injury_columns = [
    "player_id",
    "Giocatore",
    "Squadra_ESPN",
    "Injury_Status",
    "Raw_Status",
    "Injury_Type",
    "Injury_Location",
    "Return_Date",
    "Comment",
    "Report_Date",
    "Fetched_At",
    "Source"
]

injuries = pd.DataFrame(
    injury_rows,
    columns=injury_columns
)

if not injuries.empty:

    injuries["_report_time"] = pd.to_datetime(
        injuries["Report_Date"],
        utc=True,
        errors="coerce",
        format="mixed"
    )

    injuries = injuries.sort_values(
        "_report_time",
        na_position="first"
    ).drop_duplicates(
        "player_id",
        keep="last"
    ).drop(
        columns="_report_time"
    )

print("Feed disponibile:", injury_feed_ok)
print("Infortuni identificati:", len(injuries))

# ============================================================
# 5. CARICAMENTO DATABASE STORICO
# ============================================================

with sqlite3.connect(DB_PATH) as conn:

    box = pd.read_sql_query(
        "SELECT * FROM espn_player_boxscores",
        conn
    )

    games = pd.read_sql_query(
        "SELECT * FROM espn_games",
        conn
    )

for data in [box, games]:

    data["game_id"] = (
        data["game_id"].astype(str)
    )

    data["game_date"] = pd.to_datetime(
        data["game_date"],
        utc=True,
        errors="coerce",
        format="mixed"
    )

for col in ["player_id", "team_id"]:

    box[col] = box[col].astype(str)

numeric_columns = [
    "minutes",
    "pts",
    "reb",
    "ast",
    "fg3m",
    "starter",
    "did_not_play"
]

for col in numeric_columns:

    box[col] = pd.to_numeric(
        box[col],
        errors="coerce"
    )

games["completed"] = pd.to_numeric(
    games["completed"],
    errors="coerce"
)

completed_ids = set(
    games.loc[
        (games["completed"] == 1)
        & (games["game_date"] < NOW),
        "game_id"
    ]
)

box = box[
    box["game_id"].isin(completed_ids)
    & (box["game_date"] < NOW)
    & (
        box["game_date"]
        >= NOW - pd.Timedelta(days=HISTORY_DAYS)
    )
].drop_duplicates(
    ["game_id", "player_id"]
).sort_values(
    "game_date"
).copy()

box["PRA"] = (
    box["pts"] + box["reb"] + box["ast"]
)

played = box[
    (box["minutes"] > 0)
    & (box["did_not_play"] == 0)
].copy()

if played.empty:

    raise ValueError(
        "Nessuna presenza storica valida."
    )

print("\nPresenze storiche:", len(played))

# ============================================================
# 6. ANALISI ROTAZIONI E MINUTI
# ============================================================

rotation_rows = []

for player_id, group in played.groupby(
    "player_id"
):

    group = group.sort_values(
        "game_date"
    )

    last = group.iloc[-1]

    # Utilizziamo la squadra piu recente
    # per evitare di mescolare rotazioni
    # precedenti a un eventuale trasferimento.

    current_team = group[
        group["team_id"] == last["team_id"]
    ]

    last5 = current_team.tail(5)
    last10 = current_team.tail(10)
    last20 = current_team.tail(20)

    minutes5 = last5["minutes"].mean()
    minutes10 = last10["minutes"].mean()
    minutes20 = last20["minutes"].mean()

    starter_rate = last10["starter"].mean()

    if pd.isna(starter_rate):

        role = "Unknown"

    elif starter_rate >= 0.7:

        role = "Likely Starter"

    elif starter_rate >= 0.3:

        role = "Rotation / Starter"

    else:

        role = "Bench"

    rotation_rows.append({
        "player_id": player_id,
        "team_id": last["team_id"],
        "Last_Team": last["team_name"],
        "Last_Appearance": (
            last["game_date"].isoformat()
        ),
        "Minutes_Last5": minutes5,
        "Minutes_Last10": minutes10,
        "Minutes_Last20": minutes20,
        "Minutes_Trend": minutes5 - minutes20,
        "Starter_Rate_Last10": starter_rate,
        "Estimated_Role": role,
        "Rotation_Sample": len(last10),
        "PTS_Last10": last10["pts"].mean(),
        "REB_Last10": last10["reb"].mean(),
        "AST_Last10": last10["ast"].mean()
    })

rotations = pd.DataFrame(rotation_rows)

print("Profili rotazione:", len(rotations))

# ============================================================
# 7. INTEGRAZIONE INJURY REPORT E FORECAST
# ============================================================

injury_fields = [
    "player_id",
    "Injury_Status",
    "Raw_Status",
    "Injury_Type",
    "Injury_Location",
    "Return_Date",
    "Comment",
    "Report_Date",
    "Fetched_At"
]

context = forecast.merge(
    injuries[injury_fields],
    on="player_id",
    how="left",
    validate="many_to_one"
)

context = context.merge(
    rotations,
    on="player_id",
    how="left",
    validate="many_to_one"
)

context["Injury_Status"] = (
    context["Injury_Status"].fillna("Unknown")
)

context["Injury_Feed_OK"] = injury_feed_ok

context["Listed_In_Injury_Feed"] = (
    context["player_id"].isin(
        injuries["player_id"]
    )
)

def availability_flag(row):

    status = row["Injury_Status"]

    if status == "Out":
        return "EXCLUDED"

    if status in ["Doubtful", "Questionable"]:
        return "REVIEW_REQUIRED"

    if status == "Probable":
        return "PROBABLE"

    if status == "Available":
        return "AVAILABLE"

    if not row["Injury_Feed_OK"]:
        return "FEED_UNAVAILABLE"

    return "NOT_VERIFIED"


context["Availability_Flag"] = context.apply(
    availability_flag,
    axis=1
)

# ============================================================
# 8. ANALISI IMPATTO ASSENZE COMPAGNI
# ============================================================

# Confronto descrittivo delle prestazioni
# con compagno presente e con compagno
# esplicitamente registrato come DNP.
#
# Non assumiamo che un giocatore assente
# dal boxscore fosse necessariamente infortunato.

impact_rows = []

out_players = context[
    context["Injury_Status"] == "Out"
][[
    "game_id",
    "player_id",
    "Giocatore",
    "team_id"
]].dropna(
    subset=["team_id"]
).drop_duplicates()

for absent in out_players.itertuples(
    index=False
):

    absent_history = box[
        (box["player_id"] == absent.player_id)
        & (box["team_id"] == absent.team_id)
    ]

    games_with = set(
        absent_history.loc[
            (absent_history["minutes"] > 0)
            & (absent_history["did_not_play"] == 0),
            "game_id"
        ]
    )

    games_without = set(
        absent_history.loc[
            absent_history["did_not_play"] == 1,
            "game_id"
        ]
    )

    games_without -= games_with

    if min(
        len(games_with),
        len(games_without)
    ) < MIN_IMPACT_SAMPLE:

        continue

    teammates = context[
        (context["game_id"] == absent.game_id)
        & (context["team_id"] == absent.team_id)
        & (context["player_id"] != absent.player_id)
        & (context["Injury_Status"] != "Out")
    ]

    for player in teammates.itertuples(
        index=False
    ):

        hist = played[
            (played["player_id"] == player.player_id)
            & (played["team_id"] == absent.team_id)
        ]

        with_player = hist[
            hist["game_id"].isin(games_with)
        ]

        without_player = hist[
            hist["game_id"].isin(games_without)
        ]

        if min(
            len(with_player),
            len(without_player)
        ) < MIN_IMPACT_SAMPLE:

            continue

        for target, col in {
            **TARGETS,
            "MIN": "minutes"
        }.items():

            with_values = (
                with_player[col].dropna()
            )

            without_values = (
                without_player[col].dropna()
            )

            if min(
                len(with_values),
                len(without_values)
            ) < MIN_IMPACT_SAMPLE:

                continue

            mean_with = with_values.mean()

            mean_without = (
                without_values.mean()
            )

            delta = mean_without - mean_with

            delta_pct = (
                100 * delta / mean_with
                if mean_with > 0
                else np.nan
            )

            impact_rows.append({
                "game_id": absent.game_id,
                "player_id": player.player_id,
                "Giocatore": player.Giocatore,
                "Absent_Player_ID": absent.player_id,
                "Absent_Player": absent.Giocatore,
                "Target": target,
                "Games_With": len(with_values),
                "Games_Without": len(without_values),
                "Mean_With": mean_with,
                "Mean_Without": mean_without,
                "Historical_Delta": delta,
                "Historical_Delta_pct": delta_pct
            })

impact_columns = [
    "game_id",
    "player_id",
    "Giocatore",
    "Absent_Player_ID",
    "Absent_Player",
    "Target",
    "Games_With",
    "Games_Without",
    "Mean_With",
    "Mean_Without",
    "Historical_Delta",
    "Historical_Delta_pct"
]

impact = pd.DataFrame(
    impact_rows,
    columns=impact_columns
)

print("\nTEammate Impact Analysis")
print("Confronti disponibili:", len(impact))

# ============================================================
# 9. CONTEXT-AWARE FORECAST
# ============================================================

# Non modifichiamo arbitrariamente
# le previsioni ML in base alle assenze.
#
# Manteniamo i forecast originali e
# rimuoviamo quelli dei giocatori Out
# dalla lista delle previsioni utilizzabili.

for target in TARGETS:

    context[f"Context_{target}"] = (
        context[f"Forecast_{target}"]
    )

    context.loc[
        context["Injury_Status"] == "Out",
        f"Context_{target}"
    ] = np.nan

context["Context_Generated_At"] = (
    NOW.isoformat()
)

context["Context_Version"] = "3.2"

context["Forecast_Condition"] = (
    "Condizionato alla partecipazione"
)

context["Impact_Adjustment_Applied"] = False

context["Confirmed_Lineup"] = False

# Conteggio compagni indisponibili.

out_counts = context[
    context["Injury_Status"] == "Out"
].groupby(
    ["game_id", "Squadra"]
)["player_id"].nunique().rename(
    "Listed_Out_Teammates"
).reset_index()

context = context.merge(
    out_counts,
    on=["game_id", "Squadra"],
    how="left"
)

context["Listed_Out_Teammates"] = (
    context["Listed_Out_Teammates"]
    .fillna(0)
    .astype(int)
)

context.loc[
    context["Injury_Status"] == "Out",
    "Listed_Out_Teammates"
] -= 1

context["Listed_Out_Teammates"] = (
    context["Listed_Out_Teammates"].clip(lower=0)
)

if not injury_feed_ok:

    context["Listed_Out_Teammates"] = np.nan

# ============================================================
# 10. INTEGRAZIONE PROBABILITA STEP 18
# ============================================================

probability_context = pd.DataFrame()

if os.path.isfile(PROBABILITY_PATH):

    probabilities = pd.read_csv(
        PROBABILITY_PATH,
        dtype={
            "game_id": str,
            "player_id": str
        }
    )

    metadata = context[[
        "game_id",
        "player_id",
        "generated_at",
        "Injury_Status",
        "Availability_Flag",
        "Estimated_Role",
        "Listed_Out_Teammates"
    ]].rename(columns={
        "generated_at": "Current_Forecast_Generated_At"
    })

    probability_context = probabilities.merge(
        metadata,
        on=["game_id", "player_id"],
        how="inner",
        validate="many_to_one"
    )

    # Verifica che le probabilita derivino
    # dalla stessa versione del forecast.

    if "Forecast_Generated_At" in (
        probability_context.columns
    ):

        source_time = pd.to_datetime(
            probability_context[
                "Forecast_Generated_At"
            ],
            utc=True,
            errors="coerce",
            format="mixed"
        )

        current_time = pd.to_datetime(
            probability_context[
                "Current_Forecast_Generated_At"
            ],
            utc=True,
            errors="coerce",
            format="mixed"
        )

        probability_context = (
            probability_context[
                source_time.notna()
                & (source_time == current_time)
            ].copy()
        )

    else:

        print(
            "Probabilita senza timestamp sorgente: "
            "riesegui STEP 18 versione 3.1."
        )

        probability_context = pd.DataFrame()

    if not probability_context.empty:

        probability_context[
            "Conditional_Probability"
        ] = probability_context[
            "Probability_At_Least"
        ]

        probability_context.loc[
            probability_context["Injury_Status"] == "Out",
            "Conditional_Probability"
        ] = np.nan

        probability_context[
            "Conditional_Probability_pct"
        ] = (
            100
            * probability_context[
                "Conditional_Probability"
            ]
        )

        probability_context[
            "Participation_Probability"
        ] = np.nan

        probability_context[
            "Probability_Condition"
        ] = "Se il giocatore partecipa"

print(
    "\nProbabilita integrate:",
    len(probability_context)
)

# ============================================================
# 11. ANALISI QUINTETTI STIMATI
# ============================================================

lineup = context[
    context["Injury_Status"] != "Out"
].copy()

lineup = lineup.sort_values(
    [
        "game_id",
        "Squadra",
        "Starter_Rate_Last10",
        "Minutes_Last10"
    ],
    ascending=[
        True, True, False, False
    ]
)

lineup = lineup.groupby(
    ["game_id", "Squadra"],
    group_keys=False
).head(5).copy()

lineup["Lineup_Status"] = (
    "Proiezione storica non confermata"
)

lineup_columns = [
    "game_id",
    "date",
    "Squadra",
    "Avversario",
    "Giocatore",
    "Injury_Status",
    "Estimated_Role",
    "Starter_Rate_Last10",
    "Minutes_Last10",
    "Lineup_Status"
]

lineup = lineup[lineup_columns]

# ============================================================
# 12. CONTROLLO QUALITA'
# ============================================================

if context.duplicated(
    ["game_id", "player_id"]
).any():

    raise ValueError(
        "Forecast duplicati."
    )

out_mask = (
    context["Injury_Status"] == "Out"
)

for target in TARGETS:

    if not context.loc[
        out_mask,
        f"Context_{target}"
    ].isna().all():

        raise ValueError(
            "Giocatore Out non escluso."
        )

if not probability_context.empty:

    valid_p = probability_context[
        "Conditional_Probability"
    ].dropna()

    if not valid_p.between(0, 1).all():

        raise ValueError(
            "Probabilita non valide."
        )

FINISHED_AT = pd.Timestamp.now(tz="UTC")

context = context[
    context["date"] > FINISHED_AT
].copy()

valid_games = set(context["game_id"])

lineup = lineup[
    lineup["game_id"].isin(valid_games)
].copy()

impact = impact[
    impact["game_id"].isin(valid_games)
].copy()

if not probability_context.empty:

    probability_context = (
        probability_context[
            probability_context["game_id"]
            .isin(valid_games)
        ].copy()
    )

if context.empty:

    raise ValueError(
        "Nessun forecast futuro valido."
    )

print("\nCONTROLLO QUALITA' SUPERATO!")

# ============================================================
# 13. ARCHIVIAZIONE SQLITE
# ============================================================

with sqlite3.connect(
    DB_PATH,
    timeout=60
) as conn:

    conn.execute("""
        CREATE TABLE IF NOT EXISTS
        nba_injury_snapshots (
            player_id TEXT NOT NULL,
            fetched_at TEXT NOT NULL,
            player_name TEXT,
            team_name TEXT,
            injury_status TEXT,
            raw_status TEXT,
            injury_type TEXT,
            injury_location TEXT,
            report_date TEXT,
            source TEXT,
            PRIMARY KEY (
                player_id,
                fetched_at
            )
        )
    """)

    records = []

    for r in injuries.itertuples(
        index=False
    ):

        records.append((
            r.player_id,
            r.Fetched_At,
            r.Giocatore,
            r.Squadra_ESPN,
            r.Injury_Status,
            str(r.Raw_Status),
            str(r.Injury_Type),
            str(r.Injury_Location),
            str(r.Report_Date),
            r.Source
        ))

    conn.executemany("""
        INSERT OR IGNORE INTO
        nba_injury_snapshots
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, records)

    conn.execute("""
        CREATE TABLE IF NOT EXISTS
        nba_injury_feed_runs (
            fetched_at TEXT PRIMARY KEY,
            feed_ok INTEGER,
            injury_count INTEGER,
            source TEXT
        )
    """)

    conn.execute("""
        INSERT OR IGNORE INTO
        nba_injury_feed_runs
        VALUES (?, ?, ?, ?)
    """, (
        NOW.isoformat(),
        int(injury_feed_ok),
        len(injuries),
        ESPN_INJURIES_URL
    ))

    conn.execute("""
        CREATE TABLE IF NOT EXISTS
        nba_availability_history (
            game_id TEXT NOT NULL,
            player_id TEXT NOT NULL,
            generated_at TEXT NOT NULL,
            game_date TEXT,
            player_name TEXT,
            team_name TEXT,
            injury_status TEXT,
            availability_flag TEXT,
            feed_ok INTEGER,
            PRIMARY KEY (
                game_id,
                player_id,
                generated_at
            )
        )
    """)

    availability_records = []

    for r in context.itertuples(
        index=False
    ):

        availability_records.append((
            r.game_id,
            r.player_id,
            NOW.isoformat(),
            r.date.isoformat(),
            r.Giocatore,
            r.Squadra,
            r.Injury_Status,
            r.Availability_Flag,
            int(injury_feed_ok)
        ))

    conn.executemany("""
        INSERT OR IGNORE INTO
        nba_availability_history
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, availability_records)

    total_injury_history = conn.execute("""
        SELECT COUNT(*)
        FROM nba_injury_snapshots
    """).fetchone()[0]

# ============================================================
# 14. ESPORTAZIONE FILE
# ============================================================

stamp = NOW.strftime(
    "%Y%m%d_%H%M%S_%f"
)

export = context.copy()

export["date"] = export["date"].map(
    lambda x: x.isoformat()
)

lineup_export = lineup.copy()

lineup_export["date"] = (
    lineup_export["date"].map(
        lambda x: x.isoformat()
    )
)

injuries.to_csv(
    INJURY_PATH,
    index=False,
    encoding="utf-8-sig"
)

injuries.to_csv(
    os.path.join(
        ARCHIVE_DIR,
        f"NBA_Injuries_{stamp}.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

export.to_csv(
    CSV_PATH + ".tmp",
    index=False,
    encoding="utf-8-sig"
)

os.replace(
    CSV_PATH + ".tmp",
    CSV_PATH
)

export.to_csv(
    os.path.join(
        ARCHIVE_DIR,
        f"NBA_Context_{stamp}.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

impact.to_csv(
    os.path.join(
        BASE_DIR,
        "NBA_Teammate_Impact.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

lineup_export.to_csv(
    os.path.join(
        BASE_DIR,
        "NBA_Estimated_Lineups.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

probability_context.to_csv(
    os.path.join(
        BASE_DIR,
        "NBA_Context_Probabilities.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

status_report = context.groupby(
    [
        "Injury_Status",
        "Availability_Flag"
    ],
    as_index=False
).agg(
    Forecast_Count=("player_id", "size"),
    Unique_Players=("player_id", "nunique")
)

with pd.ExcelWriter(
    OUTPUT_PATH + ".tmp.xlsx",
    engine="openpyxl"
) as writer:

    sheets = {
        "Context Forecast": export,
        "Injury Report": injuries,
        "Estimated Lineups": lineup_export,
        "Rotation Analysis": rotations,
        "Teammate Impact": impact,
        "Context Probabilities": probability_context,
        "Availability Summary": status_report
    }

    for name, data in sheets.items():

        data.to_excel(
            writer,
            sheet_name=name,
            index=False
        )

        sheet = writer.sheets[name]

        sheet.freeze_panes = "A2"
        sheet.auto_filter.ref = sheet.dimensions

os.replace(
    OUTPUT_PATH + ".tmp.xlsx",
    OUTPUT_PATH
)

# ============================================================
# 15. VISUALIZZAZIONE
# ============================================================

print("\nAVAILABILITY SUMMARY")

display(status_report)

fig = px.bar(
    status_report,
    x="Injury_Status",
    y="Forecast_Count",
    color="Injury_Status",
    title="NBA — Player Availability",
    template="plotly_dark",
    color_discrete_map={
        "Available": "#22c55e",
        "Probable": "#84cc16",
        "Questionable": "#eab308",
        "Doubtful": "#f97316",
        "Out": "#ef4444",
        "Unknown": "#64748b"
    }
)

fig.show()

print("\nGIOCATORI DA VERIFICARE")

review = context[
    context["Injury_Status"].isin([
        "Out",
        "Doubtful",
        "Questionable"
    ])
].sort_values(
    "Forecast_PTS",
    ascending=False
)

display(
    review[[
        "Giocatore",
        "Squadra",
        "Avversario",
        "Injury_Status",
        "Injury_Type",
        "Forecast_PTS",
        "Availability_Flag"
    ]].round(2)
)

# ============================================================
# 16. PLAYER CONTEXT EXPLORER
# ============================================================

player_options = context[[
    "player_id",
    "Giocatore"
]].drop_duplicates(
    "player_id"
).sort_values(
    "Giocatore"
)

players = [
    (
        f"{r.Giocatore} ({r.player_id})",
        r.player_id
    )
    for r in player_options.itertuples(
        index=False
    )
]

player_widget = widgets.Dropdown(
    options=players,
    description="Player:"
)

output = widgets.Output()

def show_player(change=None):

    with output:

        clear_output(wait=True)

        selected = context[
            context["player_id"]
            == player_widget.value
        ].sort_values("date")

        if selected.empty:
            return

        r = selected.iloc[0]

        print("=" * 60)
        print(r["Giocatore"].upper())
        print("=" * 60)

        print("Squadra:", r["Squadra"])
        print("Avversario:", r["Avversario"])

        print(
            "Injury Status:",
            r["Injury_Status"]
        )

        print(
            "Availability:",
            r["Availability_Flag"]
        )

        print(
            "Injury Type:",
            r["Injury_Type"]
        )

        print(
            "Ruolo stimato:",
            r["Estimated_Role"]
        )

        print(
            "Minuti ultime 10:",
            round(r["Minutes_Last10"], 2)
        )

        print(
            "Compagni Out:",
            r["Listed_Out_Teammates"]
        )

        print("\nFORECAST")

        rows = []

        for target in TARGETS:

            rows.append({
                "Target": target,
                "Forecast_ML": r[
                    f"Forecast_{target}"
                ],
                "Context_Forecast": r[
                    f"Context_{target}"
                ]
            })

        display(
            pd.DataFrame(rows).round(2)
        )

        player_impact = impact[
            (impact["player_id"] == r["player_id"])
            & (impact["game_id"] == r["game_id"])
        ]

        if not player_impact.empty:

            print("\nIMPATTO STORICO COMPAGNI ASSENTI")

            display(
                player_impact[[
                    "Absent_Player",
                    "Target",
                    "Games_With",
                    "Games_Without",
                    "Mean_With",
                    "Mean_Without",
                    "Historical_Delta"
                ]].round(2)
            )

            fig = px.bar(
                player_impact,
                x="Target",
                y="Historical_Delta",
                color="Absent_Player",
                barmode="group",
                title="Historical Teammate Impact",
                template="plotly_dark"
            )

            fig.add_hline(y=0)
            fig.show()

        else:

            print(
                "\nCampione assenze insufficiente "
                "o nessun compagno Out identificato."
            )

        if not probability_context.empty:

            selected_probs = probability_context[
                (
                    probability_context["player_id"]
                    == r["player_id"]
                )
                & (
                    probability_context["game_id"]
                    == r["game_id"]
                )
            ]

            if not selected_probs.empty:

                print("\nPROBABILITA CONDIZIONATE")

                display(
                    selected_probs[[
                        "Target",
                        "Threshold",
                        "Conditional_Probability_pct"
                    ]].round(2)
                )

player_widget.observe(
    show_player,
    names="value"
)

display(
    player_widget,
    output
)

show_player()

# ============================================================
# 17. RIEPILOGO FINALE
# ============================================================

print("\n" + "=" * 60)
print("STEP 19 COMPLETATO!")
print("INJURY & LINEUP INTELLIGENCE V3.2")
print("=" * 60)

print("Feed ESPN disponibile:", injury_feed_ok)

print(
    "Partite future:",
    context["game_id"].nunique()
)

print(
    "Giocatori:",
    context["player_id"].nunique()
)

print(
    "Forecast esclusi per Out:",
    int(
        (context["Injury_Status"] == "Out").sum()
    )
)

print(
    "Forecast da verificare:",
    int(
        context["Injury_Status"].isin([
            "Questionable", "Doubtful"
        ]).sum()
    )
)

print(
    "Disponibilita sconosciuta:",
    int(
        (context["Injury_Status"] == "Unknown").sum()
    )
)

print(
    "Quintetti stimati - righe:",
    len(lineup)
)

print(
    "Analisi impatto:",
    len(impact)
)

print(
    "Injury snapshot storici:",
    total_injury_history
)

print("\nFILE SALVATI")

print(OUTPUT_PATH)
print(CSV_PATH)
print(INJURY_PATH)

if not injury_feed_ok:

    print(
        "\nATTENZIONE: modulo injury degradato. "
        "Verificare la disponibilita del feed ESPN."
    )

print("\nPRONTO PER STEP 20!")


In [ ]:

# ============================================================
# STEP 20 — AUTOMATIC MODEL RETRAINING
# NBA DATA SCIENCE ANALYTICS
# VERSION 4.0
# ============================================================

import os
import json
import copy
import shutil
import hashlib
import sqlite3
import joblib
import numpy as np
import pandas as pd
import plotly.express as px

from pathlib import Path
from google.colab import drive
from IPython.display import display

from sklearn.base import clone
from sklearn.ensemble import (
    RandomForestRegressor,
    HistGradientBoostingRegressor
)
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

# ============================================================
# 1. RIPRISTINO AUTOMATICO AMBIENTE
# ============================================================

mounts = [
    "/content/drive",
    "/content/gdrive_nba",
    "/content/nba_drive"
]

BASE_DIR = None

for mount in mounts:

    candidate = os.path.join(
        mount, "MyDrive", "NBA_Data_Science"
    )

    if os.path.isfile(
        os.path.join(candidate, "nba_database.db")
    ):
        BASE_DIR = candidate
        break

if BASE_DIR is None:

    for mount in mounts:

        if os.path.isdir(mount) and os.listdir(mount):
            continue

        drive.mount(mount)

        candidate = os.path.join(
            mount, "MyDrive", "NBA_Data_Science"
        )

        if os.path.isfile(
            os.path.join(candidate, "nba_database.db")
        ):
            BASE_DIR = candidate
            break

if BASE_DIR is None:

    raise FileNotFoundError(
        "Cartella NBA_Data_Science non trovata."
    )

NOW = pd.Timestamp.now(tz="UTC")

VERSION_ID = NOW.strftime(
    "%Y%m%d_%H%M%S_%f"
)

print("Google Drive collegato.")
print("Cartella:", BASE_DIR)

# ============================================================
# 2. CONFIGURAZIONE
# ============================================================

DATA_PATH = os.path.join(
    BASE_DIR,
    "NBA_Advanced_ML_Features.csv"
)

MODEL_PATH = os.path.join(
    BASE_DIR,
    "nba_player_models_v3.joblib"
)

HISTORY_PATH = os.path.join(
    BASE_DIR,
    "NBA_Advanced_Test_Predictions.csv"
)

DB_PATH = os.path.join(
    BASE_DIR,
    "nba_database.db"
)

REGISTRY_DIR = os.path.join(
    BASE_DIR,
    "Model_Registry"
)

REPORT_PATH = os.path.join(
    BASE_DIR,
    "NBA_Model_Retraining_Report.xlsx"
)

CANDIDATE_PATH = os.path.join(
    REGISTRY_DIR,
    f"nba_player_models_candidate_{VERSION_ID}.joblib"
)

os.makedirs(
    REGISTRY_DIR,
    exist_ok=True
)

MIN_NEW_DAYS = 30
MIN_TRAIN_ROWS = 1000
MIN_VALIDATION_ROWS = 200
MIN_TEST_ROWS = 200

DRIFT_THRESHOLD = 0.15
MIN_IMPROVEMENT = 0.02

FORCE_RETRAIN = False

RANDOM_STATE = 42

TARGETS = {
    "PTS": "pts",
    "REB": "reb",
    "AST": "ast",
    "3PM": "fg3m",
    "PRA": "PRA"
}

for path in [DATA_PATH, MODEL_PATH, HISTORY_PATH]:

    if not os.path.isfile(path):

        raise FileNotFoundError(
            f"File mancante: {path}"
        )

# ============================================================
# 3. CARICAMENTO MODELLO CHAMPION
# ============================================================

champion = joblib.load(MODEL_PATH)

if not isinstance(champion, dict):

    raise TypeError(
        "Formato modello V3 non riconosciuto."
    )

required_keys = [
    "models",
    "features",
    "targets"
]

for key in required_keys:

    if key not in champion:

        raise KeyError(
            f"Chiave modello mancante: {key}"
        )

FEATURES = list(champion["features"])

print("\nMODELLO CHAMPION")

print(
    "Versione:",
    champion.get("model_version", "V3")
)

print("Numero feature:", len(FEATURES))

# ============================================================
# 4. CARICAMENTO FEATURE ENGINEERING
# ============================================================

data = pd.read_csv(
    DATA_PATH,
    dtype={
        "game_id": str,
        "player_id": str
    }
)

data["game_date"] = pd.to_datetime(
    data["game_date"],
    utc=True,
    errors="coerce",
    format="mixed"
)

if "PRA" not in data.columns:

    data["PRA"] = (
        pd.to_numeric(data["pts"], errors="coerce")
        + pd.to_numeric(data["reb"], errors="coerce")
        + pd.to_numeric(data["ast"], errors="coerce")
    )

missing_features = [
    feature for feature in FEATURES
    if feature not in data.columns
]

if missing_features:

    raise ValueError(
        f"Feature mancanti: {missing_features}"
    )

# Evita l'inclusione accidentale
# dei target tra le feature.

for target in TARGETS.values():

    if target in FEATURES:

        raise ValueError(
            f"Possibile target leakage: {target}"
        )

for col in set(
    FEATURES + list(TARGETS.values())
):

    data[col] = pd.to_numeric(
        data[col],
        errors="coerce"
    )

data = data.replace(
    [np.inf, -np.inf],
    np.nan
)

data = data[
    data["game_date"].notna()
    & (data["game_date"] < NOW)
].drop_duplicates(
    ["game_id", "player_id"]
).sort_values(
    "game_date"
).reset_index(drop=True)

data["NBA_Day"] = (
    data["game_date"]
    .dt.tz_convert("America/New_York")
    .dt.strftime("%Y-%m-%d")
)

print("\nDATASET DISPONIBILE")

print("Osservazioni:", len(data))
print("Giocatori:", data["player_id"].nunique())
print("Partite:", data["game_id"].nunique())
print("Ultimo dato:", data["game_date"].max())

# ============================================================
# 5. VERIFICA AGGIORNAMENTO DATABASE
# ============================================================

with sqlite3.connect(DB_PATH) as conn:

    latest_database = pd.read_sql_query("""
        SELECT MAX(game_date) AS latest_date
        FROM espn_games
        WHERE completed = 1
    """, conn)

latest_db_date = pd.to_datetime(
    latest_database.iloc[0]["latest_date"],
    utc=True,
    errors="coerce"
)

latest_feature_date = data["game_date"].max()

if pd.notna(latest_db_date):

    lag_days = (
        latest_db_date.normalize()
        - latest_feature_date.normalize()
    ).days

    print("\nRitardo feature rispetto al DB:", lag_days)

    if lag_days > 1:

        print(
            "ATTENZIONE: aggiorna STEP 15 "
            "prima del prossimo retraining."
        )

# ============================================================
# 6. STORICO PERFORMANCE CHAMPION
# ============================================================

history = pd.read_csv(
    HISTORY_PATH,
    dtype={
        "game_id": str,
        "player_id": str
    }
)

history["game_date"] = pd.to_datetime(
    history["game_date"],
    utc=True,
    errors="coerce",
    format="mixed"
)

for col in ["Actual", "Forecast"]:

    history[col] = pd.to_numeric(
        history[col],
        errors="coerce"
    )

history = history.replace(
    [np.inf, -np.inf],
    np.nan
).dropna(
    subset=[
        "game_date",
        "Target",
        "Actual",
        "Forecast"
    ]
)

history = history[
    history["game_date"] < NOW
].copy()

history["Absolute_Error"] = (
    history["Actual"]
    - history["Forecast"]
).abs()

history["NBA_Day"] = (
    history["game_date"]
    .dt.tz_convert("America/New_York")
    .dt.strftime("%Y-%m-%d")
)

historical_days = sorted(
    history["NBA_Day"].unique()
)

if not historical_days:

    raise ValueError(
        "Storico valutazione vuoto."
    )

# Usiamo la fine del test precedente
# come limite conservativo dei dati
# gia utilizzati per valutare il Champion.

previous_cutoff = historical_days[-1]

new_days = sorted(
    day for day in data["NBA_Day"].unique()
    if day > previous_cutoff
)

print("\nCONTROLLO NUOVI DATI")

print("Fine test precedente:", previous_cutoff)
print("Nuovi giorni NBA:", len(new_days))

# ============================================================
# 7. DRIFT DETECTION
# ============================================================

drift_rows = []

if len(historical_days) >= 20:

    split = int(
        len(historical_days) * 0.70
    )

    reference_days = historical_days[:split]
    recent_days = historical_days[split:]

    for target in TARGETS:

        ref = history[
            (history["Target"] == target)
            & history["NBA_Day"].isin(reference_days)
        ]

        recent = history[
            (history["Target"] == target)
            & history["NBA_Day"].isin(recent_days)
        ]

        if min(len(ref), len(recent)) < 100:
            continue

        reference_mae = ref["Absolute_Error"].mean()
        recent_mae = recent["Absolute_Error"].mean()

        drift = (
            recent_mae / reference_mae - 1
            if reference_mae > 0
            else np.nan
        )

        drift_rows.append({
            "Target": target,
            "Reference_MAE": reference_mae,
            "Recent_MAE": recent_mae,
            "Drift_pct": 100 * drift,
            "Reference_Sample": len(ref),
            "Recent_Sample": len(recent),
            "Drift_Detected": bool(
                drift > DRIFT_THRESHOLD
            )
        })

drift_report = pd.DataFrame(
    drift_rows,
    columns=[
        "Target",
        "Reference_MAE",
        "Recent_MAE",
        "Drift_pct",
        "Reference_Sample",
        "Recent_Sample",
        "Drift_Detected"
    ]
)

print("\nDRIFT REPORT")

display(drift_report.round(4))

drift_detected = bool(
    drift_report["Drift_Detected"].any()
)

# ============================================================
# 8. DECISIONE RETRAINING
# ============================================================

enough_new_data = (
    len(new_days) >= MIN_NEW_DAYS
)

retrain_required = (
    enough_new_data
    and (
        drift_detected
        or FORCE_RETRAIN
        or len(new_days) >= 60
    )
)

print("\nRETRAINING DECISION")

print("Drift storico:", drift_detected)
print("Nuovi dati sufficienti:", enough_new_data)
print("Retraining richiesto:", retrain_required)

# ============================================================
# 9. DEFINIZIONE MODELLI CANDIDATI
# ============================================================

def build_candidates():

    return {
        "Ridge": make_pipeline(
            SimpleImputer(strategy="median"),
            StandardScaler(),
            Ridge(alpha=10.0)
        ),

        "Random Forest": make_pipeline(
            SimpleImputer(strategy="median"),
            RandomForestRegressor(
                n_estimators=150,
                max_depth=12,
                min_samples_leaf=10,
                n_jobs=-1,
                random_state=RANDOM_STATE
            )
        ),

        "Hist Gradient Boosting": make_pipeline(
            SimpleImputer(strategy="median"),
            HistGradientBoostingRegressor(
                max_iter=200,
                learning_rate=0.05,
                max_leaf_nodes=31,
                min_samples_leaf=20,
                l2_regularization=1.0,
                early_stopping=False,
                random_state=RANDOM_STATE
            )
        )
    }

# ============================================================
# 10. FUNZIONI DI VALUTAZIONE
# ============================================================

def evaluate(y_true, y_pred):

    y_pred = np.maximum(
        np.asarray(y_pred, dtype=float),
        0
    )

    if not np.isfinite(y_pred).all():

        raise ValueError(
            "Predizioni non finite."
        )

    return {
        "MAE": mean_absolute_error(
            y_true, y_pred
        ),

        "RMSE": np.sqrt(
            mean_squared_error(
                y_true, y_pred
            )
        ),

        "Bias": np.mean(
            y_pred - np.asarray(y_true)
        )
    }


def champion_predict(target, X):

    entry = champion["models"][target]

    if isinstance(entry, dict):

        model = entry.get("model")

    else:

        model = entry

    if model is None:

        raise ValueError(
            f"Champion {target}: baseline "
            "senza estimatore compatibile."
        )

    return np.maximum(
        model.predict(X),
        0
    )

# ============================================================
# 11. CHAMPION VS CHALLENGER
# ============================================================

validation_rows = []
comparison_rows = []
prediction_rows = []

candidate_bundle = copy.deepcopy(champion)

accepted_targets = []

if retrain_required:

    # I giorni nuovi vengono divisi
    # cronologicamente in validation e test.

    validation_split = len(new_days) // 2

    validation_days = new_days[:validation_split]
    test_days = new_days[validation_split:]

    train = data[
        data["NBA_Day"] <= previous_cutoff
    ].copy()

    validation = data[
        data["NBA_Day"].isin(validation_days)
    ].copy()

    test = data[
        data["NBA_Day"].isin(test_days)
    ].copy()

    print("\nTEMPORAL SPLIT")

    print("Training:", len(train))
    print("Validation:", len(validation))
    print("Independent Test:", len(test))

    assert (
        train["game_date"].max()
        < validation["game_date"].min()
    )

    assert (
        validation["game_date"].max()
        < test["game_date"].min()
    )

    for target, target_col in TARGETS.items():

        print("\n" + "=" * 50)
        print("TRAINING TARGET:", target)

        tr = train.dropna(
            subset=[target_col]
        )

        va = validation.dropna(
            subset=[target_col]
        )

        te = test.dropna(
            subset=[target_col]
        )

        if (
            len(tr) < MIN_TRAIN_ROWS
            or len(va) < MIN_VALIDATION_ROWS
            or len(te) < MIN_TEST_ROWS
        ):

            print("Campione insufficiente.")
            continue

        X_train = tr[FEATURES]
        y_train = tr[target_col]

        X_val = va[FEATURES]
        y_val = va[target_col]

        X_test = te[FEATURES]
        y_test = te[target_col]

        best_name = None
        best_model = None
        best_mae = np.inf

        for name, model in build_candidates().items():

            print("Addestramento:", name)

            model.fit(
                X_train,
                y_train
            )

            pred = model.predict(X_val)

            metrics = evaluate(
                y_val,
                pred
            )

            validation_rows.append({
                "Target": target,
                "Model": name,
                **metrics
            })

            print(
                "Validation MAE:",
                round(metrics["MAE"], 4)
            )

            if metrics["MAE"] < best_mae:

                best_mae = metrics["MAE"]
                best_name = name
                best_model = model

        # Riaddestramento del candidato
        # selezionato su train + validation.
        # Il test rimane escluso.

        final_train = pd.concat(
            [tr, va],
            ignore_index=True
        )

        challenger = clone(best_model)

        challenger.fit(
            final_train[FEATURES],
            final_train[target_col]
        )

        challenger_pred = np.maximum(
            challenger.predict(X_test),
            0
        )

        try:

            champion_pred = champion_predict(
                target,
                X_test
            )

        except Exception as e:

            print(
                "Confronto non disponibile:",
                str(e)
            )

            continue

        old_metrics = evaluate(
            y_test,
            champion_pred
        )

        new_metrics = evaluate(
            y_test,
            challenger_pred
        )

        improvement = (
            1 - new_metrics["MAE"] / old_metrics["MAE"]
            if old_metrics["MAE"] > 0
            else 0
        )

        accepted = (
            improvement >= MIN_IMPROVEMENT
            and new_metrics["RMSE"] <= old_metrics["RMSE"]
        )

        comparison_rows.append({
            "Target": target,
            "Challenger": best_name,
            "Champion_MAE": old_metrics["MAE"],
            "Challenger_MAE": new_metrics["MAE"],
            "Champion_RMSE": old_metrics["RMSE"],
            "Challenger_RMSE": new_metrics["RMSE"],
            "Improvement_pct": improvement * 100,
            "Test_Sample": len(te),
            "Accepted": accepted
        })

        print(
            "Champion MAE:",
            round(old_metrics["MAE"], 4)
        )

        print(
            "Challenger MAE:",
            round(new_metrics["MAE"], 4)
        )

        print(
            "Miglioramento:",
            round(100 * improvement, 2),
            "%"
        )

        if accepted:

            old_entry = champion["models"][target]

            if isinstance(old_entry, dict):

                new_entry = copy.deepcopy(old_entry)

                new_entry["model"] = challenger
                new_entry["name"] = best_name

            else:

                new_entry = challenger

            candidate_bundle["models"][target] = (
                new_entry
            )

            accepted_targets.append(target)

            print("CANDIDATO APPROVATO")

        else:

            print("CHAMPION MANTENUTO")

        for i, (_, row) in enumerate(te.iterrows()):

            prediction_rows.append({
                "game_id": row["game_id"],
                "player_id": row["player_id"],
                "game_date": row["game_date"].isoformat(),
                "Target": target,
                "Actual": float(row[target_col]),
                "Champion_Forecast": float(
                    champion_pred[i]
                ),
                "Challenger_Forecast": float(
                    challenger_pred[i]
                )
            })

else:

    print(
        "\nRetraining non necessario "
        "o nuovi dati insufficienti."
    )

    print(
        "Il modello V3 viene mantenuto."
    )

# ============================================================
# 12. MODEL REGISTRY
# ============================================================

comparison_report = pd.DataFrame(
    comparison_rows
)

validation_report = pd.DataFrame(
    validation_rows
)

test_predictions = pd.DataFrame(
    prediction_rows
)

candidate_saved = False

def file_hash(path):

    digest = hashlib.sha256()

    with open(path, "rb") as f:

        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):

            digest.update(block)

    return digest.hexdigest()


if accepted_targets:

    candidate_bundle["model_version"] = (
        f"V4_candidate_{VERSION_ID}"
    )

    candidate_bundle["retraining_metadata"] = {
        "generated_at": NOW.isoformat(),
        "parent_model": os.path.basename(MODEL_PATH),
        "parent_sha256": file_hash(MODEL_PATH),
        "accepted_targets": accepted_targets,
        "training_end": validation_days[-1],
        "evaluation_end": test_days[-1],
        "status": "PENDING_COMPATIBILITY_CHECK"
    }

    joblib.dump(
        candidate_bundle,
        CANDIDATE_PATH + ".tmp",
        compress=3
    )

    os.replace(
        CANDIDATE_PATH + ".tmp",
        CANDIDATE_PATH
    )

    # Verifica integrita del file.

    restored = joblib.load(CANDIDATE_PATH)

    assert restored["features"] == champion["features"]

    backup_path = os.path.join(
        REGISTRY_DIR,
        f"champion_backup_{VERSION_ID}.joblib"
    )

    shutil.copy2(
        MODEL_PATH,
        backup_path
    )

    candidate_saved = True

    print("\nCANDIDATO SALVATO")
    print(CANDIDATE_PATH)

# ============================================================
# 13. ARCHIVIAZIONE SQLITE
# ============================================================

with sqlite3.connect(
    DB_PATH,
    timeout=60
) as conn:

    conn.execute("""
        CREATE TABLE IF NOT EXISTS
        nba_model_retraining_runs (
            run_id TEXT PRIMARY KEY,
            generated_at TEXT,
            new_days INTEGER,
            drift_detected INTEGER,
            retrain_required INTEGER,
            candidate_saved INTEGER,
            accepted_targets TEXT,
            candidate_path TEXT
        )
    """)

    conn.execute("""
        INSERT INTO nba_model_retraining_runs
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        VERSION_ID,
        NOW.isoformat(),
        len(new_days),
        int(drift_detected),
        int(retrain_required),
        int(candidate_saved),
        json.dumps(accepted_targets),
        CANDIDATE_PATH if candidate_saved else None
    ))

    conn.execute("""
        CREATE TABLE IF NOT EXISTS
        nba_model_registry (
            version_id TEXT PRIMARY KEY,
            created_at TEXT,
            model_path TEXT,
            sha256 TEXT,
            status TEXT,
            accepted_targets TEXT
        )
    """)

    if candidate_saved:

        conn.execute("""
            INSERT INTO nba_model_registry
            VALUES (?, ?, ?, ?, ?, ?)
        """, (
            VERSION_ID,
            NOW.isoformat(),
            CANDIDATE_PATH,
            file_hash(CANDIDATE_PATH),
            "PENDING_COMPATIBILITY_CHECK",
            json.dumps(accepted_targets)
        ))

# ============================================================
# 14. ESPORTAZIONE REPORT
# ============================================================

run_summary = pd.DataFrame([{
    "Run_ID": VERSION_ID,
    "Generated_At": NOW.isoformat(),
    "Champion_Version": champion.get(
        "model_version", "V3"
    ),
    "Historical_Rows": len(data),
    "New_Days": len(new_days),
    "Drift_Detected": drift_detected,
    "Retraining_Required": retrain_required,
    "Candidate_Saved": candidate_saved,
    "Accepted_Targets": ", ".join(
        accepted_targets
    ),
    "Production_Model_Changed": False
}])

with pd.ExcelWriter(
    REPORT_PATH + ".tmp.xlsx",
    engine="openpyxl"
) as writer:

    sheets = {
        "Run Summary": run_summary,
        "Drift Detection": drift_report,
        "Validation": validation_report,
        "Champion vs Challenger": comparison_report,
        "Independent Test": test_predictions
    }

    for name, frame in sheets.items():

        frame.to_excel(
            writer,
            sheet_name=name,
            index=False
        )

        sheet = writer.sheets[name]

        sheet.freeze_panes = "A2"
        sheet.auto_filter.ref = sheet.dimensions

os.replace(
    REPORT_PATH + ".tmp.xlsx",
    REPORT_PATH
)

registry_log = os.path.join(
    REGISTRY_DIR,
    f"Retraining_Run_{VERSION_ID}.json"
)

with open(
    registry_log,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        run_summary.iloc[0].to_dict(),
        f,
        indent=4,
        default=str
    )

# ============================================================
# 15. DASHBOARD PERFORMANCE
# ============================================================

if not drift_report.empty:

    fig = px.bar(
        drift_report,
        x="Target",
        y="Drift_pct",
        color="Drift_Detected",
        title="NBA — Model Performance Drift",
        template="plotly_dark"
    )

    fig.add_hline(
        y=100 * DRIFT_THRESHOLD,
        line_dash="dash",
        annotation_text="Drift Threshold"
    )

    fig.add_hline(y=0)

    fig.show()

if not comparison_report.empty:

    display(
        comparison_report.round(4)
    )

    chart_data = comparison_report.melt(
        id_vars=["Target"],
        value_vars=[
            "Champion_MAE",
            "Challenger_MAE"
        ],
        var_name="Model",
        value_name="MAE"
    )

    fig = px.bar(
        chart_data,
        x="Target",
        y="MAE",
        color="Model",
        barmode="group",
        title="Champion vs Challenger — Test MAE",
        template="plotly_dark"
    )

    fig.show()

    fig = px.bar(
        comparison_report,
        x="Target",
        y="Improvement_pct",
        color="Accepted",
        title="Model Improvement",
        template="plotly_dark"
    )

    fig.add_hline(
        y=100 * MIN_IMPROVEMENT,
        line_dash="dash"
    )

    fig.show()

# ============================================================
# 16. RIEPILOGO FINALE
# ============================================================

print("\n" + "=" * 60)

print("STEP 20 COMPLETATO!")
print("AUTOMATIC MODEL RETRAINING V4")

print("=" * 60)

print("Nuovi giorni:", len(new_days))
print("Drift rilevato:", drift_detected)
print("Retraining richiesto:", retrain_required)
print("Candidato salvato:", candidate_saved)
print("Target migliorati:", accepted_targets)

print(
    "Modello di produzione:",
    "V3 INVARIATO"
)

print("\nREPORT SALVATO")

print(REPORT_PATH)
print(registry_log)

if candidate_saved:

    print("\nNUOVO MODELLO CANDIDATO")
    print(CANDIDATE_PATH)

    print(
        "In attesa di verifica compatibilita."
    )

print("\nPRONTO PER STEP 21!")


In [ ]:
!pip -q install gradio plotly openpyxl

In [ ]:

# ============================================================
# STEP 22A — PIPELINE BUILDER
# NBA AI PREDICTION SYSTEM
# ============================================================

import os
import re
import json
import hashlib
from pathlib import Path
from google.colab import files, drive

# ============================================================
# 1. GOOGLE DRIVE
# ============================================================

mounts = [
    "/content/drive",
    "/content/gdrive_nba",
    "/content/nba_drive"
]

BASE_DIR = None

for mount in mounts:

    candidate = os.path.join(
        mount, "MyDrive", "NBA_Data_Science"
    )

    if os.path.isfile(
        os.path.join(candidate, "nba_database.db")
    ):
        BASE_DIR = candidate
        break

if BASE_DIR is None:

    for mount in mounts:

        if os.path.isdir(mount) and os.listdir(mount):
            continue

        drive.mount(mount)

        candidate = os.path.join(
            mount, "MyDrive", "NBA_Data_Science"
        )

        if os.path.isfile(
            os.path.join(candidate, "nba_database.db")
        ):
            BASE_DIR = candidate
            break

if BASE_DIR is None:
    raise FileNotFoundError(
        "Cartella NBA_Data_Science non trovata."
    )

PIPELINE_DIR = os.path.join(
    BASE_DIR, "Production_Pipeline"
)

os.makedirs(
    PIPELINE_DIR,
    exist_ok=True
)

print("Cartella pipeline:", PIPELINE_DIR)

# ============================================================
# 2. IMPORTAZIONE NOTEBOOK
# ============================================================

print("\nCarica il notebook NBA completo (.ipynb)")

uploaded = files.upload()

notebooks = [
    name for name in uploaded
    if name.endswith(".ipynb")
]

if len(notebooks) != 1:

    raise ValueError(
        "Carica esattamente un notebook .ipynb."
    )

notebook_name = notebooks[0]

notebook = json.loads(
    uploaded[notebook_name].decode("utf-8")
)

# ============================================================
# 3. IDENTIFICAZIONE CELLE
# ============================================================

detected_steps = {}

for index, cell in enumerate(
    notebook.get("cells", [])
):

    if cell.get("cell_type") != "code":
        continue

    source = cell.get("source", "")

    if isinstance(source, list):
        source = "".join(source)

    # Cerca lo STEP nell'intestazione
    # della cella, non nei print finali.

    header = "\n".join(
        source.splitlines()[:15]
    )

    match = re.search(
        r"(?im)^\s*#\s*STEP\s+(\d+)\b",
        header
    )

    if not match:
        continue

    step = int(match.group(1))

    if step >= 22:
        continue

    detected_steps.setdefault(
        step, []
    ).append({
        "cell_index": index,
        "source": source,
        "lines": len(source.splitlines()),
        "header": header
    })

print("\nSTEP IDENTIFICATI")

for step, entries in sorted(
    detected_steps.items()
):

    print(
        f"STEP {step:02d}: "
        f"{len(entries)} celle"
    )

    for entry in entries:

        print(
            "  Indice:", entry["cell_index"],
            "| Righe:", entry["lines"]
        )

# ============================================================
# 4. CONFIGURAZIONE PIPELINE
# ============================================================

DAILY_STEPS = [

    12,  # Aggiornamento database
    14,  # Accuracy Tracker
    15,  # Feature Engineering
    6,   # Forecast squadre
    17,  # Forecast giocatori
    18,  # Probability Engine
    19   # Injury Intelligence
]

# STEP 20 temporaneamente escluso:
# necessita di verifica compatibilita
# con gli ensemble V3.
#
# STEP 21 escluso:
# avvia un server Gradio interattivo.

missing_steps = [
    step for step in DAILY_STEPS
    if step not in detected_steps
]

duplicate_steps = [
    step for step in DAILY_STEPS
    if len(detected_steps.get(step, [])) > 1
]

print("\nCONTROLLO PIPELINE")

print("Step mancanti:", missing_steps)
print("Step con piu celle:", duplicate_steps)

if missing_steps:

    raise ValueError(
        f"Step non identificati: {missing_steps}. "
        "Controllare le intestazioni del notebook."
    )

if duplicate_steps:

    raise ValueError(
        f"Step ambigui: {duplicate_steps}. "
        "Identificare le celle corrette prima "
        "di procedere."
    )

# ============================================================
# 5. ESPORTAZIONE MODULI
# ============================================================

manifest = []

for step in DAILY_STEPS:

    entry = detected_steps[step][0]

    source = entry["source"]

    filename = f"step_{step:02d}.py"

    path = os.path.join(
        PIPELINE_DIR,
        filename
    )

    with open(
        path, "w", encoding="utf-8"
    ) as f:

        f.write(source)

    checksum = hashlib.sha256(
        source.encode("utf-8")
    ).hexdigest()

    manifest.append({
        "step": step,
        "filename": filename,
        "source_cell": entry["cell_index"],
        "lines": entry["lines"],
        "sha256": checksum
    })

manifest_path = os.path.join(
    PIPELINE_DIR,
    "pipeline_manifest.json"
)

with open(
    manifest_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=4
    )

print("\nMODULI ESPORTATI")

for entry in manifest:

    print(
        entry["filename"],
        "| Righe:", entry["lines"]
    )

print("\nSTEP 22A COMPLETATO!")
print("Manifest:", manifest_path)


In [ ]:
# STEP 22A — DIAGNOSTICA DUPLICATI

for step in [12, 18]:

    print("\n" + "=" * 70)
    print(f"STEP {step} — VERSIONI DISPONIBILI")
    print("=" * 70)

    for entry in detected_steps.get(step, []):

        print("\nIndice cella:", entry["cell_index"])
        print("Numero righe:", entry["lines"])

        print("\nINTESTAZIONE:")
        print(entry["header"])

        print("\nULTIME 15 RIGHE:")
        print(
            "\n".join(
                entry["source"].splitlines()[-15:]
            )
        )

        print("-" * 70)

In [ ]:
for step in [12, 18]:

    print(f"\nSTEP {step}")

    for entry in detected_steps[step]:

        print("Indice:", entry["cell_index"])
        print("Righe:", entry["lines"])

        print("\n".join(
            entry["source"].splitlines()[:6]
        ))

        print("-" * 50)

In [ ]:
for entry in detected_steps[12]:

    print("Indice:", entry["cell_index"])
    print("Righe:", entry["lines"])
    print(entry["header"])
    print("-" * 50)

In [ ]:

# ============================================================
# STEP 22A V2 — PIPELINE BUILDER FIXED
# ============================================================

import os
import json
import hashlib
import ipywidgets as widgets

from IPython.display import display, clear_output

# ============================================================
# 1. CONTROLLO AMBIENTE
# ============================================================

if "detected_steps" not in globals():

    raise RuntimeError(
        "Riesegui prima lo STEP 22A originale "
        "per importare il notebook."
    )

BASE_DIR = next(
    (
        os.path.join(
            mount, "MyDrive", "NBA_Data_Science"
        )
        for mount in [
            "/content/drive",
            "/content/gdrive_nba",
            "/content/nba_drive"
        ]
        if os.path.isfile(
            os.path.join(
                mount,
                "MyDrive",
                "NBA_Data_Science",
                "nba_database.db"
            )
        )
    ),
    None
)

if BASE_DIR is None:

    raise FileNotFoundError(
        "Google Drive non collegato."
    )

PIPELINE_DIR = os.path.join(
    BASE_DIR,
    "Production_Pipeline"
)

os.makedirs(
    PIPELINE_DIR,
    exist_ok=True
)

# ============================================================
# 2. ORDINE PIPELINE
# ============================================================

DAILY_STEPS = [
    12,
    14,
    15,
    6,
    17,
    18,
    19
]

STEP_NAMES = {
    12: "Database Update",
    14: "Accuracy Tracker",
    15: "Feature Engineering",
    6: "Team Forecast",
    17: "Player Forecast",
    18: "Probability Engine",
    19: "Injury Intelligence"
}

missing = [
    step for step in DAILY_STEPS
    if not detected_steps.get(step)
]

if missing:

    raise ValueError(
        f"Step mancanti: {missing}"
    )

# ============================================================
# 3. SELEZIONE VERSIONI
# ============================================================

selectors = {}

for step in DAILY_STEPS:

    entries = detected_steps[step]

    options = []

    for entry in entries:

        index = entry["cell_index"]

        lines = entry["lines"]

        label = (
            f"Cella {index} | {lines} righe"
        )

        if "FIXED" in entry["header"].upper():

            label += " | FIXED"

        options.append(
            (label, index)
        )

    if step == 18:

        valid = [
            e for e in entries
            if e["cell_index"] == 44
            and "3.1" in e["header"]
            and "FIXED" in e["header"].upper()
        ]

        if len(valid) != 1:

            raise ValueError(
                "STEP 18 V3.1 non identificato."
            )

        default = 44

    elif len(entries) == 1:

        default = entries[0]["cell_index"]

    else:

        options.insert(
            0,
            ("Seleziona versione corretta", None)
        )

        default = None

    selectors[step] = widgets.Dropdown(
        options=options,
        value=default,
        description=f"STEP {step}",
        layout=widgets.Layout(width="95%")
    )

# ============================================================
# 4. INTERFACCIA DI SELEZIONE
# ============================================================

output = widgets.Output()

button = widgets.Button(
    description="CREA PIPELINE",
    button_style="success",
    icon="check",
    layout=widgets.Layout(
        width="300px",
        height="45px"
    )
)

print("=" * 60)
print("NBA PIPELINE BUILDER V2")
print("=" * 60)

for step in DAILY_STEPS:

    print(STEP_NAMES[step])

    display(selectors[step])

print("\nVERSIONI STEP 12 DISPONIBILI")

for entry in detected_steps[12]:

    print("\nCella:", entry["cell_index"])
    print("Righe:", entry["lines"])
    print(entry["header"])

    print("\nUltime righe:")

    print(
        "\n".join(
            entry["source"].splitlines()[-10:]
        )
    )

    print("-" * 50)

# ============================================================
# 5. ESPORTAZIONE MODULI
# ============================================================

def create_pipeline(change=None):

    with output:

        clear_output(wait=True)

        manifest = []

        selected = {}

        for step in DAILY_STEPS:

            index = selectors[step].value

            if index is None:

                print(
                    f"Seleziona una versione "
                    f"per STEP {step}."
                )

                return

            entries = [
                entry
                for entry in detected_steps[step]
                if entry["cell_index"] == index
            ]

            if len(entries) != 1:

                print(
                    f"Versione non valida: STEP {step}"
                )

                return

            selected[step] = entries[0]

        # Tutte le selezioni sono valide.
        # Esportiamo i moduli.

        for step in DAILY_STEPS:

            entry = selected[step]

            source = entry["source"]

            filename = f"step_{step:02d}.py"

            path = os.path.join(
                PIPELINE_DIR,
                filename
            )

            with open(
                path + ".tmp",
                "w",
                encoding="utf-8"
            ) as f:

                f.write(source)

            os.replace(
                path + ".tmp",
                path
            )

            checksum = hashlib.sha256(
                source.encode("utf-8")
            ).hexdigest()

            manifest.append({
                "step": step,
                "filename": filename,
                "source_cell": entry["cell_index"],
                "lines": entry["lines"],
                "sha256": checksum
            })

            print(
                f"OK — STEP {step:02d} | "
                f"Cella {entry['cell_index']} | "
                f"{entry['lines']} righe"
            )

        manifest_path = os.path.join(
            PIPELINE_DIR,
            "pipeline_manifest.json"
        )

        with open(
            manifest_path + ".tmp",
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                manifest,
                f,
                indent=4
            )

        os.replace(
            manifest_path + ".tmp",
            manifest_path
        )

        print("\n" + "=" * 60)
        print("STEP 22A V2 COMPLETATO!")
        print("=" * 60)

        print("Moduli esportati:", len(manifest))
        print("Manifest:", manifest_path)

        print("\nPRONTO PER STEP 22B!")

button.on_click(create_pipeline)

display(button, output)


In [ ]:

# ============================================================
# STEP 22B — PIPELINE VALIDATION
# ============================================================

import os
import json
import hashlib
import sqlite3
import pandas as pd

from IPython import get_ipython
from IPython.display import display

# 1. Individuazione Google Drive

BASE_DIR = next(
    (
        os.path.join(mount, "MyDrive", "NBA_Data_Science")
        for mount in [
            "/content/drive",
            "/content/gdrive_nba",
            "/content/nba_drive"
        ]
        if os.path.isfile(
            os.path.join(
                mount,
                "MyDrive",
                "NBA_Data_Science",
                "nba_database.db"
            )
        )
    ),
    None
)

if BASE_DIR is None:
    raise FileNotFoundError("Google Drive non collegato.")

PIPELINE_DIR = os.path.join(
    BASE_DIR, "Production_Pipeline"
)

DB_PATH = os.path.join(
    BASE_DIR, "nba_database.db"
)

MANIFEST_PATH = os.path.join(
    PIPELINE_DIR, "pipeline_manifest.json"
)

if not os.path.isfile(MANIFEST_PATH):
    raise FileNotFoundError(
        "Completa prima lo STEP 22A V2."
    )

with open(MANIFEST_PATH, encoding="utf-8") as f:
    manifest = json.load(f)

EXPECTED_STEPS = [12, 14, 15, 6, 17, 18, 19]

if [m["step"] for m in manifest] != EXPECTED_STEPS:
    raise ValueError("Ordine pipeline non corretto.")

# 2. Controllo moduli

results = []

ip = get_ipython()

print("=" * 60)
print("NBA PIPELINE VALIDATION")
print("=" * 60)

for entry in manifest:

    path = os.path.join(
        PIPELINE_DIR, entry["filename"]
    )

    result = {
        "Step": entry["step"],
        "Modulo": entry["filename"],
        "Cella": entry["source_cell"],
        "File": "OK",
        "Integrita": "OK",
        "Sintassi": "OK",
        "Colab_Dependencies": "",
        "Errore": ""
    }

    try:

        if not os.path.isfile(path):
            result["File"] = "FAILED"
            raise FileNotFoundError(path)

        with open(path, encoding="utf-8") as f:
            source = f.read()

        checksum = hashlib.sha256(
            source.encode("utf-8")
        ).hexdigest()

        if checksum != entry["sha256"]:
            result["Integrita"] = "FAILED"
            raise ValueError("Checksum non valido.")

        try:

            transformed = ip.transform_cell(source)

            compile(
                transformed,
                path,
                "exec"
            )

        except Exception:
            result["Sintassi"] = "FAILED"
            raise

        dependencies = []

        for pattern in [
            "google.colab",
            "drive.mount",
            "files.upload",
            "ipywidgets",
            "get_ipython",
            "input(",
            "app.launch"
        ]:

            if pattern in source:
                dependencies.append(pattern)

        result["Colab_Dependencies"] = ", ".join(
            dependencies
        )

    except Exception as e:

        result["Errore"] = str(e)

    results.append(result)

    status = (
        "OK"
        if not result["Errore"]
        else "FAILED"
    )

    print(f"STEP {entry['step']:02d}: {status}")

report = pd.DataFrame(results)

display(report)

# 3. Controllo database in sola lettura

print("\nCONTROLLO DATABASE")

with sqlite3.connect(
    f"file:{DB_PATH}?mode=ro",
    uri=True,
    timeout=30
) as conn:

    integrity = conn.execute(
        "PRAGMA quick_check"
    ).fetchall()

    database_ok = integrity == [("ok",)]

    tables = pd.read_sql_query(
        """
        SELECT name
        FROM sqlite_master
        WHERE type = 'table'
        ORDER BY name
        """,
        conn
    )

    required_tables = [
        "espn_games",
        "espn_player_boxscores"
    ]

    missing_tables = [
        t for t in required_tables
        if t not in tables["name"].tolist()
    ]

print("Integrita database:", integrity)
print("Tabelle presenti:", len(tables))
print("Tabelle mancanti:", missing_tables)

display(tables)

# 4. Salvataggio report

REPORT_PATH = os.path.join(
    PIPELINE_DIR,
    "Pipeline_Validation_Report.csv"
)

report.to_csv(
    REPORT_PATH,
    index=False
)

passed = (
    len(report) == 7
    and report["Errore"].eq("").all()
    and database_ok
    and not missing_tables
)

print("\n" + "=" * 60)

if passed:

    print("STEP 22B COMPLETATO!")
    print("TUTTI I CONTROLLI SUPERATI")
    print("PRONTO PER STEP 22C")

else:

    print("VALIDAZIONE NON SUPERATA")
    print("Controllare gli errori.")

print("=" * 60)

print("Report salvato:", REPORT_PATH)


In [ ]:

# ============================================================
# STEP 22C — FIRST PRODUCTION RUN
# NBA AI PREDICTION SYSTEM
# ============================================================

import os
import json
import time
import sqlite3
import hashlib
import traceback
import pandas as pd

from pathlib import Path
from datetime import datetime, timezone
from IPython import get_ipython
from IPython.display import display

# ============================================================
# 1. CONFIGURAZIONE
# ============================================================

BASE_DIR = next(
    (
        os.path.join(mount, "MyDrive", "NBA_Data_Science")
        for mount in [
            "/content/drive",
            "/content/gdrive_nba",
            "/content/nba_drive"
        ]
        if os.path.isfile(
            os.path.join(
                mount,
                "MyDrive",
                "NBA_Data_Science",
                "nba_database.db"
            )
        )
    ),
    None
)

if BASE_DIR is None:
    raise FileNotFoundError("Google Drive non collegato.")

PIPELINE_DIR = Path(BASE_DIR) / "Production_Pipeline"
DB_PATH = Path(BASE_DIR) / "nba_database.db"

LOG_DIR = PIPELINE_DIR / "Logs"
BACKUP_DIR = PIPELINE_DIR / "Backups"

LOG_DIR.mkdir(parents=True, exist_ok=True)
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_PATH = PIPELINE_DIR / "pipeline_manifest.json"

with open(MANIFEST_PATH, encoding="utf-8") as f:
    manifest = json.load(f)

EXPECTED_STEPS = [12, 14, 15, 6, 17, 18, 19]

STEP_NAMES = {
    12: "Database Update",
    14: "Accuracy Tracker",
    15: "Feature Engineering",
    6: "Team Forecast",
    17: "Player Forecast",
    18: "Probability Engine",
    19: "Injury Intelligence"
}

# ============================================================
# 2. PRODUCTION RUNNER
# ============================================================

def production_run(base_dir, pipeline_dir, db_path, manifest):

    ip = get_ipython()

    if ip is None:
        raise RuntimeError("Ambiente IPython non disponibile.")

    if [m["step"] for m in manifest] != EXPECTED_STEPS:
        raise ValueError("Ordine pipeline non valido.")

    run_id = datetime.now(timezone.utc).strftime(
        "%Y%m%d_%H%M%S_%f"
    )

    log_dir = pipeline_dir / "Logs"
    backup_dir = pipeline_dir / "Backups"

    records = []
    sources = {}

    # --------------------------------------------------------
    # 3. VALIDAZIONE PRELIMINARE
    # --------------------------------------------------------

    print("=" * 65)
    print("NBA PRODUCTION PIPELINE")
    print("RUN ID:", run_id)
    print("=" * 65)

    for entry in manifest:

        path = pipeline_dir / entry["filename"]

        source = path.read_text(encoding="utf-8")

        checksum = hashlib.sha256(
            source.encode("utf-8")
        ).hexdigest()

        if checksum != entry["sha256"]:
            raise ValueError(
                f"Checksum non valido: {path.name}"
            )

        compile(
            ip.transform_cell(source),
            str(path),
            "exec"
        )

        sources[entry["step"]] = source

        print("VALIDATO:", path.name)

    # --------------------------------------------------------
    # 4. BACKUP SQLITE
    # --------------------------------------------------------

    backup_path = backup_dir / f"NBA_Backup_{run_id}.db"

    print("\nCREAZIONE BACKUP DATABASE...")

    with sqlite3.connect(
        f"file:{db_path}?mode=ro",
        uri=True,
        timeout=60
    ) as original:

        with sqlite3.connect(backup_path) as backup:

            original.backup(backup)

            integrity = backup.execute(
                "PRAGMA quick_check"
            ).fetchall()

            if integrity != [("ok",)]:
                raise RuntimeError("Backup non valido.")

    print("Backup completato:", backup_path)

    # --------------------------------------------------------
    # 5. STATO DATABASE
    # --------------------------------------------------------

    def database_snapshot():

        with sqlite3.connect(
            f"file:{db_path}?mode=ro",
            uri=True,
            timeout=60
        ) as conn:

            games = conn.execute(
                "SELECT COUNT(*) FROM espn_games"
            ).fetchone()[0]

            completed = conn.execute(
                """
                SELECT COUNT(*)
                FROM espn_games
                WHERE completed = 1
                """
            ).fetchone()[0]

            boxscores = conn.execute(
                """
                SELECT COUNT(*)
                FROM espn_player_boxscores
                """
            ).fetchone()[0]

        return {
            "Games": games,
            "Completed_Games": completed,
            "Player_Boxscores": boxscores
        }

    before = database_snapshot()

    print("\nDATABASE PRIMA DELL'AGGIORNAMENTO")

    display(pd.DataFrame([before]))

    pipeline_start = time.monotonic()

    # --------------------------------------------------------
    # 6. SALVATAGGIO LOG
    # --------------------------------------------------------

    def save_logs():

        report = pd.DataFrame(records)

        report.to_csv(
            log_dir / f"Production_Run_{run_id}.csv",
            index=False
        )

    # --------------------------------------------------------
    # 7. ESECUZIONE SEQUENZIALE
    # --------------------------------------------------------

    for index, entry in enumerate(manifest, start=1):

        step = entry["step"]
        source = sources[step]

        print("\n" + "=" * 65)

        print(
            f"MODULO {index}/{len(manifest)}"
        )

        print(
            f"STEP {step} — {STEP_NAMES[step]}"
        )

        print("=" * 65)

        started_at = datetime.now(
            timezone.utc
        ).isoformat()

        start = time.monotonic()

        status = "SUCCESS"
        error = ""

        try:

            # Rimuove eventuali contatori residui
            # prima dell'aggiornamento incrementale.

            if step == 12:

                for name in ["failed", "failed_dates"]:

                    ip.user_ns.pop(name, None)

            result = ip.run_cell(
                source,
                store_history=False
            )

            if result.error_before_exec:
                raise result.error_before_exec

            if result.error_in_exec:
                raise result.error_in_exec

            # Controllo aggiornamento parziale STEP 12.

            if step == 12:

                failed = ip.user_ns.get("failed")
                failed_dates = ip.user_ns.get("failed_dates")

                if failed is None or failed_dates is None:

                    raise RuntimeError(
                        "Impossibile verificare i contatori "
                        "dei download falliti dello STEP 12."
                    )

                if failed or failed_dates:

                    raise RuntimeError(
                        "Aggiornamento database parziale. "
                        f"Download falliti: {failed}; "
                        f"Date fallite: {failed_dates}"
                    )

        except KeyboardInterrupt:

            status = "INTERRUPTED"
            error = "Esecuzione interrotta dall'utente."

        except Exception:

            status = "FAILED"
            error = traceback.format_exc()

        duration = round(
            time.monotonic() - start, 2
        )

        records.append({
            "Run_ID": run_id,
            "Step": step,
            "Module": STEP_NAMES[step],
            "Started_At": started_at,
            "Finished_At": datetime.now(
                timezone.utc
            ).isoformat(),
            "Duration_Seconds": duration,
            "Status": status,
            "Error": error
        })

        save_logs()

        print("\nRISULTATO STEP", step)
        print("Status:", status)
        print("Durata:", duration, "secondi")

        if status != "SUCCESS":

            print(error)

            print(
                "\nPIPELINE INTERROTTA PER SICUREZZA."
            )

            break

    # --------------------------------------------------------
    # 8. DATABASE DOPO AGGIORNAMENTO
    # --------------------------------------------------------

    after = database_snapshot()

    comparison = pd.DataFrame({
        "Indicatore": list(before.keys()),
        "Prima": list(before.values()),
        "Dopo": [after[k] for k in before]
    })

    comparison["Differenza"] = (
        comparison["Dopo"] - comparison["Prima"]
    )

    print("\nDATABASE UPDATE SUMMARY")

    display(comparison)

    comparison.to_csv(
        log_dir / f"Database_Update_{run_id}.csv",
        index=False
    )

    # --------------------------------------------------------
    # 9. CONTROLLO FILE GENERATI
    # --------------------------------------------------------

    expected_files = [
        "NBA_Forecast_Prossime_Partite.csv",
        "NBA_Advanced_ML_Features.csv",
        "NBA_Forecast_Engine_V3.csv",
        "NBA_Probability_Engine_V3.csv",
        "NBA_Probability_Summary_V3.csv",
        "NBA_Injury_Report.csv"
    ]

    file_status = []

    for filename in expected_files:

        path = Path(base_dir) / filename

        exists = path.is_file()

        modified = None

        if exists:

            modified = datetime.fromtimestamp(
                path.stat().st_mtime,
                tz=timezone.utc
            ).isoformat()

        file_status.append({
            "File": filename,
            "Exists": exists,
            "Size_KB": (
                round(path.stat().st_size / 1024, 2)
                if exists else 0
            ),
            "Last_Modified_UTC": modified
        })

    files_report = pd.DataFrame(file_status)

    print("\nOUTPUT FILES")

    display(files_report)

    files_report.to_csv(
        log_dir / f"Output_Files_{run_id}.csv",
        index=False
    )

    # --------------------------------------------------------
    # 10. REPORT FINALE
    # --------------------------------------------------------

    total_duration = round(
        time.monotonic() - pipeline_start, 2
    )

    completed = sum(
        r["Status"] == "SUCCESS"
        for r in records
    )

    success = (
        completed == len(manifest)
        and files_report["Exists"].all()
        and files_report["Size_KB"].gt(0).all()
    )

    summary = {
        "Run_ID": run_id,
        "Success": bool(success),
        "Completed_Steps": completed,
        "Total_Steps": len(manifest),
        "Duration_Seconds": total_duration,
        "Backup_Path": str(backup_path),
        "Finished_At": datetime.now(
            timezone.utc
        ).isoformat()
    }

    with open(
        log_dir / "Latest_Run.json",
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(summary, f, indent=4)

    print("\n" + "=" * 65)
    print("NBA PRODUCTION RUN — FINAL REPORT")
    print("=" * 65)

    display(pd.DataFrame(records))

    print("Moduli completati:", completed, "/", len(manifest))

    print(
        "Durata totale:",
        round(total_duration / 60, 2),
        "minuti"
    )

    if success:

        print("\nSTEP 22C COMPLETATO!")
        print("PRONTO PER STEP 22D!")

    else:

        print("\nSTEP 22C NON COMPLETATO.")
        print("Controllare i log prima di proseguire.")

    return summary


# ============================================================
# 11. AVVIO PIPELINE
# ============================================================

production_result = production_run(
    BASE_DIR,
    PIPELINE_DIR,
    DB_PATH,
    manifest
)


In [ ]:

# ============================================================
# STEP 22D — OUTPUT QUALITY CONTROL
# NBA AI PREDICTION SYSTEM
# ============================================================

import os
import json
import sqlite3
import numpy as np
import pandas as pd
import plotly.graph_objects as go

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

# ============================================================
# 1. CONFIGURAZIONE
# ============================================================

BASE_DIR = next(
    (
        Path(mount) / "MyDrive" / "NBA_Data_Science"
        for mount in [
            "/content/drive",
            "/content/gdrive_nba",
            "/content/nba_drive"
        ]
        if (
            Path(mount)
            / "MyDrive"
            / "NBA_Data_Science"
            / "nba_database.db"
        ).is_file()
    ),
    None
)

if BASE_DIR is None:
    raise FileNotFoundError(
        "Google Drive non collegato."
    )

DB_PATH = BASE_DIR / "nba_database.db"

PIPELINE_DIR = BASE_DIR / "Production_Pipeline"

QC_DIR = PIPELINE_DIR / "Quality_Control"

QC_DIR.mkdir(
    parents=True,
    exist_ok=True
)

NOW = pd.Timestamp.now(tz="UTC")

RUN_ID = NOW.strftime("%Y%m%d_%H%M%S_%f")

TARGETS = ["PTS", "REB", "AST", "3PM", "PRA"]

MAX_FILE_AGE_HOURS = 26

CHECKS = []

# ============================================================
# 2. FUNZIONI DI SUPPORTO
# ============================================================

def add_check(
    category,
    name,
    status,
    details="",
    critical=True
):

    CHECKS.append({
        "Category": category,
        "Check": name,
        "Status": status,
        "Critical": critical,
        "Details": str(details)
    })

    print(
        f"[{status}] {name}: {details}"
    )


def load_csv(filename):

    path = BASE_DIR / filename

    if not path.is_file():

        add_check(
            "Files",
            filename,
            "FAIL",
            "File mancante"
        )

        return pd.DataFrame()

    age = (
        NOW.timestamp()
        - path.stat().st_mtime
    ) / 3600

    fresh = (
        -0.1 <= age <= MAX_FILE_AGE_HOURS
    )

    add_check(
        "Freshness",
        filename,
        "PASS" if fresh else "FAIL",
        f"Ultima modifica: {age:.2f} ore fa"
    )

    try:

        data = pd.read_csv(
            path,
            dtype={
                "game_id": str,
                "player_id": str
            }
        )

        add_check(
            "Files",
            f"Lettura {filename}",
            "PASS",
            f"{len(data)} righe"
        )

        return data

    except Exception as e:

        add_check(
            "Files",
            f"Lettura {filename}",
            "FAIL",
            str(e)
        )

        return pd.DataFrame()


def require_columns(data, columns, name):

    missing = [
        c for c in columns
        if c not in data.columns
    ]

    add_check(
        "Schema",
        name,
        "FAIL" if missing else "PASS",
        f"Colonne mancanti: {missing}"
        if missing else "Schema valido"
    )

    return not missing


def check_duplicates(data, keys, name):

    duplicates = int(
        data.duplicated(keys).sum()
    )

    missing_keys = int(
        data[keys].isna().any(axis=1).sum()
    )

    add_check(
        "Duplicates",
        name,
        "PASS"
        if duplicates == 0 and missing_keys == 0
        else "FAIL",
        f"Duplicati: {duplicates}; "
        f"chiavi mancanti: {missing_keys}"
    )


# ============================================================
# 3. CONTROLLO ULTIMA PIPELINE
# ============================================================

print("=" * 65)
print("NBA OUTPUT QUALITY CONTROL")
print("=" * 65)

latest_run = (
    PIPELINE_DIR
    / "Logs"
    / "Latest_Run.json"
)

run_start = None

if latest_run.is_file():

    try:

        with open(
            latest_run,
            encoding="utf-8"
        ) as f:

            run = json.load(f)

        success = run.get(
            "Success",
            run.get("success", False)
        )

        finished = pd.to_datetime(
            run.get(
                "Finished_At",
                run.get("finished_at")
            ),
            utc=True,
            errors="coerce"
        )

        duration = run.get(
            "Duration_Seconds",
            run.get("duration_seconds")
        )

        recent = (
            pd.notna(finished)
            and 0 <= (
                NOW - finished
            ).total_seconds() / 3600
            <= MAX_FILE_AGE_HOURS
        )

        if pd.notna(finished) and duration is not None:

            run_start = finished - pd.Timedelta(
                seconds=float(duration)
            )

        add_check(
            "Pipeline",
            "Ultima esecuzione",
            "PASS" if success and recent else "FAIL",
            f"Success: {success}; "
            f"Fine: {finished}"
        )

    except Exception as e:

        add_check(
            "Pipeline",
            "Ultima esecuzione",
            "FAIL",
            str(e)
        )

else:

    add_check(
        "Pipeline",
        "Ultima esecuzione",
        "FAIL",
        "Latest_Run.json non trovato"
    )

# ============================================================
# 4. DATABASE QUALITY
# ============================================================

future_games = pd.DataFrame()

try:

    with sqlite3.connect(
        f"file:{DB_PATH}?mode=ro",
        uri=True,
        timeout=60
    ) as conn:

        integrity = conn.execute(
            "PRAGMA quick_check"
        ).fetchall()

        add_check(
            "Database",
            "SQLite Integrity",
            "PASS"
            if integrity == [("ok",)]
            else "FAIL",
            integrity
        )

        games = pd.read_sql_query(
            """
            SELECT
                game_id,
                game_date,
                completed,
                status
            FROM espn_games
            """,
            conn,
            dtype={"game_id": str}
        )

        games["parsed_date"] = pd.to_datetime(
            games["game_date"],
            utc=True,
            errors="coerce",
            format="mixed"
        )

        invalid_dates = int(
            games["parsed_date"].isna().sum()
        )

        add_check(
            "Database",
            "Game Dates",
            "PASS" if invalid_dates == 0 else "FAIL",
            f"Date non valide: {invalid_dates}"
        )

        future_games = games[
            (games["completed"] == 0)
            & (games["status"] == "STATUS_SCHEDULED")
            & (games["parsed_date"] > NOW)
            & (
                games["parsed_date"]
                <= NOW + pd.Timedelta(days=3)
            )
        ].copy()

        add_check(
            "Database",
            "Future Games",
            "PASS" if len(future_games) else "WARN",
            f"{len(future_games)} partite nei prossimi 3 giorni",
            critical=False
        )

        duplicate_boxscores = conn.execute("""
            SELECT COUNT(*)
            FROM (
                SELECT game_id, player_id
                FROM espn_player_boxscores
                GROUP BY game_id, player_id
                HAVING COUNT(*) > 1
            )
        """).fetchone()[0]

        add_check(
            "Database",
            "Duplicate Boxscores",
            "PASS" if duplicate_boxscores == 0 else "FAIL",
            f"Gruppi duplicati: {duplicate_boxscores}"
        )

        missing_boxscores = conn.execute("""
            SELECT COUNT(*)
            FROM espn_games g
            WHERE g.completed = 1
            AND NOT EXISTS (
                SELECT 1
                FROM espn_player_boxscores b
                WHERE b.game_id = g.game_id
            )
        """).fetchone()[0]

        add_check(
            "Database",
            "Completed Games Without Boxscores",
            "PASS" if missing_boxscores == 0 else "FAIL",
            f"Partite senza boxscore: {missing_boxscores}"
        )

except Exception as e:

    add_check(
        "Database",
        "Database Access",
        "FAIL",
        str(e)
    )

# ============================================================
# 5. CARICAMENTO OUTPUT
# ============================================================

forecast = load_csv(
    "NBA_Forecast_Engine_V3.csv"
)

probabilities = load_csv(
    "NBA_Probability_Engine_V3.csv"
)

summary = load_csv(
    "NBA_Probability_Summary_V3.csv"
)

# Controlla che gli output siano stati
# effettivamente aggiornati nell'ultima run.

if run_start is not None:

    for filename in [
        "NBA_Forecast_Engine_V3.csv",
        "NBA_Probability_Engine_V3.csv",
        "NBA_Probability_Summary_V3.csv"
    ]:

        path = BASE_DIR / filename

        if path.is_file():

            modified = pd.Timestamp(
                path.stat().st_mtime,
                unit="s",
                tz="UTC"
            )

            updated = modified >= (
                run_start - pd.Timedelta(seconds=5)
            )

            add_check(
                "Freshness",
                f"Current Run — {filename}",
                "PASS" if updated else "FAIL",
                f"Modificato: {modified}"
            )

else:

    add_check(
        "Freshness",
        "Current Run Verification",
        "FAIL",
        "Orario iniziale della pipeline non verificabile"
    )

# ============================================================
# 6. FORECAST QUALITY
# ============================================================

forecast_columns = [
    "game_id",
    "player_id",
    "date",
    "generated_at"
] + [
    f"Forecast_{t}"
    for t in TARGETS
]

forecast_valid = require_columns(
    forecast,
    forecast_columns,
    "Player Forecast Schema"
)

future_forecast = pd.DataFrame()

if forecast_valid:

    check_duplicates(
        forecast,
        ["game_id", "player_id"],
        "Player Forecast Keys"
    )

    for target in TARGETS:

        column = f"Forecast_{target}"

        values = pd.to_numeric(
            forecast[column],
            errors="coerce"
        )

        invalid = int(
            (
                ~np.isfinite(values)
                | (values < 0)
            ).sum()
        )

        add_check(
            "Forecast",
            f"{target} Numeric Validation",
            "PASS" if invalid == 0 else "FAIL",
            f"Valori non validi: {invalid}"
        )

    forecast["parsed_date"] = pd.to_datetime(
        forecast["date"],
        utc=True,
        errors="coerce",
        format="mixed"
    )

    generated = pd.to_datetime(
        forecast["generated_at"],
        utc=True,
        errors="coerce",
        format="mixed"
    )

    invalid_times = int(
        (
            generated.isna()
            | forecast["parsed_date"].isna()
            | (generated >= forecast["parsed_date"])
            | (
                generated
                > NOW + pd.Timedelta(minutes=5)
            )
        ).sum()
    )

    add_check(
        "Forecast",
        "Pregame Timestamp Validation",
        "PASS" if invalid_times == 0 else "FAIL",
        f"Timestamp non validi: {invalid_times}"
    )

    future_forecast = forecast[
        forecast["parsed_date"] > NOW
    ].copy()

    add_check(
        "Forecast",
        "Future Player Forecasts",
        "PASS" if len(future_forecast) else "WARN",
        f"{len(future_forecast)} forecast futuri",
        critical=False
    )

    expired = len(forecast) - len(future_forecast)

    add_check(
        "Forecast",
        "Expired Forecasts",
        "PASS" if expired == 0 else "WARN",
        f"{expired} forecast relativi a partite iniziate",
        critical=False
    )

    if not future_games.empty:

        expected = set(
            future_games["game_id"]
        )

        available = set(
            future_forecast["game_id"]
        )

        missing = expected - available

        coverage = (
            len(expected & available)
            / len(expected)
            * 100
        )

        add_check(
            "Coverage",
            "Future Game Coverage",
            "PASS" if not missing else "FAIL",
            f"Copertura: {coverage:.2f}%; "
            f"Game ID mancanti: {sorted(missing)}"
        )

# ============================================================
# 7. PROBABILITY QUALITY
# ============================================================

probability_columns = [
    "game_id",
    "player_id",
    "Target",
    "Threshold",
    "Probability_At_Least",
    "Forecast_Generated_At"
]

probability_valid = require_columns(
    probabilities,
    probability_columns,
    "Probability Schema"
)

if probability_valid:

    check_duplicates(
        probabilities,
        [
            "game_id",
            "player_id",
            "Target",
            "Threshold"
        ],
        "Probability Keys"
    )

    probabilities["Probability_At_Least"] = (
        pd.to_numeric(
            probabilities["Probability_At_Least"],
            errors="coerce"
        )
    )

    probabilities["Threshold"] = pd.to_numeric(
        probabilities["Threshold"],
        errors="coerce"
    )

    p = probabilities["Probability_At_Least"]

    invalid = int(
        (
            ~np.isfinite(p)
            | (p < 0)
            | (p > 1)
        ).sum()
    )

    add_check(
        "Probability",
        "Probability Range",
        "PASS" if invalid == 0 else "FAIL",
        f"Probabilita non valide: {invalid}"
    )

    thresholds = probabilities["Threshold"]

    invalid_thresholds = int(
        (
            ~np.isfinite(thresholds)
            | (thresholds < 0)
            | (thresholds % 1 != 0)
        ).sum()
    )

    add_check(
        "Probability",
        "Threshold Validation",
        "PASS" if invalid_thresholds == 0 else "FAIL",
        f"Soglie non valide: {invalid_thresholds}"
    )

    invalid_targets = int(
        (~probabilities["Target"].isin(TARGETS)).sum()
    )

    add_check(
        "Probability",
        "Target Validation",
        "PASS" if invalid_targets == 0 else "FAIL",
        f"Target non validi: {invalid_targets}"
    )

    ordered = probabilities.sort_values([
        "game_id",
        "player_id",
        "Target",
        "Threshold"
    ])

    differences = ordered.groupby(
        ["game_id", "player_id", "Target"]
    )["Probability_At_Least"].diff()

    violations = int(
        (differences > 1e-10).sum()
    )

    add_check(
        "Probability",
        "Monotonicity",
        "PASS" if violations == 0 else "FAIL",
        f"Violazioni: {violations}"
    )

    if "Probability_Below" in probabilities.columns:

        below = pd.to_numeric(
            probabilities["Probability_Below"],
            errors="coerce"
        )

        violations = int(
            (
                ~np.isfinite(below)
                | (below < 0)
                | (below > 1)
                | ((p + below - 1).abs() > 1e-8)
            ).sum()
        )

        add_check(
            "Probability",
            "Complement Consistency",
            "PASS" if violations == 0 else "FAIL",
            f"Violazioni: {violations}"
        )

# ============================================================
# 8. COERENZA FORECAST E PROBABILITA
# ============================================================

if forecast_valid and probability_valid:

    forecast_keys = forecast[
        [
            "game_id",
            "player_id",
            "generated_at"
        ]
    ].rename(columns={
        "generated_at": "Forecast_Generated_At"
    }).drop_duplicates()

    probability_keys = probabilities[
        [
            "game_id",
            "player_id",
            "Forecast_Generated_At"
        ]
    ].drop_duplicates()

    for frame in [forecast_keys, probability_keys]:

        frame["Forecast_Generated_At"] = pd.to_datetime(
            frame["Forecast_Generated_At"],
            utc=True,
            errors="coerce",
            format="mixed"
        )

    merged = probability_keys.merge(
        forecast_keys,
        how="left",
        indicator=True
    )

    mismatches = int(
        (
            (merged["_merge"] != "both")
            | merged["Forecast_Generated_At"].isna()
        ).sum()
    )

    add_check(
        "Consistency",
        "Forecast Version Matching",
        "PASS" if mismatches == 0 else "FAIL",
        f"Forecast non corrispondenti: {mismatches}"
    )

    expected_targets = {
        (g, p, t)
        for g, p in future_forecast[
            ["game_id", "player_id"]
        ].itertuples(index=False, name=None)
        for t in TARGETS
    }

    available_targets = set(
        probabilities[
            ["game_id", "player_id", "Target"]
        ].itertuples(index=False, name=None)
    )

    missing_targets = (
        expected_targets - available_targets
    )

    add_check(
        "Consistency",
        "Probability Target Coverage",
        "PASS" if not missing_targets else "FAIL",
        f"Distribuzioni mancanti: {len(missing_targets)}"
    )

# ============================================================
# 9. PREDICTION INTERVALS
# ============================================================

interval_columns = [
    "game_id",
    "player_id",
    "Target"
]

for level in [50, 80, 90, 95]:

    interval_columns.extend([
        f"Lower_{level}",
        f"Upper_{level}"
    ])

summary_valid = require_columns(
    summary,
    interval_columns,
    "Prediction Interval Schema"
)

if summary_valid:

    check_duplicates(
        summary,
        ["game_id", "player_id", "Target"],
        "Distribution Summary Keys"
    )

    for level in [50, 80, 90, 95]:

        lower = pd.to_numeric(
            summary[f"Lower_{level}"],
            errors="coerce"
        )

        upper = pd.to_numeric(
            summary[f"Upper_{level}"],
            errors="coerce"
        )

        invalid = int(
            (
                ~np.isfinite(lower)
                | ~np.isfinite(upper)
                | (lower < 0)
                | (upper < lower)
            ).sum()
        )

        add_check(
            "Intervals",
            f"{level}% Prediction Interval",
            "PASS" if invalid == 0 else "FAIL",
            f"Intervalli non validi: {invalid}"
        )

    nesting_errors = 0

    for small, large in [
        (50, 80),
        (80, 90),
        (90, 95)
    ]:

        low_small = pd.to_numeric(
            summary[f"Lower_{small}"],
            errors="coerce"
        )

        low_large = pd.to_numeric(
            summary[f"Lower_{large}"],
            errors="coerce"
        )

        high_small = pd.to_numeric(
            summary[f"Upper_{small}"],
            errors="coerce"
        )

        high_large = pd.to_numeric(
            summary[f"Upper_{large}"],
            errors="coerce"
        )

        nesting_errors += int(
            (
                (low_large > low_small)
                | (high_large < high_small)
            ).sum()
        )

    add_check(
        "Intervals",
        "Interval Nesting",
        "PASS" if nesting_errors == 0 else "FAIL",
        f"Violazioni: {nesting_errors}"
    )

# ============================================================
# 10. INJURY REPORT
# ============================================================

injury_path = BASE_DIR / "NBA_Injury_Report.csv"

if injury_path.is_file():

    injury_age = (
        NOW.timestamp()
        - injury_path.stat().st_mtime
    ) / 3600

    add_check(
        "Injuries",
        "Injury Report File Freshness",
        "PASS" if 0 <= injury_age <= 6 else "WARN",
        f"File modificato {injury_age:.2f} ore fa",
        critical=False
    )

else:

    add_check(
        "Injuries",
        "Injury Report",
        "WARN",
        "Report non disponibile",
        critical=False
    )

# La presenza del file non certifica
# la disponibilita dei giocatori.

# ============================================================
# 11. QUALITY SCORE
# ============================================================

report = pd.DataFrame(CHECKS)

passed = int(
    (report["Status"] == "PASS").sum()
)

warnings = int(
    (report["Status"] == "WARN").sum()
)

failed = int(
    (report["Status"] == "FAIL").sum()
)

critical_failed = int(
    (
        (report["Status"] == "FAIL")
        & report["Critical"]
    ).sum()
)

total = len(report)

quality_score = (
    100 * passed / total
    if total else 0
)

if critical_failed > 0:

    system_status = "BLOCKED"

elif future_forecast.empty:

    system_status = "NO_FUTURE_FORECASTS"

elif warnings > 0 or failed > 0:

    system_status = "WARNING"

else:

    system_status = "QC_PASSED"

# ============================================================
# 12. REPORT GRAFICO
# ============================================================

fig = go.Figure(
    go.Indicator(
        mode="gauge+number",
        value=quality_score,
        number={"suffix": "%"},
        title={"text": "NBA Data Quality Score"},
        gauge={
            "axis": {"range": [0, 100]},
            "bar": {"color": "#38bdf8"},
            "steps": [
                {
                    "range": [0, 60],
                    "color": "#7f1d1d"
                },
                {
                    "range": [60, 90],
                    "color": "#854d0e"
                },
                {
                    "range": [90, 100],
                    "color": "#14532d"
                }
            ]
        }
    )
)

fig.update_layout(
    template="plotly_dark",
    height=400
)

fig.show()

display(report)

# ============================================================
# 13. ESPORTAZIONE
# ============================================================

CSV_PATH = QC_DIR / "NBA_Quality_Control.csv"

EXCEL_PATH = QC_DIR / "NBA_Quality_Control.xlsx"

JSON_PATH = QC_DIR / "Latest_Quality_Status.json"

report.to_csv(
    CSV_PATH,
    index=False
)

report.to_csv(
    QC_DIR / f"Quality_Control_{RUN_ID}.csv",
    index=False
)

quality_summary = {
    "Generated_At": NOW.isoformat(),
    "Quality_Score": round(quality_score, 2),
    "Total_Checks": total,
    "Passed": passed,
    "Warnings": warnings,
    "Failed": failed,
    "Critical_Failed": critical_failed,
    "System_Status": system_status
}

with pd.ExcelWriter(
    EXCEL_PATH,
    engine="openpyxl"
) as writer:

    pd.DataFrame(
        [quality_summary]
    ).to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )

    report.to_excel(
        writer,
        sheet_name="All Checks",
        index=False
    )

    report[
        report["Status"] != "PASS"
    ].to_excel(
        writer,
        sheet_name="Issues",
        index=False
    )

with open(
    JSON_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        quality_summary,
        f,
        indent=4
    )

# ============================================================
# 14. RISULTATO FINALE
# ============================================================

print("\n" + "=" * 65)

print("NBA QUALITY CONTROL — FINAL REPORT")

print("=" * 65)

print("Quality Score:", round(quality_score, 2), "%")
print("Controlli totali:", total)
print("Superati:", passed)
print("Warnings:", warnings)
print("Falliti:", failed)
print("Errori critici:", critical_failed)

print("\nSYSTEM STATUS:", system_status)

print("\nFILE SALVATI:")

print(CSV_PATH)
print(EXCEL_PATH)
print(JSON_PATH)

print("\nSTEP 22D COMPLETATO!")

if system_status == "QC_PASSED":

    print("PRONTO PER STEP 23 — CLOUD DEPLOYMENT")

elif system_status == "WARNING":

    print("Verificare gli avvisi prima del deployment.")

elif system_status == "NO_FUTURE_FORECASTS":

    print("Nessun forecast futuro disponibile.")

else:

    print("DEPLOYMENT BLOCCATO.")
    print("Correggere gli errori critici.")


In [ ]:

# ============================================================
# STEP 23 — NBA ONE CLICK CONTROL CENTER
# ============================================================

import os
import json
import time
import hashlib
import traceback
import pandas as pd
import ipywidgets as widgets

from pathlib import Path
from datetime import datetime, timezone
from IPython import get_ipython
from IPython.display import display, HTML, clear_output

# ============================================================
# 1. CONFIGURAZIONE
# ============================================================

BASE_DIR = next(
    (
        Path(mount) / "MyDrive" / "NBA_Data_Science"
        for mount in [
            "/content/drive",
            "/content/gdrive_nba",
            "/content/nba_drive"
        ]
        if (
            Path(mount)
            / "MyDrive"
            / "NBA_Data_Science"
            / "nba_database.db"
        ).is_file()
    ),
    None
)

if BASE_DIR is None:
    raise FileNotFoundError(
        "Collega prima Google Drive."
    )

PIPELINE_DIR = BASE_DIR / "Production_Pipeline"

MANIFEST_PATH = (
    PIPELINE_DIR / "pipeline_manifest.json"
)

if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(
        "Completa prima lo STEP 22A."
    )

with open(MANIFEST_PATH, encoding="utf-8") as f:
    CONTROL_MANIFEST = json.load(f)

CONTROL_STEPS = {
    entry["step"]: entry
    for entry in CONTROL_MANIFEST
}

CONTROL_IP = get_ipython()

CONTROL_BUSY = False

# ============================================================
# 2. INTERFACCIA GRAFICA
# ============================================================

header = widgets.HTML("""
<div style="
    background:#0b1120;
    padding:25px;
    border-radius:14px;
    color:white;
    font-family:Arial;
">
    <div style="
        color:#38bdf8;
        font-size:13px;
        letter-spacing:3px;
    ">
        MACHINE LEARNING PLATFORM
    </div>
    <h1 style="margin:10px 0;">
        NBA INTELLIGENCE V4
    </h1>
    <div style="color:#94a3b8;">
        Central Control Center | Google Colab
    </div>
</div>
""")

status = widgets.HTML(
    "<b style='color:green'>SYSTEM READY</b>"
)

progress = widgets.IntProgress(
    value=0,
    min=0,
    max=7,
    description="Progress:",
    bar_style="",
    layout=widgets.Layout(width="100%")
)

output = widgets.Output(
    layout=widgets.Layout(
        border="1px solid #64748b",
        padding="12px",
        width="100%",
        max_height="600px",
        overflow="auto"
    )
)

# ============================================================
# 3. CREAZIONE PULSANTI
# ============================================================

def make_button(label, icon, style=""):

    return widgets.Button(
        description=label,
        icon=icon,
        button_style=style,
        layout=widgets.Layout(
            width="100%",
            height="48px"
        )
    )

btn_all = make_button(
    "AGGIORNA TUTTO",
    "refresh",
    "primary"
)

btn_db = make_button(
    "Aggiorna database",
    "database"
)

btn_accuracy = make_button(
    "Accuracy Tracker",
    "check"
)

btn_features = make_button(
    "Feature Engineering",
    "cogs"
)

btn_teams = make_button(
    "Team Predictions",
    "users"
)

btn_players = make_button(
    "Player Forecast V3",
    "line-chart"
)

btn_prob = make_button(
    "Probability Engine",
    "calculator"
)

btn_injury = make_button(
    "Injury Intelligence",
    "medkit"
)

btn_quality = make_button(
    "Controllo qualità",
    "shield",
    "warning"
)

btn_dashboard = make_button(
    "APRI DASHBOARD V4",
    "desktop",
    "success"
)

btn_files = make_button(
    "Visualizza file e stato",
    "folder-open"
)

btn_clear = make_button(
    "Pulisci console",
    "trash"
)

CONTROL_BUTTONS = [
    btn_all,
    btn_db,
    btn_accuracy,
    btn_features,
    btn_teams,
    btn_players,
    btn_prob,
    btn_injury,
    btn_quality,
    btn_dashboard,
    btn_files,
    btn_clear
]

# ============================================================
# 4. GESTIONE STATO
# ============================================================

def set_busy(value):

    global CONTROL_BUSY

    CONTROL_BUSY = value

    for button in CONTROL_BUTTONS:
        button.disabled = value


def set_status(message, color="#38bdf8"):

    status.value = (
        f"<b style='color:{color}'>"
        f"{message}</b>"
    )


# ============================================================
# 5. ESECUZIONE MODULI
# ============================================================

def run_exported_step(step):

    if step not in CONTROL_STEPS:

        raise ValueError(
            f"STEP {step} non disponibile."
        )

    entry = CONTROL_STEPS[step]

    path = PIPELINE_DIR / entry["filename"]

    source = path.read_text(
        encoding="utf-8"
    )

    checksum = hashlib.sha256(
        source.encode("utf-8")
    ).hexdigest()

    if checksum != entry["sha256"]:

        raise RuntimeError(
            f"Integrita STEP {step} non valida."
        )

    print(f"\nAVVIO STEP {step}")
    print("=" * 60)

    if step == 12:

        for name in ["failed", "failed_dates"]:

            CONTROL_IP.user_ns.pop(
                name, None
            )

    result = CONTROL_IP.run_cell(
        source,
        store_history=False
    )

    if result.error_before_exec:

        raise result.error_before_exec

    if result.error_in_exec:

        raise result.error_in_exec

    if step == 12:

        failed = CONTROL_IP.user_ns.get(
            "failed"
        )

        failed_dates = CONTROL_IP.user_ns.get(
            "failed_dates"
        )

        if failed is None or failed_dates is None:

            raise RuntimeError(
                "Contatori aggiornamento "
                "non verificabili."
            )

        if failed or failed_dates:

            raise RuntimeError(
                "Aggiornamento database parziale."
            )

    print(f"\nSTEP {step} COMPLETATO")


# ============================================================
# 6. RECUPERO CELLE DAL NOTEBOOK
# ============================================================

import re

def find_notebook_cell(step_label):

    history = CONTROL_IP.user_ns.get(
        "In", []
    )

    pattern = re.compile(
        r"^\s*#\s*STEP\s+"
        + re.escape(step_label)
        + r"\b",
        re.IGNORECASE | re.MULTILINE
    )

    for source in reversed(history):

        header = "\n".join(
            source.splitlines()[:15]
        )

        if pattern.search(header):

            return source

    raise RuntimeError(
        f"STEP {step_label} non presente "
        "nella cronologia del runtime. "
        "Esegui prima la relativa cella."
    )


def run_notebook_step(step_label):

    source = find_notebook_cell(
        step_label
    )

    result = CONTROL_IP.run_cell(
        source,
        store_history=False
    )

    if result.error_before_exec:

        raise result.error_before_exec

    if result.error_in_exec:

        raise result.error_in_exec


# ============================================================
# 7. ESECUZIONE CONTROLLATA
# ============================================================

def execute_action(
    action,
    label,
    success_message="OPERAZIONE COMPLETATA"
):

    if CONTROL_BUSY:
        return

    set_busy(True)

    set_status(
        f"RUNNING — {label}",
        "#38bdf8"
    )

    progress.value = 0
    progress.bar_style = "info"

    with output:

        clear_output(wait=True)

        started = time.monotonic()

        try:

            print("=" * 60)
            print(label)
            print("=" * 60)

            action()

            elapsed = (
                time.monotonic() - started
            )

            progress.value = progress.max
            progress.bar_style = "success"

            set_status(
                success_message,
                "#22c55e"
            )

            print(
                f"\nDurata: {elapsed:.2f} secondi"
            )

        except KeyboardInterrupt:

            progress.bar_style = "warning"

            set_status(
                "OPERAZIONE INTERROTTA",
                "#eab308"
            )

        except Exception:

            progress.bar_style = "danger"

            set_status(
                "OPERAZIONE FALLITA",
                "#ef4444"
            )

            traceback.print_exc()

        finally:

            set_busy(False)


# ============================================================
# 8. AGGIORNAMENTO COMPLETO
# ============================================================

def full_update():

    # Utilizza il production runner
    # dello STEP 22C, incluso il backup.

    run_notebook_step("22C")

    result = CONTROL_IP.user_ns.get(
        "production_result"
    )

    if not isinstance(result, dict):

        raise RuntimeError(
            "Report production run non disponibile."
        )

    if not result.get("Success", False):

        raise RuntimeError(
            "Pipeline non completata."
        )

    progress.value = 6

    print("\nAVVIO QUALITY CONTROL")

    run_notebook_step("22D")

    quality_path = (
        PIPELINE_DIR
        / "Quality_Control"
        / "Latest_Quality_Status.json"
    )

    with open(
        quality_path,
        encoding="utf-8"
    ) as f:

        quality = json.load(f)

    system_status = quality.get(
        "System_Status"
    )

    print(
        "\nQUALITY SCORE:",
        quality.get("Quality_Score")
    )

    print(
        "SYSTEM STATUS:",
        system_status
    )

    if system_status != "QC_PASSED":

        raise RuntimeError(
            "Controllare il report qualità: "
            + str(system_status)
        )

    progress.value = 7

    print("\nAGGIORNAMENTO COMPLETO!")
    print("Puoi aprire la dashboard.")


# ============================================================
# 9. FILE MONITOR
# ============================================================

def show_files():

    files = [
        "nba_database.db",
        "nba_player_models_v3.joblib",
        "NBA_Forecast_Prossime_Partite.csv",
        "NBA_Forecast_Engine_V3.csv",
        "NBA_Probability_Engine_V3.csv",
        "NBA_Probability_Summary_V3.csv",
        "NBA_Injury_Report.csv",
        "NBA_Context_Aware_Forecast.csv"
    ]

    rows = []

    for filename in files:

        path = BASE_DIR / filename

        exists = path.is_file()

        rows.append({
            "File": filename,
            "Status": (
                "AVAILABLE"
                if exists else "MISSING"
            ),
            "Size_MB": (
                round(
                    path.stat().st_size / 1024**2,
                    3
                )
                if exists else 0
            ),
            "Last_Update": (
                datetime.fromtimestamp(
                    path.stat().st_mtime,
                    tz=timezone.utc
                ).isoformat()
                if exists else None
            )
        })

    display(pd.DataFrame(rows))

    reports = [
        PIPELINE_DIR
        / "Logs"
        / "Latest_Run.json",

        PIPELINE_DIR
        / "Quality_Control"
        / "Latest_Quality_Status.json"
    ]

    for path in reports:

        if path.is_file():

            print("\n", path.name)

            with open(
                path,
                encoding="utf-8"
            ) as f:

                display(
                    pd.DataFrame([json.load(f)])
                )


# ============================================================
# 10. COLLEGAMENTO PULSANTI
# ============================================================

btn_all.on_click(
    lambda b: execute_action(
        full_update,
        "FULL SYSTEM UPDATE"
    )
)

individual_buttons = [
    (btn_db, 12),
    (btn_accuracy, 14),
    (btn_features, 15),
    (btn_teams, 6),
    (btn_players, 17),
    (btn_prob, 18),
    (btn_injury, 19)
]

for button, step in individual_buttons:

    button.on_click(
        lambda b, s=step: execute_action(
            lambda: run_exported_step(s),
            f"STEP {s}"
        )
    )

btn_quality.on_click(
    lambda b: execute_action(
        lambda: run_notebook_step("22D"),
        "QUALITY CONTROL",
        "CONTROLLO ESEGUITO — LEGGI IL REPORT"
    )
)

btn_dashboard.on_click(
    lambda b: execute_action(
        lambda: run_notebook_step("21"),
        "DASHBOARD V4"
    )
)

btn_files.on_click(
    lambda b: execute_action(
        show_files,
        "SYSTEM FILE MONITOR"
    )
)

def clear_console(button):

    with output:
        clear_output()

btn_clear.on_click(clear_console)

# ============================================================
# 11. VISUALIZZAZIONE CONTROL CENTER
# ============================================================

grid = widgets.GridBox(
    children=[
        btn_db,
        btn_accuracy,
        btn_features,
        btn_teams,
        btn_players,
        btn_prob,
        btn_injury,
        btn_quality,
        btn_files,
        btn_clear
    ],
    layout=widgets.Layout(
        grid_template_columns="repeat(2, 1fr)",
        grid_gap="10px",
        width="100%"
    )
)

control_center = widgets.VBox([
    header,
    status,
    progress,
    btn_all,
    grid,
    btn_dashboard,
    widgets.HTML("<h3>Execution Console</h3>"),
    output
])

display(control_center)

print("\nSTEP 23 COMPLETATO!")
print("NBA ONE CLICK CONTROL CENTER READY")


In [ ]:

# ============================================================
# STEP 24 — NBA INTELLIGENCE V5
# ============================================================

%pip install -q gradio plotly pandas numpy openpyxl

from google.colab import files

print("Carica il file STEP_24_Dashboard_V5.py")

uploaded = files.upload()

filename = "STEP_24_Dashboard_V5.py"

if filename not in uploaded:
    raise FileNotFoundError(
        "Seleziona il file Python corretto."
    )

source = uploaded[filename].decode("utf-8")

exec(compile(source, filename, "exec"))


In [ ]:

# ============================================================
# STEP 24 — DASHBOARD V5 COMPATIBILITY FIX
# ============================================================

import gradio as gr
import inspect
import ast

from pathlib import Path

path = Path("/content/STEP_24_Dashboard_V5.py")

if not path.exists():
    raise FileNotFoundError("File dashboard non trovato.")

source = path.read_text(encoding="utf-8")

# Rilevamento parametri supportati da Gradio

supported = set(
    inspect.signature(gr.themes.Soft().set).parameters
)

# Rimozione dei parametri grafici incompatibili

tree = ast.parse(source)

class ThemeCompatibilityFix(ast.NodeTransformer):

    def visit_Call(self, node):

        self.generic_visit(node)

        if (
            isinstance(node.func, ast.Attribute)
            and node.func.attr == "set"
            and isinstance(node.func.value, ast.Call)
            and isinstance(node.func.value.func, ast.Attribute)
            and node.func.value.func.attr == "Soft"
        ):

            removed = [
                kw.arg for kw in node.keywords
                if kw.arg is not None
                and kw.arg not in supported
            ]

            node.keywords = [
                kw for kw in node.keywords
                if kw.arg is None or kw.arg in supported
            ]

            print("Parametri incompatibili rimossi:", removed)

        return node

tree = ThemeCompatibilityFix().visit(tree)

ast.fix_missing_locations(tree)

print("Versione Gradio:", gr.__version__)
print("Correzione grafica applicata.")
print("Avvio NBA Intelligence V5...")

exec(
    compile(tree, str(path), "exec"),
    globals()
)


In [ ]:
import pandas as pd
from pathlib import Path

path = Path(
    "/content/drive/MyDrive/NBA_Data_Science/"
    "NBA_Forecast_Prossime_Partite.csv"
)

if path.exists():
    df = pd.read_csv(path)

    print("Numero previsioni:", len(df))
    print("Colonne:", df.columns.tolist())

    print("\nUltimo aggiornamento:")
    print(pd.Timestamp(path.stat().st_mtime, unit="s", tz="UTC"))

    display(df.head(10))

else:
    print("File previsioni non trovato.")

In [ ]:
# Aggiornamento STEP 6

%run /content/drive/MyDrive/NBA_Data_Science/Production_Pipeline/step_6.py

In [ ]:

from pathlib import Path
from IPython import get_ipython

roots = [
    Path("/content/drive/MyDrive"),
    Path("/content/gdrive_nba/MyDrive"),
    Path("/content/nba_drive/MyDrive")
]

matches = []

for root in roots:
    if root.exists():
        matches.extend(root.rglob("step_6.py"))

matches = list(dict.fromkeys(matches))

if len(matches) == 1:

    path = matches[0]

    print("STEP 6 TROVATO:")
    print(path)

    source = path.read_text(encoding="utf-8")

    result = get_ipython().run_cell(
        source,
        store_history=False
    )

    if result.error_before_exec:
        raise result.error_before_exec

    if result.error_in_exec:
        raise result.error_in_exec

    print("\nSTEP 6 ESEGUITO")

elif len(matches) > 1:

    print("Trovate più versioni dello STEP 6:")

    for path in matches:
        print(path)

else:

    print("STEP 6 NON ESPORTATO")
    print("Esegui la cella originale STEP 6 del notebook.")


In [ ]:

import json
import re
import hashlib

from pathlib import Path
from google.colab import files
from IPython import get_ipython

BASE_DIR = Path(
    "/content/drive/MyDrive/NBA_Data_Science"
)

if not (BASE_DIR / "nba_database.db").exists():
    raise FileNotFoundError("Database non trovato.")

PIPELINE_DIR = BASE_DIR / "Production_Pipeline"
PIPELINE_DIR.mkdir(parents=True, exist_ok=True)

print("Carica il notebook NBA_Data_Science_Analytics_V4.ipynb")

uploaded = files.upload()

filename = next(
    (f for f in uploaded if f.endswith(".ipynb")),
    None
)

if filename is None:
    raise ValueError("Carica un notebook .ipynb")

notebook = json.loads(uploaded[filename])

required_steps = [12, 14, 15, 6, 17, 18, 19]

candidates = {step: [] for step in required_steps}

for index, cell in enumerate(notebook["cells"]):

    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))

    header = "\n".join(source.splitlines()[:15])

    match = re.search(
        r"(?im)^\s*#\s*STEP\s+(\d+)\b",
        header
    )

    if match:

        step = int(match.group(1))

        if step in required_steps:

            candidates[step].append({
                "index": index,
                "source": source,
                "header": header
            })

manifest = []
selected = {}

for step in required_steps:

    options = candidates[step]

    if not options:
        raise RuntimeError(f"STEP {step} mancante.")

    if step == 18:

        fixed = [
            x for x in options
            if "3.1" in x["header"]
            and "FIXED" in x["header"].upper()
        ]

        if len(fixed) != 1:
            raise RuntimeError(
                "Versione STEP 18 FIXED non identificata."
            )

        choice = fixed[0]

    elif step == 12:

        fixed = [
            x for x in options
            if "failed_dates" in x["source"]
            and "Nuovi box score" in x["source"]
        ]

        if len(fixed) != 1:
            raise RuntimeError(
                "Versione STEP 12 ambigua."
            )

        choice = fixed[0]

    else:

        if len(options) != 1:
            raise RuntimeError(
                f"STEP {step}: più versioni disponibili."
            )

        choice = options[0]

    source = choice["source"]

    compile(
        get_ipython().transform_cell(source),
        f"step_{step}.py",
        "exec"
    )

    selected[step] = choice

# Esportazione dopo la validazione di tutti i moduli

for step, choice in selected.items():

    source = choice["source"]

    filename = f"step_{step}.py"

    path = PIPELINE_DIR / filename

    path.write_text(source, encoding="utf-8")

    manifest.append({
        "step": step,
        "filename": filename,
        "source_cell": choice["index"],
        "lines": len(source.splitlines()),
        "sha256": hashlib.sha256(
            source.encode("utf-8")
        ).hexdigest()
    })

    print(f"STEP {step}: ESPORTATO")

with open(
    PIPELINE_DIR / "pipeline_manifest.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(manifest, f, indent=4)

print("\n" + "=" * 50)
print("PRODUCTION PIPELINE RICOSTRUITA")
print("Moduli esportati:", len(manifest))
print("=" * 50)


In [ ]:
from pathlib import Path

path = Path(
    "/content/drive/MyDrive/NBA_Data_Science/"
    "Production_Pipeline/step_6.py"
)

result = get_ipython().run_cell(
    path.read_text(encoding="utf-8"),
    store_history=False
)

if result.error_before_exec:
    raise result.error_before_exec

if result.error_in_exec:
    raise result.error_in_exec

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:

# ============================================================
# STEP 24.3 — SYSTEM RECOVERY
# ============================================================

from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

BASE_DIR = Path(
    "/content/drive/MyDrive/NBA_Data_Science"
)

if not BASE_DIR.is_dir():
    raise FileNotFoundError("Cartella NBA non trovata.")

print("=" * 60)
print("NBA INTELLIGENCE V5 — SYSTEM RECOVERY")
print("=" * 60)

required_files = [
    "nba_database.db",
    "nba_win_prediction_model.joblib",
    "nba_player_models_v3.joblib",
    "NBA_Forecast_Prossime_Partite.csv",
    "NBA_Forecast_Engine_V3.csv",
    "NBA_Probability_Engine_V3.csv",
    "NBA_Probability_Summary_V3.csv",
    "NBA_Injury_Report.csv",
    "NBA_Context_Aware_Forecast.csv"
]

print("\nDATABASE, MODELLI E PREVISIONI\n")

for filename in required_files:

    path = BASE_DIR / filename

    if path.is_file():
        print(f"OK | {filename}")
    else:
        print(f"MANCANTE | {filename}")

print("\nPRODUCTION PIPELINE\n")

pipeline = BASE_DIR / "Production_Pipeline"

for step in [12, 14, 15, 6, 17, 18, 19]:

    path = pipeline / f"step_{step}.py"

    if path.is_file():
        print(f"OK | STEP {step}")
    else:
        print(f"MANCANTE | STEP {step}")

print("\nMANIFEST:")

print(
    "OK" if (pipeline / "pipeline_manifest.json").is_file()
    else "MANCANTE"
)

print("\nDASHBOARD V5:")

dashboard = Path("/content/STEP_24_Dashboard_V5.py")

print("OK" if dashboard.is_file() else "DA RIPRISTINARE")

print("\n" + "=" * 60)
print("VERIFICA COMPLETATA")
print("=" * 60)


In [ ]:

# ============================================================
# STEP 24.4 — PRODUCTION PIPELINE RECOVERY
# ============================================================

import json
import re
import hashlib

from pathlib import Path
from google.colab import files
from IPython import get_ipython

BASE_DIR = Path(
    "/content/drive/MyDrive/NBA_Data_Science"
)

if not (BASE_DIR / "nba_database.db").is_file():
    raise FileNotFoundError("Database NBA non trovato.")

PIPELINE_DIR = BASE_DIR / "Production_Pipeline"
PIPELINE_DIR.mkdir(parents=True, exist_ok=True)

print("Carica il notebook NBA originale")

uploaded = files.upload()

filename = next(
    (name for name in uploaded if name.endswith(".ipynb")),
    None
)

if filename is None:
    raise ValueError("Carica un file .ipynb")

notebook = json.loads(uploaded[filename])

REQUIRED_STEPS = [12, 14, 15, 6, 17, 18, 19]

candidates = {step: [] for step in REQUIRED_STEPS}

for index, cell in enumerate(notebook["cells"]):

    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))
    header = "\n".join(source.splitlines()[:15])

    match = re.search(
        r"(?im)^\s*#\s*STEP\s+(\d+)\b",
        header
    )

    if match:

        step = int(match.group(1))

        if step in candidates:

            candidates[step].append({
                "index": index,
                "source": source,
                "header": header
            })

selected = {}

for step in REQUIRED_STEPS:

    options = candidates[step]

    if not options:
        raise RuntimeError(f"STEP {step} mancante")

    if step == 18 and len(options) > 1:

        options = [
            x for x in options
            if "3.1" in x["header"]
            and "FIXED" in x["header"].upper()
        ]

    elif step == 12 and len(options) > 1:

        options = [
            x for x in options
            if "failed_dates" in x["source"]
            and "Nuovi box score" in x["source"]
        ]

    if len(options) != 1:
        raise RuntimeError(
            f"STEP {step}: versione non identificabile"
        )

    choice = options[0]

    compile(
        get_ipython().transform_cell(choice["source"]),
        f"step_{step}.py",
        "exec"
    )

    selected[step] = choice

manifest = []

for step, choice in selected.items():

    source = choice["source"]
    filename = f"step_{step}.py"

    path = PIPELINE_DIR / filename

    temporary = path.with_suffix(".tmp")
    temporary.write_text(source, encoding="utf-8")
    temporary.replace(path)

    checksum = hashlib.sha256(
        path.read_bytes()
    ).hexdigest()

    manifest.append({
        "step": step,
        "filename": filename,
        "source_cell": choice["index"],
        "lines": len(source.splitlines()),
        "sha256": checksum
    })

    print(f"OK | STEP {step} RIPRISTINATO")

manifest_path = PIPELINE_DIR / "pipeline_manifest.json"

temporary = manifest_path.with_suffix(".tmp")

temporary.write_text(
    json.dumps(manifest, indent=4),
    encoding="utf-8"
)

temporary.replace(manifest_path)

print("\n" + "=" * 55)
print("PRODUCTION PIPELINE RIPRISTINATA")
print("Moduli disponibili:", len(manifest))
print("Percorso:", PIPELINE_DIR)
print("=" * 55)


In [ ]:

# ============================================================
# STEP 24.5 — NBA FULL UPDATE
# ============================================================

from pathlib import Path
from IPython import get_ipython

def update_nba_system():

    import json
    import hashlib
    import sqlite3
    import time
    import pandas as pd

    from datetime import datetime, timezone

    base = Path(
        "/content/drive/MyDrive/NBA_Data_Science"
    )

    pipeline = base / "Production_Pipeline"
    database = base / "nba_database.db"

    steps = [12, 14, 15, 6, 17, 18, 19]

    names = {
        12: "Database Update",
        14: "Accuracy Tracker",
        15: "Feature Engineering",
        6: "Team Predictions",
        17: "Player Forecast V3",
        18: "Probability Engine",
        19: "Injury Intelligence"
    }

    # 1. Controllo preliminare

    print("NBA INTELLIGENCE V5")
    print("=" * 60)

    manifest_path = pipeline / "pipeline_manifest.json"

    if not manifest_path.is_file():
        raise FileNotFoundError(
            "Manifest mancante. Eseguire STEP 24.4."
        )

    manifest = json.loads(
        manifest_path.read_text(encoding="utf-8")
    )

    manifest = {
        int(item["step"]): item
        for item in manifest
    }

    sources = {}

    for step in steps:

        path = pipeline / f"step_{step}.py"

        if not path.is_file():
            raise FileNotFoundError(
                f"STEP {step} mancante."
            )

        checksum = hashlib.sha256(
            path.read_bytes()
        ).hexdigest()

        if checksum != manifest[step]["sha256"]:
            raise RuntimeError(
                f"STEP {step}: integrità non valida."
            )

        sources[step] = path.read_text(
            encoding="utf-8"
        )

        print(f"OK | STEP {step}")

    # 2. Backup del database

    if not database.is_file():
        raise FileNotFoundError("Database mancante.")

    backup_dir = base / "Backups"
    backup_dir.mkdir(exist_ok=True)

    stamp = datetime.now(timezone.utc).strftime(
        "%Y%m%d_%H%M%S"
    )

    backup_path = backup_dir / f"nba_backup_{stamp}.db"

    original = sqlite3.connect(
        database.as_uri() + "?mode=ro",
        uri=True
    )

    backup = sqlite3.connect(backup_path)

    try:
        original.backup(backup)
    finally:
        backup.close()
        original.close()

    print("\nBackup database completato.")

    # 3. Aggiornamento dei moduli

    records = []
    ip = get_ipython()

    for index, step in enumerate(steps, 1):

        print("\n" + "=" * 60)
        print(f"[{index}/7] STEP {step}")
        print(names[step])
        print("=" * 60)

        started = time.time()

        status = "SUCCESS"
        error = ""

        try:

            result = ip.run_cell(
                sources[step],
                store_history=False
            )

            if result.error_before_exec:
                raise result.error_before_exec

            if result.error_in_exec:
                raise result.error_in_exec

            if step == 12:

                for flag in ["failed", "failed_dates"]:

                    if ip.user_ns.get(flag):
                        raise RuntimeError(
                            "Aggiornamento database parziale: "
                            + flag
                        )

        except Exception as exc:

            status = "FAILED"
            error = str(exc)

        records.append({
            "Step": step,
            "Module": names[step],
            "Status": status,
            "Seconds": round(time.time() - started, 2),
            "Error": error
        })

        print(f"STEP {step}: {status}")

        if status == "FAILED":
            print(error)
            print("Pipeline interrotta.")
            break

    # 4. Salvataggio report

    report = pd.DataFrame(records)

    logs = pipeline / "Logs"
    logs.mkdir(exist_ok=True)

    report.to_csv(
        logs / f"Update_{stamp}.csv",
        index=False
    )

    report.to_csv(
        logs / "Latest_Update.csv",
        index=False
    )

    print("\n" + "=" * 60)
    print("NBA UPDATE REPORT")
    print("=" * 60)

    display(report)

    success = (
        len(records) == len(steps)
        and all(
            row["Status"] == "SUCCESS"
            for row in records
        )
    )

    if success:
        print("AGGIORNAMENTO COMPLETATO")
        print("PRONTO PER STEP 24.6")
    else:
        print("AGGIORNAMENTO INCOMPLETO")

    return report

nba_update_report = update_nba_system()


In [ ]:
# ============================================================
# STEP 24.6 FIX — RECUPERO FILE DASHBOARD
# ============================================================

from google.colab import drive, files
from pathlib import Path
import ast

drive.mount("/content/drive")

folder = Path(
    "/content/drive/MyDrive/NBA_Data_Science/Dashboard"
)

folder.mkdir(parents=True, exist_ok=True)

destination = folder / "STEP_24_Dashboard_V5.py"

if not destination.is_file():

    print("Seleziona STEP_24_Dashboard_V5.py")

    uploaded = files.upload()

    python_files = [
        name for name in uploaded
        if name.lower().endswith(".py")
    ]

    if len(python_files) != 1:
        raise ValueError(
            "Devi caricare un solo file Python .py, "
            "non il notebook .ipynb."
        )

    filename = python_files[0]

    source = uploaded[filename].decode("utf-8")

    ast.parse(source)

    if "gradio" not in source.lower():
        raise ValueError("File dashboard non riconosciuto.")

    destination.write_text(source, encoding="utf-8")

print("=" * 50)
print("DASHBOARD SALVATA CORRETTAMENTE")
print("Percorso:", destination)
print("Dimensione:", destination.stat().st_size, "bytes")
print("PRONTO PER STEP 24.6")
print("=" * 50)

In [ ]:
from pathlib import Path

dashboard = Path(
    "/content/drive/MyDrive/NBA_Data_Science/"
    "Dashboard/STEP_24_Dashboard_V5.py"
)

if not dashboard.is_file():
    raise FileNotFoundError("Dashboard non ancora salvata.")

print("Dashboard V5 disponibile")
print("Dimensione:", dashboard.stat().st_size, "bytes")
print("Rieseguire STEP 24.6 per avviare il server.")

In [ ]:

# ============================================================
# STEP 25A — ADVANCED PROBABILITY ENGINE
# ============================================================

from pathlib import Path
from google.colab import drive
from IPython import get_ipython

import ast
import pandas as pd

drive.mount("/content/drive")

BASE_DIR = Path(
    "/content/drive/MyDrive/NBA_Data_Science"
)

STEP18_PATH = (
    BASE_DIR / "Production_Pipeline" / "step_18.py"
)

if not STEP18_PATH.is_file():
    raise FileNotFoundError(
        "STEP 18 non trovato nella Production Pipeline."
    )

source = STEP18_PATH.read_text(encoding="utf-8")

# Nuove soglie con incrementi unitari

ADVANCED_THRESHOLDS = {
    "PTS": list(range(0, 101)),
    "REB": list(range(0, 41)),
    "AST": list(range(0, 41)),
    "3PM": list(range(0, 21)),
    "PRA": list(range(0, 151))
}

# Analisi del codice originale

ip = get_ipython()

tree = ast.parse(
    ip.transform_cell(source)
)

class ThresholdUpgrade(ast.NodeTransformer):

    def __init__(self):
        self.replacements = 0

    def visit_Assign(self, node):

        self.generic_visit(node)

        for target in node.targets:

            if (
                isinstance(target, ast.Name)
                and target.id == "THRESHOLDS"
            ):

                node.value = ast.parse(
                    repr(ADVANCED_THRESHOLDS),
                    mode="eval"
                ).body

                self.replacements += 1

        return node

upgrade = ThresholdUpgrade()

tree = upgrade.visit(tree)

ast.fix_missing_locations(tree)

if upgrade.replacements != 1:
    raise RuntimeError(
        "Configurazione THRESHOLDS non identificata "
        "in maniera univoca."
    )

advanced_source = ast.unparse(tree)

# Salvataggio separato del nuovo modulo

advanced_path = (
    BASE_DIR / "Production_Pipeline" /
    "step_25_probability.py"
)

advanced_path.write_text(
    advanced_source,
    encoding="utf-8"
)

print("=" * 60)
print("ADVANCED PROBABILITY ENGINE")
print("=" * 60)

for target, thresholds in ADVANCED_THRESHOLDS.items():

    print(
        f"{target}: "
        f"{min(thresholds)} - {max(thresholds)} "
        f"| {len(thresholds)} soglie"
    )

print("\nNuovo modulo salvato:")
print(advanced_path)

print("\nEsecuzione del motore probabilistico...")

result = ip.run_cell(
    advanced_source,
    store_history=False
)

if result.error_before_exec:
    raise result.error_before_exec

if result.error_in_exec:
    raise result.error_in_exec

# Verifica dei risultati

output_path = (
    BASE_DIR / "NBA_Probability_Engine_V3.csv"
)

if not output_path.is_file():
    raise FileNotFoundError(
        "File delle probabilità non generato."
    )

probabilities = pd.read_csv(output_path)

print("\n" + "=" * 60)
print("VERIFICA ADVANCED PROBABILITY ENGINE")
print("=" * 60)

print("Probabilità disponibili:", len(probabilities))

checks = []

for target, expected in ADVANCED_THRESHOLDS.items():

    subset = probabilities[
        probabilities["Target"] == target
    ]

    found = set(
        subset["Threshold"].dropna().astype(int)
    )

    complete = bool(found) and set(expected).issubset(found)

    checks.append(complete)

    print(
        target,
        "| Soglie:", len(found),
        "|", "OK" if complete else "INCOMPLETO"
    )

if not all(checks):
    raise RuntimeError(
        "Alcune soglie non sono state generate."
    )

if not probabilities[
    "Probability_At_Least"
].between(0, 1).all():

    raise ValueError(
        "Probabilità non valide."
    )

print("\nSTEP 25A COMPLETATO")
print("PRONTO PER ADVANCED PROBABILITY EXPLORER")


In [ ]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path(
    "/content/drive/MyDrive/NBA_Data_Science"
)

output_path = BASE_DIR / "NBA_Probability_Engine_V3.csv"

if not output_path.is_file():
    raise FileNotFoundError(output_path)

probabilities = pd.read_csv(output_path)

expected = {
    "PTS": 101,
    "REB": 41,
    "AST": 41,
    "3PM": 21,
    "PRA": 151
}

print("=" * 60)
print("ADVANCED PROBABILITY ENGINE — VERIFICA")
print("=" * 60)

print("Probabilità generate:", len(probabilities))

for target, count in expected.items():

    subset = probabilities[
        probabilities["Target"] == target
    ]

    found = set(subset["Threshold"].dropna())

    if found != set(range(count)):
        raise ValueError(f"Soglie incomplete: {target}")

    print(f"{target}: {len(found)} soglie — OK")

if not probabilities[
    "Probability_At_Least"
].between(0, 1).all():
    raise ValueError("Probabilità non valide")

print("=" * 60)
print("STEP 25A COMPLETATO")
print("PRONTO PER STEP 25B")
print("=" * 60)

In [ ]:

# ============================================================
# STEP 26A — NBA INJURY INTELLIGENCE V2
# ============================================================

!pip -q install pymupdf beautifulsoup4

import requests
import pandas as pd
import fitz
import re
import unicodedata

from pathlib import Path
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from datetime import datetime, timezone
from zoneinfo import ZoneInfo

ROOT26 = Path(
    "/content/drive/MyDrive/NBA_Data_Science"
)

FORECAST26 = ROOT26 / "NBA_Forecast_Engine_V3.csv"

NBA_URL = "https://www.nba.com/players/injury-report"

ESPN_URL = (
    "https://site.api.espn.com/apis/site/v2/"
    "sports/basketball/nba/injuries"
)

session26 = requests.Session()

session26.headers.update({
    "User-Agent": "Mozilla/5.0",
    "Accept": "*/*"
})

now26 = datetime.now(timezone.utc)

today_et = now26.astimezone(
    ZoneInfo("America/New_York")
).date()

VALID_STATUS = [
    "Out",
    "Doubtful",
    "Questionable",
    "Probable",
    "Available"
]

# ============================================================
# 1. NORMALIZZAZIONE
# ============================================================

def normalize26(value):

    value = unicodedata.normalize(
        "NFKD", str(value)
    )

    value = "".join(
        c for c in value
        if not unicodedata.combining(c)
    )

    return re.sub(
        r"[^a-z0-9]", "", value.lower()
    )


def status26(value):

    value = str(value).strip().lower()

    mapping = {
        "out": "Out",
        "doubtful": "Doubtful",
        "questionable": "Questionable",
        "probable": "Probable",
        "available": "Available",
        "day-to-day": "Questionable",
        "game time decision": "Questionable"
    }

    return mapping.get(value, "Unknown")


# ============================================================
# 2. CARICAMENTO FORECAST
# ============================================================

forecast26 = pd.read_csv(
    FORECAST26,
    dtype={
        "game_id": str,
        "player_id": str
    }
)

forecast26["date"] = pd.to_datetime(
    forecast26["date"],
    utc=True,
    errors="coerce"
)

forecast26 = forecast26[
    forecast26["date"] > pd.Timestamp(now26)
].copy()

if forecast26.empty:
    raise RuntimeError(
        "Nessun forecast futuro. Aggiornare STEP 17."
    )

forecast26 = forecast26.sort_values(
    "generated_at"
).drop_duplicates(
    ["game_id", "player_id"],
    keep="last"
)

players26 = forecast26[
    ["player_id", "Giocatore", "Squadra"]
].drop_duplicates()

print("Giocatori da verificare:", len(players26))

# ============================================================
# 3. ESPN INJURY REPORT
# ============================================================

espn_records26 = []
espn_ok26 = False

try:

    response = session26.get(
        ESPN_URL,
        timeout=30
    )

    response.raise_for_status()

    payload = response.json()

    groups = payload.get("injuries")

    if not isinstance(groups, list):
        raise ValueError(
            "Struttura ESPN non riconosciuta."
        )

    for group in groups:

        team = group.get(
            "displayName", ""
        )

        for item in group.get("injuries", []):

            athlete = item.get("athlete", {})

            details = item.get("details") or {}

            raw_status = item.get("status", "")

            if isinstance(raw_status, dict):
                raw_status = raw_status.get(
                    "name", ""
                )

            espn_records26.append({
                "player_id": str(
                    athlete.get("id", "")
                ),
                "Giocatore": athlete.get(
                    "displayName", ""
                ),
                "Squadra_ESPN": team,
                "Injury_Status": status26(
                    raw_status
                ),
                "Raw_Status": str(raw_status),
                "Injury_Type": details.get(
                    "type", ""
                ),
                "Injury_Location": details.get(
                    "location", ""
                ),
                "Return_Date": details.get(
                    "returnDate", ""
                ),
                "Comment": item.get(
                    "longComment",
                    item.get("shortComment", "")
                ),
                "Report_Date": item.get(
                    "date", ""
                ),
                "Fetched_At": now26.isoformat(),
                "Source": "ESPN"
            })

    espn_ok26 = bool(espn_records26)

    print(
        "ESPN:",
        len(espn_records26),
        "segnalazioni"
    )

except Exception as e:
    print("ESPN non disponibile:", e)


# ============================================================
# 4. REPORT UFFICIALE NBA
# ============================================================

nba_records26 = []
nba_ok26 = False
official_pdf26 = None
report_time26 = None

try:

    page = session26.get(
        NBA_URL,
        timeout=30
    )

    page.raise_for_status()

    soup = BeautifulSoup(
        page.text,
        "html.parser"
    )

    links = []

    for a in soup.find_all("a", href=True):

        url = urljoin(
            NBA_URL, a["href"]
        )

        match = re.search(
            r"Injury-Report_"
            r"(\d{4}-\d{2}-\d{2})_"
            r"(\d{2})_(\d{2})(AM|PM)",
            url,
            re.I
        )

        if not match:
            continue

        stamp = datetime.strptime(
            f"{match[1]} "
            f"{match[2]}:{match[3]} "
            f"{match[4].upper()}",
            "%Y-%m-%d %I:%M %p"
        ).replace(
            tzinfo=ZoneInfo("America/New_York")
        )

        if (
            stamp.date() == today_et
            and stamp <= now26
            and "ak-static.cms.nba.com" in url
        ):

            links.append((stamp, url))

    if not links:
        raise RuntimeError(
            "Nessun PDF ufficiale odierno trovato."
        )

    report_time26, official_pdf26 = max(
        links,
        key=lambda x: x[0]
    )

    response = session26.get(
        official_pdf26,
        timeout=40
    )

    response.raise_for_status()

    document = fitz.open(
        stream=response.content,
        filetype="pdf"
    )

    text26 = "\n".join(
        page.get_text(sort=True)
        for page in document
    )

    document.close()

    (
        ROOT26 / "NBA_Official_Injury_Report.txt"
    ).write_text(
        text26,
        encoding="utf-8"
    )

    print("Report ufficiale NBA:")
    print(official_pdf26)

    # Riconoscimento dei nomi nel formato
    # Cognome, Nome utilizzato dal report NBA.

    for _, player in players26.iterrows():

        name = player["Giocatore"]

        parts = name.split()

        if len(parts) < 2:
            continue

        variants = []

        for split in range(1, len(parts)):

            first = " ".join(parts[:split])
            last = " ".join(parts[split:])

            variants.append(
                last + ", " + first
            )

        variants.append(name)

        for variant in variants:

            pattern = (
                re.escape(variant)
                + r"\s+"
                + r"(Out|Doubtful|Questionable|"
                  r"Probable|Available)\b"
            )

            match = re.search(
                pattern,
                text26,
                re.I
            )

            if not match:
                continue

            nba_records26.append({
                "player_id": player["player_id"],
                "Giocatore": name,
                "Squadra_ESPN": player["Squadra"],
                "Injury_Status": status26(
                    match.group(1)
                ),
                "Raw_Status": match.group(1),
                "Injury_Type": "",
                "Injury_Location": "",
                "Return_Date": "",
                "Comment": (
                    "Da verificare nel PDF ufficiale"
                ),
                "Report_Date": (
                    report_time26.isoformat()
                ),
                "Fetched_At": now26.isoformat(),
                "Source": "NBA Official"
            })

            break

    nba_ok26 = bool(nba_records26)

    print(
        "NBA Official:",
        len(nba_records26),
        "giocatori riconosciuti"
    )

except Exception as e:
    print("Report NBA non disponibile:", e)


# ============================================================
# 5. UNIFICAZIONE FONTI
# ============================================================

columns26 = [
    "player_id",
    "Giocatore",
    "Squadra_ESPN",
    "Injury_Status",
    "Raw_Status",
    "Injury_Type",
    "Injury_Location",
    "Return_Date",
    "Comment",
    "Report_Date",
    "Fetched_At",
    "Source"
]

injuries26 = pd.DataFrame(
    espn_records26 + nba_records26,
    columns=columns26
)

injuries26 = injuries26[
    injuries26["player_id"].astype(str) != ""
].copy()

injuries26 = injuries26[
    injuries26["Injury_Status"] != "Unknown"
]

# Non sovrascrivere i dati precedenti
# se entrambi i recuperi falliscono.

if injuries26.empty:

    raise RuntimeError(
        "Nessuna segnalazione valida recuperata. "
        "Controllare le fonti e il parser."
    )

# Conservazione di entrambe le fonti

injuries26.to_csv(
    ROOT26 / "NBA_Injury_Sources_V2.csv",
    index=False
)

# ============================================================
# 6. DISPONIBILITA PER SINGOLA PARTITA
# ============================================================

availability26 = []

for _, r in forecast26.iterrows():

    pid = str(r["player_id"])

    player_rows = injuries26[
        injuries26["player_id"].astype(str) == pid
    ]

    team26 = normalize26(r["Squadra"])

    official = player_rows[
        (player_rows["Source"] == "NBA Official")
        & (
            player_rows["Squadra_ESPN"]
            .map(normalize26) == team26
        )
    ]

    espn = player_rows[
        player_rows["Source"] == "ESPN"
    ]

    game_day = r["date"].tz_convert(
        "America/New_York"
    ).date()

    status = "Unknown"
    source = "Not verified"
    note = "Disponibilità non confermata"

    if game_day == today_et:

        candidates = (
            official
            if not official.empty
            else espn
        )

        if not candidates.empty:

            statuses = set(
                candidates["Injury_Status"]
            )

            if len(statuses) == 1:

                status = next(iter(statuses))
                source = candidates.iloc[-1]["Source"]

                note = "Segnalazione recuperata"

                if (
                    not official.empty
                    and not espn.empty
                    and set(espn["Injury_Status"])
                    != statuses
                ):

                    status = "Unknown"
                    source = "NBA / ESPN conflict"
                    note = (
                        "Fonti discordanti: "
                        "verifica manuale necessaria"
                    )

            else:
                note = "Segnalazioni discordanti"

    else:
        note = "Report non verificato per questa data"

    flag = {
        "Out": "EXCLUDE",
        "Doubtful": "HIGH_RISK",
        "Questionable": "UNCERTAIN",
        "Probable": "PROBABLE",
        "Available": "AVAILABLE",
        "Unknown": "UNVERIFIED"
    }[status]

    availability26.append({
        "game_id": r["game_id"],
        "player_id": pid,
        "date": r["date"],
        "Giocatore": r["Giocatore"],
        "Squadra": r["Squadra"],
        "Avversario": r["Avversario"],
        "Injury_Status": status,
        "Availability_Flag": flag,
        "Injury_Source": source,
        "Availability_Note": note,
        "Fetched_At": now26.isoformat()
    })

availability26 = pd.DataFrame(
    availability26
)

availability26.to_csv(
    ROOT26 / "NBA_Availability_V2.csv",
    index=False
)

# ============================================================
# 7. FORECAST OPERATIVO
# ============================================================

context26 = forecast26.drop(
    columns=[
        "Injury_Status",
        "Availability_Flag"
    ],
    errors="ignore"
).merge(
    availability26[
        [
            "game_id",
            "player_id",
            "Injury_Status",
            "Availability_Flag",
            "Injury_Source",
            "Availability_Note"
        ]
    ],
    on=["game_id", "player_id"],
    how="left",
    validate="one_to_one"
)

context26["Roster_Status"] = (
    context26["Availability_Flag"]
)

context26.to_csv(
    ROOT26 / "NBA_Forecast_With_Injuries_V2.csv",
    index=False
)

operational26 = context26[
    context26["Availability_Flag"] != "EXCLUDE"
].copy()

operational26.to_csv(
    ROOT26 / "NBA_Operational_Forecast_V2.csv",
    index=False
)

# ============================================================
# 8. REPORT
# ============================================================

print("\n" + "=" * 60)
print("NBA INJURY INTELLIGENCE V2")
print("=" * 60)

print("ESPN:", "OK" if espn_ok26 else "FAILED")
print("NBA:", "OK" if nba_ok26 else "FAILED")

print("\nStati disponibilità:")

display(
    availability26[
        "Injury_Status"
    ].value_counts()
)

print("\nGiocatori OUT:")

display(
    availability26[
        availability26["Injury_Status"] == "Out"
    ]
)

print("\nCONTROLLO LAMELO BALL")

display(
    availability26[
        availability26["Giocatore"].str.contains(
            "LaMelo Ball",
            case=False,
            na=False
        )
    ]
)

print("\nFile salvati su Google Drive:")

for file in [
    "NBA_Injury_Sources_V2.csv",
    "NBA_Availability_V2.csv",
    "NBA_Forecast_With_Injuries_V2.csv",
    "NBA_Operational_Forecast_V2.csv"
]:
    print(file)

print("\nSTEP 26A COMPLETATO")


In [ ]:

# ============================================================
# STEP 26B — INJURY INTEGRATION & DASHBOARD RESTART
# ============================================================

from google.colab import drive
from pathlib import Path
from datetime import datetime
import pandas as pd
import shutil

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/NBA_Data_Science")

BACKUP = ROOT / "Backup_Injury_V2" / datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

BACKUP.mkdir(parents=True, exist_ok=True)

# ============================================================
# 1. CARICAMENTO NUOVI DATI
# ============================================================

keys = ["game_id", "player_id"]

availability = pd.read_csv(
    ROOT / "NBA_Availability_V2.csv",
    dtype=str
)

injuries = pd.read_csv(
    ROOT / "NBA_Injury_Sources_V2.csv",
    dtype=str
)

availability = availability.drop_duplicates(
    keys, keep="last"
)

columns = [
    "Injury_Status",
    "Availability_Flag",
    "Injury_Source",
    "Availability_Note"
]

def backup_file(path):
    if path.exists():
        shutil.copy2(path, BACKUP / path.name)

def save_csv(df, path):
    backup_file(path)
    temp = path.with_suffix(".tmp")
    df.to_csv(temp, index=False)
    temp.replace(path)

def integrate_availability(df):

    df = df.drop(
        columns=columns,
        errors="ignore"
    )

    df = df.merge(
        availability[keys + columns],
        on=keys,
        how="left",
        validate="many_to_one"
    )

    df["Injury_Status"] = (
        df["Injury_Status"].fillna("Unknown")
    )

    df["Availability_Flag"] = (
        df["Availability_Flag"].fillna("UNVERIFIED")
    )

    df["Roster_Status"] = df["Availability_Flag"]

    return df

# ============================================================
# 2. AGGIORNAMENTO INJURY REPORT
# ============================================================

save_csv(
    injuries,
    ROOT / "NBA_Injury_Report.csv"
)

# ============================================================
# 3. AGGIORNAMENTO FORECAST CONTESTUALE
# ============================================================

raw = pd.read_csv(
    ROOT / "NBA_Forecast_Engine_V3.csv",
    dtype={k: str for k in keys}
)

context_path = ROOT / "NBA_Context_Aware_Forecast.csv"

context = raw.copy()

if context_path.exists():

    previous = pd.read_csv(
        context_path,
        dtype={k: str for k in keys}
    )

    join_keys = keys + ["generated_at"]

    if all(k in previous.columns for k in join_keys):

        extra = [
            c for c in previous.columns
            if c not in raw.columns
            and c not in columns
        ]

        context = raw.merge(
            previous[join_keys + extra].drop_duplicates(
                join_keys, keep="last"
            ),
            on=join_keys,
            how="left"
        )

context = integrate_availability(context)

save_csv(context, context_path)

# ============================================================
# 4. AGGIORNAMENTO PROBABILITA
# ============================================================

prob = pd.read_csv(
    ROOT / "NBA_Probability_Engine_V3.csv",
    dtype={k: str for k in keys}
)

prob = integrate_availability(prob)

prob = prob[
    prob["Injury_Status"] != "Out"
].copy()

save_csv(
    prob,
    ROOT / "NBA_Context_Probabilities.csv"
)

# ============================================================
# 5. AGGIORNAMENTO FORMAZIONI
# ============================================================

lineup_path = ROOT / "NBA_Estimated_Lineups.csv"

if lineup_path.exists():

    lineup = pd.read_csv(
        lineup_path,
        dtype={k: str for k in keys}
    )

    if all(k in lineup.columns for k in keys):

        lineup = integrate_availability(lineup)

        lineup = lineup[
            lineup["Injury_Status"] != "Out"
        ].copy()

        save_csv(lineup, lineup_path)

# ============================================================
# 6. CONTROLLO FINALE
# ============================================================

print("=" * 60)
print("INJURY INTELLIGENCE V2 INTEGRATO")
print("=" * 60)

print("\nStati giocatori:")

display(
    context["Injury_Status"].value_counts()
)

print("\nGiocatori OUT:")

display(
    context.loc[
        context["Injury_Status"] == "Out",
        ["Giocatore", "Squadra", "Injury_Source"]
    ]
)

print("\nCONTROLLO LAMELO BALL:")

display(
    context.loc[
        context["Giocatore"].str.contains(
            "LaMelo Ball",
            case=False,
            na=False
        ),
        [
            "Giocatore",
            "Injury_Status",
            "Availability_Flag",
            "Injury_Source",
            "Forecast_PTS"
        ]
    ]
)

print("\nBackup:", BACKUP)
print("\nSTEP 26B COMPLETATO")


In [ ]:

# STEP 26C — DASHBOARD RESTART

from google.colab import drive
from pathlib import Path
from IPython import get_ipython
import gradio as gr

drive.mount("/content/drive")

root = Path("/content/drive/MyDrive/NBA_Data_Science")

matches = list(root.rglob("STEP_24_Dashboard_V5.py"))

if not matches:
    raise FileNotFoundError(
        "Dashboard V5 non trovata. "
        "Esegui la cella originale STEP 24."
    )

if len(matches) > 1:
    print("Versioni disponibili:")
    for i, path in enumerate(matches):
        print(i, path)
    index = int(input("Seleziona versione: "))
    dashboard_path = matches[index]
else:
    dashboard_path = matches[0]

source = dashboard_path.read_text(encoding="utf-8")

print("Dashboard:", dashboard_path)

# Chiude i server Gradio precedenti
gr.close_all()

# Avvia la dashboard completa
result = get_ipython().run_cell(
    source,
    store_history=False
)

if result.error_before_exec:
    raise result.error_before_exec

if result.error_in_exec:
    raise result.error_in_exec


In [ ]:

# ============================================================
# STEP 26C FIX — DASHBOARD V5
# ============================================================

from pathlib import Path
from google.colab import drive
from IPython import get_ipython
from datetime import datetime

import gradio as gr
import re
import shutil
import inspect

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/NBA_Data_Science")

DASHBOARD = ROOT / "Dashboard" / "STEP_24_Dashboard_V5.py"

if not DASHBOARD.exists():
    raise FileNotFoundError(DASHBOARD)

# 1. Backup del codice originale

backup = DASHBOARD.with_name(
    "Dashboard_Backup_"
    + datetime.now().strftime("%Y%m%d_%H%M%S")
    + ".py"
)

shutil.copy2(DASHBOARD, backup)

# 2. Lettura dashboard

source = DASHBOARD.read_text(encoding="utf-8")

# 3. Correzione parametri incompatibili

unsupported = [
    "input_text_color",
    "input_text_color_dark"
]

supported = inspect.signature(
    gr.themes.Soft().set
).parameters

for parameter in unsupported:

    if parameter not in supported:

        source, count = re.subn(
            rf""",\s*{parameter}\s*=\s*["'][^"']*["']""",
            "",
            source
        )

        print("Corretto:", parameter, count)

# 4. Controllo sintassi

compile(source, str(DASHBOARD), "exec")

# 5. Salvataggio permanente su Drive

DASHBOARD.write_text(source, encoding="utf-8")

print("Dashboard corretta e salvata.")
print("Versione Gradio:", gr.__version__)

# 6. Chiusura vecchi server

gr.close_all()

# 7. Avvio dashboard completa

print("\nAVVIO NBA INTELLIGENCE V5...\n")

result = get_ipython().run_cell(
    source,
    store_history=False
)

if result.error_before_exec:
    raise result.error_before_exec

if result.error_in_exec:
    raise result.error_in_exec

print("\nDASHBOARD AVVIATA")


In [ ]:

# STEP 26D — SECURE LOGIN

import gradio as gr
import hashlib
import hmac
import secrets
import inspect

from getpass import getpass
from pathlib import Path

USERNAME = "admin"

password = getpass("Inserisci la password precedente: ")

SALT = secrets.token_bytes(32)

PASSWORD_HASH = hashlib.pbkdf2_hmac(
    "sha256",
    password.encode(),
    SALT,
    600_000
)

del password

def secure_auth(username, password):

    candidate = hashlib.pbkdf2_hmac(
        "sha256",
        password.encode(),
        SALT,
        600_000
    )

    return (
        username == USERNAME
        and hmac.compare_digest(
            candidate, PASSWORD_HASH
        )
    )

DASHBOARD = Path(
    "/content/drive/MyDrive/NBA_Data_Science/"
    "Dashboard/STEP_24_Dashboard_V5.py"
)

source = DASHBOARD.read_text(encoding="utf-8")

gr.close_all()

original_launch = gr.Blocks.launch

def secure_launch(self, *args, **kwargs):

    kwargs["auth"] = secure_auth
    kwargs["share"] = True
    kwargs["inline"] = False
    kwargs["debug"] = False

    return original_launch(
        self, *args, **kwargs
    )

gr.Blocks.launch = secure_launch

try:

    exec(
        compile(source, str(DASHBOARD), "exec"),
        {
            "__name__": "__main__",
            "__file__": str(DASHBOARD)
        }
    )

finally:

    gr.Blocks.launch = original_launch


In [ ]:

# ============================================================
# STEP 27 — PROBABILITY BACKTESTING ENGINE
# ============================================================

from google.colab import drive
from pathlib import Path
import sqlite3
import pandas as pd
import numpy as np
import plotly.express as px

drive.mount("/content/drive")

BT_ROOT = Path("/content/drive/MyDrive/NBA_Data_Science")
BT_DB = BT_ROOT / "nba_database.db"

def probability_backtest():

    with sqlite3.connect(BT_DB) as conn:

        predictions = pd.read_sql_query("""
            SELECT *
            FROM nba_probability_history_v3
        """, conn)

        games = pd.read_sql_query("""
            SELECT *
            FROM espn_games
        """, conn)

        actual = pd.read_sql_query("""
            SELECT *
            FROM espn_player_boxscores
        """, conn)

    for df in [predictions, games, actual]:
        df["game_id"] = df["game_id"].astype(str)

    for df in [predictions, actual]:
        df["player_id"] = df["player_id"].astype(str)

    games = games[
        pd.to_numeric(
            games["completed"], errors="coerce"
        ) == 1
    ].copy()

    actual = actual[
        pd.to_numeric(
            actual["did_not_play"], errors="coerce"
        ) == 0
    ].copy()

    actual = actual[
        pd.to_numeric(
            actual["minutes"], errors="coerce"
        ) > 0
    ].copy()

    games["Tipoff"] = pd.to_datetime(
        games["game_date"], utc=True, errors="coerce"
    )

    predictions["generated_at"] = pd.to_datetime(
        predictions["generated_at"],
        utc=True,
        errors="coerce"
    )

    # Classificazione stagione: senza deduzioni arbitrarie

    games["Season_Type"] = "Unknown"

    for column in [
        "season_type", "season_type_id",
        "season_type_name"
    ]:

        if column in games.columns:

            values = games[column].astype(str).str.lower()

            games.loc[
                values.isin([
                    "2", "2.0", "regular",
                    "regular season"
                ]),
                "Season_Type"
            ] = "Regular Season"

            games.loc[
                values.isin([
                    "3", "3.0", "postseason",
                    "playoffs"
                ]),
                "Season_Type"
            ] = "Postseason"

    games = games.drop_duplicates("game_id")

    predictions = predictions.merge(
        games[
            ["game_id", "Tipoff", "Season_Type"]
        ],
        on="game_id",
        how="inner"
    )

    # Vietate previsioni successive all'inizio

    predictions = predictions[
        predictions["generated_at"] <
        predictions["Tipoff"]
    ].copy()

    predictions["probability"] = pd.to_numeric(
        predictions["probability"],
        errors="coerce"
    )

    predictions["threshold"] = pd.to_numeric(
        predictions["threshold"],
        errors="coerce"
    )

    predictions = predictions[
        predictions["probability"].between(0, 1)
        & predictions["threshold"].notna()
        & (predictions["threshold"] > 0)
    ]

    # Una previsione per evento:
    # ultima disponibile prima della partita

    predictions = predictions.sort_values(
        "generated_at"
    ).drop_duplicates(
        [
            "game_id", "player_id",
            "target", "threshold"
        ],
        keep="last"
    )

    target_map = {
        "PTS": "pts",
        "REB": "reb",
        "AST": "ast",
        "3PM": "fg3m",
        "PRA": "PRA"
    }

    for col in ["pts", "reb", "ast", "fg3m"]:
        actual[col] = pd.to_numeric(
            actual[col], errors="coerce"
        )

    actual["PRA"] = (
        actual["pts"] +
        actual["reb"] +
        actual["ast"]
    )

    actual = actual.drop_duplicates(
        ["game_id", "player_id"]
    )

    actual["Played_Date"] = pd.to_datetime(
        actual["game_date"],
        utc=True,
        errors="coerce"
    )

    actual = actual.sort_values("Played_Date")

    long_actual = actual.melt(
        id_vars=["game_id", "player_id"],
        value_vars=list(target_map.values()),
        var_name="Stat",
        value_name="Actual"
    )

    reverse_map = {
        v: k for k, v in target_map.items()
    }

    long_actual["target"] = (
        long_actual["Stat"].map(reverse_map)
    )

    result = predictions.merge(
        long_actual[
            ["game_id", "player_id", "target", "Actual"]
        ],
        on=["game_id", "player_id", "target"],
        how="inner"
    )

    result = result.dropna(subset=["Actual"])

    if result.empty:
        raise RuntimeError(
            "Nessuna previsione pre-partita "
            "con risultato effettivo disponibile."
        )

    result["Success"] = (
        result["Actual"] >= result["threshold"]
    ).astype(int)

    result["Brier"] = (
        result["probability"] - result["Success"]
    ) ** 2

    result["Correct"] = (
        (result["probability"] >= 0.5).astype(int)
        == result["Success"]
    ).astype(int)

    result["Baseline_Probability"] = np.nan

    # Benchmark storico: ultime 20 partite
    # esclusivamente anteriori alla previsione.
    # Margine prudenziale di 6 ore per evitare
    # partite iniziate ma non ancora concluse.

    history = {
        pid: group
        for pid, group in actual.groupby("player_id")
    }

    grouping = [
        "game_id", "player_id",
        "target", "generated_at"
    ]

    for key, subset in result.groupby(grouping):

        gid, pid, target, generated = key

        player_history = history.get(pid)

        if player_history is None:
            continue

        previous = player_history[
            player_history["Played_Date"]
            < generated - pd.Timedelta(hours=6)
        ].tail(20)

        if target not in target_map:
            continue

        values = previous[
            target_map[target]
        ].dropna().to_numpy()

        if len(values) < 10:
            continue

        thresholds = subset["threshold"].to_numpy()

        baseline = (
            values[:, None] >= thresholds[None, :]
        ).mean(axis=0)

        result.loc[
            subset.index,
            "Baseline_Probability"
        ] = baseline

    result["Baseline_Brier"] = (
        result["Baseline_Probability"]
        - result["Success"]
    ) ** 2

    result["Probability_Band"] = pd.cut(
        result["probability"],
        bins=np.linspace(0, 1, 11),
        include_lowest=True,
        labels=[
            f"{i*10}-{(i+1)*10}%"
            for i in range(10)
        ]
    )

    # Funzione di aggregazione

    def summarize(group_columns):

        report = result.groupby(
            group_columns,
            observed=True
        ).agg(
            Predictions=("Success", "size"),
            Successes=("Success", "sum"),
            Mean_Probability=("probability", "mean"),
            Hit_Rate=("Success", "mean"),
            Classification_Accuracy=("Correct", "mean"),
            Brier_Score=("Brier", "mean")
        ).reset_index()

        report["Calibration_Gap"] = (
            report["Hit_Rate"] -
            report["Mean_Probability"]
        ).abs()

        return report

    calibration = summarize(["Probability_Band"])

    players = summarize(
        ["player_id", "player_name", "target"]
    )

    targets = summarize(["target"])

    thresholds = summarize(
        ["target", "threshold"]
    )

    seasons = summarize(
        ["Season_Type", "target"]
    )

    paired = result.dropna(
        subset=["Baseline_Brier"]
    )

    baseline_report = paired.groupby(
        "target"
    ).agg(
        Comparisons=("Brier", "size"),
        ML_Brier=("Brier", "mean"),
        Baseline_Brier=("Baseline_Brier", "mean")
    ).reset_index()

    baseline_report["Improvement_pct"] = np.where(
        baseline_report["Baseline_Brier"] > 0,
        100 * (
            1 - baseline_report["ML_Brier"]
            / baseline_report["Baseline_Brier"]
        ),
        np.nan
    )

    ece = np.average(
        calibration["Calibration_Gap"],
        weights=calibration["Predictions"]
    )

    overview = pd.DataFrame([{
        "Predictions": len(result),
        "Unique_Games": result["game_id"].nunique(),
        "Player_Games": result[
            ["player_id", "game_id"]
        ].drop_duplicates().shape[0],
        "Unique_Players": result["player_id"].nunique(),
        "Brier_Score": result["Brier"].mean(),
        "ECE": ece,
        "Updated_At": pd.Timestamp.now(
            tz="UTC"
        ).isoformat()
    }])

    reports = {
        "Overview": overview,
        "Calibration": calibration,
        "Players": players,
        "Targets": targets,
        "Thresholds": thresholds,
        "Season Types": seasons,
        "Baseline": baseline_report
    }

    output = BT_ROOT / "Probability_Backtesting"
    output.mkdir(exist_ok=True)

    for name, table in reports.items():
        table.to_csv(
            output / f"{name}.csv",
            index=False
        )

    result.to_csv(
        output / "Evaluated_Predictions.csv",
        index=False
    )

    with pd.ExcelWriter(
        output / "NBA_Probability_Backtesting.xlsx"
    ) as writer:

        for name, table in reports.items():
            table.to_excel(
                writer,
                sheet_name=name,
                index=False
            )

    print("STEP 27 COMPLETATO")
    display(overview)
    display(calibration)
    display(baseline_report)

    fig = px.scatter(
        calibration,
        x="Mean_Probability",
        y="Hit_Rate",
        size="Predictions",
        hover_name="Probability_Band",
        title="Probability Calibration",
        template="plotly_dark"
    )

    fig.add_shape(
        type="line",
        x0=0, y0=0,
        x1=1, y1=1,
        line=dict(dash="dash")
    )

    fig.update_xaxes(range=[0, 1])
    fig.update_yaxes(range=[0, 1])
    fig.show()

    return result

bt_results = probability_backtest()


In [ ]:
# ============================================================
# STEP 27A — DATABASE & BACKTEST DIAGNOSTICS
# ============================================================

from pathlib import Path
import pandas as pd
import sqlite3

ROOT = Path("/content/drive/MyDrive/NBA_Data_Science")
DB = ROOT / "nba_database.db"

with sqlite3.connect(DB) as conn:

    games = pd.read_sql_query(
        "SELECT * FROM espn_games", conn
    )

    actual = pd.read_sql_query(
        "SELECT * FROM espn_player_boxscores", conn
    )

    predictions = pd.read_sql_query(
        "SELECT * FROM nba_probability_history_v3",
        conn
    )

print("=" * 60)
print("1. STATO DEL DATABASE")
print("=" * 60)

print("Partite:", len(games))
print("Boxscore giocatori:", len(actual))
print("Probabilità archiviate:", len(predictions))

for df in [games, actual, predictions]:
    df["game_id"] = df["game_id"].astype(str)

for df in [actual, predictions]:
    df["player_id"] = df["player_id"].astype(str)

games["Date"] = pd.to_datetime(
    games["game_date"],
    format="mixed",
    utc=True,
    errors="coerce"
)

actual["Date"] = pd.to_datetime(
    actual["game_date"],
    format="mixed",
    utc=True,
    errors="coerce"
)

predictions["Generated"] = pd.to_datetime(
    predictions["generated_at"],
    format="mixed",
    utc=True,
    errors="coerce"
)

completed = games[
    games["completed"].astype(str).str.lower()
    .isin(["1", "1.0", "true"])
].copy()

print("\nUltima partita completata:")
print(completed["Date"].max())

print("\nUltimo boxscore disponibile:")
print(actual["Date"].max())

print("\nUltima previsione generata:")
print(predictions["Generated"].max())

print("\nDate previsioni non valide:")
print(predictions["Generated"].isna().sum())

print("\nDate partite non valide:")
print(games["Date"].isna().sum())

print("\n" + "=" * 60)
print("2. CONTROLLO ASSOCIAZIONI")
print("=" * 60)

print(
    "Partite previste:",
    predictions["game_id"].nunique()
)

print(
    "Partite previste e completate:",
    predictions[
        predictions["game_id"].isin(
            completed["game_id"]
        )
    ]["game_id"].nunique()
)

matched = predictions.merge(
    completed[["game_id", "Date"]]
    .drop_duplicates("game_id"),
    on="game_id",
    how="inner"
)

print("Probabilità con partita conclusa:", len(matched))

pregame = matched[
    matched["Generated"] < matched["Date"]
].copy()

print("Probabilità valide pre-partita:", len(pregame))

player_matches = pregame.merge(
    actual[
        [
            "game_id", "player_id",
            "minutes", "did_not_play"
        ]
    ].drop_duplicates(["game_id", "player_id"]),
    on=["game_id", "player_id"],
    how="inner"
)

print(
    "Probabilità associate ai boxscore:",
    len(player_matches)
)

print("\nFormato minutaggio:")
print(actual["minutes"].tail(10).tolist())

print("\nValori did_not_play:")
print(actual["did_not_play"].value_counts(dropna=False))

print("\n" + "=" * 60)
print("3. CONTROLLO WEMBANYAMA E LAMELO BALL")
print("=" * 60)

players = actual[
    actual["player_name"].str.contains(
        "Wembanyama|LaMelo Ball",
        case=False,
        na=False
    )
].sort_values("Date", ascending=False)

display(
    players[
        [
            "game_date",
            "player_name",
            "team_name",
            "minutes",
            "pts",
            "reb",
            "ast",
            "fg3m"
        ]
    ].head(15)
)

print("\n" + "=" * 60)
print("4. ULTIME PARTITE REGISTRATE")
print("=" * 60)

display(
    games.sort_values(
        "Date", ascending=False
    )[
        [
            "game_date",
            "home_team",
            "away_team",
            "completed",
            "status"
        ]
    ].head(20)
)

print("\nDIAGNOSTICA COMPLETATA")

In [ ]:

# ============================================================
# STEP 27B — DATABASE UPDATE & BACKTEST RECOVERY
# ============================================================

from pathlib import Path
import sqlite3
import pandas as pd
from IPython import get_ipython

ROOT = Path("/content/drive/MyDrive/NBA_Data_Science")
DB = ROOT / "nba_database.db"
PIPE = ROOT / "Production_Pipeline"

print("NBA INTELLIGENCE — RECOVERY")
print("=" * 60)

# 1. Backup database

backup_dir = ROOT / "Backups"
backup_dir.mkdir(exist_ok=True)

stamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")

backup_path = backup_dir / f"NBA_Backup_{stamp}.db"

with sqlite3.connect(DB) as source:
    with sqlite3.connect(backup_path) as destination:
        source.backup(destination)

print("Backup completato:", backup_path.name)

# 2. Controllo previsioni archiviate

with sqlite3.connect(DB) as conn:

    before = pd.read_sql_query("""
        SELECT COUNT(*) AS Records
        FROM nba_probability_history_v3
    """, conn).iloc[0, 0]

print("Probabilità archiviate:", before)

# 3. Aggiornamento risultati ESPN

updater = PIPE / "step_12.py"

if not updater.exists():
    raise FileNotFoundError(
        "step_12.py non trovato. "
        "Esegui manualmente lo STEP 12 originale."
    )

print("\nAGGIORNAMENTO DATABASE NBA...")

ip = get_ipython()

execution = ip.run_cell(
    updater.read_text(encoding="utf-8"),
    store_history=False
)

error = (
    execution.error_before_exec
    or execution.error_in_exec
)

if error:
    raise RuntimeError(
        f"Errore aggiornamento ESPN: {error}"
    )

# 4. Controllo risultati aggiornati

with sqlite3.connect(DB) as conn:

    games = pd.read_sql_query("""
        SELECT * FROM espn_games
    """, conn)

    actual = pd.read_sql_query("""
        SELECT * FROM espn_player_boxscores
    """, conn)

    probabilities = pd.read_sql_query("""
        SELECT * FROM nba_probability_history_v3
    """, conn)

print("\nCONTROLLO DATABASE")

print("Partite:", len(games))
print("Boxscore:", len(actual))
print("Probabilità archiviate:", len(probabilities))

if len(probabilities) < before:
    raise RuntimeError(
        "Archivio probabilità ridotto. "
        "Controllare il backup prima di procedere."
    )

# 5. Normalizzazione date

games["Date"] = pd.to_datetime(
    games["game_date"],
    format="mixed",
    utc=True,
    errors="coerce"
)

actual["Date"] = pd.to_datetime(
    actual["game_date"],
    format="mixed",
    utc=True,
    errors="coerce"
)

probabilities["Generated"] = pd.to_datetime(
    probabilities["generated_at"],
    format="mixed",
    utc=True,
    errors="coerce"
)

for df in [games, actual, probabilities]:
    df["game_id"] = df["game_id"].astype(str)

for df in [actual, probabilities]:
    df["player_id"] = df["player_id"].astype(str)

completed = games[
    games["completed"].astype(str)
    .str.lower().isin(["1", "1.0", "true"])
].copy()

print("\nUltima partita completata:")
print(completed["Date"].max())

print("\nUltimo boxscore:")
print(actual["Date"].max())

# 6. Verifica forecast pre-partita

matched = probabilities.merge(
    completed[["game_id", "Date"]]
    .drop_duplicates("game_id"),
    on="game_id",
    how="inner"
)

matched = matched[
    matched["Generated"] < matched["Date"]
]

matched = matched.merge(
    actual[["game_id", "player_id"]]
    .drop_duplicates(),
    on=["game_id", "player_id"],
    how="inner"
)

print("\nProbabilità associabili:", len(matched))
print("Partite valutabili:", matched["game_id"].nunique())

# 7. Risultati giocatori

players = actual[
    actual["player_name"].str.contains(
        "Wembanyama|LaMelo Ball",
        case=False,
        na=False
    )
].sort_values("Date", ascending=False)

display(
    players[
        [
            "game_date",
            "player_name",
            "team_name",
            "minutes",
            "pts",
            "reb",
            "ast",
            "fg3m"
        ]
    ].head(10)
)

# 8. Nuovo tentativo di backtesting

if matched.empty:

    print("\nATTENZIONE")
    print("Nessuna previsione valutabile.")
    print("Non verranno inventati risultati.")

else:

    try:
        bt_results = probability_backtest()

        print("\nBACKTESTING COMPLETATO")
        print("Previsioni valutate:", len(bt_results))

    except Exception as e:

        print("\nBACKTESTING DA CORREGGERE")
        print(type(e).__name__, str(e))

print("\nSTEP 27B TERMINATO")


In [ ]:

# ============================================================
# STEP 28 — DAILY PERFORMANCE TRACKER V2
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px
import ipywidgets as widgets

from IPython.display import display, clear_output

ROOT28 = Path(
    "/content/drive/MyDrive/NBA_Data_Science"
)

PERF28 = ROOT28 / "Probability_Backtesting"

SOURCE28 = PERF28 / "Evaluated_Predictions.csv"


def daily_performance():

    if not SOURCE28.exists():

        raise RuntimeError(
            "Completare prima lo STEP 27B. "
            "Archivio backtesting non disponibile."
        )

    df = pd.read_csv(
        SOURCE28,
        dtype={
            "game_id": str,
            "player_id": str
        }
    )

    if df.empty:
        raise RuntimeError(
            "Nessuna previsione valutabile."
        )

    df["Tipoff"] = pd.to_datetime(
        df["Tipoff"],
        format="mixed",
        utc=True,
        errors="coerce"
    )

    df = df.dropna(subset=["Tipoff"])

    df["NBA_Date"] = (
        df["Tipoff"]
        .dt.tz_convert("America/New_York")
        .dt.strftime("%Y-%m-%d")
    )

    df["Italy_Date"] = (
        df["Tipoff"]
        .dt.tz_convert("Europe/Rome")
        .dt.strftime("%Y-%m-%d")
    )

    numeric = [
        "forecast",
        "Actual",
        "threshold",
        "probability",
        "Success",
        "Correct",
        "Brier",
        "Baseline_Brier"
    ]

    for col in numeric:

        if col in df.columns:

            df[col] = pd.to_numeric(
                df[col],
                errors="coerce"
            )

    df["Probability_pct"] = (
        df["probability"] * 100
    )

    df["Outcome"] = np.where(
        df["Success"] == 1,
        "RAGGIUNTA",
        "NON RAGGIUNTA"
    )

    df["Error"] = (
        df["forecast"] - df["Actual"]
    )

    df["Absolute_Error"] = df["Error"].abs()

    # Una previsione numerica per statistica

    point = (
        df.sort_values("generated_at")
        .drop_duplicates(
            ["game_id", "player_id", "target"],
            keep="last"
        )
        .copy()
    )

    # Performance giornaliera

    daily = df.groupby("NBA_Date").agg(
        Predictions=("Success", "size"),
        Games=("game_id", "nunique"),
        Players=("player_id", "nunique"),
        Expected_Success=("probability", "mean"),
        Actual_Success=("Success", "mean"),
        Accuracy=("Correct", "mean"),
        Brier_Score=("Brier", "mean")
    ).reset_index()

    daily["Calibration_Gap"] = (
        daily["Expected_Success"]
        - daily["Actual_Success"]
    ).abs()

    # Accuratezza forecast numerici

    mae = point.groupby(
        ["NBA_Date", "target"]
    ).agg(
        Samples=("Actual", "size"),
        MAE=("Absolute_Error", "mean"),
        Bias=("Error", "mean")
    ).reset_index()

    # Performance per fascia probabilistica

    df["Confidence_Band"] = pd.cut(
        df["probability"],
        bins=[
            0, .50, .60, .70,
            .80, .90, .95, 1
        ],
        labels=[
            "0-50%",
            "50-60%",
            "60-70%",
            "70-80%",
            "80-90%",
            "90-95%",
            "95-100%"
        ],
        include_lowest=True
    )

    confidence = df.groupby(
        ["NBA_Date", "Confidence_Band"],
        observed=True
    ).agg(
        Predictions=("Success", "size"),
        Expected=("probability", "mean"),
        Actual=("Success", "mean"),
        Brier_Score=("Brier", "mean")
    ).reset_index()

    confidence["Gap"] = (
        confidence["Expected"]
        - confidence["Actual"]
    ).abs()

    # Performance per giocatore

    players = df.groupby(
        ["player_id", "player_name", "target"]
    ).agg(
        Predictions=("Success", "size"),
        Games=("game_id", "nunique"),
        Expected=("probability", "mean"),
        Hit_Rate=("Success", "mean"),
        Accuracy=("Correct", "mean"),
        Brier_Score=("Brier", "mean")
    ).reset_index()

    # Confronto benchmark sulle stesse osservazioni

    paired = df.dropna(
        subset=["Baseline_Brier", "Brier"]
    )

    benchmark = paired.groupby(
        ["NBA_Date", "target"]
    ).agg(
        Samples=("Brier", "size"),
        ML_Brier=("Brier", "mean"),
        Baseline_Brier=("Baseline_Brier", "mean")
    ).reset_index()

    benchmark["Improvement_pct"] = np.where(
        benchmark["Baseline_Brier"] > 0,
        100 * (
            1 - benchmark["ML_Brier"]
            / benchmark["Baseline_Brier"]
        ),
        np.nan
    )

    # Ultima giornata valutabile

    latest_date = df["NBA_Date"].max()

    latest = df[
        df["NBA_Date"] == latest_date
    ].copy()

    latest_point = point[
        point["NBA_Date"] == latest_date
    ].copy()

    latest_point = latest_point.sort_values(
        "Absolute_Error"
    )

    # Salvataggio report

    reports = {
        "Daily_Performance": daily,
        "Daily_MAE": mae,
        "Daily_Confidence": confidence,
        "Player_Reliability": players,
        "Daily_Benchmark": benchmark,
        "Latest_Matches": latest,
        "Latest_Player_Forecasts": latest_point
    }

    for name, table in reports.items():

        temp = PERF28 / f"{name}.tmp"

        table.to_csv(temp, index=False)

        temp.replace(
            PERF28 / f"{name}.csv"
        )

    with pd.ExcelWriter(
        PERF28 / "NBA_Daily_Performance.xlsx",
        engine="openpyxl"
    ) as writer:

        for name, table in reports.items():

            table.to_excel(
                writer,
                sheet_name=name[:31],
                index=False
            )

    print("=" * 65)
    print("NBA INTELLIGENCE — DAILY PERFORMANCE")
    print("=" * 65)

    print("Ultima giornata valutata:", latest_date)

    print(
        "Partite:",
        latest["game_id"].nunique()
    )

    print(
        "Giocatori:",
        latest["player_id"].nunique()
    )

    print(
        "Soglie analizzate:",
        len(latest)
    )

    print(
        "Brier Score:",
        round(latest["Brier"].mean(), 4)
    )

    print("\nPERFORMANCE PER FASCIA")

    display(
        confidence[
            confidence["NBA_Date"] == latest_date
        ]
    )

    print("\nMIGLIORI FORECAST NUMERICI")

    columns = [
        "player_name",
        "target",
        "forecast",
        "Actual",
        "Absolute_Error"
    ]

    display(
        latest_point[columns].head(15)
    )

    print("\nPEGGIORI FORECAST NUMERICI")

    display(
        latest_point[columns]
        .sort_values(
            "Absolute_Error",
            ascending=False
        ).head(15)
    )

    print("\nSTEP 28 COMPLETATO")

    return df


daily_results = daily_performance()


# ============================================================
# PLAYER PERFORMANCE EXPLORER
# ============================================================

player_selector = widgets.Dropdown(
    options=sorted(
        daily_results["player_name"]
        .dropna().unique().tolist()
    ),
    description="Player:",
    layout=widgets.Layout(width="420px")
)

target_selector = widgets.Dropdown(
    options=["PTS", "REB", "AST", "3PM", "PRA"],
    value="PTS",
    description="Stat:"
)

date_selector = widgets.Dropdown(
    options=sorted(
        daily_results["NBA_Date"].unique(),
        reverse=True
    ),
    description="Date:"
)

output28 = widgets.Output()


def show_player_performance(change=None):

    with output28:

        clear_output(wait=True)

        selected = daily_results[
            (
                daily_results["player_name"]
                == player_selector.value
            )
            & (
                daily_results["target"]
                == target_selector.value
            )
            & (
                daily_results["NBA_Date"]
                == date_selector.value
            )
        ].copy()

        if selected.empty:

            print(
                "Nessuna previsione valutabile "
                "per questa selezione."
            )

            return

        selected = selected.sort_values(
            "threshold"
        )

        for game_id, game in selected.groupby("game_id"):

            row = game.iloc[0]

            print("=" * 55)
            print(row["player_name"])
            print("Partita:", game_id)
            print("Statistica:", row["target"])
            print("Forecast:", round(row["forecast"], 2))
            print("Risultato:", row["Actual"])

            print(
                "Errore assoluto:",
                round(row["Absolute_Error"], 2)
            )

            display(
                game[
                    [
                        "threshold",
                        "Probability_pct",
                        "Actual",
                        "Outcome"
                    ]
                ].round(2)
            )

            fig = px.bar(
                game,
                x="threshold",
                y="Probability_pct",
                color="Outcome",
                color_discrete_map={
                    "RAGGIUNTA": "#22c55e",
                    "NON RAGGIUNTA": "#ef4444"
                },
                template="plotly_dark",
                title="Forecast vs Actual — Probability Engine"
            )

            fig.update_yaxes(range=[0, 100])

            fig.show()


for control in [
    player_selector,
    target_selector,
    date_selector
]:

    control.observe(
        show_player_performance,
        names="value"
    )

display(
    widgets.VBox([
        player_selector,
        target_selector,
        date_selector,
        output28
    ])
)

show_player_performance()


In [ ]:

# ============================================================
# STEP 28 FIX — EXCEL TIMEZONE COMPATIBILITY
# ============================================================

import pandas as pd

from pathlib import Path
from IPython import get_ipython

ip = get_ipython()

# 1. Recupero codice originale STEP 28

source28 = None

for cell in reversed(ip.user_ns.get("In", [])):

    if (
        "# STEP 28 — DAILY PERFORMANCE TRACKER V2"
        in cell.splitlines()[:8]
    ):
        source28 = cell
        break

if source28 is None:
    raise RuntimeError(
        "Cella originale STEP 28 non trovata."
    )

print("STEP 28 originale recuperato")

# 2. Funzione compatibilità Excel

def excel_safe_dataframe(df):

    safe = df.copy()

    for col in safe.columns:

        if isinstance(
            safe[col].dtype,
            pd.DatetimeTZDtype
        ):

            safe[col] = (
                safe[col]
                .dt.tz_convert("UTC")
                .dt.tz_localize(None)
            )

        elif safe[col].dtype == "object":

            safe[col] = safe[col].map(
                lambda value: (
                    value.isoformat()
                    if getattr(value, "tzinfo", None)
                    is not None
                    else value
                )
            )

    return safe


# 3. Correzione automatica esportazione

old_code = "table.to_excel("

new_code = "excel_safe_dataframe(table).to_excel("

if old_code not in source28:

    raise RuntimeError(
        "Blocco esportazione Excel non trovato."
    )

fixed_source28 = source28.replace(
    old_code,
    new_code
)

compile(
    fixed_source28,
    "step_28_fixed.py",
    "exec"
)

# 4. Salvataggio permanente su Drive

ROOT_FIX28 = Path(
    "/content/drive/MyDrive/NBA_Data_Science"
)

PIPE_FIX28 = ROOT_FIX28 / "Production_Pipeline"

PIPE_FIX28.mkdir(
    parents=True,
    exist_ok=True
)

fixed_path = PIPE_FIX28 / "step_28_fixed.py"

fixed_path.write_text(
    fixed_source28,
    encoding="utf-8"
)

print("Codice corretto salvato su Drive")

# 5. Riesecuzione completa

execution28 = ip.run_cell(
    fixed_source28,
    store_history=False
)

error28 = (
    execution28.error_before_exec
    or execution28.error_in_exec
)

if error28:

    raise RuntimeError(
        f"Errore residuo: {error28}"
    )

print("\n" + "=" * 60)
print("STEP 28 FIX COMPLETATO")
print("ESPORTAZIONE EXCEL CORRETTA")
print("DAILY PERFORMANCE TRACKER ATTIVO")
print("=" * 60)


In [ ]:

# ============================================================
# STEP 29A — AUTOMATIC PIPELINE PREFLIGHT
# ============================================================

from pathlib import Path
import pandas as pd
import sqlite3
import hashlib

ROOT29 = Path(
    "/content/drive/MyDrive/NBA_Data_Science"
)

PIPE29 = ROOT29 / "Production_Pipeline"
DB29 = ROOT29 / "nba_database.db"

PIPE29.mkdir(parents=True, exist_ok=True)

print("=" * 65)
print("NBA INTELLIGENCE — AUTOMATION PREFLIGHT")
print("=" * 65)

if not DB29.is_file():
    raise FileNotFoundError(DB29)

modules29 = {
    "Database Update": ["step_12.py"],
    "Team Forecast": ["step_6.py"],
    "Player Forecast": ["step_17.py"],
    "Probability Engine": [
        "step_25_probability.py",
        "step_18.py"
    ],
    "Injury Fetch": ["injury_fetch.py"],
    "Injury Integration": ["injury_merge.py"],
    "Daily Performance": ["step_28_fixed.py"]
}

selected29 = {}
checks29 = []

for name, candidates in modules29.items():

    path = next(
        (
            PIPE29 / filename
            for filename in candidates
            if (PIPE29 / filename).is_file()
        ),
        None
    )

    status = "MISSING"

    if path is not None:

        try:

            source = path.read_text(
                encoding="utf-8"
            )

            compile(source, str(path), "exec")

            selected29[name] = path
            status = "READY"

        except Exception as e:

            status = f"REVIEW: {e}"

    checks29.append({
        "Module": name,
        "File": path.name if path else "-",
        "Status": status
    })

display(pd.DataFrame(checks29))

print("\nBACKTESTING FUNCTION")

print(
    "READY"
    if callable(globals().get("probability_backtest"))
    else "MISSING"
)

print("\nDAILY PERFORMANCE FUNCTION")

print(
    "READY"
    if callable(globals().get("daily_performance"))
    else "MISSING"
)

print("\nDATABASE STATUS")

with sqlite3.connect(
    f"file:{DB29}?mode=ro",
    uri=True
) as conn:

    display(pd.read_sql_query("""
        SELECT
            MAX(game_date) AS Latest_Boxscore,
            COUNT(DISTINCT game_id) AS Games,
            COUNT(*) AS Player_Records
        FROM espn_player_boxscores
    """, conn))

print("\nAVAILABLE PIPELINE FILES")

for path in sorted(PIPE29.glob("*.py")):

    print(
        path.name,
        "|",
        round(path.stat().st_size / 1024, 1),
        "KB"
    )

print("\nSTEP 29A COMPLETATO")
